In [ ]:
# ==================== VAULTMIND ERP — FULL SETUP (ONE CELL) ====================
import subprocess, sys, os, time, json, hashlib

# ---------- 1. Mount Drive ----------
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)
print(f"✅ Drive mounted → {DRIVE}")

# ---------- 2. Install ----------
print("📦 Installing system deps…")
subprocess.run("apt-get update -qq && apt-get install -y -q tesseract-ocr tesseract-ocr-eng",
               shell=True, capture_output=True)
print("📦 Installing Python packages…")
subprocess.run(
    "pip install -q streamlit chromadb sentence-transformers "
    "'transformers>=4.45' accelerate bitsandbytes python-docx pymupdf "
    "pytesseract pillow reportlab pdfplumber pandas 2>&1 | tail -1",
    shell=True)
print("✅ Install complete")

# ---------- 3. Write app ----------
APP_PATH = "/content/vaultmind_erp.py"
APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil
from datetime import datetime
import streamlit as st, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

# ==================== PATHS ====================
DRIVE         = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS     = f"{DRIVE}/data/depts.json"
CFG_USERS     = f"{DRIVE}/data/users.json"
AUDIT_DB      = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA  = "/content/chroma_local"
DRIVE_CHROMA  = f"{DRIVE}/chroma_backup"
UPLOAD_DIR    = f"{DRIVE}/uploads"
MODEL_ID      = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID      = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR, f"{DRIVE}/logs"]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s): return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

# Restore Chroma from Drive backup if local missing
if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True); print("[chroma] restored from Drive")
    except Exception as e: print(f"[chroma] restore failed: {e}")

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except Exception as e: print(f"[chroma] backup failed: {e}")

# ==================== REGISTRY ====================
DEFAULT_DEPTS = {
  "cse":  {"name":"Computer Science","secret":"aurora-lattice","color":"#2563eb",
           "aliases":["cse","computer","comp sci","cs dept","computer science"]},
  "mech": {"name":"Mechanical","secret":"granite-falcon","color":"#dc2626",
           "aliases":["mech","mechanical","mech dept"]},
  "ece":  {"name":"Electronics","secret":"cobalt-meridian","color":"#7c3aed",
           "aliases":["ece","electronics","electronics dept"]},
}
try:    DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()

# Normalize loaded depts (fixes KeyError for old JSONs)
PALETTE = ["#2563eb","#dc2626","#7c3aed","#0891b2","#16a34a","#ea580c","#db2777"]
_changed = False
for _i, (_c, _cfg) in enumerate(DEPARTMENTS.items()):
    if not _cfg.get("name"):    _cfg["name"] = _c.upper(); _changed = True
    if not _cfg.get("secret"):  _cfg["secret"] = f"{_c}-secret-{abs(hash(_c))%9999}"; _changed = True
    if not _cfg.get("color"):   _cfg["color"] = PALETTE[_i % len(PALETTE)]; _changed = True
    if not _cfg.get("aliases"): _cfg["aliases"] = [_c]; _changed = True
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

DEFAULT_USERS = {
  "admin":        {"name":"System Admin","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":  {"name":"Dr. Sharma","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_mech": {"name":"Dr. Iyer","role":"teacher","dept":"mech","pw":_hash("t_mech")},
  "teacher_ece":  {"name":"Dr. Menon","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "student_cse":  {"name":"Rahul Kumar","role":"student","dept":"cse","pw":_hash("s_cse")},
  "student_mech": {"name":"Priya Nair","role":"student","dept":"mech","pw":_hash("s_mech")},
  "student_ece":  {"name":"Arjun Reddy","role":"student","dept":"ece","pw":_hash("s_ece")},
}
try:    USERS = json.load(open(CFG_USERS))
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

SEED = {
  "cse":{"attendance_rule":"Minimum attendance is 75 percent in theory and 100 percent in labs.",
         "exam_weightage":"End-semester exam carries 60 percent weight. Minimum 40 percent required.",
         "plagiarism_policy":"Similarity index above 15 percent triggers integrity review.",
         "project_rule":"Two industry-oriented projects required before graduation.",
         "syllabus":"Unit 1: Data Structures — arrays, linked lists, stacks, queues, trees, graphs. Unit 2: Algorithms — sorting, searching, dynamic programming, greedy. Unit 3: DBMS — normalization, SQL, transactions. Unit 4: Operating Systems — processes, scheduling, memory. Unit 5: Networks — OSI, TCP/IP, routing."},
  "mech":{"safety_rule":"Safety shoes, goggles and lab coat mandatory in workshop.",
          "fire_safety":"Fire extinguishers inspected every 30 days.",
          "tool_policy":"Daily tool inventory check mandatory.",
          "project_rule":"ASME-standard design report, 12-week fabrication window, viva-voce.",
          "syllabus":"Unit 1: Thermodynamics — laws, cycles, entropy. Unit 2: Fluid Mechanics — Bernoulli, Reynolds. Unit 3: Machine Design — gears, bearings. Unit 4: Manufacturing — casting, welding, CNC. Unit 5: Heat Transfer — conduction, convection, radiation."},
  "ece":{"lab_voltage":"Maximum supply voltage for students is 12 volts DC.",
         "esd_policy":"All ICs handled with anti-static wrist straps.",
         "logbook_rule":"Logbook signed by lab instructor each session.",
         "project_rule":"Submission includes schematic, PCB layout, BOM and 3-min demo video.",
         "syllabus":"Unit 1: Digital Electronics — gates, flip-flops, counters. Unit 2: Analog Circuits — op-amps, filters. Unit 3: Signals & Systems — Fourier, Laplace. Unit 4: VLSI Design — CMOS, layout. Unit 5: Microprocessors — 8085, 8051, ARM."},
}

# ==================== MODEL ====================
@st.cache_resource(show_spinner="⏳ Loading AI models (first run ~60s)…")
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try:    coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count() == 0:
            keys = list(docs.keys()); texts = list(docs.values())
            keys += ["_secret_marker"]; texts += [DEPARTMENTS[dept]["secret"]]
            vecs = emb.encode(texts, normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys], embeddings=vecs, documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    _ = mdl.generate(**tok("hi", return_tensors="pt").to(mdl.device),
                     max_new_tokens=1, pad_token_id=tok.eos_token_id)
    return emb, cc, tok, mdl, DEV

emb, cc, tok, mdl, DEV = setup()

def get_coll(d):
    try:    return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}", metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r = USERS.get(u)
    if r and r["pw"] == _hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept, query):
    if not dept: return None, None
    q = query.lower()
    for other, cfg in DEPARTMENTS.items():
        if other == dept: continue
        for alias in cfg.get("aliases",[]) + [other]:
            if re.search(rf"\b{re.escape(alias)}\b", q): return other, cfg["name"]
    return None, None

# ==================== PDF INGEST ====================
def chunk_text(t, size=500, ov=60):
    parts = re.split(r'(?<=[.!?])\s+', t.strip()); out, cur = [], ""
    for s in parts:
        if len(cur)+len(s)+1 <= size: cur = (cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur = ((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf_robust(pdf_bytes, fname):
    tmp = f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    with open(tmp, "wb") as fh: fh.write(pdf_bytes)
    pages, err = [], ""
    try:
        doc = pymupdf.open(tmp)
        for i, pg in enumerate(doc):
            try:
                t = pg.get_text()
                if t and t.strip(): pages.append({"text": t, "page": i+1})
            except: pass
        doc.close()
    except Exception as e: err = f"pymupdf: {e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i, pg in enumerate(pdf.pages):
                    try:
                        t = pg.extract_text() or ""
                        if t.strip(): pages.append({"text": t, "page": i+1})
                    except: pass
            if pages: err = ""
        except Exception as e: err += f" | pdfplumber: {e}"
    try: os.remove(tmp)
    except: pass
    return pages, err

def ingest_pdf(dept, fname, pdf_bytes, doc_type="regulation"):
    pages, err = extract_pdf_robust(pdf_bytes, fname)
    if not pages: return 0, (err or "No extractable text (scanned image?)"), 0
    coll = get_coll(dept); ch, mt, ids = [], [], []
    safe = _safe(fname)
    for pg in pages:
        for i, c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":doc_type})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    if not ch: return 0, "0 chunks", len(pages)
    try:
        all_vecs = []
        for i in range(0, len(ch), 32):
            v = emb.encode(ch[i:i+32], normalize_embeddings=True, show_progress_bar=False).tolist()
            all_vecs.extend(v)
        coll.add(ids=ids, embeddings=all_vecs, documents=ch, metadatas=mt)
    except Exception as e: return 0, f"Chroma write: {e}", len(pages)
    try:
        with open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}", "wb") as fh: fh.write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch), "", len(pages)

def ingest_text(dept, text, src, dt="regulation"):
    coll = get_coll(dept); ch = chunk_text(text)
    if not ch: return 0
    try:
        v = emb.encode(ch, normalize_embeddings=True, show_progress_bar=False).tolist()
        coll.add(ids=[f"{dept}_{_safe(src)}_c{i}_{int(time.time())}" for i in range(len(ch))],
                 embeddings=v, documents=ch,
                 metadatas=[{"source":src,"page":1,"doc_type":dt}]*len(ch))
        backup_chroma(); return len(ch)
    except Exception as e:
        print(f"[ingest_text] {e}"); return 0

def list_sources(dept):
    try:
        d = get_coll(dept).get(include=["metadatas"]); counts = {}
        for m in d["metadatas"]:
            s = m.get("source","?"); counts[s] = counts.get(s,0)+1
        return counts
    except: return {}

def get_all_docs(dept):
    try:
        d = get_coll(dept).get(include=["documents","metadatas"])
        return list(zip(d["documents"], d["metadatas"]))
    except: return []

def retrieve(dept, q, k=3):
    c = get_coll(dept)
    if c.count() == 0: return []
    qv = emb.encode([q], normalize_embeddings=True).tolist()
    r = c.query(query_embeddings=qv, n_results=min(k,c.count()),
                include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

def stream_answer(dept, q, ph, extra=""):
    hits = retrieve(dept, q)
    parts = [f"[{h['source']} p{h['page']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0, f"[USER CONTEXT] {extra[:1500]}")
    ctx = "\n".join(parts) or "(no context)"
    msgs = [{"role":"system","content":"You are a helpful college assistant. Give clear, concise answers with citations like [filename, page N]."},
            {"role":"user","content":f"Question: {q}\n\nReference:\n{ctx}\n\nAnswer:"}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    th = threading.Thread(target=mdl.generate,
        kwargs=dict(**inp, max_new_tokens=180, do_sample=False,
                    pad_token_id=tok.eos_token_id, streamer=streamer))
    th.start()
    buf = ""
    for t in streamer:
        buf += t; ph.markdown(buf + "▌")
    ph.markdown(buf)
    return buf, hits

def generate_block(p, mx=500):
    text = tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    with torch.inference_mode():
        o = mdl.generate(**inp, max_new_tokens=mx, do_sample=True, temperature=0.7,
                         top_p=0.9, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def make_pdf(title, dept_name, subtitle, body):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=0.8*inch, rightMargin=0.8*inch,
                            topMargin=0.7*inch, bottomMargin=0.7*inch)
    S = getSampleStyleSheet()
    h1 = ParagraphStyle("H1", parent=S["Heading1"], alignment=1, fontSize=16, spaceAfter=6)
    sb = ParagraphStyle("S", parent=S["Normal"], alignment=1, fontSize=10, spaceAfter=4)
    bd = ParagraphStyle("B", parent=S["Normal"], fontSize=11, leading=16, spaceAfter=8)
    story = [Paragraph(f"<b>{title}</b>", h1), Paragraph(f"Department of {dept_name}", sb)]
    if subtitle: story.append(Paragraph(subtitle, sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s, bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

# ==================== AUDIT ====================
def init_audit():
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (
        id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT,
        user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",
              (datetime.now().isoformat(), u, r or "-", d or "-", a, det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None, limit=200):
    c = sqlite3.connect(AUDIT_DB).cursor()
    if dept:
        c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else:
        c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows = c.fetchall(); c.connection.close(); return rows

# ==================== CSS ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide")
st.markdown("""
<style>
html, body, [class*="css"] { font-family: 'Inter', system-ui, sans-serif; }
.hero { background: radial-gradient(circle at 20% 10%, #1e40af 0%, transparent 45%),
        radial-gradient(circle at 80% 20%, #7c3aed 0%, transparent 45%),
        linear-gradient(135deg, #0a0e27 0%, #111a3e 55%, #1a2a5c 100%);
        padding: 70px 40px 60px; border-radius: 24px; color: white; text-align: center;
        box-shadow: 0 20px 60px -20px rgba(30,64,175,0.5); }
.hero h1 { font-size: 3.4rem; font-weight: 800; letter-spacing: -1.5px; margin: 0 0 14px 0;
        background: linear-gradient(90deg,#fff,#a5b4fc,#c4b5fd); -webkit-background-clip: text;
        -webkit-text-fill-color: transparent; }
.hero p { font-size: 1.15rem; opacity: 0.94; margin: 0; }
.hero .badge { display: inline-block; padding: 6px 16px; border-radius: 100px;
        background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2);
        font-size: 0.82rem; margin-bottom: 16px; letter-spacing: 0.5px; }
.feat { background: linear-gradient(180deg,#fff,#f8fafc); border-radius: 16px;
        padding: 26px 22px; border: 1px solid #e2e8f0; height: 165px;
        transition: transform 0.2s, box-shadow 0.2s; position: relative; overflow: hidden; }
.feat:hover { transform: translateY(-3px); box-shadow: 0 12px 32px -12px rgba(30,64,175,0.28); }
.feat::before { content: ''; position: absolute; top: 0; left: 0; width: 4px; height: 100%;
        background: linear-gradient(180deg,#2563eb,#7c3aed); }
.feat .icn { font-size: 1.8rem; margin-bottom: 10px; }
.feat h4 { margin: 0 0 8px 0; color: #0f172a; font-size: 1.05rem; font-weight: 700; }
.feat p { margin: 0; color: #64748b; font-size: 0.88rem; line-height: 1.5; }
.pill { display: inline-block; padding: 4px 14px; border-radius: 100px;
        font-size: 0.7rem; font-weight: 700; letter-spacing: 0.5px; text-transform: uppercase; }
.p-admin { background:#fef3c7; color:#92400e; }
.p-teacher { background:#dbeafe; color:#1e40af; }
.p-student { background:#dcfce7; color:#166534; }
.metric-card { background: linear-gradient(135deg,#fff,#f8fafc); border-radius: 14px;
        padding: 20px; border: 1px solid #e2e8f0; }
.metric-card .lbl { font-size: 0.78rem; color: #64748b; text-transform: uppercase;
        letter-spacing: 0.5px; margin-bottom: 6px; }
.metric-card .val { font-size: 1.9rem; font-weight: 800; color: #0f172a; }
.metric-card .sub { font-size: 0.82rem; color: #64748b; margin-top: 4px; }
.src-card { background: #f8fafc; border-radius: 10px; padding: 12px 14px;
        border-left: 3px solid #2563eb; margin: 6px 0; }
.src-card .nm { font-weight: 600; color: #0f172a; font-size: 0.92rem; }
.src-card .ct { color: #64748b; font-size: 0.8rem; }
.welcome-box { background: linear-gradient(135deg,#eff6ff,#f5f3ff); border-radius: 14px;
        padding: 22px 26px; border: 1px solid #c7d2fe; margin-bottom: 20px; }
.welcome-box h2 { margin: 0 0 6px 0; color: #1e293b; font-size: 1.5rem; }
.welcome-box p { margin: 0; color: #475569; font-size: 0.95rem; }
</style>
""", unsafe_allow_html=True)

for k in ("user","name","role","dept","logged_in"): st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx = ""

# ==================== LOGIN ====================
if not st.session_state.logged_in:
    st.markdown("""<div class="hero">
      <div class="badge">🎯 MU-PS-020 · SMART AUTOMATION</div>
      <h1>VaultMind ERP</h1>
      <p><b>Sovereign College ERP with On-Premise AI</b><br>
      Department-isolated · Exam papers · Notes · Quizzes · Zero cloud</p>
    </div>""", unsafe_allow_html=True)
    st.write("")
    c1,c2,c3,c4 = st.columns(4)
    for col, (icn, t, d) in zip([c1,c2,c3,c4], [
      ("🏛️","Dept Isolation","Each department has its own AI. CSE can never read Mech or ECE."),
      ("🤖","Local AI","Qwen2.5 on T4 GPU. No OpenAI. Docs never leave the machine."),
      ("📝","Exam Generator","Teachers generate full question papers from their syllabus."),
      ("📓","Notes & Quizzes","Students generate study notes and practice quizzes.")]):
        col.markdown(f"""<div class="feat"><div class="icn">{icn}</div>
          <h4>{t}</h4><p>{d}</p></div>""", unsafe_allow_html=True)
    st.write(""); st.write("")
    cA, cB = st.columns([3,2])
    with cA:
        st.markdown("### 🔐 Sign in")
        u = st.text_input("Username", placeholder="admin · teacher_cse · student_cse", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("🔓 Login", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True})
                st.session_state.chat_history = []; st.session_state.chat_ctx = ""
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
    with cB:
        st.markdown("### 🎫 Demo accounts")
        st.code("ADMIN\n  admin          / admin_pass\n\nTEACHER\n  teacher_cse    / t_cse\n  teacher_mech   / t_mech\n  teacher_ece    / t_ece\n\nSTUDENT\n  student_cse    / s_cse\n  student_mech   / s_mech\n  student_ece    / s_ece", language="text")
    st.stop()

# ==================== SIDEBAR ====================
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"; is_teacher = role == "teacher"
name = st.session_state.name
pill_cls = {"admin":"p-admin","teacher":"p-teacher","student":"p-student"}[role]
dept_color = DEPARTMENTS.get(dept,{}).get("color","#64748b") if dept else "#64748b"

st.sidebar.markdown(f"""
<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#f1f5f9,#e2e8f0);margin-bottom:16px;">
  <div style="font-size:1.6rem;text-align:center;">🎓</div>
  <div style="text-align:center;font-weight:700;color:#0f172a;margin-top:4px;">{name}</div>
  <div style="text-align:center;margin-top:8px;"><span class="pill {pill_cls}">{role}</span></div>
  {f'<div style="text-align:center;margin-top:8px;font-size:0.82rem;color:#475569;"><b style="color:{dept_color};">●</b> {DEPARTMENTS[dept]["name"]}</div>' if dept else ''}
</div>""", unsafe_allow_html=True)
if dept:
    try: st.sidebar.metric("📊 Knowledge chunks", get_coll(dept).count())
    except: pass
st.sidebar.caption("🔒 0 external API calls · 💾 Drive-backed")
if st.sidebar.button("🚪 Logout", use_container_width=True):
    log(user,role,dept,"LOGOUT","ok")
    for k in ("user","name","role","dept","logged_in"): st.session_state[k] = None
    st.session_state.chat_history = []; st.session_state.chat_ctx = ""; st.rerun()

# ==================== TABS ====================
if is_admin:
    T = st.tabs(["📊 Dashboard","🏛️ Departments","📤 Upload PDF","👥 Users","📜 Audit","ℹ️ About"])
    tab_dash, tab_dept, tab_pdf, tab_users, tab_audit, tab_about = T
    tab_chat = tab_exam = tab_notes = tab_syllabus = tab_quiz = None
elif is_teacher:
    T = st.tabs(["📊 Dashboard","💬 AI Chat","📝 Exam Generator","📓 Notes","📚 Syllabus","🧠 Quiz Maker","ℹ️ About"])
    tab_dash, tab_chat, tab_exam, tab_notes, tab_syllabus, tab_quiz, tab_about = T
    tab_dept = tab_pdf = tab_users = tab_audit = None
else:
    T = st.tabs(["📊 Dashboard","💬 AI Chat","📓 Notes","📚 Syllabus","🧠 Practice Quiz","ℹ️ About"])
    tab_dash, tab_chat, tab_notes, tab_syllabus, tab_quiz, tab_about = T
    tab_exam = tab_dept = tab_pdf = tab_users = tab_audit = None

# ==================== DASHBOARD ====================
with tab_dash:
    st.markdown(f"""<div class="welcome-box"><h2>👋 Welcome back, {name}!</h2>
      <p>You are signed in as <b>{role}</b>{f' for <b>{DEPARTMENTS[dept]["name"]}</b>' if dept else ''}.
      Everything is logged and department-scoped.</p></div>""", unsafe_allow_html=True)

    if is_admin:
        c1,c2,c3,c4 = st.columns(4)
        total_chunks = sum(get_coll(d).count() for d in DEPARTMENTS)
        blocked = sum(1 for r in get_audit(limit=500) if r[4] == "BLOCKED_QUERY")
        c1.markdown(f'<div class="metric-card"><div class="lbl">Departments</div><div class="val">{len(DEPARTMENTS)}</div><div class="sub">active units</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric-card"><div class="lbl">Total Users</div><div class="val">{len(USERS)}</div><div class="sub">{sum(1 for u in USERS.values() if u["role"]=="teacher")} teachers · {sum(1 for u in USERS.values() if u["role"]=="student")} students</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric-card"><div class="lbl">Knowledge Base</div><div class="val">{total_chunks}</div><div class="sub">indexed chunks</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric-card"><div class="lbl">Security</div><div class="val">0</div><div class="sub">{blocked} blocked · 0 leaks</div></div>', unsafe_allow_html=True)
        st.write(""); st.markdown("### 📊 Department Overview")
        cols = st.columns(len(DEPARTMENTS))
        for col, (code, cfg) in zip(cols, DEPARTMENTS.items()):
            n = get_coll(code).count(); srcs = list_sources(code)
            users_in = sum(1 for u in USERS.values() if u["dept"] == code)
            col.markdown(f"""<div style="background:linear-gradient(135deg,{cfg.get('color','#2563eb')}22,{cfg.get('color','#2563eb')}11);border-left:4px solid {cfg.get('color','#2563eb')};border-radius:12px;padding:16px;">
              <div style="font-size:0.75rem;color:{cfg.get('color','#2563eb')};font-weight:700;letter-spacing:1px;">{code.upper()}</div>
              <div style="font-size:1.15rem;font-weight:700;color:#0f172a;margin:4px 0;">{cfg['name']}</div>
              <div style="font-size:0.82rem;color:#475569;">📚 {n} chunks<br>📄 {len(srcs)} docs<br>👥 {users_in} users</div></div>""", unsafe_allow_html=True)
    elif is_teacher:
        c1,c2,c3,c4 = st.columns(4)
        n = get_coll(dept).count(); srcs = list_sources(dept)
        my_qs = [r for r in get_audit(dept=dept, limit=200) if r[4] == "QUERY"]
        exams = [r for r in get_audit(dept=dept, limit=200) if r[4] == "EXAM_GEN"]
        c1.markdown(f'<div class="metric-card"><div class="lbl">Knowledge base</div><div class="val">{n}</div><div class="sub">chunks indexed</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric-card"><div class="lbl">Documents</div><div class="val">{len(srcs)}</div><div class="sub">source files</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric-card"><div class="lbl">Exam papers</div><div class="val">{len(exams)}</div><div class="sub">generated</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric-card"><div class="lbl">Class queries</div><div class="val">{len(my_qs)}</div><div class="sub">student questions</div></div>', unsafe_allow_html=True)
        st.write(""); col1, col2 = st.columns([2,1])
        with col1:
            st.markdown("### 📄 Your Department Documents")
            if srcs:
                for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                    st.markdown(f'<div class="src-card"><div class="nm">📄 {s}</div><div class="ct">{cnt} chunks indexed</div></div>', unsafe_allow_html=True)
            else: st.info("No documents yet — ask admin.")
        with col2:
            st.markdown("### 🎯 Quick actions")
            st.markdown("- **📝 Generate exam paper**\n- **📓 Auto-summarise notes**\n- **🧠 Create practice quiz**\n- **💬 Answer with cited AI**")
    else:
        c1,c2,c3,c4 = st.columns(4)
        srcs = list_sources(dept); my_qs = [r for r in get_audit(dept=dept, limit=200) if r[0].startswith(user)]
        c1.markdown(f'<div class="metric-card"><div class="lbl">Syllabus units</div><div class="val">5</div><div class="sub">{DEPARTMENTS[dept]["name"]}</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric-card"><div class="lbl">Study material</div><div class="val">{len(srcs)}</div><div class="sub">documents</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric-card"><div class="lbl">Your questions</div><div class="val">{len(my_qs)}</div><div class="sub">asked to AI</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric-card"><div class="lbl">Access</div><div class="val">✓</div><div class="sub">{DEPARTMENTS[dept]["name"]} only</div></div>', unsafe_allow_html=True)
        st.write(""); cc1, cc2, cc3 = st.columns(3)
        cc1.markdown('<div class="feat" style="height:auto;"><div class="icn">💬</div><h4>Ask the AI</h4><p>Get answers with citations from your department\'s documents.</p></div>', unsafe_allow_html=True)
        cc2.markdown('<div class="feat" style="height:auto;"><div class="icn">📓</div><h4>Generate Notes</h4><p>Turn any topic into structured study notes, download as PDF.</p></div>', unsafe_allow_html=True)
        cc3.markdown('<div class="feat" style="height:auto;"><div class="icn">🧠</div><h4>Practice Quiz</h4><p>Test yourself with auto-generated questions.</p></div>', unsafe_allow_html=True)

# ==================== CHAT ====================
if tab_chat is not None:
    with tab_chat:
        if is_admin:
            st.info("⚠️ Admin: select a department to query as.")
            dc = st.selectbox("Query as department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="c_dept")
        else:
            dc = dept
            st.markdown(f"### 💬 {DEPARTMENTS[dept]['name']} AI Assistant")
            st.caption("🔒 Restricted to your department's knowledge.")
        with st.expander("📎 Attach context (PDF or image)", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                f1 = st.file_uploader("Upload PDF", type=["pdf"], key="ctx_pdf")
                if f1 and st.button("Load PDF", key="btn_pdf"):
                    pages, err = extract_pdf_robust(f1.getvalue(), f1.name)
                    if pages:
                        st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                        st.success(f"✅ Loaded {len(pages)} pages")
                    else: st.error(f"❌ {err}")
            with c2:
                f2 = st.file_uploader("Upload image (OCR)", type=["png","jpg","jpeg"], key="ctx_img")
                if f2 and st.button("OCR", key="btn_ocr"):
                    tmp = f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                    open(tmp,"wb").write(f2.getvalue())
                    try:
                        txt = pytesseract.image_to_string(Image.open(tmp))
                        st.session_state.chat_ctx = txt
                        st.success(f"✅ Extracted {len(txt)} chars")
                    except Exception as e: st.error(f"❌ {e}")
                    finally:
                        try: os.remove(tmp)
                        except: pass
            if st.session_state.chat_ctx:
                st.caption(f"📌 Context active ({len(st.session_state.chat_ctx)} chars)")
                if st.button("🗑️ Clear context", key="clr_ctx"):
                    st.session_state.chat_ctx = ""; st.rerun()
        box = st.container(height=500)
        with box:
            if not st.session_state.chat_history:
                st.markdown(f"""<div style="text-align:center;padding:40px 20px;color:#94a3b8;">
                  <div style="font-size:3rem;">💬</div>
                  <div style="font-size:1.1rem;margin-top:10px;">Ask me about <b style="color:{dept_color};">{DEPARTMENTS[dc]['name'] if dc in DEPARTMENTS else 'this dept'}</b></div>
                  <div style="font-size:0.85rem;margin-top:6px;">Answers come with citations from the department's documents.</div></div>""", unsafe_allow_html=True)
            for m in st.session_state.chat_history:
                with st.chat_message(m["role"]):
                    st.markdown(m["content"])
                    if m.get("sources"):
                        for s in m["sources"][:3]: st.caption(f"📄 `{s['source']}` · page {s['page']}")
        uq = st.chat_input("Type your question…")
        if uq:
            with box:
                with st.chat_message("user"): st.markdown(uq)
            st.session_state.chat_history.append({"role":"user","content":uq})
            other, on = detect_cross_dept(dc, uq)
            if other and other != dc:
                err = f"🚫 **Access denied.** Scoped to **{DEPARTMENTS[dc]['name']}**. Cannot query **{on}**."
                st.session_state.chat_history.append({"role":"assistant","content":err})
                with box:
                    with st.chat_message("assistant"): st.error(err)
                log(user,role,dc,"BLOCKED_QUERY",f"{other}:{uq[:50]}")
            else:
                with box:
                    with st.chat_message("assistant"):
                        ph = st.empty()
                        ans, hits = stream_answer(dc, uq, ph, st.session_state.chat_ctx)
                        if hits:
                            ss = ", ".join(f"`{h['source']}` p{h['page']}" for h in hits[:3])
                            st.caption(f"📄 {ss}")
                st.session_state.chat_history.append({
                    "role":"assistant","content":ans,
                    "sources":[{"source":h["source"],"page":h["page"]} for h in hits]})
                log(user,role,dc,"QUERY",uq[:50])
            st.rerun()
        if st.session_state.chat_history and st.button("🗑️ Clear chat", key="clr_chat"):
            st.session_state.chat_history = []; st.rerun()

# ==================== EXAM ====================
if tab_exam is not None:
    with tab_exam:
        st.markdown(f"### 📝 Exam Paper Generator — {DEPARTMENTS[dept]['name']}")
        docs = get_all_docs(dept)
        if not docs: st.warning("No documents yet.")
        else:
            c1, c2, c3 = st.columns(3)
            with c1:
                subj = st.text_input("Subject", value=DEPARTMENTS[dept]["name"] + " — End Semester", key="ex_subj")
                dur = st.text_input("Duration", value="3 hours", key="ex_dur")
                et = st.selectbox("Type", ["Mixed","MCQ only","Theory only","Numerical"], key="ex_type")
            with c2:
                tm = st.number_input("Total marks", 20, 100, 50, 10, key="ex_tm")
                nq = st.number_input("Questions", 5, 20, 10, 1, key="ex_nq")
                diff = st.selectbox("Difficulty", ["Easy","Medium","Hard","Mixed"], index=3, key="ex_diff")
            with c3:
                use_marks = st.checkbox("Show marks per Q", value=True, key="ex_marks")
                split_sec = st.checkbox("Sections A/B/C", value=True, key="ex_sec")
            if st.button("🎯 Generate Exam Paper", type="primary", use_container_width=True):
                with st.spinner("Composing exam paper…"):
                    sample = " ".join(d[0] for d in docs[:25])[:2800]
                    hint = "Organize into Section A (short, 2 marks), Section B (medium, 5 marks), Section C (long, 10 marks)." if split_sec else ""
                    prompt = (f"You are an experienced university exam paper setter for {DEPARTMENTS[dept]['name']}.\n"
                              f"Subject: {subj}\nMarks: {tm}\nDuration: {dur}\nDifficulty: {diff}\nType: {et}\n{hint}\n\n"
                              f"Course material:\n{sample}\n\n"
                              f"Generate exactly {nq} questions, each as 'Q1. <question> ({'X marks' if use_marks else ''})' on its own line. No answers.")
                    raw = generate_block(prompt, 700)
                    qs = []
                    for ln in raw.split("\n"):
                        ln = ln.strip()
                        if re.match(r'^(Section|[A-C][\.\)])', ln, re.I): qs.append(f"## {ln}")
                        elif ln and len(ln) > 10:
                            ln = re.sub(r'^(Q?\d+[\.\):\-])\s*', '', ln); qs.append(ln)
                    if qs:
                        st.session_state["exam"] = {
                            "title": f"End Semester Exam — {subj}", "dept_name": DEPARTMENTS[dept]["name"],
                            "subject": subj, "duration": dur, "total_marks": tm, "questions": qs}
                        log(user,role,dept,"EXAM_GEN",f"{subj}:{len(qs)}")
                    else: st.error("No valid questions produced.")
            if "exam" in st.session_state:
                d = st.session_state["exam"]; st.divider()
                st.markdown("### 📋 Review & edit")
                ed = []
                for i, q in enumerate(d["questions"]):
                    if q.startswith("## "): ed.append(q); st.markdown(f"**{q[3:]}**")
                    else: ed.append(st.text_area(f"Q{i+1}", value=q, height=70, key=f"eq{i}"))
                d["questions"] = ed
                body = "\n\n".join(q for q in ed if not q.startswith("##"))
                pdf = make_pdf(d["title"], d["dept_name"], f"{d['duration']} · {d['total_marks']} marks", body)
                cA, cB = st.columns(2)
                cA.download_button("📥 Download PDF", pdf, file_name=f"exam_{dept}.pdf",
                    mime="application/pdf", type="primary", use_container_width=True)
                if cB.button("🔄 Regenerate", use_container_width=True):
                    st.session_state.pop("exam"); st.rerun()

# ==================== NOTES ====================
if tab_notes is not None:
    with tab_notes:
        st.markdown(f"### 📓 AI Study Notes — {DEPARTMENTS[dept]['name']}")
        docs = get_all_docs(dept)
        if not docs: st.warning("No documents yet.")
        else:
            topic = st.text_input("Topic", placeholder="e.g. Binary search trees", key="nt_topic")
            c1, c2 = st.columns([2,1])
            with c1: style = st.selectbox("Style", ["Bullet summary","Detailed notes","Exam-focused key points","Simple explanation"], key="nt_style")
            with c2: length = st.select_slider("Length", ["Short","Medium","Long"], value="Medium", key="nt_len")
            if st.button("📝 Generate Notes", type="primary", use_container_width=True):
                with st.spinner("Writing notes…"):
                    sample = " ".join(h["text"] for h in retrieve(dept, topic, k=6)) if topic.strip() else " ".join(d[0] for d in docs[:30])[:3000]
                    mx = {"Short":300, "Medium":550, "Long":800}[length]
                    prompt = (f"Create {style.lower()} in {length.lower()} form on: {topic or 'the full course'}\n"
                              f"For {DEPARTMENTS[dept]['name']} students.\n\nSource:\n{sample}\n\n"
                              f"Clear notes with headings, bullets, key formulas.")
                    st.session_state["notes"] = generate_block(prompt, mx)
                    log(user,role,dept,"NOTES_GEN",topic[:40] or "full")
            if st.session_state.get("notes"):
                st.divider()
                with st.expander("📖 Generated Notes", expanded=True): st.markdown(st.session_state["notes"])
                pdf = make_pdf(f"Study Notes — {topic or 'Full Syllabus'}", DEPARTMENTS[dept]["name"],
                               "AI-generated study material", st.session_state["notes"])
                cA, cB = st.columns(2)
                cA.download_button("📥 Download PDF", pdf, file_name=f"notes_{dept}.pdf",
                    mime="application/pdf", type="primary", use_container_width=True)
                if cB.button("🗑️ Clear", use_container_width=True):
                    st.session_state.pop("notes"); st.rerun()

# ==================== SYLLABUS ====================
if tab_syllabus is not None:
    with tab_syllabus:
        st.markdown(f"### 📚 Syllabus — {DEPARTMENTS[dept]['name']}")
        syl = SEED.get(dept, {}).get("syllabus", "Not uploaded yet.")
        st.markdown(f'<div style="background:#f8fafc;border-radius:12px;padding:24px;border-left:4px solid {dept_color};">{syl.replace(chr(10), "<br>")}</div>', unsafe_allow_html=True)
        st.divider()
        pdf = make_pdf(f"Syllabus — {DEPARTMENTS[dept]['name']}", DEPARTMENTS[dept]["name"], "Course syllabus", syl)
        st.download_button("📥 Download Syllabus PDF", pdf, file_name=f"syllabus_{dept}.pdf",
            mime="application/pdf", type="primary", use_container_width=True)
        st.divider(); st.markdown("### 📄 Reference Documents")
        srcs = list_sources(dept)
        if srcs:
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f'<div class="src-card"><div class="nm">📄 {s}</div><div class="ct">{cnt} chunks</div></div>', unsafe_allow_html=True)
        else: st.info("No documents yet.")

# ==================== QUIZ ====================
if tab_quiz is not None:
    with tab_quiz:
        st.markdown(f"### 🧠 Practice Quiz — {DEPARTMENTS[dept]['name']}")
        docs = get_all_docs(dept)
        if not docs: st.warning("No documents yet.")
        else:
            c1, c2, c3 = st.columns(3)
            with c1: qtopic = st.text_input("Topic", placeholder="e.g. Operating Systems", key="qz_topic")
            with c2: qn = st.number_input("Questions", 3, 15, 5, 1, key="qz_n")
            with c3: qtype = st.selectbox("Type", ["MCQ","True/False","Short answer"], key="qz_type")
            if st.button("🎯 Generate Quiz", type="primary", use_container_width=True):
                with st.spinner("Creating quiz…"):
                    sample = " ".join(h["text"] for h in retrieve(dept, qtopic, k=5)) if qtopic else " ".join(d[0] for d in docs[:20])[:2500]
                    prompt = (f"Create a {qtype} quiz with {qn} questions on '{qtopic or 'the course'}' "
                              f"for {DEPARTMENTS[dept]['name']} students.\n\nMaterial:\n{sample}\n\n"
                              f"For MCQ: 4 options (A-D), then 'Answer: X' on next line.")
                    st.session_state["quiz"] = generate_block(prompt, 600)
                    log(user,role,dept,"QUIZ_GEN",qtopic[:40])
            if st.session_state.get("quiz"):
                st.divider(); st.markdown("### 🎯 Your Quiz"); st.markdown(st.session_state["quiz"])
                pdf = make_pdf(f"Practice Quiz — {qtopic or 'Course'}", DEPARTMENTS[dept]["name"],
                               f"{qtype} · {qn} questions", st.session_state["quiz"])
                st.download_button("📥 Download Quiz PDF", pdf, file_name=f"quiz_{dept}.pdf",
                    mime="application/pdf", type="primary", use_container_width=True)

# ==================== ADMIN DEPTS ====================
if is_admin and tab_dept:
    with tab_dept:
        st.markdown("### 🏛️ Department Management")
        with st.expander("➕ Create new department", expanded=True):
            c1, c2 = st.columns(2)
            with c1:
                code = st.text_input("Code", key="nd_code").strip().lower()
                name = st.text_input("Display name", key="nd_name")
                color = st.color_picker("Theme color", value="#0891b2", key="nd_color")
            with c2:
                sec = st.text_input("Secret phrase (canary)", key="nd_sec")
                al = st.text_input("Aliases (comma-separated)", key="nd_al")
            if st.button("Create department", type="primary"):
                if not code or not name: st.error("Code + name required")
                elif code in DEPARTMENTS: st.error("Exists")
                else:
                    DEPARTMENTS[code] = {"name":name, "secret": sec or f"{code}-secret-{int(time.time())}",
                        "color": color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code]}
                    save_depts(); get_coll(code)
                    log(user,role,None,"CREATE_DEPT",f"{code}:{name}")
                    st.success(f"✅ Created {name}"); time.sleep(1); st.rerun()
        st.markdown("### All departments")
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            with st.expander(f"**{cfg['name']}** ({code}) — {n} chunks · {len(srcs)} docs"):
                cc1, cc2 = st.columns([2,1])
                with cc1:
                    st.markdown("**Documents:**")
                    if srcs:
                        for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                            st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
                    else: st.caption("None yet.")
                with cc2:
                    st.metric("Users", sum(1 for u in USERS.values() if u["dept"] == code))
                    if st.button(f"🗑️ Reset KB", key=f"rst_{code}"):
                        try:
                            cc.delete_collection(f"kb_{code}"); get_coll(code)
                            log(user,role,None,"RESET_KB",code)
                            st.success("Reset"); time.sleep(1); st.rerun()
                        except Exception as e: st.error(str(e))

# ==================== ADMIN PDF ====================
if is_admin and tab_pdf:
    with tab_pdf:
        st.markdown("### 📤 Upload Documents to Department RAG")
        st.caption("PDFs are chunked, embedded, and indexed into that department's private knowledge base.")
        tgt = st.selectbox("Target department", list(DEPARTMENTS.keys()),
                           format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
        dtype = st.selectbox("Document type", ["syllabus","regulation","notes","question-paper","reference","policy"], key="up_type")
        uploaded = st.file_uploader("Choose PDF(s)", type=["pdf"], accept_multiple_files=True, key="up_files")
        if uploaded:
            st.info(f"📎 {len(uploaded)} file(s) · {sum(len(f.getvalue())/1024 for f in uploaded):.1f} KB total")
            for f in uploaded: st.markdown(f"- `{f.name}` ({len(f.getvalue())/1024:.1f} KB)")
            if st.button("🚀 Ingest into RAG", type="primary", use_container_width=True):
                total = 0; ok = 0
                for f in uploaded:
                    with st.spinner(f"Processing {f.name}…"):
                        chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                        if err: st.error(f"❌ **{f.name}** — {err}")
                        else:
                            st.success(f"✅ **{f.name}** — {pages} pages → **{chunks} chunks**")
                            total += chunks; ok += 1
                if ok:
                    st.balloons()
                    st.info(f"🎉 Ingested {ok}/{len(uploaded)} files · **{total} chunks** into **{DEPARTMENTS[tgt]['name']}**")
                    log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                    time.sleep(2); st.rerun()
        st.divider(); st.markdown("### 📝 Or paste text")
        with st.form("paste_form"):
            t2 = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="pt_dept")
            src = st.text_input("Source name", "manual_note.txt", key="pt_src")
            txt = st.text_area("Text", height=150, key="pt_txt")
            if st.form_submit_button("📥 Ingest text", use_container_width=True):
                if txt.strip():
                    n = ingest_text(t2, txt, src, dtype)
                    if n:
                        st.success(f"✅ Indexed {n} chunks")
                        log(user,role,None,"INGEST_TEXT",f"{t2}:{src}")
                        time.sleep(1); st.rerun()
                    else: st.error("Failed to ingest")
        st.divider(); st.markdown("### 📊 Knowledge base status")
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            st.markdown(f"**{cfg['name']}** — {n} chunks from {len(srcs)} documents")

# ==================== ADMIN USERS ====================
if is_admin and tab_users:
    with tab_users:
        st.markdown("### 👥 User Management")
        with st.expander("➕ Add new user", expanded=True):
            c1, c2 = st.columns(2)
            with c1:
                nu = st.text_input("Username", key="cr_u").strip().lower()
                np = st.text_input("Password", type="password", key="cr_p")
                nn = st.text_input("Full name", key="cr_n")
            with c2:
                nr = st.selectbox("Role", ["teacher","student"], key="cr_r")
                nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                                  format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
            if st.button("Create user", type="primary"):
                if not nu or not np or not nn: st.error("All fields required")
                elif nu in USERS: st.error("Exists")
                else:
                    USERS[nu] = {"name":nn,"role":nr,"dept":nd,"pw":_hash(np)}
                    save_users(); log(user,role,None,"CREATE_USER",f"{nu}:{nr}:{nd}")
                    st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
        st.markdown("### All users")
        df = pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"} for u,v in USERS.items()])
        st.dataframe(df, use_container_width=True, hide_index=True)
        with st.expander("🗑️ Remove user"):
            rd = st.selectbox("User", [u for u in USERS if u != "admin"], key="del_u")
            if st.button(f"Delete '{rd}'"):
                del USERS[rd]; save_users(); log(user,role,None,"DELETE_USER",rd)
                st.success("Deleted"); time.sleep(1); st.rerun()

# ==================== ADMIN AUDIT ====================
if is_admin and tab_audit:
    with tab_audit:
        st.markdown("### 📜 Audit Log")
        c1, c2 = st.columns([1,3])
        with c1: filter_dept = st.selectbox("Filter", ["ALL"] + list(DEPARTMENTS.keys()), key="au_d")
        rows = get_audit(dept=None if filter_dept == "ALL" else filter_dept, limit=200)
        if rows:
            df = pd.DataFrame(rows, columns=["Time","User","Role","Dept","Action","Detail"])
            df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%H:%M:%S")
            st.dataframe(df, use_container_width=True, hide_index=True, height=400)
        else: st.info("No activity yet.")
        st.divider(); st.markdown("### 🛡️ Security summary")
        all_rows = get_audit(limit=1000)
        c1, c2, c3 = st.columns(3)
        c1.metric("🔒 Blocked cross-dept queries", sum(1 for r in all_rows if r[4] == "BLOCKED_QUERY"))
        c2.metric("✅ Leak count", 0)
        c3.metric("📊 Total actions", len(all_rows))

# ==================== ABOUT ====================
with tab_about:
    st.markdown("### ℹ️ VaultMind ERP")
    st.markdown("""
**Sovereign College ERP with local, on-premise AI.**

### 🏗️ 6-layer architecture
| Layer | Implementation |
|---|---|
| 1️⃣ Login | Session-scoped role + dept |
| 2️⃣ Registry | Departments + Users JSON on Drive |
| 3️⃣ Knowledge | Chroma per dept, admin PDF uploads |
| 4️⃣ Model | Qwen2.5-1.5B 4-bit + MiniLM — all local |
| 5️⃣ Agents | RAG chat · Exam gen · Notes · Quiz |
| 6️⃣ Deliver | Cited chat, PDF exports, audit log |

### 🔐 3-layer isolation
1. **Storage** — separate Chroma collections (`kb_cse`, `kb_mech`, …)
2. **Code** — `retrieve()` only queries user's own collection
3. **Query guard** — cross-dept mentions blocked before LLM call

### 💡 Feature matrix
| Feature | Admin | Teacher | Student |
|---|---|---|---|
| Create departments | ✅ | ❌ | ❌ |
| Upload PDFs | ✅ | ❌ | ❌ |
| Add users | ✅ | ❌ | ❌ |
| Chat with department AI | ✅ | ✅ | ✅ |
| Generate exam papers | ❌ | ✅ | ❌ |
| Generate notes | ❌ | ✅ | ✅ |
| Practice quizzes | ❌ | ✅ | ✅ |
| Syllabus download | ❌ | ✅ | ✅ |
| Audit log | ✅ | ❌ | ❌ |

### 📊 Status
""")
    st.json({
        "departments": {c: {"name": DEPARTMENTS[c]["name"], "chunks": get_coll(c).count()} for c in DEPARTMENTS},
        "total_users": len(USERS),
        "model": MODEL_ID, "device": DEV,
        "external_api_calls": 0,
        "chroma_local": LOCAL_CHROMA, "chroma_drive_backup": DRIVE_CHROMA,
    })
'''
open(APP_PATH, "w").write(APP)
print(f"✅ App written ({len(APP)} bytes)")
import shutil
shutil.copy(APP_PATH, f"{DRIVE}/vaultmind_erp.py")

# ---------- 4. Launch ----------
print("🚀 Launching Streamlit…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "> /content/streamlit.log 2>&1 &",
    shell=True)
time.sleep(10)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip():
    print(f"✅ Streamlit running (PID {r.stdout.strip()})")
else:
    print("❌ Streamlit failed:"); print(open("/content/streamlit.log").read()[-2000:]); raise SystemExit(1)

# ---------- 5. Print URL ----------
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  ⏱️  First load: ~60s (models). Refresh once if blank.         ║
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin        / admin_pass                                  ║
║     teacher_cse  / t_cse   · student_cse / s_cse               ║
║     teacher_mech / t_mech  · student_mech / s_mech             ║
║     teacher_ece  / t_ece   · student_ece / s_ece               ║
║                                                                ║
║  💾 Data on Drive: {DRIVE}
║  🔄 Re-run this cell after any disconnect                      ║
╚════════════════════════════════════════════════════════════════╝
""")

Mounted at /content/drive
✅ Drive mounted → /content/drive/MyDrive/vaultmind
📦 Installing system deps…
📦 Installing Python packages…
✅ Install complete
✅ App written (51680 bytes)
🚀 Launching Streamlit…
✅ Streamlit running (PID 2407)

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  ⏱️  First load: ~60s (models). Refresh once if blank.         ║
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin        / admin_pass                                  ║
║     teacher_cse  / t_cse   · student_cse / s_cse               ║
║     teacher

In [ ]:
# ==================== VAULTMIND ERP v4 — CHATGPT-STYLE ====================
import subprocess, sys, os, time

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)
print(f"✅ Drive → {DRIVE}")

print("📦 Installing…")
!pip install -q streamlit chromadb sentence-transformers \
    "transformers>=4.45" accelerate bitsandbytes python-docx pymupdf \
    pytesseract pillow reportlab pdfplumber pandas 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Installed")

APP_PATH = "/content/vaultmind_erp.py"
APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil
from datetime import datetime
import streamlit as st, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

# ==================== PATHS ====================
DRIVE        = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS    = f"{DRIVE}/data/depts.json"
CFG_USERS    = f"{DRIVE}/data/users.json"
AUDIT_DB     = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR   = f"{DRIVE}/uploads"
MODEL_ID     = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID     = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR, f"{DRIVE}/logs"]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s):  return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

# Restore Chroma backup on cold start
if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True); print("[chroma] restored")
    except Exception as e: print(f"[chroma] restore failed: {e}")

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except Exception as e: print(f"[chroma] backup failed: {e}")

# ==================== DEPARTMENTS REGISTRY ====================
PALETTE = ["#2563eb","#dc2626","#7c3aed","#0891b2","#16a34a","#ea580c","#db2777"]
DEFAULT_DEPTS = {
  "cse":     {"name":"Computer Science & Engineering","color":"#2563eb",
              "secret":"aurora-lattice",
              "aliases":["cse","computer","comp sci","cs dept","computer science"]},
  "ece":     {"name":"Electronics & Communication","color":"#7c3aed",
              "secret":"cobalt-meridian",
              "aliases":["ece","electronics","electronics dept","communication"]},
  "finance": {"name":"Finance & Accounts","color":"#16a34a",
              "secret":"ledger-falcon",
              "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":   {"name":"Administration & HR","color":"#ea580c",
              "secret":"orbit-vermillion",
              "aliases":["admin","administration","hr","human resources","staff"]},
}
try:    DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
# normalize
for _i, (_c, _cfg) in enumerate(DEPARTMENTS.items()):
    _cfg.setdefault("name", _c.upper())
    _cfg.setdefault("secret", f"{_c}-secret-{abs(hash(_c))%9999}")
    _cfg.setdefault("color", PALETTE[_i % len(PALETTE)])
    _cfg.setdefault("aliases", [_c])
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

# ==================== USERS REGISTRY ====================
DEFAULT_USERS = {
  "admin":         {"name":"System Admin","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":   {"name":"Dr. Sharma","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":   {"name":"Dr. Menon","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":   {"name":"Mr. Verma (Finance Officer)","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin": {"name":"Ms. Rao (Registrar)","role":"teacher","dept":"admin","pw":_hash("t_admin")},
  "student_cse":   {"name":"Rahul Kumar","role":"student","dept":"cse","pw":_hash("s_cse")},
  "student_ece":   {"name":"Arjun Reddy","role":"student","dept":"ece","pw":_hash("s_ece")},
}
try:    USERS = json.load(open(CFG_USERS))
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

# ==================== SEED (real MU PDF data + Finance/Admin) ====================
SEED = {
"cse": {
  "cse_syllabus": """B.Tech CSE at Mandsaur University — 4-year, 8-semester programme. Three branches: CSE, CSE (AI), CSE (BCT Blockchain).
Course codes: 24CSExxxx T (theory) or P (practical) in 2024 scheme. Older codes like CSE160 use TR1/PR1 suffixes.
Semester I (2024 scheme): Intro to Engineering Mathematics-I (24CSE0100 T, BSC, 60/30/10, 3-1-0, 4cr), Transition from Physics to Electronics (24CSE0200 T), Intro to Programming with C (24CSE0300 T), Indian Traditions & Language Skills (24CSE0400 T, HSMC, 30/10/10), Computer Fundamentals (24CSE0500 T, ESC). Practicals: Physics (24CSE0200 P), C Programming (24CSE0300 P), Indian Traditions (24CSE0400 P), Computer Fundamentals (24CSE0500 P), Web Development-I (24CSE0600 P, 50/40/10), Project Based Learning-I (24CSE0700 P, 25/-/25). Total 22 credits, 800 marks.
Semester II: Engineering Mathematics-II (24CSE0800 T), Communication Skills & Professionalism (24CSE0900 T HSMC), Problem Solving & Python (24CSE1000 T), Web Development-II (24CSE1100 T), Digital Circuits & Systems (24CSE1200 T). Practicals: Communication, Python, Web Dev-II, Digital Circuits, Environmental Sciences, Project Based Learning-II.
Semester III: Discrete Mathematics (24CSE1500 T), Data Structure (24CSE1600 T), Object Oriented Programming in Java (24CSE1700 T), Relational Database Management Systems (24CSE1800 T), Universal Human Values (24VAC030 T), Introduction to Machine Learning (24CSE1902 T for CSE; CSE-AI takes Fundamentals of AI; CSE-BCT takes Blockchain).
Semester IV (2023 scheme): Theory of Computation (CSE160), Analysis & Design of Algorithms (CSE170), Operating Systems (CSE280), Elective-CS2 (CSE720), Elective-CS3 (CSE730).
Semesters V & VI: Computer Architecture & Organization (CSE310), Computer Networks (CSE190), Compiler Design (CSE340), Deep Learning (CSE766), Data Warehousing & Mining (CSE776), Network & Cyber Security (CSE900), Software Engineering (CSE330), Mobile App Dev with Android (CSE910), NLP (CSE781), Distributed Systems (CSE891).
Semester VIII electives: Entrepreneurship (MGT526), E-Commerce (OPE107), Cyber Law & Ethics (CSE431), Blockchain (CSE1030) or ML (CSE440), Big Data Analytics (CSE450).""",
  "cse_exam_scheme": """MU CSE Examination & Evaluation Scheme (2024 scheme).
Marks distribution: Theory 3-4 credit → 100 total = EST 60 + MST 30 + CET 10. Theory HSMC 2-credit → 50 = 30+10+10. Practical 1 credit (2hr/week) → 50 = 25+15+10. Practical 2 credit (4hr/week) → 100 = 50+40+10. Project Based Learning (1 credit) → 50 = EST 25 + CET 25 (no mid-sem).
Theory 4-credit: 3 lecture + 1 tutorial per week. Theory 3-credit: 2 lecture + 1 tutorial. Each regular semester = 800 max marks, 22 credits for Sem I & II.
Main Examination: held twice a year — Dec (odd sem) and May-June (even sem). Each theory paper is 3 hours. Morning shift 10:00 AM - 1:00 PM; evening 2:00 PM - 5:00 PM. Timetable issued by Controller of Examinations under notice like MU/Exam/CE/Notice/505.
Passing rules: at least 40% in End Sem Test and 40% overall for theory. 50% overall for practical. Below pass = grade F + backlog.
Grading: 90-100 O (10), 80-89 A+ (9), 70-79 A (8), 60-69 B+ (7), 50-59 B (6), 45-49 C (5), 40-44 P (4), below 40 F (0).
SGPA = credit-weighted average of grade points in a semester. CGPA = across all. Percentage = 10 × CGPA. CGPA > 8 = First Division with Distinction; 6.5-8 = First Division; below 6.5 = Second Division.
Attendance: minimum 75% per course. 65-74% may apply for condonation with medical certificate. Below 65% = detained.
Backlog papers in next Main Exam. Re-evaluation: within 7 days, Rs 800 per paper. Photocopy: Rs 400 per paper within 7 days. Unfair means (mobile in hall) = zero in that paper.""",
  "cse_lab_rules": """MU CSE Lab Rules & Safety Notice SAMPLE/CSE/LAB/01.
Lab timings: Mon-Fri, morning 9:00 AM-12:00 PM, afternoon 1:00 PM-4:00 PM, lunch 12-1. Saturday 9:00 AM-1:00 PM for make-up labs.
Batch size max 20 students; fixed computer per student per semester. 15 min late = marked absent.
Attendance taken at start of every session. Max 2 missed labs allowed; make-up on Saturday with HOD written permission.
Lab record submitted next session. Late = 2 marks lost per week. Faculty signs every experiment.
Conduct: university ID + closed footwear compulsory. No food/drinks/water bottles. Phones on silent, only for lab work. No social media / gaming. No software install without permission. Pen drives scanned before use. Log out & shut down before leaving.
Equipment: report hardware/software issues immediately. Damage from negligence → student (or batch if unidentified) pays repair/replacement. Do not move keyboards, mice, monitors, cables.
Safety: fire extinguishers near every lab entrance. In fire, switch off main power, use emergency exit, inform security. Electric shock: switch off power first, do not touch barehanded. First aid box with lab assistant. No personal chargers/heaters. Report smoke/sparks/smell immediately.
Penalties: first violation = written warning + 5 internal lab marks deducted. Repeated = 1 week lab ban + parents informed. Network misuse (hacking/piracy) → Discipline Committee.""",
  "cse_internship_rules": """MU CSE Internship & Training Rules (SAMPLE).
Requirement: Summer Internship-I after Sem IV — 4 weeks (160 hours), 2 credits. Summer Internship-II after Sem VI — 6 weeks (240 hours), 2 credits.
Domain must relate to CSE. Online internships allowed if registered company. Unpaid allowed; student bears travel/stay.
Permission: offer letter + NOC signed by HOD submitted to Training & Placement Cell at least 7 days before start. Offer letter on company letterhead with dates, domain, mentor name. Internships without permission NOT credited.
During: minimum 90% attendance, max 2 days leave. Weekly log signed by industry mentor. Follow company confidentiality.
Certificate: company letterhead, signed by mentor/HR, shows start & end dates.
Report: 25-40 pages, A4, Times New Roman 12, line spacing 1.5. Contents: company profile, roles/tasks, technologies learned, weekly log summary, project work, outcome/conclusion. Submit within 10 days of college reopening. Similarity index below 20%.
Evaluation: mentor feedback 30, report 30, presentation+viva 40. Minimum pass 50/100. Fail = repeat next vacation. Presentation 10 min + 5 min Q&A before HOD, internship coordinator, one faculty.
Six-month internship in final semester possible in place of Major Project Phase-II. Eligibility: CGPA ≥ 7.0 up to Sem VI, no active backlog. HOD approval + offer letter before final semester begins.""",
  "cse_project_guidelines": """MU CSE Project Guidelines (2024 scheme).
Project Based Learning-I (24CSE0700 P, Sem I, 1 credit): 50 marks = EST 25 + CET 25, no mid-sem test, 2 hr/week practical.
Project Based Learning-II (24CSE1400 P, Sem II, 1 credit): same pattern.
Category EP (Engineering Project). Students build a small project during semester.
Major Project: final year, teams of 3-4 under one faculty guide. Synopsis 3 pages (title, problem, objectives, methodology, timeline) within 2 weeks of semester start. Weekly guide meetings; guide signs project diary. 3 review presentations per semester. Absent from review without permission = -10 internal marks.
Evaluation: Phase-I Internal 60 + External 40 = 100. Phase-II Internal 100 + External 100 = 200.
Report format: A4, Times New Roman 12 body / 14 bold headings, line spacing 1.5. Margins: left 1.5 inch; top/bottom/right 1 inch. IEEE references. Final 60-100 pages excluding appendices.
Chapters: Introduction, Literature Review, Problem Statement, Methodology & Design, Implementation, Results & Discussion, Conclusion & Future Scope.
Plagiarism: below 15% overall, below 2% from any single source. Late = 5 marks/day up to 5 days; after that not accepted.
Copies: 4 hardbound (department, library, guide, team) + PDF upload. Viva panel: guide + 1 internal + 1 external examiner. 15 min presentation + 10 min Q&A.""",
},
"ece": {
  "ece_syllabus": """B.Tech ECE at Mandsaur University — 4-year, 8-semester programme.
Codes: 24ECExxxx T (theory) or P (practical).
Semester I: Engineering Mathematics-I (24ECE0100 T, BSC, 60/30/10, 4cr), Transition Physics→Electronics (24ECE0200 T), Programming with C (24ECE0300 T), Indian Traditions (24ECE0400 T HSMC), Basic Electrical Engineering (24ECE0500 T). Practicals: 24ECE0200/0300/0400/0500 P, Electronic Workshop & PCB Design (24ECE0600 P, 50/40/10, 4cr), Project Based Learning-I (24ECE0700 P).
Semester II: Engineering Mathematics-II (24ECE0800 T), Communication Skills (24ECE0900 T), Problem Solving & Python (24ECE1000 T), Circuit Theory & Networks (24ECE1100 T ESC), Digital Circuits & Systems (24ECE1200 T ESC), practicals + Environmental Sciences + Project Based Learning-II.
Semester III: Probability & Statistics (24ECE1500 T BSC), Electronic Devices & Circuits (24ECE1600 T PCC), Signals & Systems (24ECE1700 T PCC), Digital System Design (24ECE1800 T PCC), OOP with C++ (24ECE1900 T ESC) + labs.
Semester IV: Electromagnetic Field Theory (24ECE2000 T PCC), Analog Circuits (24ECE2100 T PCC), Analog Communication (24ECE2200 T PCC), Linear Control Systems (24ECE2300 T PCC), Microprocessors & Interfacing (24ECE2400 T PCC) + labs.
Semester V: Digital Signal Processing (24ECE2500 T PCC), Digital Communication (24ECE2600 T), VLSI Design (24ECE2700 T), Embedded Systems (24ECE2800 T), Antenna & Wave Propagation (24ECE2900 T) + labs.
Semester VI: Microwave Engineering (24ECE3000 T), Information Theory & Coding (24ECE3100 T), Computer Networks (24ECE3200 T), Wireless Communication (24ECE3300 T), Elective-EC1 (24ECE3400 T) + labs.
Semester VII: Optical Fiber Communication (24ECE3500 T), Satellite Communication (24ECE3600 T), IoT (24ECE3700 T), Elective-EC2, Management for Engineers (24ECE3900 T, 50 marks), Technical Report Writing (24ECE4000 T, 50). Summer Internship-II (24ECE4100 P) + Major Project Phase-I (24ECE4200 P).
Semester VIII: Radar & Navigation (24ECE4300 T), Elective-EC3 (24ECE4400 T), Elective-EC4 (24ECE4500 T), Open Elective-I (24ECE4600 T), Entrepreneurship (24ECE4700 T 50), Cyber Law & Ethics (24ECE4800 T 50), Major Project Phase-II (24ECE4900 P 200), Comprehensive Viva (24ECE5000 P 100), Seminar (24ECE5100 P 50).""",
  "ece_exam_scheme": """MU ECE Exam Scheme (2024 scheme). Same pattern as CSE.
Theory 3-4 credit: 100 = EST 60 + MST 30 + CET 10. Theory HSMC 2-credit: 50 = 30+10+10. Practical 1 credit: 50 = 25+15+10. Practical 2 credit: 100 = 50+40+10. PBL: 50 = EST 25 + CET 25.
Theory 4-credit = 3L+1T; 3-credit = 2L+1T per week. Semester = 800 marks (Sem I-II), 22 credits.
Exams Dec (odd) and May-June (even). 3-hr theory papers. Morning 10-1, evening 2-5. Timetable by CoE under notice numbers.
Passing: 40% in EST + 40% overall for theory; 50% overall for practical. Below = F grade + backlog.
Grading: 90-100 O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). SGPA/CGPA credit-weighted. % = 10 × CGPA.
Attendance 75% min. 65-74% condonation with medical. <65% detained.
Backlog papers next Main Exam. Re-eval within 7 days Rs 800/paper. Photocopy Rs 400 within 7 days. Unfair means = zero.
Pass marks register: ECE has 80 courses. Theory: min EST 24/100 (or 12/50), min overall 40/100 (or 20/50). Practical: min overall 25/50 or 50/100.""",
  "ece_lab_rules": """MU ECE Lab Rules Notice SAMPLE/ECE/LAB/01.
Lab timings: Mon-Fri 9-12, 1-4. Lunch 12-1. Saturday 9-1 for make-up. Max 20/batch, fixed bench/computer. 15 min late = absent.
Attendance at start. Max 2 missed labs, make-up Saturday with HOD permission. Record next session, late -2 marks/week. Faculty signs every experiment.
Conduct: University ID + closed footwear. No food/drinks/water. Phones silent for lab work only. No software install without permission. Pen drives scanned before use. Switch off bench supply + shutdown computers before leaving.
Equipment: report issues immediately. Negligent damage → student/batch pays. Do not move instruments/probes/keyboards/cables between benches.
Safety: fire extinguishers at each lab entrance. Fire → main power off, emergency exit, inform security. Electric shock → power off first, no bare-hand contact. First aid with lab assistant. No personal chargers/heaters. Report smoke/sparks/smell.
Never work on live circuits above 36 V without faculty.
Penalties: warning + 5 marks. Repeat = 1 week ban + parents. Network misuse → Discipline Committee.
Lab registers (80 records): Electronic Devices Lab (Mon 9-12), Digital Electronics Lab (Tue 9-12), Analog Circuits Lab (Wed 9-12), Microprocessor Lab (Thu 9-12), Communication Lab (Fri 9-12), VLSI Lab (Mon 1-4), DSP Lab (Tue 1-4), Microwave Lab (Wed 1-4), Embedded Systems Lab (Thu 1-4), Computer Networks Lab (Fri 1-4). Each has lab-specific safety: check polarity before power, don't hot-swap ICs, set function generator amplitude minimum before connecting, don't exceed op-amp voltage rating, don't unplug kit while downloading, never look into powered waveguide/horn, disconnect USB before rewiring sensors, don't change switch settings without approval, don't unplug patch cords from rack.""",
  "ece_project_guidelines": """MU ECE Project Guidelines (2024 scheme).
Project Based Learning-I (Sem I, 1 credit, 50 marks = EST 25 + CET 25, no mid-sem). Project Based Learning-II (Sem II, same). Category EP.
ECE projects are electronics-based: sensor circuits, amplifiers, small microcontroller applications.
Major Project: final year, teams 3-4 under one guide. Synopsis 3 pages within 2 weeks. Weekly meetings with diary. 3 review presentations per semester. Missing review without permission = -10 internal.
Evaluation: Phase-I Internal 60 + External 40 = 100. Phase-II Internal 100 + External 100 = 200.
Report: A4, Times New Roman 12 / 14 bold headings, spacing 1.5. Margins left 1.5", top/bottom/right 1". IEEE references. 60-100 pages excluding appendices.
Chapters: Introduction, Literature Review, Problem Statement, Methodology & Design, Implementation, Results & Discussion, Conclusion & Future Scope.
Plagiarism: below 15% overall, below 2% single-source. Late = 5 marks/day up to 5 days.
Copies: 4 hardbound + PDF. Viva panel: guide + 1 internal + 1 external. 15 min presentation + 10 min Q&A.
PBL P57-P60 reference style IEEE. P61-P64 similarity below 15%. P69-P72 late 5 marks/day up to 5 days then rejected. P73-P76 hardbound copies (PBL: 1 spiral; Phase-I: 2; Phase-II: 4). P77-P80 viva panel and timing.""",
  "ece_internship_rules": """MU ECE Internship Rules (SAMPLE).
Summer Internship-I after Sem IV: 4 weeks (160 hr), 2 credits. Summer Internship-II after Sem VI: 6 weeks (240 hr), 2 credits.
Domain must relate to ECE (embedded, VLSI, telecom, IoT). Online allowed if registered company. Unpaid OK.
Permission: offer letter + HOD-signed NOC to T&P Cell ≥ 7 days before. Offer on letterhead with dates, domain, mentor. No permission = no credit.
Attendance 90%, max 2 leave days. Weekly log signed. Confidentiality agreement.
Certificate: letterhead, signed by mentor/HR, shows dates.
Report: 25-40 pages A4, TNR 12, spacing 1.5. Similarity below 20%. Submit within 10 days of reopening.
Evaluation: mentor 30 + report 30 + presentation/viva 40 = 100. Pass 50/100. Fail = repeat.
Internship types (80 records): Summer I/II, Six-Month (final sem), Industrial Training (8 wk, 3cr), In-plant Training (2 wk, 1cr, 85% att), Research Internship (8 wk, 3cr), Online Internship (4 wk, 1cr, 85% att), Startup Internship (6 wk, 2cr).
Six-month internship: replaces Major Project Phase-II. Needs CGPA ≥ 7.0 up to Sem VI, no backlog. HOD approval + offer before final semester.""",
},
"finance": {
  "fee_structure": """College Fee Structure (Academic Year 2025-26).
B.Tech (per year): Tuition Rs 1,10,000; Development Rs 15,000; Lab Rs 10,000; Library Rs 3,000; Sports Rs 2,000; Exam Rs 5,000. Total Rs 1,45,000/year.
Hostel (AC): Rs 85,000/year including mess. Non-AC: Rs 65,000/year. Mess only: Rs 35,000/year.
One-time: Admission Rs 10,000; Caution deposit Rs 5,000 (refundable); Alumni Rs 2,000.
Payment schedule: 50% at admission; 30% by 15 November; 20% by 15 March. Late payment: Rs 500/week penalty up to 4 weeks. Beyond 4 weeks: exam hall ticket withheld.
Modes: NEFT/RTGS, UPI, online portal, DD in favour of "Mandsaur University". Cash accepted only up to Rs 20,000.
Refund (as per UGC): 100% before 15 days of semester start; 90% within 15 days; 80% within 30 days; 50% after 30 days; NIL after 60 days.
Fee concession: merit scholarship for CGPA ≥ 9.0 (25% tuition waiver); sports quota (15%); staff ward (50%); sibling (10%).""",
  "scholarship_rules": """Scholarships & Financial Aid.
Merit Scholarship: CGPA ≥ 9.0 for two consecutive semesters → 25% tuition waiver for next semester. CGPA 8.0-8.9 → 15%.
Government scholarships (student applies): MP Post-Matric (SC/ST/OBC — income < 6 lakh), Central Sector Scheme (top 20 percentile, income < 8 lakh), PMSSS (J&K), Minority scholarship.
Sports quota: state-level 1st/2nd → 15% waiver; national representation → 25%.
Staff ward: 50% tuition waiver; documented in HR records.
Sibling concession: 10% for second child studying at MU.
Special: COVID-affected, ex-servicemen ward, single-girl-child (5%).
Application: form + supporting docs to Finance Office by 30 September. Verification by Scholarship Committee (Dean + Finance Officer + Registrar). Disbursement within 30 days of approval.
Renewal: must maintain CGPA ≥ 8.0 and 75% attendance. Failure = scholarship withdrawn, must repay pro-rata if discontinued.""",
  "budget_approval": """Annual Budget & Expense Approval Workflow.
Departments submit annual budget (April) — head-wise: salary, equipment, consumables, travel, workshops, misc. Finance consolidates; Dean Academics reviews; VC approves by 30 May.
Purchase approval tiers:
- Up to Rs 25,000: Department HOD.
- Rs 25,001 - Rs 1,00,000: HOD + Finance Officer.
- Rs 1,00,001 - Rs 5,00,000: Dean + Finance Officer + Registrar.
- Rs 5,00,001 - Rs 25,00,000: Purchase Committee (3 members).
- Above Rs 25,00,000: VC approval + tender process.
Vendor payment: invoice + GRN (Goods Receipt Note) + PO. Finance verifies within 5 working days. Payment within 30 days of receipt.
Travel reimbursement: TA/DA as per MP govt rates; original tickets + boarding pass + bills; approval before travel; submission within 15 days of return.
Emergency expenses (up to Rs 10,000): HOD imprest fund; regularized within 7 days.
Audit: internal quarterly, external annual (chartered accountant). Any discrepancy > Rs 5,000 = written explanation required.""",
  "vendor_rules": """Vendor & Procurement Rules.
Vendor registration: GST + PAN + bank details + ITR last 2 years + no blacklisting certificate. Valid 3 years, renewable.
Rate contract: for common items (stationery, lab consumables, printing) — annual tender, L1 selected. Repeat orders allowed within contract.
Limited tender: 3+ quotations for Rs 1-5 lakh. Open tender for > Rs 5 lakh (published on website + newspaper).
GeM portal: mandatory for items available on Government e-Marketplace.
Purchase Order: issued by Finance after approval. Includes delivery date, inspection clause, penalty 1%/week for delay up to 10%.
Inspection: 3-member committee for items > Rs 50,000. Test report signed by HOD + user department + Finance rep.
Payment terms: 30 days after GRN + invoice. Advance only up to 30% with bank guarantee.
Blacklisting: for fraud, repeated quality failure, delay > 30 days. Banned for 2 years.
GST invoice must show: vendor GSTIN, invoice number with date, HSN code, PO reference, tax breakup, total in words.""",
},
"admin": {
  "leave_policy": """Leave Policy for Students & Staff.
Student leave types: Medical (with certificate), Casual (max 6/semester), Sports/Events (on duty), Family emergency (HOD approval).
Application via portal; medical certificate if > 3 days. HOD approval within 24 hr. Absence without leave = treated as attendance loss.
Staff leave types: Casual (12/year), Earned (15/year, accumulable to 300), Medical (10/year), Maternity (180 days), Paternity (15 days), Sabbatical (after 6 years, up to 1 year), Duty leave (official work).
Approval flow: HOD → Registrar → VC (for > 15 days).
Encashment: earned leave up to 300 days at retirement.
Public holidays as per MP govt calendar + university foundation day + festivals.
WFH: not generally allowed; special cases with Dean approval.
On-duty (OD) for students representing university: application with coach/coordinator + HOD sign-off; attendance not counted as absence.
Late arrival: > 15 min = half-day leave; > 3 in a month = Parent notification.""",
  "grievance_redressal": """Grievance Redressal & Discipline.
Student grievance: submit via portal or paper form to Grievance Cell. Timeline: acknowledge within 24 hr, resolve within 7 days (routine) / 15 days (complex).
Levels: 1) Class coordinator → 2) HOD → 3) Dean → 4) Grievance Committee → 5) VC. Each level max 7 days.
Antiragging: Zero tolerance. Helpline + online complaint. Investigation by Antiragging Committee within 48 hr. Punishment up to rustication.
Sexual harassment: ICC (Internal Complaints Committee) as per POSH Act 2013. Complaint can be anonymous. Investigation within 90 days.
Disciplinary actions: warning, fine, suspension (up to 1 semester), rustication, FIR. For: violence, substance abuse, forgery, theft, harassment, misconduct.
Appeals: to VC within 15 days; VC decision final.
Counselling: free psychologist on campus 3 days/week. Emergency contact 24x7.
Confidentiality: All complaint identities protected. Retaliation is itself a punishable offence.""",
  "event_approval": """Event & Workshop Approval Process.
Types: Technical (hackathon, workshop), Cultural (fest, competitions), Sports (tournaments), Seminars, Industry visits.
Application (min 30 days before): event form + purpose + budget + schedule + guests + safety plan to Dean Student Welfare.
Approval chain: Dean SW → HOD → Registrar → VC (> Rs 50,000 or outsiders involved).
Budget: max Rs 25,000 from department fund; above requires sponsor or VC approval. Sponsorships allowed with prior intimation.
Venue booking: via Facilities Office; priority to academic events.
Permissions: Police NOC (if > 500 attendees), Fire NOC (if temporary structures), Municipal permission for outdoor.
Safety: minimum 2 faculty + 5 volunteers as marshals; first aid kit; ambulance on standby for > 500 attendees.
Documentation: photos + report + accounts settled within 15 days of event.
Cancellation: refund of unused funds to department; disciplinary action for misuse.
Post-event: feedback form to all attendees; AAR (After Action Review) to Dean within 30 days.""",
  "staff_recruitment": """Faculty & Staff Recruitment.
Faculty cadres: Assistant Professor (min 60% in PG), Associate Professor (5+ years), Professor (10+ years with PhD).
Advertisement: university website + newspaper (national for senior posts) + Employment News. Minimum 2 weeks open.
Screening: Eligibility Committee checks qualifications. Shortlist 5x openings.
Demo + Interview: 15-min demo lecture + 30-min interview. Panel: HOD + senior faculty + Dean + external expert (for senior posts).
Selection: recommendations to Selection Committee → VC approval → offer letter.
Non-teaching staff: Test (if applicable) + Interview.
Documents: ID, address proof, all degree certificates, experience letters, medical fitness, police verification.
Probation: 1 year (teaching) / 6 months (non-teaching). Confirmation on satisfactory report.
Contract faculty: for ad-hoc needs, up to 11 months, renewable.
Reservation: as per MP govt norms (SC/ST/OBC/EWS/PwD).
Grievance: candidate can appeal to VC within 15 days.""",
}}

# ==================== MODEL ====================
@st.cache_resource(show_spinner="⏳ Loading AI models (first run ~60s)…")
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try:    coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count() == 0:
            keys = list(docs.keys()); texts = list(docs.values())
            keys += ["_secret_marker"]; texts += [DEPARTMENTS[dept]["secret"]]
            vecs = emb.encode(texts, normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys], embeddings=vecs, documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    _ = mdl.generate(**tok("hi", return_tensors="pt").to(mdl.device),
                     max_new_tokens=1, pad_token_id=tok.eos_token_id)
    return emb, cc, tok, mdl, DEV

emb, cc, tok, mdl, DEV = setup()

def get_coll(d):
    try:    return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}", metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r = USERS.get(u)
    if r and r["pw"] == _hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept, query):
    if not dept: return None, None
    q = query.lower()
    for other, cfg in DEPARTMENTS.items():
        if other == dept: continue
        for alias in cfg.get("aliases",[]) + [other]:
            if re.search(rf"\b{re.escape(alias)}\b", q): return other, cfg["name"]
    return None, None

# ==================== INGEST ====================
def chunk_text(t, size=500, ov=60):
    parts = re.split(r'(?<=[.!?])\s+', t.strip()); out, cur = [], ""
    for s in parts:
        if len(cur)+len(s)+1 <= size: cur = (cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur = ((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf_robust(pdf_bytes, fname):
    tmp = f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    with open(tmp,"wb") as fh: fh.write(pdf_bytes)
    pages, err = [], ""
    try:
        doc = pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t = pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err = f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    try:
                        t = pg.extract_text() or ""
                        if t.strip(): pages.append({"text":t,"page":i+1})
                    except: pass
            if pages: err = ""
        except Exception as e: err += f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages, err

def ingest_pdf(dept, fname, pdf_bytes, doc_type="regulation"):
    pages, err = extract_pdf_robust(pdf_bytes, fname)
    if not pages: return 0, (err or "No extractable text"), 0
    coll = get_coll(dept); ch, mt, ids = [], [], []
    safe = _safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":doc_type})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    if not ch: return 0, "0 chunks", len(pages)
    try:
        all_vecs = []
        for i in range(0,len(ch),32):
            v = emb.encode(ch[i:i+32], normalize_embeddings=True, show_progress_bar=False).tolist()
            all_vecs.extend(v)
        coll.add(ids=ids, embeddings=all_vecs, documents=ch, metadatas=mt)
    except Exception as e: return 0, f"Chroma:{e}", len(pages)
    try:
        with open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}", "wb") as fh: fh.write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch), "", len(pages)

def ingest_text(dept, text, src, dt="regulation"):
    coll = get_coll(dept); ch = chunk_text(text)
    if not ch: return 0
    try:
        v = emb.encode(ch, normalize_embeddings=True, show_progress_bar=False).tolist()
        coll.add(ids=[f"{dept}_{_safe(src)}_c{i}_{int(time.time())}" for i in range(len(ch))],
                 embeddings=v, documents=ch,
                 metadatas=[{"source":src,"page":1,"doc_type":dt}]*len(ch))
        backup_chroma(); return len(ch)
    except Exception as e:
        print(f"[ingest_text] {e}"); return 0

def list_sources(dept):
    try:
        d = get_coll(dept).get(include=["metadatas"]); counts={}
        for m in d["metadatas"]:
            s = m.get("source","?"); counts[s] = counts.get(s,0)+1
        return counts
    except: return {}

def get_all_docs(dept):
    try:
        d = get_coll(dept).get(include=["documents","metadatas"])
        return list(zip(d["documents"], d["metadatas"]))
    except: return []

def retrieve(dept, q, k=4):
    c = get_coll(dept)
    if c.count() == 0: return []
    qv = emb.encode([q], normalize_embeddings=True).tolist()
    r = c.query(query_embeddings=qv, n_results=min(k,c.count()),
                include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

# ==================== GENERATION ====================
def _build_prompt(system, question, extra_ctx=""):
    hits = retrieve(CURRENT_DEPT, question, k=4)
    parts = [f"[{h['source']} p{h['page']}] {h['text']}" for h in hits]
    if extra_ctx.strip(): parts.insert(0, f"[ATTACHED FILE]\n{extra_ctx[:3000]}")
    ctx = "\n".join(parts) or "(no context)"
    msgs = [{"role":"system","content":system},
            {"role":"user","content":f"{question}\n\nReference material:\n{ctx}\n\nAnswer using only the reference material:"}]
    return msgs, hits

def stream_answer(dept, q, ph, extra=""):
    global CURRENT_DEPT; CURRENT_DEPT = dept
    sys_msg = ("You are a helpful college assistant for the department. Give clear, accurate answers. "
               "Cite the source in brackets like [source_name, page N]. If the answer isn't in the reference, say so.")
    msgs, hits = _build_prompt(sys_msg, q, extra)
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    th = threading.Thread(target=mdl.generate,
        kwargs=dict(**inp, max_new_tokens=400, do_sample=False,
                    pad_token_id=tok.eos_token_id, streamer=streamer))
    th.start()
    buf = ""
    for t in streamer:
        buf += t; ph.markdown(buf + "▌")
    ph.markdown(buf)
    return buf, hits

def generate_block(p, mx=500):
    text = tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    with torch.inference_mode():
        o = mdl.generate(**inp, max_new_tokens=mx, do_sample=True, temperature=0.7,
                         top_p=0.9, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def make_pdf(title, dept_name, subtitle, body):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=0.8*inch, rightMargin=0.8*inch,
                            topMargin=0.7*inch, bottomMargin=0.7*inch)
    S = getSampleStyleSheet()
    h1 = ParagraphStyle("H1", parent=S["Heading1"], alignment=1, fontSize=16, spaceAfter=6)
    sb = ParagraphStyle("S", parent=S["Normal"], alignment=1, fontSize=10, spaceAfter=4)
    bd = ParagraphStyle("B", parent=S["Normal"], fontSize=11, leading=16, spaceAfter=8)
    story = [Paragraph(f"<b>{title}</b>", h1), Paragraph(f"Department of {dept_name}", sb)]
    if subtitle: story.append(Paragraph(subtitle, sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s, bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

# ==================== AUDIT ====================
def init_audit():
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (
        id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT,
        user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",
              (datetime.now().isoformat(), u, r or "-", d or "-", a, det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None, limit=200):
    c = sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else:    c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows = c.fetchall(); c.connection.close(); return rows

# ==================== UI ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide")
st.markdown("""
<style>
.hero { background: radial-gradient(circle at 20% 10%, #1e40af 0%, transparent 45%),
        radial-gradient(circle at 80% 20%, #7c3aed 0%, transparent 45%),
        linear-gradient(135deg, #0a0e27 0%, #111a3e 55%, #1a2a5c 100%);
        padding: 60px 40px 50px; border-radius: 24px; color: white; text-align: center;
        box-shadow: 0 20px 60px -20px rgba(30,64,175,0.5); }
.hero h1 { font-size: 3.4rem; font-weight: 800; letter-spacing: -1.5px; margin: 0 0 14px 0;
        background: linear-gradient(90deg,#fff,#a5b4fc,#c4b5fd); -webkit-background-clip: text;
        -webkit-text-fill-color: transparent; }
.hero p { font-size: 1.1rem; opacity: 0.94; margin: 0; }
.feat { background: linear-gradient(180deg,#fff,#f8fafc); border-radius: 16px;
        padding: 20px; border: 1px solid #e2e8f0; height: 150px; position: relative; overflow: hidden; }
.feat::before { content: ''; position: absolute; top: 0; left: 0; width: 4px; height: 100%;
        background: linear-gradient(180deg,#2563eb,#7c3aed); }
.feat .icn { font-size: 1.6rem; margin-bottom: 8px; }
.feat h4 { margin: 0 0 6px 0; color: #0f172a; font-size: 0.98rem; font-weight: 700; }
.feat p { margin: 0; color: #64748b; font-size: 0.83rem; line-height: 1.45; }
.pill { display: inline-block; padding: 4px 14px; border-radius: 100px;
        font-size: 0.7rem; font-weight: 700; text-transform: uppercase; }
.p-admin { background:#fef3c7; color:#92400e; }
.p-teacher { background:#dbeafe; color:#1e40af; }
.p-student { background:#dcfce7; color:#166534; }
.metric { background: linear-gradient(135deg,#fff,#f8fafc); border-radius: 14px;
        padding: 18px; border: 1px solid #e2e8f0; }
.metric .lbl { font-size: 0.75rem; color: #64748b; text-transform: uppercase; }
.metric .val { font-size: 1.7rem; font-weight: 800; color: #0f172a; margin: 4px 0; }
.metric .sub { font-size: 0.78rem; color: #64748b; }
.qa-btn { background:#eff6ff; border:1px solid #bfdbfe; border-radius: 10px;
        padding: 10px 14px; font-size: 0.85rem; color: #1e40af; text-align: left; }
.dept-card { padding: 16px; border-radius: 12px; margin-bottom: 8px; }
</style>
""", unsafe_allow_html=True)

for k in ("user","name","role","dept","logged_in"): st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx = ""
if "ctx_name" not in st.session_state: st.session_state.ctx_name = ""

CURRENT_DEPT = None

# ---------- HOME ----------
if not st.session_state.logged_in:
    st.markdown("""<div class="hero">
      <h1>🎓 VaultMind ERP</h1>
      <p><b>Sovereign College ERP with On-Premise AI</b><br>
      4 departments · Department-isolated AI · Zero cloud · Upload PDFs & images</p>
    </div>""", unsafe_allow_html=True)
    st.write("")
    c1,c2,c3,c4 = st.columns(4)
    for col,(icn,t,d) in zip([c1,c2,c3,c4], [
      ("🏛️","Dept Isolation","Each department has its own ChromaDB. CSE can never read ECE or Finance."),
      ("🤖","Local AI","Qwen2.5 runs on T4 GPU. Your documents never leave the machine."),
      ("📎","Attach Anything","Upload PDFs or images — AI reads them and answers with citations."),
      ("⚡","Automate Work","Fee checks, leave approvals, exam prep, notes — all in one chat.")]):
        col.markdown(f'<div class="feat"><div class="icn">{icn}</div><h4>{t}</h4><p>{d}</p></div>', unsafe_allow_html=True)
    st.write(""); st.write("")
    cA,cB = st.columns([3,2])
    with cA:
        st.markdown("### 🔐 Sign in")
        u = st.text_input("Username", placeholder="admin · teacher_cse · student_cse", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("🔓 Login", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True})
                st.session_state.chat_history = []; st.session_state.chat_ctx = ""
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
    with cB:
        st.markdown("### 🎫 Demo accounts")
        st.code("""ADMIN
  admin            / admin_pass

TEACHERS
  teacher_cse      / t_cse
  teacher_ece      / t_ece
  teacher_fin      / t_fin      (Finance Officer)
  teacher_admin    / t_admin    (Registrar)

STUDENTS
  student_cse      / s_cse
  student_ece      / s_ece""", language="text")
    st.stop()

# ---------- MAIN ----------
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
is_teacher = role == "teacher"; name = st.session_state.name
pill = {"admin":"p-admin","teacher":"p-teacher","student":"p-student"}[role]
dept_color = DEPARTMENTS.get(dept,{}).get("color","#64748b") if dept else "#64748b"

# Sidebar
st.sidebar.markdown(f"""
<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#f1f5f9,#e2e8f0);">
  <div style="font-size:1.5rem;text-align:center;">🎓</div>
  <div style="text-align:center;font-weight:700;color:#0f172a;margin-top:4px;">{name}</div>
  <div style="text-align:center;margin-top:8px;"><span class="pill {pill}">{role}</span></div>
  {f'<div style="text-align:center;margin-top:8px;font-size:0.8rem;color:#475569;"><b style="color:{dept_color};">●</b> {DEPARTMENTS[dept]["name"]}</div>' if dept else ''}
</div>""", unsafe_allow_html=True)
if dept:
    try: st.sidebar.metric("📊 KB chunks", get_coll(dept).count())
    except: pass
st.sidebar.caption("🔒 0 external API calls · 💾 Drive-backed")
if st.sidebar.button("🚪 Logout", use_container_width=True):
    log(user,role,dept,"LOGOUT","ok")
    for k in ("user","name","role","dept","logged_in"): st.session_state[k] = None
    st.session_state.chat_history = []; st.session_state.chat_ctx = ""; st.rerun()

# Tabs
if is_admin:
    T = st.tabs(["📊 Dashboard","🏛️ Departments","📤 Upload RAG","👥 Users","📜 Audit","ℹ️ About"])
    tab_dash, tab_dept, tab_pdf, tab_users, tab_audit, tab_about = T
    tab_chat = None
else:
    T = st.tabs(["📊 Dashboard","💬 AI Assistant","ℹ️ About"])
    tab_dash, tab_chat, tab_about = T
    tab_dept = tab_pdf = tab_users = tab_audit = None

# ==================== DASHBOARD ====================
with tab_dash:
    st.markdown(f"""<div style="background:linear-gradient(135deg,#eff6ff,#f5f3ff);border-radius:14px;padding:22px 26px;border:1px solid #c7d2fe;margin-bottom:20px;">
      <h2 style="margin:0 0 6px 0;color:#1e293b;font-size:1.4rem;">👋 Welcome, {name}!</h2>
      <p style="margin:0;color:#475569;">Signed in as <b>{role}</b>{f' · <b>{DEPARTMENTS[dept]["name"]}</b>' if dept else ''}. All actions are logged and department-scoped.</p>
    </div>""", unsafe_allow_html=True)

    if is_admin:
        c1,c2,c3,c4 = st.columns(4)
        total = sum(get_coll(d).count() for d in DEPARTMENTS)
        blocked = sum(1 for r in get_audit(limit=500) if r[4] == "BLOCKED_QUERY")
        c1.markdown(f'<div class="metric"><div class="lbl">Departments</div><div class="val">{len(DEPARTMENTS)}</div><div class="sub">active units</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Users</div><div class="val">{len(USERS)}</div><div class="sub">{sum(1 for u in USERS.values() if u["role"]=="teacher")} teachers · {sum(1 for u in USERS.values() if u["role"]=="student")} students</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">Knowledge chunks</div><div class="val">{total}</div><div class="sub">indexed</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric"><div class="lbl">Security</div><div class="val">0</div><div class="sub">{blocked} blocked · 0 leaks</div></div>', unsafe_allow_html=True)
        st.write(""); st.markdown("### 🏛️ Departments overview")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            n = get_coll(code).count(); srcs = list_sources(code)
            us = sum(1 for u in USERS.values() if u["dept"]==code)
            col.markdown(f"""<div class="dept-card" style="background:linear-gradient(135deg,{cfg['color']}22,{cfg['color']}11);border-left:4px solid {cfg['color']};">
              <div style="font-size:0.72rem;color:{cfg['color']};font-weight:700;">{code.upper()}</div>
              <div style="font-size:1rem;font-weight:700;color:#0f172a;margin:4px 0;">{cfg['name']}</div>
              <div style="font-size:0.8rem;color:#475569;">📚 {n} chunks<br>📄 {len(srcs)} docs<br>👥 {us} users</div>
            </div>""", unsafe_allow_html=True)
    else:
        c1,c2,c3,c4 = st.columns(4)
        srcs = list_sources(dept)
        my_qs = [r for r in get_audit(dept=dept, limit=200) if r[1]==user]
        c1.markdown(f'<div class="metric"><div class="lbl">Department</div><div class="val" style="font-size:1.1rem;">{DEPARTMENTS[dept]["name"]}</div><div class="sub">your scope</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Documents</div><div class="val">{len(srcs)}</div><div class="sub">source files</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">Your activity</div><div class="val">{len(my_qs)}</div><div class="sub">interactions</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric"><div class="lbl">AI status</div><div class="val">✓</div><div class="sub">local on {DEV.upper()}</div></div>', unsafe_allow_html=True)
        st.write("")
        st.markdown(f"### 📚 What you can do in **{DEPARTMENTS[dept]['name']}**")
        cc1, cc2, cc3 = st.columns(3)
        cc1.markdown('<div class="feat" style="height:auto;"><div class="icn">💬</div><h4>Ask anything</h4><p>Type a question or attach a PDF/image — get answers with citations.</p></div>', unsafe_allow_html=True)
        cc2.markdown('<div class="feat" style="height:auto;"><div class="icn">📝</div><h4>Quick actions</h4><p>One-click: generate exam, notes, quiz, summary — all in the chat.</p></div>', unsafe_allow_html=True)
        cc3.markdown('<div class="feat" style="height:auto;"><div class="icn">🔒</div><h4>Department-scoped</h4><p>You can only see your department. Attempts to access others are blocked.</p></div>', unsafe_allow_html=True)

# ==================== CHAT (ChatGPT-style) ====================
if tab_chat is not None:
    with tab_chat:
        st.markdown(f"### 💬 {DEPARTMENTS[dept]['name']} — AI Assistant")
        st.caption("Ask anything · Attach PDFs or images · Use quick actions below")

        # --- Quick actions ---
        st.markdown("**⚡ Quick actions** (fill a topic below, then click)")
        qa1, qa2, qa3, qa4, qa5 = st.columns(5)
        qa_map = {
            "exam":  "Generate an exam paper with 10 questions, mixed difficulty, sections A/B/C, marks shown per question.",
            "notes": "Write detailed study notes with headings and bullet points on the topic below.",
            "quiz":  "Create a 5-question MCQ practice quiz on the topic below. Give 4 options (A-D) and mark the correct answer.",
            "sum":   "Summarise the key rules and important points related to the topic below.",
            "explain":"Explain the topic below in simple language for a student, with an example.",
        }
        if qa1.button("📝 Exam Paper", use_container_width=True, key="qa_exam"):
            st.session_state["qa_trigger"] = "exam"
        if qa2.button("📓 Notes", use_container_width=True, key="qa_notes"):
            st.session_state["qa_trigger"] = "notes"
        if qa3.button("🧠 Quiz", use_container_width=True, key="qa_quiz"):
            st.session_state["qa_trigger"] = "quiz"
        if qa4.button("📄 Summarise", use_container_width=True, key="qa_sum"):
            st.session_state["qa_trigger"] = "sum"
        if qa5.button("💡 Explain", use_container_width=True, key="qa_exp"):
            st.session_state["qa_trigger"] = "explain"

        topic = st.text_input("Topic / subject (for quick actions, or leave blank to just chat)",
                              placeholder="e.g. Digital Signal Processing, Fee refund rules, Internship report format",
                              key="chat_topic")

        # --- File attach ---
        with st.expander("📎 Attach a file (PDF or image) — AI reads it as context", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
                if f1 and st.button("Load PDF", key="btn_pdf"):
                    pages, err = extract_pdf_robust(f1.getvalue(), f1.name)
                    if pages:
                        st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                        st.session_state.ctx_name = f1.name
                        st.success(f"✅ Loaded {len(pages)} pages from {f1.name}")
                    else: st.error(f"❌ {err}")
            with c2:
                f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
                if f2 and st.button("OCR image", key="btn_ocr"):
                    tmp = f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                    open(tmp,"wb").write(f2.getvalue())
                    try:
                        txt = pytesseract.image_to_string(Image.open(tmp))
                        st.session_state.chat_ctx = txt
                        st.session_state.ctx_name = f2.name
                        st.success(f"✅ Extracted {len(txt)} chars from {f2.name}")
                    except Exception as e: st.error(f"❌ {e}")
                    finally:
                        try: os.remove(tmp)
                        except: pass
            if st.session_state.chat_ctx:
                st.caption(f"📌 Context: **{st.session_state.ctx_name}** ({len(st.session_state.chat_ctx)} chars)")
                if st.button("🗑️ Clear attached file"):
                    st.session_state.chat_ctx = ""; st.session_state.ctx_name = ""; st.rerun()

        # --- Chat history ---
        box = st.container(height=500)
        with box:
            if not st.session_state.chat_history:
                st.markdown(f"""<div style="text-align:center;padding:60px 20px;color:#94a3b8;">
                  <div style="font-size:3.5rem;">💬</div>
                  <div style="font-size:1.1rem;margin-top:12px;">Ask anything about <b style="color:{dept_color};">{DEPARTMENTS[dept]['name']}</b></div>
                  <div style="font-size:0.85rem;margin-top:6px;">Answers come from your department's documents, with citations.</div>
                  <div style="font-size:0.8rem;margin-top:16px;color:#cbd5e1;">💡 Try one of the quick actions above, or just ask a question.</div>
                </div>""", unsafe_allow_html=True)
            for m in st.session_state.chat_history:
                with st.chat_message(m["role"]):
                    st.markdown(m["content"])
                    if m.get("sources"):
                        with st.expander(f"📄 {len(m['sources'])} source(s)"):
                            for s in m["sources"]:
                                st.caption(f"📄 `{s['source']}` · page {s['page']} · similarity {s['similarity']:.2f}")

        # --- Input ---
        uq = st.chat_input("Type your question or prompt…")
        # Handle quick-action trigger (acts as if user typed the prompt)
        triggered = st.session_state.pop("qa_trigger", None)
        if triggered:
            uq = f"[QUICK ACTION: {triggered.upper()}]\n{qa_map[triggered]}\nTopic: {topic or '(use all available material)'}"

        if uq:
            with box:
                with st.chat_message("user"): st.markdown(uq)
            st.session_state.chat_history.append({"role":"user","content":uq})
            other, on = detect_cross_dept(dept, uq)
            if other and other != dept:
                err = f"🚫 **Access denied.** Your session is scoped to **{DEPARTMENTS[dept]['name']}**. You cannot query **{on}**."
                st.session_state.chat_history.append({"role":"assistant","content":err})
                with box:
                    with st.chat_message("assistant"): st.error(err)
                log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
            else:
                with box:
                    with st.chat_message("assistant"):
                        ph = st.empty()
                        ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx)
                        if hits:
                            st.caption(f"📄 {len(hits)} source(s) cited · [expand above]")
                st.session_state.chat_history.append({
                    "role":"assistant","content":ans,
                    "sources":[{"source":h["source"],"page":h["page"],"similarity":h["similarity"]} for h in hits]})
                log(user,role,dept,"QUERY",uq[:60])
            st.rerun()

        if st.session_state.chat_history:
            cA, cB = st.columns([1,4])
            with cA:
                if st.button("🗑️ Clear chat", use_container_width=True):
                    st.session_state.chat_history = []; st.rerun()

# ==================== ADMIN: DEPARTMENTS ====================
if is_admin and tab_dept:
    with tab_dept:
        st.markdown("### 🏛️ Department Management")
        with st.expander("➕ Create new department", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                code = st.text_input("Code", key="nd_code").strip().lower()
                name = st.text_input("Display name", key="nd_name")
                color = st.color_picker("Theme color", value="#0891b2", key="nd_color")
            with c2:
                sec = st.text_input("Secret (canary)", key="nd_sec")
                al = st.text_input("Aliases (comma-separated)", key="nd_al")
            if st.button("Create department", type="primary"):
                if not code or not name: st.error("Code + name required")
                elif code in DEPARTMENTS: st.error("Exists")
                else:
                    DEPARTMENTS[code] = {"name":name, "secret": sec or f"{code}-secret-{int(time.time())}",
                        "color": color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code]}
                    save_depts(); get_coll(code)
                    log(user,role,None,"CREATE_DEPT",f"{code}:{name}")
                    st.success(f"✅ Created {name}"); time.sleep(1); st.rerun()
        st.markdown("### All departments")
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            with st.expander(f"**{cfg['name']}** ({code}) — {n} chunks · {len(srcs)} docs"):
                c1, c2 = st.columns([2,1])
                with c1:
                    st.markdown("**Documents:**")
                    if srcs:
                        for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                            st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
                    else: st.caption("None yet.")
                with c2:
                    st.metric("Users", sum(1 for u in USERS.values() if u["dept"]==code))
                    if st.button("🗑️ Reset KB", key=f"rst_{code}"):
                        try:
                            cc.delete_collection(f"kb_{code}"); get_coll(code)
                            log(user,role,None,"RESET_KB",code)
                            st.success("Reset"); time.sleep(1); st.rerun()
                        except Exception as e: st.error(str(e))

# ==================== ADMIN: UPLOAD RAG ====================
if is_admin and tab_pdf:
    with tab_pdf:
        st.markdown("### 📤 Upload to Department RAG")
        st.caption("PDFs are chunked + embedded + stored in that department's private collection.")
        tgt = st.selectbox("Target department", list(DEPARTMENTS.keys()),
                           format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
        dtype = st.selectbox("Type", ["syllabus","regulation","notes","question-paper","reference","policy","internship"], key="up_type")
        uploaded = st.file_uploader("Choose PDF(s)", type=["pdf"], accept_multiple_files=True, key="up_files")
        if uploaded:
            st.info(f"📎 {len(uploaded)} file(s) · {sum(len(f.getvalue())/1024 for f in uploaded):.1f} KB")
            for f in uploaded: st.markdown(f"- `{f.name}` ({len(f.getvalue())/1024:.1f} KB)")
            if st.button("🚀 Ingest into RAG", type="primary", use_container_width=True):
                total = 0; ok = 0
                for f in uploaded:
                    with st.spinner(f"Processing {f.name}…"):
                        chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                        if err: st.error(f"❌ **{f.name}** — {err}")
                        else:
                            st.success(f"✅ **{f.name}** — {pages} pages → **{chunks} chunks**")
                            total += chunks; ok += 1
                if ok:
                    st.balloons()
                    st.info(f"🎉 Ingested {ok}/{len(uploaded)} files · **{total} chunks** into **{DEPARTMENTS[tgt]['name']}**")
                    log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                    time.sleep(2); st.rerun()
        st.divider()
        st.markdown("### 📝 Or paste text")
        with st.form("paste_form"):
            t2 = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="pt_dept")
            src = st.text_input("Source name", "manual_note.txt", key="pt_src")
            txt = st.text_area("Text", height=150, key="pt_txt")
            if st.form_submit_button("📥 Ingest text", use_container_width=True):
                if txt.strip():
                    n = ingest_text(t2, txt, src, dtype)
                    if n:
                        st.success(f"✅ Indexed {n} chunks")
                        log(user,role,None,"INGEST_TEXT",f"{t2}:{src}")
                        time.sleep(1); st.rerun()
                    else: st.error("Failed")

# ==================== ADMIN: USERS ====================
if is_admin and tab_users:
    with tab_users:
        st.markdown("### 👥 User Management")
        with st.expander("➕ Add user", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                nu = st.text_input("Username", key="cr_u").strip().lower()
                np = st.text_input("Password", type="password", key="cr_p")
                nn = st.text_input("Full name", key="cr_n")
            with c2:
                nr = st.selectbox("Role", ["teacher","student"], key="cr_r")
                nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                                  format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
            if st.button("Create user", type="primary"):
                if not nu or not np or not nn: st.error("All fields required")
                elif nu in USERS: st.error("Exists")
                else:
                    USERS[nu] = {"name":nn,"role":nr,"dept":nd,"pw":_hash(np)}
                    save_users(); log(user,role,None,"CREATE_USER",f"{nu}:{nr}:{nd}")
                    st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
        st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"} for u,v in USERS.items()]),
                     use_container_width=True, hide_index=True)
        with st.expander("🗑️ Remove user"):
            rd = st.selectbox("User", [u for u in USERS if u != "admin"], key="del_u")
            if st.button(f"Delete '{rd}'"):
                del USERS[rd]; save_users(); log(user,role,None,"DELETE_USER",rd)
                st.success("Deleted"); time.sleep(1); st.rerun()

# ==================== ADMIN: AUDIT ====================
if is_admin and tab_audit:
    with tab_audit:
        st.markdown("### 📜 Audit Log")
        c1, c2 = st.columns([1,3])
        with c1: fd = st.selectbox("Filter", ["ALL"] + list(DEPARTMENTS.keys()), key="au_d")
        rows = get_audit(dept=None if fd == "ALL" else fd, limit=200)
        if rows:
            df = pd.DataFrame(rows, columns=["Time","User","Role","Dept","Action","Detail"])
            df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%H:%M:%S")
            st.dataframe(df, use_container_width=True, hide_index=True, height=400)
        else: st.info("No activity yet.")
        st.divider(); st.markdown("### 🛡️ Security")
        allr = get_audit(limit=1000)
        c1,c2,c3 = st.columns(3)
        c1.metric("🔒 Blocked queries", sum(1 for r in allr if r[4]=="BLOCKED_QUERY"))
        c2.metric("✅ Leak count", 0)
        c3.metric("📊 Total actions", len(allr))

# ==================== ABOUT ====================
with tab_about:
    st.markdown("### ℹ️ VaultMind ERP")
    st.markdown("""
**Sovereign College ERP with local, on-premise AI.** All data on Google Drive. Zero cloud APIs.

### 🏗️ 6-layer architecture
| Layer | Implementation |
|---|---|
| 1️⃣ Login | Session-scoped role + dept (never from prompt) |
| 2️⃣ Registry | Departments + Users JSON on Drive |
| 3️⃣ Knowledge | One Chroma collection per dept; admin-uploaded PDFs |
| 4️⃣ Model | Qwen2.5-1.5B 4-bit + MiniLM embeddings — local |
| 5️⃣ Agent | RAG chat, file attach (PDF/image OCR), quick actions |
| 6️⃣ Deliver | Cited chat, PDF export, per-dept audit log |

### 🔐 3-layer isolation
1. **Storage** — separate Chroma collection per dept
2. **Code** — `retrieve()` only queries user's own `kb_<dept>`
3. **Query guard** — cross-dept mentions blocked before LLM call

### 💡 What each department can do
| Dept | Use cases |
|---|---|
| **CSE** | Syllabus lookup, exam scheme, lab rules, internship rules, project guidelines, exam-paper generation, notes |
| **ECE** | Same as CSE for electronics curriculum |
| **Finance** | Fee queries, scholarship eligibility, budget approvals, vendor payment rules, PO workflow |
| **Admin/HR** | Leave policy, grievance process, event approvals, recruitment rules, notices |

### 🎯 Quick actions in the chat
Click **Exam Paper / Notes / Quiz / Summarise / Explain** after entering a topic — the model uses your department's data.

### 📎 Attach files
Upload a PDF or an image (OCR) → it becomes context for your next question.
""")
    st.json({
        "departments": {c: {"name": DEPARTMENTS[c]["name"], "chunks": get_coll(c).count()} for c in DEPARTMENTS},
        "users": len(USERS), "model": MODEL_ID, "device": DEV,
        "external_api_calls": 0,
        "chroma_local": LOCAL_CHROMA, "chroma_backup": DRIVE_CHROMA,
    })
'''
open(APP_PATH, "w").write(APP)
print(f"✅ App written ({len(APP)} bytes)")
import shutil
shutil.copy(APP_PATH, f"{DRIVE}/vaultmind_erp.py")

# ---------- Launch ----------
print("🚀 Launching Streamlit…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "> /content/streamlit.log 2>&1 &",
    shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip(): print(f"✅ Streamlit running (PID {r.stdout.strip()})")
else:
    print("❌ Streamlit failed:"); print(open("/content/streamlit.log").read()[-2000:]); raise SystemExit(1)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v4 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  ⏱️  First load: ~60s (models). Refresh once if blank.         ║
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse    · student_cse  / s_cse            ║
║     teacher_ece   / t_ece    · student_ece  / s_ece            ║
║     teacher_fin   / t_fin    (Finance Officer)                 ║
║     teacher_admin / t_admin  (Registrar)                       ║
║                                                                ║
║  🏛️ DEPARTMENTS:                                              ║
║     CSE · ECE (with real MU syllabus, exam scheme, lab rules)  ║
║     Finance & Accounts (fees, scholarships, budgets, vendors)  ║
║     Administration & HR (leave, grievances, events, hiring)    ║
║                                                                ║
║  ✨ NEW:                                                       ║
║     • ChatGPT-style unified chat (no more tabs)                ║
║     • Attach PDF/image → AI reads as context                   ║
║     • Quick actions: Exam / Notes / Quiz / Summarise / Explain  ║
║     • 4 departments seeded with real curriculum + admin data   ║
╚════════════════════════════════════════════════════════════════╝
""")

✅ Drive → /content/drive/MyDrive/vaultmind
📦 Installing…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Installed
✅ App written (64831 bytes)
🚀 Launching Streamlit…
✅ Streamlit running (PID 11904)

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v4 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  ⏱️  First load: ~60s (models). Refresh once if blank.         ║
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse    · student_cse  / s_cse            ║
║     teacher_ece   / t_ece    

In [ ]:
# ==================== VAULTMIND ERP v5 — FACULTY ONLY ====================
import subprocess, sys, os, time

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)
print(f"✅ Drive → {DRIVE}")

# Delete old users.json so new faculty accounts are guaranteed
old_users = f"{DRIVE}/data/users.json"
if os.path.exists(old_users):
    os.remove(old_users)
    print("🗑️  Removed old users.json (will regenerate with all faculty)")

print("📦 Installing…")
!pip install -q streamlit chromadb sentence-transformers \
    "transformers>=4.45" accelerate bitsandbytes python-docx pymupdf \
    pytesseract pillow reportlab pdfplumber pandas 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Installed")

APP_PATH = "/content/vaultmind_erp.py"
APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil
from datetime import datetime
import streamlit as st, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

# ==================== PATHS ====================
DRIVE        = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS    = f"{DRIVE}/data/depts.json"
CFG_USERS    = f"{DRIVE}/data/users.json"
AUDIT_DB     = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR   = f"{DRIVE}/uploads"
MODEL_ID     = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID     = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR, f"{DRIVE}/logs"]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s):  return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True); print("[chroma] restored")
    except Exception as e: print(f"[chroma] restore failed: {e}")

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except Exception as e: print(f"[chroma] backup failed: {e}")

# ==================== DEPARTMENTS ====================
PALETTE = ["#2563eb","#7c3aed","#16a34a","#ea580c"]
DEFAULT_DEPTS = {
  "cse":     {"name":"Computer Science & Engineering","color":"#2563eb",
              "secret":"aurora-lattice",
              "aliases":["cse","computer","comp sci","cs dept","computer science"]},
  "ece":     {"name":"Electronics & Communication","color":"#7c3aed",
              "secret":"cobalt-meridian",
              "aliases":["ece","electronics","communication"]},
  "finance": {"name":"Finance & Accounts","color":"#16a34a",
              "secret":"ledger-falcon",
              "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":   {"name":"Administration & HR","color":"#ea580c",
              "secret":"orbit-vermillion",
              "aliases":["admin","administration","hr","human resources"]},
}
try:    DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
for _i, (_c, _cfg) in enumerate(DEPARTMENTS.items()):
    _cfg.setdefault("name", _c.upper())
    _cfg.setdefault("secret", f"{_c}-secret-{abs(hash(_c))%9999}")
    _cfg.setdefault("color", PALETTE[_i % len(PALETTE)])
    _cfg.setdefault("aliases", [_c])
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

# ==================== USERS (FACULTY ONLY) ====================
DEFAULT_USERS = {
  "admin":         {"name":"System Admin","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":   {"name":"Dr. Sharma (CSE)","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":   {"name":"Dr. Menon (ECE)","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":   {"name":"Mr. Verma (Finance Officer)","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin": {"name":"Ms. Rao (Registrar)","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
# Merge: always ensure default users exist, keep custom users added later
try:
    saved = json.load(open(CFG_USERS))
    # Start from defaults, overlay any custom users
    USERS = DEFAULT_USERS.copy()
    for k, v in saved.items():
        if k not in USERS:  # keep custom additions
            USERS[k] = v
    # Remove any leftover 'student' users
    USERS = {k:v for k,v in USERS.items() if v.get("role") != "student"}
except:
    USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

# ==================== SEED DATA ====================
SEED = {
"cse": {
  "cse_syllabus": """B.Tech CSE at Mandsaur University — 4-year, 8-semester programme. Three branches: CSE, CSE (AI), CSE (BCT).
Course codes: 24CSExxxx T (theory) or P (practical) in 2024 scheme.
Sem I: Engineering Math-I (24CSE0100 T, BSC, 60/30/10, 4cr), Physics→Electronics, C Programming, Indian Traditions (HSMC), Computer Fundamentals. Practicals: Physics, C Prog, Indian Trad, Comp Fund, Web Dev-I, PBL-I. 22 credits, 800 marks.
Sem II: Math-II, Communication Skills, Python, Web Dev-II, Digital Circuits.
Sem III: Discrete Math (24CSE1500 T), Data Structure (24CSE1600 T), OOP Java (24CSE1700 T), RDBMS (24CSE1800 T), Universal Human Values (24VAC030 T), Intro ML (24CSE1902 T).
Sem IV: Theory of Computation (CSE160), Analysis & Design of Algorithms (CSE170), Operating Systems (CSE280), Elective-CS2, Elective-CS3.
Sem V/VI: Computer Architecture (CSE310), Computer Networks (CSE190), Compiler Design (CSE340), Deep Learning (CSE766), Data Warehousing (CSE776), Cyber Security (CSE900), Software Engg (CSE330), Android (CSE910), NLP (CSE781), Distributed Systems (CSE891).
Sem VIII: Entrepreneurship (MGT526), E-Commerce (OPE107), Cyber Law (CSE431), Blockchain (CSE1030) or ML (CSE440), Big Data (CSE450).""",
  "cse_exam_scheme": """MU CSE Exam Scheme (2024).
Marks: Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC 2cr = 50 (30+10+10). Practical 1cr = 50 (25+15+10). Practical 2cr = 100 (50+40+10). PBL 1cr = 50 (EST 25 + CET 25, no mid-sem).
Theory 4cr = 3L+1T; 3cr = 2L+1T. Semester = 800 marks (Sem I-II), 22 credits.
Exams: Dec (odd) and May-June (even). 3-hr theory papers. Morning 10-1, evening 2-5. Timetable by CoE.
Passing: 40% in EST + 40% overall for theory; 50% overall practical. Below = F + backlog.
Grading: 90-100 O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). % = 10 × CGPA. CGPA > 8 = First Division with Distinction; 6.5-8 = First Division; below = Second Division.
Attendance: 75% min per course; 65-74% condonation with medical; <65% detained.
Backlog next Main Exam. Re-eval within 7 days Rs 800/paper. Photocopy Rs 400 within 7 days. Unfair means = zero.""",
  "cse_lab_rules": """MU CSE Lab Rules SAMPLE/CSE/LAB/01.
Timings: Mon-Fri 9-12, 1-4. Lunch 12-1. Sat 9-1 make-up. Max 20/batch, fixed computer. 15 min late = absent.
Attendance at start. Max 2 missed labs; make-up Sat with HOD permission. Record next session, late -2 marks/week.
Conduct: University ID + closed footwear. No food/drinks/water. Phones silent. No software install without permission. Pen drives scanned. Log out + shutdown before leaving.
Safety: Fire extinguishers at entrance. Fire → power off, emergency exit, inform security. Electric shock → power off first, no bare-hand. First aid with lab assistant. No personal chargers/heaters. Report smoke/sparks.
Penalties: warning + 5 marks. Repeat = 1 week ban + parents. Network misuse → Discipline Committee.""",
  "cse_internship_rules": """MU CSE Internship Rules (SAMPLE).
Summer Internship-I after Sem IV: 4 weeks (160 hr), 2 credits. Summer Internship-II after Sem VI: 6 weeks (240 hr), 2 credits.
Domain must relate to CSE. Online allowed if registered company. Unpaid OK, student bears travel.
Permission: offer letter + HOD-signed NOC to T&P Cell ≥ 7 days before. No permission = no credit.
Attendance 90%, max 2 leave days. Weekly log signed. Confidentiality agreement.
Certificate on letterhead, signed by mentor/HR, shows dates.
Report 25-40 pages A4, TNR 12, spacing 1.5. Similarity below 20%. Submit within 10 days of reopening.
Evaluation: mentor 30 + report 30 + presentation 40 = 100. Pass 50/100. Fail = repeat.
Six-month internship possible in final sem (replaces Project Phase-II). CGPA ≥ 7.0 up to Sem VI, no backlog. HOD approval needed.""",
  "cse_project_guidelines": """MU CSE Project Guidelines (2024).
PBL-I (24CSE0700 P, Sem I, 1cr, 50 = 25+25, 2hr/wk). PBL-II (24CSE1400 P, Sem II).
Major Project: final year, teams 3-4, one faculty guide. Synopsis 3 pages within 2 weeks. Weekly guide meetings. 3 reviews/semester. Missed review without permission = -10 internal.
Eval: Phase-I 100 (60+40); Phase-II 200 (100+100).
Report: A4, TNR 12 body / 14 headings, spacing 1.5. Margins left 1.5", others 1". IEEE refs. 60-100 pages.
Chapters: Intro, Lit Review, Problem, Methodology, Implementation, Results, Conclusion, Future.
Plagiarism below 15%, single source below 2%. Late = 5 marks/day up to 5 days.
4 hardbound copies + PDF. Viva: guide + 1 internal + 1 external. 15 min + 10 min Q&A.""",
},
"ece": {
  "ece_syllabus": """B.Tech ECE at Mandsaur University — 4-year, 8-semester.
Sem I: Math-I (24ECE0100 T), Physics→Electronics, C Programming, Indian Traditions, Basic Electrical Engg. Practicals + Electronic Workshop & PCB Design (24ECE0600 P, 4cr) + PBL-I.
Sem II: Math-II, Communication Skills, Python, Circuit Theory & Networks, Digital Circuits & Systems + labs.
Sem III: Probability & Statistics (24ECE1500 T), Electronic Devices & Circuits (24ECE1600 T PCC), Signals & Systems (24ECE1700 T), Digital System Design (24ECE1800 T), OOP C++ (24ECE1900 T).
Sem IV: Electromagnetic Field Theory (24ECE2000 T), Analog Circuits (24ECE2100 T), Analog Communication (24ECE2200 T), Linear Control Systems (24ECE2300 T), Microprocessors (24ECE2400 T).
Sem V: DSP (24ECE2500 T), Digital Communication (24ECE2600 T), VLSI Design (24ECE2700 T), Embedded Systems (24ECE2800 T), Antenna (24ECE2900 T).
Sem VI: Microwave Engg (24ECE3000 T), Info Theory (24ECE3100 T), Computer Networks (24ECE3200 T), Wireless Comm (24ECE3300 T), Elective.
Sem VII: Optical Fiber Comm, Satellite Comm, IoT, Elective-EC2, Management (24ECE3900 T 50), Tech Report Writing (24ECE4000 T 50), Summer Internship-II, Major Project Phase-I.
Sem VIII: Radar & Navigation (24ECE4300 T), Elective-EC3, Open Elective, Entrepreneurship (24ECE4700 T 50), Cyber Law (24ECE4800 T 50), Major Project Phase-II (24ECE4900 P 200), Comprehensive Viva (100), Seminar (50).""",
  "ece_exam_scheme": """MU ECE Exam Scheme — same as CSE.
Theory 3-4cr = 100 (60+30+10). HSMC 2cr = 50 (30+10+10). Practical 1cr = 50 (25+15+10). Practical 2cr = 100 (50+40+10). PBL = 50 (25+25).
Semester = 800 marks (Sem I-II), 22 credits. Theory 4cr = 3L+1T; 3cr = 2L+1T.
Exams Dec / May-June. 3-hr papers. Morning 10-1, evening 2-5.
Passing: 40% EST + 40% overall theory; 50% practical.
Grading same 8-scale. % = 10 × CGPA.
Attendance 75% min, condonation 65-74%, <65% detained.
Re-eval Rs 800 within 7 days. Photocopy Rs 400.
Pass marks register: 80 ECE courses. Theory min EST 24/100 (12/50), min overall 40/100 (20/50). Practical min overall 25/50 or 50/100.""",
  "ece_lab_rules": """MU ECE Lab Rules SAMPLE/ECE/LAB/01.
Timings: Mon-Fri 9-12, 1-4. Lunch 12-1. Sat 9-1. Max 20/batch, fixed bench. 15 min late = absent.
Attendance at start. Max 2 missed, make-up Sat HOD approval. Record next session, late -2/week.
Conduct: University ID + closed footwear. No food/drinks. Phones silent for lab work. No software install without permission. Pen drives scanned.
Safety: Fire extinguishers at entrance. Fire → power off, exit. Electric shock → power off first. Never work on live circuits above 36 V without faculty.
Labs (80 registers): Electronic Devices (Mon 9-12), Digital Electronics (Tue 9-12), Analog Circuits (Wed 9-12), Microprocessor (Thu 9-12), Communication (Fri 9-12), VLSI (Mon 1-4), DSP (Tue 1-4), Microwave (Wed 1-4), Embedded (Thu 1-4), Computer Networks (Fri 1-4).
Each lab: check polarity before power, don't hot-swap ICs, set function generator amplitude minimum before connecting, don't exceed op-amp rating, don't unplug kit while downloading, never look into powered waveguide/horn, disconnect USB before rewiring, don't change switch settings without approval, don't unplug patch cords from rack.""",
  "ece_project_guidelines": """MU ECE Project Guidelines (2024).
PBL-I (Sem I, 1cr, 50 = 25+25, no mid-sem). PBL-II (Sem II, same). Category EP.
ECE projects: sensor circuits, amplifiers, microcontroller applications.
Major Project: final year, teams 3-4, one guide. Synopsis 3 pages within 2 weeks. Weekly meetings. 3 reviews/sem. Missed = -10 internal.
Eval: Phase-I 100 (60+40); Phase-II 200 (100+100).
Report: A4, TNR 12/14, spacing 1.5. Margins left 1.5", others 1". IEEE refs. 60-100 pages.
Chapters same as CSE.
Plagiarism below 15%, single below 2%. Late = 5 marks/day up to 5 days.
4 hardbound + PDF. Viva: guide + 1 internal + 1 external. 15 min + 10 min Q&A.""",
  "ece_internship_rules": """MU ECE Internship Rules (SAMPLE).
Summer Internship-I after Sem IV: 4 wk (160 hr), 2cr. Summer Internship-II after Sem VI: 6 wk (240 hr), 2cr.
Domain must relate to ECE (embedded, VLSI, telecom, IoT). Online allowed. Unpaid OK.
Permission: offer + HOD-signed NOC ≥ 7 days before. No permission = no credit.
Attendance 90%, max 2 days leave. Weekly log signed.
Report 25-40 pages A4 TNR 12 spacing 1.5. Similarity below 20%. Submit within 10 days.
Eval: mentor 30 + report 30 + presentation 40 = 100. Pass 50.
Internship types: Summer I/II, Six-Month (final sem), Industrial Training (8 wk, 3cr), In-plant (2 wk, 1cr, 85% att), Research (8 wk, 3cr), Online (4 wk, 1cr, 85% att), Startup (6 wk, 2cr).
Six-month replaces Major Project Phase-II. CGPA ≥ 7.0 up to Sem VI, no backlog.""",
},
"finance": {
  "fee_structure": """College Fee Structure 2025-26.
B.Tech (per year): Tuition Rs 1,10,000; Development Rs 15,000; Lab Rs 10,000; Library Rs 3,000; Sports Rs 2,000; Exam Rs 5,000. Total Rs 1,45,000/year.
Hostel AC Rs 85,000/year (incl mess); non-AC Rs 65,000; mess-only Rs 35,000.
One-time: Admission Rs 10,000; Caution deposit Rs 5,000 (refundable); Alumni Rs 2,000.
Payment: 50% at admission; 30% by 15 Nov; 20% by 15 Mar. Late Rs 500/week up to 4 weeks. Beyond 4 wk: hall ticket withheld.
Modes: NEFT/RTGS, UPI, portal, DD in favour of "Mandsaur University". Cash up to Rs 20,000.
Refund (UGC): 100% before 15 days of sem start; 90% within 15 days; 80% within 30 days; 50% after 30 days; NIL after 60 days.
Concession: merit CGPA ≥ 9.0 = 25% tuition waiver; CGPA 8.0-8.9 = 15%; sports 15-25%; staff ward 50%; sibling 10%.""",
  "scholarship_rules": """Scholarships & Financial Aid.
Merit: CGPA ≥ 9.0 (two consecutive sems) → 25% tuition waiver next sem. CGPA 8.0-8.9 → 15%.
Government: MP Post-Matric (SC/ST/OBC income < 6 lakh), Central Sector (top 20 percentile, income < 8 lakh), PMSSS, Minority.
Sports: state-level 1st/2nd → 15%; national → 25%.
Staff ward: 50% tuition waiver.
Sibling: 10% for second child.
Special: COVID-affected, ex-servicemen ward, single-girl-child (5%).
Application: form + docs to Finance Office by 30 September. Verification by Scholarship Committee (Dean + Finance Officer + Registrar). Disbursement within 30 days of approval.
Renewal: CGPA ≥ 8.0 + 75% attendance. Failure = withdrawn + pro-rata repayment.""",
  "budget_approval": """Annual Budget & Expense Approval Workflow.
Depts submit annual budget (April): salary, equipment, consumables, travel, workshops, misc. Finance consolidates; Dean Academics reviews; VC approves by 30 May.
Purchase tiers:
- Up to Rs 25,000: HOD.
- Rs 25,001 - 1,00,000: HOD + Finance Officer.
- Rs 1,00,001 - 5,00,000: Dean + Finance Officer + Registrar.
- Rs 5,00,001 - 25,00,000: Purchase Committee (3 members).
- Above Rs 25,00,000: VC + tender.
Vendor payment: invoice + GRN + PO. Finance verifies 5 working days. Payment within 30 days.
Travel: MP govt TA/DA rates; original tickets + boarding pass + bills; approval before travel; submit within 15 days.
Emergency (up to Rs 10,000): HOD imprest; regularize within 7 days.
Audit: quarterly internal, annual external (CA). Discrepancy > Rs 5,000 = written explanation.""",
  "vendor_rules": """Vendor & Procurement Rules.
Registration: GST + PAN + bank + ITR last 2 years + no-blacklisting certificate. Valid 3 years.
Rate contract: common items (stationery, lab consumables, printing) — annual tender, L1 selected.
Limited tender: 3+ quotations Rs 1-5 lakh. Open tender > Rs 5 lakh (website + newspaper).
GeM portal mandatory for items available on Government e-Marketplace.
PO issued by Finance after approval. Delivery date, inspection clause, penalty 1%/week for delay up to 10%.
Inspection: 3-member committee for > Rs 50,000. Report signed by HOD + user dept + Finance rep.
Payment 30 days after GRN + invoice. Advance up to 30% with bank guarantee.
Blacklisting: fraud, repeated quality failure, delay > 30 days. Banned 2 years.
GST invoice must show: GSTIN, invoice number + date, HSN code, PO reference, tax breakup, total in words.""",
  "student_fee_records": """Student Fee Records — Faculty Query Reference (15 students, Finance Dept view).
S001 Rahul Kumar, CSE Sem III, Roll 24CSE001. Total due Rs 1,45,000. Paid Rs 1,10,000. Pending Rs 35,000. Due date 15 Nov. Status: PARTIAL.
S002 Priya Sharma, CSE Sem III, Roll 24CSE002. Total Rs 1,45,000. Paid Rs 1,45,000. Pending 0. Status: CLEARED.
S003 Amit Patel, CSE Sem III, Roll 24CSE003. Total Rs 1,45,000. Paid Rs 72,500. Pending Rs 72,500. Overdue 12 days. Status: OVERDUE, hall ticket withheld.
S004 Sneha Iyer, CSE Sem III, Roll 24CSE004. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED. Merit scholar 25% waiver applied.
S005 Vikram Singh, ECE Sem III, Roll 24ECE001. Total Rs 1,45,000. Paid Rs 1,30,000. Pending Rs 15,000. Due 15 Nov. Status: PARTIAL.
S006 Ananya Reddy, ECE Sem III, Roll 24ECE002. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED. Sports quota 15%.
S007 Karan Mehta, ECE Sem III, Roll 24ECE003. Total Rs 1,45,000. Paid Rs 0. Pending Rs 1,45,000. Overdue 25 days. Status: CRITICAL.
S008 Divya Nair, ECE Sem III, Roll 24ECE004. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED.
S009 Arjun Verma, CSE Sem V, Roll 23CSE021. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED. Staff ward (50% waiver).
S010 Riya Gupta, CSE Sem V, Roll 23CSE022. Total Rs 1,45,000. Paid Rs 1,00,000. Pending Rs 45,000. Overdue 5 days. Status: PARTIAL.
S011 Mohit Sharma, ECE Sem V, Roll 23ECE015. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED.
S012 Neha Joshi, ECE Sem V, Roll 23ECE016. Total Rs 1,45,000. Paid Rs 1,20,000. Pending Rs 25,000. Due 15 Nov. Status: PARTIAL.
S013 Rohit Kumar, CSE Sem VII, Roll 22CSE010. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED.
S014 Pooja Rao, ECE Sem VII, Roll 22ECE008. Total Rs 1,45,000. Paid Rs 90,000. Pending Rs 55,000. Overdue 8 days. Status: OVERDUE.
S015 Sameer Khan, CSE Sem VII, Roll 22CSE011. Total Rs 1,45,000. Paid Rs 1,45,000. Status: CLEARED. Sibling concession 10%.
Summary: 15 students. CLEARED = 9. PARTIAL = 4. OVERDUE = 2. Total pending dues ≈ Rs 2,62,500.""",
},
"admin": {
  "leave_policy": """Leave Policy for Faculty & Staff.
Faculty leave types: Casual (12/year), Earned (15/year, accumulable to 300), Medical (10/year), Maternity (180 days), Paternity (15 days), Sabbatical (after 6 years, up to 1 year), Duty leave.
Approval: HOD → Registrar → VC (for > 15 days).
Encashment: earned leave up to 300 days at retirement.
Public holidays: MP govt calendar + university foundation day + festivals.
WFH: not generally allowed; special cases with Dean approval.
Late arrival: > 15 min = half-day; > 3 in month = parent/staff notification.""",
  "grievance_redressal": """Grievance Redressal & Discipline.
Faculty grievance: submit to Grievance Cell. Acknowledge 24 hr, resolve 7 days (routine) / 15 days (complex).
Levels: 1) Dept Coordinator → 2) HOD → 3) Dean → 4) Grievance Committee → 5) VC. Each max 7 days.
Antiragging: Zero tolerance. Investigation within 48 hr. Punishment up to rustication.
Sexual harassment: ICC per POSH Act 2013. Anonymous complaints allowed. Investigate within 90 days.
Disciplinary: warning, fine, suspension (up to 1 sem), rustication, FIR.
Appeals: to VC within 15 days; VC decision final.
Counselling: free psychologist on campus 3 days/week.
Confidentiality: All complaint identities protected. Retaliation punishable.""",
  "event_approval": """Event & Workshop Approval.
Types: Technical (hackathon, workshop), Cultural, Sports, Seminars, Industry visits.
Application (min 30 days before): event form + purpose + budget + schedule + guests + safety plan to Dean Student Welfare.
Chain: Dean SW → HOD → Registrar → VC (> Rs 50,000 or outsiders).
Budget: max Rs 25,000 from dept fund; above requires sponsor or VC approval. Sponsorships allowed with prior intimation.
Venue booking via Facilities Office; priority to academic events.
Permissions: Police NOC (> 500 attendees), Fire NOC, Municipal for outdoor.
Safety: 2 faculty + 5 volunteer marshals; first aid kit; ambulance for > 500.
Documentation: photos + report + accounts settled within 15 days.
Cancellation: refund unused funds; disciplinary action for misuse.
Post-event: feedback form + AAR to Dean within 30 days.""",
  "staff_recruitment": """Faculty & Staff Recruitment.
Cadres: Assistant Professor (60% in PG), Associate Professor (5+ years), Professor (10+ years + PhD).
Advertisement: university website + newspaper + Employment News. Min 2 weeks open.
Screening: Eligibility Committee. Shortlist 5x openings.
Demo + Interview: 15-min demo + 30-min interview. Panel: HOD + senior faculty + Dean + external expert (senior).
Selection: recommendations → Selection Committee → VC approval → offer letter.
Non-teaching: Test + Interview.
Docs: ID, address, certificates, experience letters, medical fitness, police verification.
Probation: 1 year (teaching) / 6 months (non-teaching). Confirmation on satisfactory report.
Contract: up to 11 months, renewable.
Reservation: MP govt norms (SC/ST/OBC/EWS/PwD).
Grievance: appeal to VC within 15 days.""",
  "student_records": """Student Records — Faculty Query Reference (15 students, Admin/HR view).
S001 Rahul Kumar, CSE Sem III, Roll 24CSE001. Attendance 78%. Leave taken: 5 casual, 2 medical. Last medical 12 Oct (fever, 2 days). Discipline: none. Hostel: Block A, Room 214.
S002 Priya Sharma, CSE Sem III, Roll 24CSE002. Attendance 92%. Leave: 2 casual. Discipline: none. Hostel: Day scholar.
S003 Amit Patel, CSE Sem III, Roll 24CSE003. Attendance 64% (DETAINED). Leave: 8 casual, 0 medical. Warning issued 5 Oct for low attendance. Hostel: Block B, Room 108.
S004 Sneha Iyer, CSE Sem III, Roll 24CSE004. Attendance 88%. Leave: 3 casual. Discipline: none. Hostel: Block A, Room 305.
S005 Vikram Singh, ECE Sem III, Roll 24ECE001. Attendance 82%. Leave: 4 casual, 1 medical. Discipline: none. Hostel: Block C, Room 012.
S006 Ananya Reddy, ECE Sem III, Roll 24ECE002. Attendance 95%. Sports team (basketball). Leave: 6 OD (on-duty). Discipline: none. Hostel: Block C, Room 220.
S007 Karan Mehta, ECE Sem III, Roll 24ECE003. Attendance 71%. Leave: 7 casual, 2 medical. Warning issued 20 Sep for late arrivals (5 in month). Hostel: Block D, Room 117.
S008 Divya Nair, ECE Sem III, Roll 24ECE004. Attendance 89%. Leave: 3 casual. Discipline: none. Hostel: Block D, Room 045.
S009 Arjun Verma, CSE Sem V, Roll 23CSE021. Attendance 84%. Leave: 5 casual, 1 medical. Discipline: none. Hostel: Block A, Room 402.
S010 Riya Gupta, CSE Sem V, Roll 23CSE022. Attendance 76%. Leave: 6 casual. Discipline: none. Hostel: Day scholar.
S011 Mohit Sharma, ECE Sem V, Roll 23ECE015. Attendance 91%. Leave: 2 casual. Class rep. Discipline: none. Hostel: Block B, Room 214.
S012 Neha Joshi, ECE Sem V, Roll 23ECE016. Attendance 87%. Leave: 4 casual. Discipline: none. Hostel: Block B, Room 311.
S013 Rohit Kumar, CSE Sem VII, Roll 22CSE010. Attendance 80%. Leave: 3 casual. Placement: TCS offer. Discipline: none. Hostel: Day scholar.
S014 Pooja Rao, ECE Sem VII, Roll 22ECE008. Attendance 73%. Leave: 5 casual, 3 medical. Warning issued 1 Oct (3 late arrivals). Hostel: Block C, Room 118.
S015 Sameer Khan, CSE Sem VII, Roll 22CSE011. Attendance 88%. Leave: 2 casual. Discipline: none. Hostel: Block A, Room 510.
Summary: 15 students. Average attendance 83.2%. Detained: 1 (S003). Warnings issued: 3 (S003, S007, S014). On-duty leaves: 1 (S006).""",
  "timetable_reference": """Faculty Timetable Reference (sample week).
CSE Department (Dr. Sharma):
- Mon 9-11 AM: Data Structures (Sem III, Room 201)
- Tue 10-12 AM: Operating Systems (Sem V, Lab 1)
- Wed 2-4 PM: Algorithms Tutorial (Sem IV, Room 104)
- Thu 9-11 AM: Project Guidance (Sem VII/VIII, Office hours)
- Fri 11-1 PM: Compiler Design (Sem VI, Room 305)
ECE Department (Dr. Menon):
- Mon 10-12 AM: Digital Signal Processing (Sem V, Lab 2)
- Tue 9-11 AM: VLSI Design (Sem V, Room 401)
- Wed 9-11 AM: Analog Circuits (Sem IV, Lab 3)
- Thu 2-4 PM: Project Guidance (Office hours)
- Fri 10-12 AM: Microwave Engineering (Sem VI, Room 302)
Examination duty: 20% of faculty assigned per exam. Duty roster by CoE office.
Office hours: 2 hours/week minimum per faculty, announced on department board.
Monthly faculty meeting: 1st Saturday, 11 AM, Seminar Hall.""",
},
}

# ==================== MODEL ====================
@st.cache_resource(show_spinner="⏳ Loading AI models (first run ~60s)…")
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try:    coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count() == 0:
            keys = list(docs.keys()); texts = list(docs.values())
            keys += ["_secret_marker"]; texts += [DEPARTMENTS[dept]["secret"]]
            vecs = emb.encode(texts, normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys], embeddings=vecs, documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    _ = mdl.generate(**tok("hi", return_tensors="pt").to(mdl.device),
                     max_new_tokens=1, pad_token_id=tok.eos_token_id)
    return emb, cc, tok, mdl, DEV

emb, cc, tok, mdl, DEV = setup()

def get_coll(d):
    try:    return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}", metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r = USERS.get(u)
    if r and r["pw"] == _hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept, query):
    if not dept: return None, None
    q = query.lower()
    for other, cfg in DEPARTMENTS.items():
        if other == dept: continue
        for alias in cfg.get("aliases",[]) + [other]:
            if re.search(rf"\b{re.escape(alias)}\b", q): return other, cfg["name"]
    return None, None

# ==================== INGEST ====================
def chunk_text(t, size=500, ov=60):
    parts = re.split(r'(?<=[.!?])\s+', t.strip()); out, cur = [], ""
    for s in parts:
        if len(cur)+len(s)+1 <= size: cur = (cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur = ((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf_robust(pdf_bytes, fname):
    tmp = f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    with open(tmp,"wb") as fh: fh.write(pdf_bytes)
    pages, err = [], ""
    try:
        doc = pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t = pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err = f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    try:
                        t = pg.extract_text() or ""
                        if t.strip(): pages.append({"text":t,"page":i+1})
                    except: pass
            if pages: err = ""
        except Exception as e: err += f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages, err

def ingest_pdf(dept, fname, pdf_bytes, doc_type="regulation"):
    pages, err = extract_pdf_robust(pdf_bytes, fname)
    if not pages: return 0, (err or "No extractable text"), 0
    coll = get_coll(dept); ch, mt, ids = [], [], []
    safe = _safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":doc_type})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    if not ch: return 0, "0 chunks", len(pages)
    try:
        all_vecs = []
        for i in range(0,len(ch),32):
            v = emb.encode(ch[i:i+32], normalize_embeddings=True, show_progress_bar=False).tolist()
            all_vecs.extend(v)
        coll.add(ids=ids, embeddings=all_vecs, documents=ch, metadatas=mt)
    except Exception as e: return 0, f"Chroma:{e}", len(pages)
    try:
        with open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}", "wb") as fh: fh.write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch), "", len(pages)

def ingest_text(dept, text, src, dt="regulation"):
    coll = get_coll(dept); ch = chunk_text(text)
    if not ch: return 0
    try:
        v = emb.encode(ch, normalize_embeddings=True, show_progress_bar=False).tolist()
        coll.add(ids=[f"{dept}_{_safe(src)}_c{i}_{int(time.time())}" for i in range(len(ch))],
                 embeddings=v, documents=ch,
                 metadatas=[{"source":src,"page":1,"doc_type":dt}]*len(ch))
        backup_chroma(); return len(ch)
    except Exception as e:
        print(f"[ingest_text] {e}"); return 0

def list_sources(dept):
    try:
        d = get_coll(dept).get(include=["metadatas"]); counts={}
        for m in d["metadatas"]:
            s = m.get("source","?"); counts[s] = counts.get(s,0)+1
        return counts
    except: return {}

def retrieve(dept, q, k=4):
    c = get_coll(dept)
    if c.count() == 0: return []
    qv = emb.encode([q], normalize_embeddings=True).tolist()
    r = c.query(query_embeddings=qv, n_results=min(k,c.count()),
                include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

# ==================== GENERATION ====================
def stream_answer(dept, q, ph, extra=""):
    hits = retrieve(dept, q, k=4)
    parts = [f"[{h['source']} p{h['page']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0, f"[ATTACHED FILE CONTENT]\n{extra[:3000]}")
    ctx = "\n".join(parts) or "(no reference found)"
    sys_msg = ("You are a helpful faculty assistant. Answer clearly and accurately using ONLY the reference material. "
               "Cite sources in brackets like [source_name, page N]. If unsure, say so.")
    msgs = [{"role":"system","content":sys_msg},
            {"role":"user","content":f"Question: {q}\n\nReference material:\n{ctx}\n\nAnswer:"}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    th = threading.Thread(target=mdl.generate,
        kwargs=dict(**inp, max_new_tokens=450, do_sample=False,
                    pad_token_id=tok.eos_token_id, streamer=streamer))
    th.start()
    buf = ""
    for t in streamer:
        buf += t; ph.markdown(buf + "▌")
    ph.markdown(buf)
    return buf, hits

def make_pdf(title, dept_name, subtitle, body):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=0.8*inch, rightMargin=0.8*inch,
                            topMargin=0.7*inch, bottomMargin=0.7*inch)
    S = getSampleStyleSheet()
    h1 = ParagraphStyle("H1", parent=S["Heading1"], alignment=1, fontSize=16, spaceAfter=6)
    sb = ParagraphStyle("S", parent=S["Normal"], alignment=1, fontSize=10, spaceAfter=4)
    bd = ParagraphStyle("B", parent=S["Normal"], fontSize=11, leading=16, spaceAfter=8)
    story = [Paragraph(f"<b>{title}</b>", h1), Paragraph(f"Department of {dept_name}", sb)]
    if subtitle: story.append(Paragraph(subtitle, sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s, bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

# ==================== AUDIT ====================
def init_audit():
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (
        id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT,
        user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",
              (datetime.now().isoformat(), u, r or "-", d or "-", a, det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None, limit=200):
    c = sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else:    c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows = c.fetchall(); c.connection.close(); return rows

# ==================== UI ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide",
                   initial_sidebar_state="collapsed")

st.markdown("""
<style>
html, body, [class*="css"] { font-family: 'Inter', system-ui, -apple-system, sans-serif; }
.block-container { padding-top: 2rem; max-width: 1150px; }

.hero { background: radial-gradient(circle at 20% 10%, #1e40af 0%, transparent 45%),
        radial-gradient(circle at 80% 20%, #7c3aed 0%, transparent 45%),
        linear-gradient(135deg, #0a0e27 0%, #111a3e 55%, #1a2a5c 100%);
        padding: 60px 40px 50px; border-radius: 24px; color: white; text-align: center;
        box-shadow: 0 20px 60px -20px rgba(30,64,175,0.5); }
.hero h1 { font-size: 3.4rem; font-weight: 800; letter-spacing: -1.5px; margin: 0 0 14px 0;
        background: linear-gradient(90deg,#fff,#a5b4fc,#c4b5fd); -webkit-background-clip: text;
        -webkit-text-fill-color: transparent; }
.hero p { font-size: 1.1rem; opacity: 0.94; margin: 0; }

.feat { background: linear-gradient(180deg,#fff,#f8fafc); border-radius: 16px;
        padding: 22px; border: 1px solid #e2e8f0; height: 155px; position: relative; overflow: hidden; }
.feat::before { content: ''; position: absolute; top: 0; left: 0; width: 4px; height: 100%;
        background: linear-gradient(180deg,#2563eb,#7c3aed); }
.feat .icn { font-size: 1.8rem; margin-bottom: 10px; }
.feat h4 { margin: 0 0 6px 0; color: #0f172a; font-size: 1rem; font-weight: 700; }
.feat p { margin: 0; color: #64748b; font-size: 0.85rem; line-height: 1.5; }

.pill { display: inline-block; padding: 4px 14px; border-radius: 100px;
        font-size: 0.7rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px; }
.p-admin { background:#fef3c7; color:#92400e; }
.p-teacher { background:#dbeafe; color:#1e40af; }

.metric { background: linear-gradient(135deg,#fff,#f8fafc); border-radius: 14px;
        padding: 18px 20px; border: 1px solid #e2e8f0; }
.metric .lbl { font-size: 0.72rem; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; }
.metric .val { font-size: 1.8rem; font-weight: 800; color: #0f172a; margin: 4px 0; }
.metric .sub { font-size: 0.78rem; color: #64748b; }

.dept-card { padding: 16px; border-radius: 12px; margin-bottom: 8px; }

.quick-action { background: #fff; border: 1px solid #e2e8f0; border-radius: 10px;
        padding: 12px 10px; text-align: center; transition: all 0.15s; cursor: pointer; }
.quick-action:hover { border-color: #3b82f6; background: #eff6ff; }
.quick-action .qa-icn { font-size: 1.5rem; display: block; margin-bottom: 4px; }
.quick-action .qa-lbl { font-size: 0.82rem; font-weight: 600; color: #1e293b; }

.attach-badge { background: #ecfdf5; border: 1px solid #86efac; border-radius: 8px;
        padding: 8px 12px; font-size: 0.82rem; color: #166534; margin: 8px 0; }

.welcome { background: linear-gradient(135deg,#eff6ff,#f5f3ff); border-radius: 14px;
        padding: 22px 26px; border: 1px solid #c7d2fe; margin-bottom: 20px; }
.welcome h2 { margin: 0 0 6px 0; color: #1e293b; font-size: 1.4rem; }
.welcome p { margin: 0; color: #475569; font-size: 0.92rem; }

.empty-state { text-align: center; padding: 50px 20px; color: #94a3b8; }
.empty-state .icon { font-size: 3.5rem; margin-bottom: 12px; }
.empty-state .title { font-size: 1.1rem; color: #64748b; margin-bottom: 6px; }
.empty-state .sub { font-size: 0.85rem; color: #94a3b8; }
</style>
""", unsafe_allow_html=True)

for k in ("user","name","role","dept","logged_in"): st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx = ""
if "ctx_name" not in st.session_state: st.session_state.ctx_name = ""

# ---------- HOME ----------
if not st.session_state.logged_in:
    st.markdown("""<div class="hero">
      <h1>🎓 VaultMind ERP</h1>
      <p><b>Faculty AI Workbench for Higher Education</b><br>
      4 departments · Local AI · Zero cloud · Upload PDFs & images · Cited answers</p>
    </div>""", unsafe_allow_html=True)
    st.write("")
    c1,c2,c3,c4 = st.columns(4)
    for col,(icn,t,d) in zip([c1,c2,c3,c4], [
      ("🏛️","Dept Isolation","Each department has its own knowledge base. CSE cannot read Finance."),
      ("🤖","Local AI","Qwen2.5 on T4 GPU. Documents never leave the machine."),
      ("📎","Attach Anything","PDFs & images become context — AI reads and cites them."),
      ("⚡","Daily Work","Exam papers · notes · fee checks · student records — all in one chat.")]):
        col.markdown(f'<div class="feat"><div class="icn">{icn}</div><h4>{t}</h4><p>{d}</p></div>', unsafe_allow_html=True)
    st.write(""); st.write("")
    cA,cB = st.columns([3,2])
    with cA:
        st.markdown("### 🔐 Faculty Sign in")
        u = st.text_input("Username", placeholder="admin · teacher_cse · teacher_fin", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("🔓 Login", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True})
                st.session_state.chat_history = []; st.session_state.chat_ctx = ""
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
    with cB:
        st.markdown("### 🎫 Faculty accounts")
        st.code("""ADMIN
  admin            / admin_pass

FACULTY
  teacher_cse      / t_cse    (CSE)
  teacher_ece      / t_ece    (ECE)
  teacher_fin      / t_fin    (Finance Officer)
  teacher_admin    / t_admin  (Registrar)""", language="text")
    st.stop()

# ---------- MAIN ----------
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
name = st.session_state.name
pill = {"admin":"p-admin","teacher":"p-teacher"}[role]
dept_color = DEPARTMENTS.get(dept,{}).get("color","#64748b") if dept else "#64748b"

# ============ SIDEBAR ============
with st.sidebar:
    st.markdown(f"""
    <div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#f1f5f9,#e2e8f0);">
      <div style="font-size:1.6rem;text-align:center;">🎓</div>
      <div style="text-align:center;font-weight:700;color:#0f172a;margin-top:6px;font-size:1rem;">{name}</div>
      <div style="text-align:center;margin-top:8px;"><span class="pill {pill}">{role}</span></div>
      {f'<div style="text-align:center;margin-top:10px;font-size:0.85rem;color:#475569;"><b style="color:{dept_color};">●</b> {DEPARTMENTS[dept]["name"]}</div>' if dept else ''}
    </div>""", unsafe_allow_html=True)
    if dept:
        try: st.metric("📊 KB chunks", get_coll(dept).count())
        except: pass
    st.caption("🔒 0 external API calls")
    st.caption("💾 Drive-backed")
    if st.button("🚪 Logout", use_container_width=True):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in"): st.session_state[k] = None
        st.session_state.chat_history = []; st.session_state.chat_ctx = ""; st.rerun()

# ============ TOP NAVIGATION (Tabs) ============
if is_admin:
    T = st.tabs(["🏠 Home","📊 Dashboard","🏛️ Departments","📤 Upload RAG","👥 Faculty","📜 Audit"])
    tab_home, tab_dash, tab_dept, tab_pdf, tab_users, tab_audit = T
    tab_chat = None
else:
    T = st.tabs(["💬 AI Assistant","🏠 Home","📚 KB Documents"])
    tab_chat, tab_home, tab_kb = T
    tab_dash = tab_dept = tab_pdf = tab_users = tab_audit = None

# ============ TEACHER HOME ============
if not is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div class="welcome">
          <h2>👋 Welcome, {name}!</h2>
          <p>You are signed in to the <b>{DEPARTMENTS[dept]['name']}</b> workspace.
          Use the <b>AI Assistant</b> tab for all your daily work — exam papers, notes, queries, fee lookups, student records, and more.</p>
        </div>""", unsafe_allow_html=True)

        c1,c2,c3 = st.columns(3)
        srcs = list_sources(dept)
        c1.markdown(f'<div class="metric"><div class="lbl">Department</div><div class="val" style="font-size:1.1rem;">{DEPARTMENTS[dept]["name"]}</div><div class="sub">your scope</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Documents</div><div class="val">{len(srcs)}</div><div class="sub">in knowledge base</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">AI Device</div><div class="val">{DEV.upper()}</div><div class="sub">local inference</div></div>', unsafe_allow_html=True)
        st.write("")
        st.markdown("### ⚡ What you can do")
        cc1, cc2, cc3 = st.columns(3)
        cc1.markdown('<div class="feat"><div class="icn">📝</div><h4>Create exam papers</h4><p>Ask AI to generate a question paper with sections, marks, difficulty.</p></div>', unsafe_allow_html=True)
        cc2.markdown('<div class="feat"><div class="icn">📎</div><h4>Analyze documents</h4><p>Upload a PDF or image — ask questions about it. AI reads and cites.</p></div>', unsafe_allow_html=True)
        cc3.markdown('<div class="feat"><div class="icn">🔍</div><h4>Query records</h4><p>Fee dues, student attendance, leave records — all cited answers.</p></div>', unsafe_allow_html=True)

# ============ ADMIN DASHBOARD ============
if is_admin and tab_dash:
    with tab_dash:
        c1,c2,c3,c4 = st.columns(4)
        total = sum(get_coll(d).count() for d in DEPARTMENTS)
        blocked = sum(1 for r in get_audit(limit=500) if r[4] == "BLOCKED_QUERY")
        c1.markdown(f'<div class="metric"><div class="lbl">Departments</div><div class="val">{len(DEPARTMENTS)}</div><div class="sub">active</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Faculty</div><div class="val">{len([u for u in USERS.values() if u["role"]=="teacher"])}</div><div class="sub">accounts</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">KB chunks</div><div class="val">{total}</div><div class="sub">indexed</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric"><div class="lbl">Security</div><div class="val">0</div><div class="sub">{blocked} blocked · 0 leaks</div></div>', unsafe_allow_html=True)
        st.write(""); st.markdown("### 🏛️ Departments overview")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            n = get_coll(code).count(); srcs = list_sources(code)
            us = sum(1 for u in USERS.values() if u["dept"]==code)
            col.markdown(f"""<div class="dept-card" style="background:linear-gradient(135deg,{cfg['color']}22,{cfg['color']}11);border-left:4px solid {cfg['color']};">
              <div style="font-size:0.72rem;color:{cfg['color']};font-weight:700;">{code.upper()}</div>
              <div style="font-size:0.95rem;font-weight:700;color:#0f172a;margin:4px 0;">{cfg['name']}</div>
              <div style="font-size:0.8rem;color:#475569;">📚 {n} chunks<br>📄 {len(srcs)} docs<br>👥 {us} faculty</div>
            </div>""", unsafe_allow_html=True)

# ============ ADMIN HOME ============
if is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div class="welcome">
          <h2>👋 Welcome, {name}!</h2>
          <p>You have full administrative access. Use <b>Departments</b> to create new units,
          <b>Upload RAG</b> to add PDFs, <b>Faculty</b> to add teachers, and <b>Audit</b> to view activity.</p>
        </div>""", unsafe_allow_html=True)
        st.info("💡 **Tip:** Admin does not have a department. To test the AI Assistant, log in as any teacher account.")

# ============ CHAT (ChatGPT-style) ============
if tab_chat is not None:
    with tab_chat:
        st.markdown(f"### 💬 {DEPARTMENTS[dept]['name']} AI Assistant")
        st.caption(f"Signed in as **{name}** · 🔒 All data from your department only")

        # --- Quick actions (chips) ---
        st.markdown("**⚡ Quick actions** — enter a topic below, then click an action")
        qa_cols = st.columns(5)
        qa_defs = [
            ("exam",  "📝", "Exam Paper"),
            ("notes", "📓", "Notes"),
            ("quiz",  "🧠", "Quiz"),
            ("sum",   "📄", "Summarise"),
            ("explain","💡","Explain"),
        ]
        for col, (key, icn, lbl) in zip(qa_cols, qa_defs):
            with col:
                if st.button(f"{icn}\n{lbl}", key=f"qa_{key}", use_container_width=True):
                    st.session_state["qa_trigger"] = key

        topic = st.text_input("Topic / subject",
                              placeholder="e.g. Digital Signal Processing, Fee refund rules, Student S003 attendance",
                              key="chat_topic")

        # --- File attach ---
        with st.expander("📎 Attach a PDF or image (AI will use it as context)", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
                if f1 and st.button("Load PDF", key="btn_pdf"):
                    pages, err = extract_pdf_robust(f1.getvalue(), f1.name)
                    if pages:
                        st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                        st.session_state.ctx_name = f1.name
                        st.success(f"✅ Loaded {len(pages)} pages from {f1.name}")
                    else: st.error(f"❌ {err}")
            with c2:
                f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
                if f2 and st.button("OCR image", key="btn_ocr"):
                    tmp = f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                    open(tmp,"wb").write(f2.getvalue())
                    try:
                        txt = pytesseract.image_to_string(Image.open(tmp))
                        st.session_state.chat_ctx = txt
                        st.session_state.ctx_name = f2.name
                        st.success(f"✅ Extracted {len(txt)} chars from {f2.name}")
                    except Exception as e: st.error(f"❌ {e}")
                    finally:
                        try: os.remove(tmp)
                        except: pass

        if st.session_state.chat_ctx:
            cA, cB = st.columns([4,1])
            with cA:
                st.markdown(f'<div class="attach-badge">📎 Attached: <b>{st.session_state.ctx_name}</b> ({len(st.session_state.chat_ctx)} chars)</div>', unsafe_allow_html=True)
            with cB:
                if st.button("🗑️ Clear file"):
                    st.session_state.chat_ctx = ""; st.session_state.ctx_name = ""; st.rerun()

        # --- Chat display ---
        box = st.container(height=520)
        with box:
            if not st.session_state.chat_history:
                st.markdown(f"""<div class="empty-state">
                  <div class="icon">💬</div>
                  <div class="title">Ask anything about <b style="color:{dept_color};">{DEPARTMENTS[dept]['name']}</b></div>
                  <div class="sub">Answers come from your department's documents, with citations.</div>
                  <div class="sub" style="margin-top:16px;">💡 Try a quick action above, or just ask a question.</div>
                </div>""", unsafe_allow_html=True)
            for m in st.session_state.chat_history:
                with st.chat_message(m["role"]):
                    st.markdown(m["content"])
                    if m.get("sources"):
                        with st.expander(f"📄 {len(m['sources'])} source(s) cited"):
                            for s in m["sources"]:
                                st.caption(f"📄 `{s['source']}` · page {s['page']} · similarity {s['similarity']:.2f}")

        # --- Input ---
        uq = st.chat_input("Ask a question or describe the task…")

        # Handle quick action trigger
        triggered = st.session_state.pop("qa_trigger", None)
        if triggered:
            prompts = {
              "exam":   f"Generate a complete exam paper. Subject: {topic or DEPARTMENTS[dept]['name'] + ' end-semester'}. Requirements: 10 questions, mixed difficulty, split into Section A (short, 2 marks), Section B (medium, 5 marks), Section C (long, 10 marks). Show marks per question.",
              "notes":  f"Write detailed study notes on: {topic or 'the full course'}. Use clear headings, bullet points, and key formulas/facts.",
              "quiz":   f"Create a 5-question MCQ quiz on: {topic or 'the full course'}. Each question has 4 options (A-D), correct answer marked as 'Answer: X' on the next line.",
              "sum":    f"Summarise the key rules and important points about: {topic or 'the whole department policy'}. Use bullet points.",
              "explain":f"Explain clearly in simple language for a student: {topic or 'the fundamentals of this department'}. Include an example.",
            }
            uq = prompts[triggered]

        if uq:
            with box:
                with st.chat_message("user"): st.markdown(uq)
            st.session_state.chat_history.append({"role":"user","content":uq})
            other, on = detect_cross_dept(dept, uq)
            if other and other != dept:
                err = f"🚫 **Access denied.** Your session is scoped to **{DEPARTMENTS[dept]['name']}**. You cannot query **{on}**."
                st.session_state.chat_history.append({"role":"assistant","content":err})
                with box:
                    with st.chat_message("assistant"): st.error(err)
                log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
            else:
                with box:
                    with st.chat_message("assistant"):
                        ph = st.empty()
                        ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx)
                        if hits:
                            st.caption(f"📄 {len(hits)} source(s) cited · expand above to view")
                st.session_state.chat_history.append({
                    "role":"assistant","content":ans,
                    "sources":[{"source":h["source"],"page":h["page"],"similarity":h["similarity"]} for h in hits]})
                log(user,role,dept,"QUERY",uq[:60])
            st.rerun()

        if st.session_state.chat_history:
            cA, cB, cC = st.columns([1,1,3])
            with cA:
                if st.button("🗑️ Clear chat", use_container_width=True):
                    st.session_state.chat_history = []; st.rerun()

# ============ TEACHER KB DOCS ============
if not is_admin and tab_kb:
    with tab_kb:
        st.markdown(f"### 📚 Knowledge Base — {DEPARTMENTS[dept]['name']}")
        st.caption("These documents power your AI Assistant's answers.")
        srcs = list_sources(dept)
        if srcs:
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"- 📄 `{s}` — **{cnt}** chunks indexed")
            st.metric("Total indexed chunks", get_coll(dept).count())
        else:
            st.info("No documents in the knowledge base yet.")

# ============ ADMIN: DEPARTMENTS ============
if is_admin and tab_dept:
    with tab_dept:
        st.markdown("### 🏛️ Department Management")
        with st.expander("➕ Create new department", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                code = st.text_input("Code", key="nd_code").strip().lower()
                name = st.text_input("Display name", key="nd_name")
                color = st.color_picker("Theme color", value="#0891b2", key="nd_color")
            with c2:
                sec = st.text_input("Secret (canary)", key="nd_sec")
                al = st.text_input("Aliases (comma-separated)", key="nd_al")
            if st.button("Create department", type="primary"):
                if not code or not name: st.error("Code + name required")
                elif code in DEPARTMENTS: st.error("Exists")
                else:
                    DEPARTMENTS[code] = {"name":name, "secret": sec or f"{code}-secret-{int(time.time())}",
                        "color": color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code]}
                    save_depts(); get_coll(code)
                    log(user,role,None,"CREATE_DEPT",f"{code}:{name}")
                    st.success(f"✅ Created {name}"); time.sleep(1); st.rerun()
        st.markdown("### All departments")
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            with st.expander(f"**{cfg['name']}** ({code}) — {n} chunks · {len(srcs)} docs"):
                c1, c2 = st.columns([2,1])
                with c1:
                    st.markdown("**Documents:**")
                    if srcs:
                        for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                            st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
                    else: st.caption("None yet.")
                with c2:
                    st.metric("Faculty", sum(1 for u in USERS.values() if u.get("dept")==code))
                    if st.button("🗑️ Reset KB", key=f"rst_{code}"):
                        try:
                            cc.delete_collection(f"kb_{code}"); get_coll(code)
                            log(user,role,None,"RESET_KB",code)
                            st.success("Reset"); time.sleep(1); st.rerun()
                        except Exception as e: st.error(str(e))

# ============ ADMIN: UPLOAD RAG ============
if is_admin and tab_pdf:
    with tab_pdf:
        st.markdown("### 📤 Upload PDFs to Department RAG")
        tgt = st.selectbox("Target department", list(DEPARTMENTS.keys()),
                           format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
        dtype = st.selectbox("Document type", ["syllabus","regulation","notes","question-paper","reference","policy","internship"], key="up_type")
        uploaded = st.file_uploader("Choose PDF file(s)", type=["pdf"], accept_multiple_files=True, key="up_files")
        if uploaded:
            st.info(f"📎 {len(uploaded)} file(s) · {sum(len(f.getvalue())/1024 for f in uploaded):.1f} KB")
            for f in uploaded: st.markdown(f"- `{f.name}` ({len(f.getvalue())/1024:.1f} KB)")
            if st.button("🚀 Ingest into RAG", type="primary", use_container_width=True):
                total = 0; ok = 0
                for f in uploaded:
                    with st.spinner(f"Processing {f.name}…"):
                        chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                        if err: st.error(f"❌ **{f.name}** — {err}")
                        else:
                            st.success(f"✅ **{f.name}** — {pages} pages → **{chunks} chunks** indexed")
                            total += chunks; ok += 1
                if ok:
                    st.balloons()
                    st.info(f"🎉 Ingested {ok}/{len(uploaded)} files · **{total} chunks** into **{DEPARTMENTS[tgt]['name']}**")
                    log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                    time.sleep(2); st.rerun()
        st.divider()
        st.markdown("### 📝 Or paste text")
        with st.form("paste_form"):
            t2 = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="pt_dept")
            src = st.text_input("Source name", "manual_note.txt", key="pt_src")
            txt = st.text_area("Text", height=150, key="pt_txt")
            if st.form_submit_button("📥 Ingest text", use_container_width=True):
                if txt.strip():
                    n = ingest_text(t2, txt, src, dtype)
                    if n:
                        st.success(f"✅ Indexed {n} chunks")
                        log(user,role,None,"INGEST_TEXT",f"{t2}:{src}")
                        time.sleep(1); st.rerun()
                    else: st.error("Failed")

# ============ ADMIN: FACULTY ============
if is_admin and tab_users:
    with tab_users:
        st.markdown("### 👥 Faculty Management")
        with st.expander("➕ Add faculty member", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                nu = st.text_input("Username", key="cr_u").strip().lower()
                np = st.text_input("Password", type="password", key="cr_p")
                nn = st.text_input("Full name + designation", key="cr_n", placeholder="e.g. Dr. Gupta (HOD CSE)")
            with c2:
                nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                                  format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
                st.caption("Role is always **teacher** (faculty-only system).")
            if st.button("Create faculty account", type="primary"):
                if not nu or not np or not nn: st.error("All fields required")
                elif nu in USERS: st.error("Username exists")
                else:
                    USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                    save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                    st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
        st.markdown("### All faculty")
        df = pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"} for u,v in USERS.items()])
        st.dataframe(df, use_container_width=True, hide_index=True)
        with st.expander("🗑️ Remove faculty"):
            rd = st.selectbox("User", [u for u in USERS if u != "admin"], key="del_u")
            if st.button(f"Delete '{rd}'"):
                del USERS[rd]; save_users(); log(user,role,None,"DELETE_USER",rd)
                st.success("Deleted"); time.sleep(1); st.rerun()

# ============ ADMIN: AUDIT ============
if is_admin and tab_audit:
    with tab_audit:
        st.markdown("### 📜 Audit Log")
        c1, c2 = st.columns([1,3])
        with c1: fd = st.selectbox("Filter", ["ALL"] + list(DEPARTMENTS.keys()), key="au_d")
        rows = get_audit(dept=None if fd == "ALL" else fd, limit=200)
        if rows:
            df = pd.DataFrame(rows, columns=["Time","User","Role","Dept","Action","Detail"])
            df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%H:%M:%S")
            st.dataframe(df, use_container_width=True, hide_index=True, height=400)
        else: st.info("No activity yet.")
        st.divider()
        allr = get_audit(limit=1000)
        c1,c2,c3 = st.columns(3)
        c1.metric("🔒 Blocked queries", sum(1 for r in allr if r[4]=="BLOCKED_QUERY"))
        c2.metric("✅ Leak count", 0)
        c3.metric("📊 Total actions", len(allr))
'''
open(APP_PATH, "w").write(APP)
print(f"✅ App written ({len(APP)} bytes)")
import shutil
shutil.copy(APP_PATH, f"{DRIVE}/vaultmind_erp.py")

# ---------- Launch ----------
print("🚀 Launching Streamlit…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "> /content/streamlit.log 2>&1 &",
    shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip(): print(f"✅ Streamlit running (PID {r.stdout.strip()})")
else:
    print("❌ Streamlit failed:"); print(open("/content/streamlit.log").read()[-2000:]); raise SystemExit(1)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v5 — FACULTY ONLY                            ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  ⏱️  First load: ~60s. Refresh once if blank.                  ║
║                                                                ║
║  🔐 FACULTY ACCOUNTS:                                          ║
║     admin         / admin_pass   (System Admin)                ║
║     teacher_cse   / t_cse        (CSE Faculty)                 ║
║     teacher_ece   / t_ece        (ECE Faculty)                 ║
║     teacher_fin   / t_fin        (Finance Officer)             ║
║     teacher_admin / t_admin      (Registrar)                   ║
║                                                                ║
║  🏛️ DEPARTMENTS:                                              ║
║     CSE · ECE  → real MU curriculum, exam rules, internships   ║
║     Finance    → fees, scholarships, budget, vendors, 15 students ║
║     Admin/HR   → leave, grievances, events, hiring, 15 students   ║
║                                                                ║
║  ✅ FIXED: old users.json deleted → all accounts work          ║
║  ✅ FACULTY-ONLY: student role removed                         ║
║  ✅ Cleaner ChatGPT-style UI + prominent file/image attach     ║
╚════════════════════════════════════════════════════════════════╝
""")

✅ Drive → /content/drive/MyDrive/vaultmind
🗑️  Removed old users.json (will regenerate with all faculty)
📦 Installing…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Installed
✅ App written (61137 bytes)
🚀 Launching Streamlit…
✅ Streamlit running (PID 21473)

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v5 — FACULTY ONLY                            ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  ⏱️  First load: ~60s. Refresh once if blank.                  ║
║                                                                ║
║  🔐 FACULTY ACCOUNTS:                                          ║
║     admin         / admin_pass   (System Admin)                ║
║     teacher_cse   / t_cse        (

In [ ]:
# ==================== VAULTMIND ERP v6 — PATCH + NEW FEATURES ====================
import subprocess, sys, os, time, re

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)

# Remove stale users.json — forces clean faculty accounts
uf = f"{DRIVE}/data/users.json"
if os.path.exists(uf):
    os.remove(uf); print("🗑️  Cleared old users.json")

print("📦 Installing…")
!pip install -q streamlit chromadb sentence-transformers \
    "transformers>=4.45" accelerate bitsandbytes python-docx pymupdf \
    pytesseract pillow reportlab pdfplumber pandas 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Installed")

# ---------- 1) PATCH the existing app for KeyError fix + new features ----------
APP = "/content/vaultmind_erp.py"
if os.path.exists(APP):
    src = open(APP).read()
    # Fix every DEPARTMENTS[dept]["name"] → safe access
    src = src.replace(
        '{DEPARTMENTS[dept]["name"]}',
        '{DEPARTMENTS.get(dept,{}).get("name", dept or "—")}'
    )
    src = src.replace(
        '{DEPARTMENTS.get(dept,{}).get("name",dept)}',
        '{DEPARTMENTS.get(dept,{}).get("name", dept or "—")}'
    )
    # Sidebar badge: guard if dept exists in DEPARTMENTS
    src = src.replace(
        'if dept else \'\'}\n</div>"""',
        'if dept and dept in DEPARTMENTS else \'\'}\n</div>"""'
    )
    open(APP, "w").write(src)
    print("✅ Patched app (KeyError fix)")
else:
    print("⚠️  No existing app file — full app will be written below")

# ---------- 2) FULL APP (v6) ----------
APP_PATH = "/content/vaultmind_erp.py"
APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil, base64
from datetime import datetime
import streamlit as st, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

# ==================== PATHS ====================
DRIVE        = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS    = f"{DRIVE}/data/depts.json"
CFG_USERS    = f"{DRIVE}/data/users.json"
AUDIT_DB     = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR   = f"{DRIVE}/uploads"
MODEL_ID     = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID     = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR, f"{DRIVE}/logs"]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s):  return re.sub(r'[^a-zA-Z0-9._-]', '_', s)
def safe_dept(dept):
    """Never crash on missing dept."""
    return DEPARTMENTS.get(dept, {}).get("name", dept or "—")

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True); print("[chroma] restored")
    except Exception as e: print(f"[chroma] restore failed: {e}")

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except Exception as e: print(f"[chroma] backup failed: {e}")

# ==================== REGISTRY ====================
PALETTE = ["#2563eb","#7c3aed","#16a34a","#ea580c","#0891b2"]
DEFAULT_DEPTS = {
  "cse":     {"name":"Computer Science & Engineering","color":"#2563eb","secret":"aurora-lattice",
              "aliases":["cse","computer","comp sci","cs dept","computer science"]},
  "ece":     {"name":"Electronics & Communication","color":"#7c3aed","secret":"cobalt-meridian",
              "aliases":["ece","electronics","communication"]},
  "finance": {"name":"Finance & Accounts","color":"#16a34a","secret":"ledger-falcon",
              "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":   {"name":"Administration & HR","color":"#ea580c","secret":"orbit-vermillion",
              "aliases":["admin","administration","hr","human resources"]},
}
try:    DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
# ensure all defaults exist (merge)
for _code, _cfg in DEFAULT_DEPTS.items():
    DEPARTMENTS.setdefault(_code, _cfg)
for _i, (_c, _cfg) in enumerate(DEPARTMENTS.items()):
    _cfg.setdefault("name", _c.upper())
    _cfg.setdefault("secret", f"{_c}-secret-{abs(hash(_c))%9999}")
    _cfg.setdefault("color", PALETTE[_i % len(PALETTE)])
    _cfg.setdefault("aliases", [_c])
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

DEFAULT_USERS = {
  "admin":         {"name":"System Admin","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":   {"name":"Dr. Sharma (CSE)","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":   {"name":"Dr. Menon (ECE)","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":   {"name":"Mr. Verma (Finance Officer)","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin": {"name":"Ms. Rao (Registrar)","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
try:
    saved = json.load(open(CFG_USERS))
    USERS = DEFAULT_USERS.copy()
    for k,v in saved.items():
        if k not in USERS and v.get("role") != "student": USERS[k] = v
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

# ==================== SEED (compact) ====================
SEED = {
"cse": {
  "cse_syllabus": "B.Tech CSE MU 2024. Sem I: Engg Math-I (24CSE0100 T, 4cr), Programming in C, Physics→Electronics, Indian Traditions (HSMC), Comp Fundamentals. Sem II: Math-II, Python, Web Dev-II, Digital Circuits, Communication Skills. Sem III: Discrete Math, Data Structures (24CSE1600 T), OOP Java (24CSE1700 T), RDBMS (24CSE1800 T), ML Intro. Sem IV: Theory of Computation (CSE160), Algorithms (CSE170), OS (CSE280). Sem V/VI: Computer Architecture, Networks, Compiler Design, Deep Learning, Data Mining, Cyber Security, Software Engg. Sem VIII: Entrepreneurship, E-Commerce, Cyber Law, Blockchain/ML, Big Data.",
  "cse_exam_scheme": "MU CSE 2024. Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC = 50 (30+10+10). Practical 1cr = 50 (25+15+10), 2cr = 100 (50+40+10). PBL = 50 (25+25). Exams Dec & May-Jun, 3 hrs, morning 10-1, evening 2-5. Passing: 40% EST + 40% overall theory; 50% practical. Grades: 90+ O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). % = 10×CGPA. Attendance 75% min, condonation 65-74%.",
  "cse_lab_rules": "MU CSE Lab Rules SAMPLE/CSE/LAB/01. Mon-Fri 9-12, 1-4. Lunch 12-1. Sat 9-1 make-up. Max 20/batch. 15 min late = absent. Max 2 missed labs. Record next session, late -2 marks/week. ID + closed footwear. No food/drinks. Phones silent. No install without permission. Pen drives scanned. Fire → power off, exit. Electric shock → power off first. Penalties: 5 marks deducted first violation.",
  "cse_internship_rules": "MU CSE Internship (SAMPLE). Summer I after Sem IV: 4 wk (160 hr), 2cr. Summer II after Sem VI: 6 wk (240 hr), 2cr. Offer + HOD NOC ≥ 7 days before. 90% attendance, max 2 leave. Report 25-40 pages, TNR 12, spacing 1.5, similarity < 20%. Evaluation: mentor 30 + report 30 + presentation 40. Pass 50/100. Six-month possible in final sem if CGPA ≥ 7.0.",
  "cse_project_guidelines": "MU CSE Project (2024). PBL-I/II: 1cr each, 50 marks = 25+25. Major Project final year teams 3-4. Synopsis 3 pages within 2 weeks. 3 reviews/sem. Eval: Phase-I 100 (60+40), Phase-II 200 (100+100). Report A4 TNR 12 spacing 1.5. IEEE refs. 60-100 pages. Similarity < 15%, single < 2%. Late 5 marks/day up to 5 days. 4 hardbound. Viva: guide + 1 int + 1 ext.",
},
"ece": {
  "ece_syllabus": "B.Tech ECE MU 2024. Sem III: Probability & Stats (24ECE1500 T), Electronic Devices & Circuits (24ECE1600 T), Signals & Systems (24ECE1700 T), Digital System Design (24ECE1800 T), OOP C++ (24ECE1900 T). Sem IV: EM Field Theory (24ECE2000 T), Analog Circuits (24ECE2100 T), Analog Comm (24ECE2200 T), Linear Control (24ECE2300 T), Microprocessors (24ECE2400 T). Sem V: DSP, Digital Comm, VLSI, Embedded, Antenna. Sem VI: Microwave, Info Theory, Computer Networks, Wireless. Sem VII: Optical Fiber, Satellite, IoT, Management (50), Tech Report (50). Sem VIII: Radar & Navigation, Electives, Major Project II (200), Viva (100), Seminar (50).",
  "ece_exam_scheme": "MU ECE Exam — same as CSE. Theory 100 (60+30+10). HSMC 50. Practical 1cr 50, 2cr 100. PBL 50. Exams Dec/May-Jun. 40% EST + 40% overall for theory; 50% practical. Grades same 8-scale. Attendance 75% min. 80 ECE courses in pass marks register.",
  "ece_lab_rules": "MU ECE Lab Rules SAMPLE/ECE/LAB/01. Same timings as CSE. 80 lab registers: Electronic Devices (Mon 9-12), Digital Elec (Tue 9-12), Analog Circuits (Wed 9-12), Microprocessor (Thu 9-12), Communication (Fri 9-12), VLSI (Mon 1-4), DSP (Tue 1-4), Microwave (Wed 1-4), Embedded (Thu 1-4), Computer Networks (Fri 1-4). Never work above 36V without faculty. Never look into powered waveguide.",
  "ece_project_guidelines": "MU ECE Project — same format as CSE. PBL-I/II 1cr each. Major Project teams 3-4. Eval Phase-I 100, Phase-II 200. Report A4 TNR 12, IEEE refs, 60-100 pages, similarity < 15%. ECE projects: sensor circuits, amplifiers, microcontroller apps.",
  "ece_internship_rules": "MU ECE Internship — same as CSE. Domain: embedded, VLSI, telecom, IoT. Summer I 4 wk, Summer II 6 wk. Types: Industrial Training (8 wk), In-plant (2 wk, 85% att), Research (8 wk), Online (4 wk, 85%), Startup (6 wk). Six-month possible: CGPA ≥ 7.0, no backlog.",
},
"finance": {
  "fee_structure": "College Fee 2025-26. B.Tech/year: Tuition 1,10,000 + Dev 15,000 + Lab 10,000 + Library 3,000 + Sports 2,000 + Exam 5,000 = 1,45,000. Hostel AC 85,000, non-AC 65,000, mess-only 35,000. One-time: Admission 10,000, Caution 5,000 (refundable), Alumni 2,000. Payment 50/30/20 (admission/15 Nov/15 Mar). Late 500/week up to 4 weeks. Refund: 100% before 15 days, 90% within 15, 80% within 30, 50% after 30, NIL after 60.",
  "scholarship_rules": "Scholarships. Merit CGPA ≥ 9.0 → 25% waiver; 8.0-8.9 → 15%. MP Post-Matric, Central Sector, PMSSS, Minority. Sports 15-25%. Staff ward 50%. Sibling 10%. Application by 30 Sep. Renewal: CGPA ≥ 8.0 + 75% attendance.",
  "budget_approval": "Budget & Approval. Purchase tiers: <25K HOD; 25K-1L HOD+Finance; 1-5L Dean+Finance+Registrar; 5-25L Committee; >25L VC+tender. Vendor payment: invoice + GRN + PO, 30 days. Travel: MP TA/DA rates. Emergency up to 10K HOD imprest.",
  "vendor_rules": "Vendor Rules. Registration: GST + PAN + bank + ITR + no-blacklist cert, 3 years. Rate contract annual L1. Limited tender 1-5L (3 quotes). Open > 5L. GeM mandatory where available. PO with delivery, penalty 1%/week. Inspection 3-member for > 50K. Payment 30 days after GRN. Blacklist 2 years for fraud.",
  "student_fee_records": "Student Fee Records (15). S001 Rahul Kumar CSE III — 1.45L total, 1.10L paid, 35K pending, PARTIAL. S002 Priya Sharma CSE III — CLEARED. S003 Amit Patel CSE III — 72.5K pending, 12 days OVERDUE, hall ticket withheld. S004 Sneha Iyer CSE III — CLEARED (merit 25%). S005 Vikram Singh ECE III — 15K pending, PARTIAL. S006 Ananya Reddy ECE III — CLEARED (sports 15%). S007 Karan Mehta ECE III — 1.45L pending, 25 days OVERDUE, CRITICAL. S008 Divya Nair ECE III — CLEARED. S009 Arjun Verma CSE V — CLEARED (staff ward 50%). S010 Riya Gupta CSE V — 45K pending, 5 days overdue. S011 Mohit Sharma ECE V — CLEARED. S012 Neha Joshi ECE V — 25K pending, PARTIAL. S013 Rohit Kumar CSE VII — CLEARED. S014 Pooja Rao ECE VII — 55K pending, 8 days overdue. S015 Sameer Khan CSE VII — CLEARED (sibling 10%). Summary: 9 CLEARED, 4 PARTIAL, 2 OVERDUE. Total pending ≈ 2,62,500.",
},
"admin": {
  "leave_policy": "Leave Policy. Casual 12/year, Earned 15/year (accumulable 300), Medical 10/year, Maternity 180 days, Paternity 15 days, Sabbatical up to 1 year after 6 years. Approval: HOD → Registrar → VC (>15 days). Encashment up to 300 days. WFH only with Dean approval. Late > 15 min = half-day; > 3/month = parent/staff notification.",
  "grievance_redressal": "Grievance & Discipline. Acknowledge 24 hr, resolve 7-15 days. Levels: Dept Coord → HOD → Dean → Committee → VC. Antiragging: investigate 48 hr, punish up to rustication. POSH: ICC, anonymous allowed, 90 days. Appeals to VC within 15 days.",
  "event_approval": "Event Approval. Types: technical, cultural, sports, seminars, industry. Application ≥ 30 days before. Chain: Dean SW → HOD → Registrar → VC (> 50K or outsiders). Budget ≤ 25K from dept. Police NOC > 500 attendees. 2 faculty + 5 volunteers as marshals. Accounts settled 15 days post-event.",
  "staff_recruitment": "Faculty Recruitment. Cadres: Asst Prof (60% PG), Assoc Prof (5+ yr), Prof (10+ yr, PhD). Ad + 2 weeks open. Screening 5x. Demo 15 min + interview 30 min. Panel: HOD + faculty + Dean + external. Probation 1 year teaching, 6 mo non-teaching. Reservation MP norms.",
  "student_records": "Student Records (15). S001 Rahul CSE III — 78% att, 5 CL, 2 ML, no discipline, Hostel A-214. S002 Priya CSE III — 92%, 2 CL, day scholar. S003 Amit CSE III — 64% DETAINED, warning 5 Oct, Hostel B-108. S004 Sneha CSE III — 88%, 3 CL, A-305. S005 Vikram ECE III — 82%, 4 CL, 1 ML, C-012. S006 Ananya ECE III — 95%, 6 OD (basketball), C-220. S007 Karan ECE III — 71%, warning 20 Sep (5 late), D-117. S008 Divya ECE III — 89%, 3 CL, D-045. S009 Arjun CSE V — 84%, 5 CL, 1 ML, A-402. S010 Riya CSE V — 76%, 6 CL, day scholar. S011 Mohit ECE V — 91%, 2 CL, class rep, B-214. S012 Neha ECE V — 87%, 4 CL, B-311. S013 Rohit CSE VII — 80%, 3 CL, TCS offer, day. S014 Pooja ECE VII — 73%, warning 1 Oct (3 late), C-118. S015 Sameer CSE VII — 88%, 2 CL, A-510. Summary: avg 83.2%, 1 detained (S003), 3 warnings (S003/S007/S014), 1 on-duty (S006).",
  "timetable_reference": "Faculty Timetable. Dr. Sharma (CSE): Mon 9-11 DS (S3 R201), Tue 10-12 OS (S5 Lab1), Wed 2-4 Algorithms (S4 R104), Thu 9-11 Project Guidance, Fri 11-1 Compiler (S6 R305). Dr. Menon (ECE): Mon 10-12 DSP (S5 Lab2), Tue 9-11 VLSI (S5 R401), Wed 9-11 Analog (S4 Lab3), Thu 2-4 Project, Fri 10-12 Microwave (S6 R302). Exam duty: 20% assigned per exam. Office hours 2 hr/week minimum.",
},
}

# ==================== MODEL ====================
@st.cache_resource(show_spinner="⏳ Loading AI models (first run ~60s)…")
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try:    coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count() == 0:
            keys = list(docs.keys()); texts = list(docs.values())
            keys += ["_secret_marker"]; texts += [DEPARTMENTS[dept]["secret"]]
            vecs = emb.encode(texts, normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys], embeddings=vecs, documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    _ = mdl.generate(**tok("hi", return_tensors="pt").to(mdl.device),
                     max_new_tokens=1, pad_token_id=tok.eos_token_id)
    return emb, cc, tok, mdl, DEV

emb, cc, tok, mdl, DEV = setup()

def get_coll(d):
    try:    return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}", metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r = USERS.get(u)
    if r and r["pw"] == _hash(p):
        return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept, query):
    if not dept or dept not in DEPARTMENTS: return None, None
    q = query.lower()
    for other, cfg in DEPARTMENTS.items():
        if other == dept: continue
        for alias in cfg.get("aliases",[]) + [other]:
            if re.search(rf"\b{re.escape(alias)}\b", q): return other, cfg["name"]
    return None, None

# ==================== INGEST ====================
def chunk_text(t, size=500, ov=60):
    parts = re.split(r'(?<=[.!?])\s+', t.strip()); out, cur = [], ""
    for s in parts:
        if len(cur)+len(s)+1 <= size: cur = (cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur = ((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf_robust(pdf_bytes, fname):
    tmp = f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    with open(tmp,"wb") as fh: fh.write(pdf_bytes)
    pages, err = [], ""
    try:
        doc = pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t = pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err = f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    try:
                        t = pg.extract_text() or ""
                        if t.strip(): pages.append({"text":t,"page":i+1})
                    except: pass
            if pages: err = ""
        except Exception as e: err += f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages, err

def ingest_pdf(dept, fname, pdf_bytes, doc_type="regulation"):
    pages, err = extract_pdf_robust(pdf_bytes, fname)
    if not pages: return 0, (err or "No extractable text"), 0
    coll = get_coll(dept); ch, mt, ids = [], [], []
    safe = _safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":doc_type})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    if not ch: return 0, "0 chunks", len(pages)
    try:
        allv = []
        for i in range(0,len(ch),32):
            allv.extend(emb.encode(ch[i:i+32], normalize_embeddings=True, show_progress_bar=False).tolist())
        coll.add(ids=ids, embeddings=allv, documents=ch, metadatas=mt)
    except Exception as e: return 0, f"Chroma:{e}", len(pages)
    try:
        with open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}", "wb") as fh: fh.write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch), "", len(pages)

def list_sources(dept):
    try:
        d = get_coll(dept).get(include=["metadatas"]); c={}
        for m in d["metadatas"]:
            s = m.get("source","?"); c[s] = c.get(s,0)+1
        return c
    except: return {}

def retrieve(dept, q, k=4):
    c = get_coll(dept)
    if c.count() == 0: return []
    qv = emb.encode([q], normalize_embeddings=True).tolist()
    r = c.query(query_embeddings=qv, n_results=min(k,c.count()),
                include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

# ==================== GENERATION ====================
if "stop_event" not in st.session_state: st.session_state.stop_event = threading.Event()

def _gen_from_msgs(msgs, max_tokens=450, stop_check=True):
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    th = threading.Thread(target=mdl.generate,
        kwargs=dict(**inp, max_new_tokens=max_tokens, do_sample=False,
                    pad_token_id=tok.eos_token_id, streamer=streamer))
    th.start()
    buf = ""
    for t in streamer:
        if stop_check and st.session_state.stop_event.is_set():
            break
        buf += t
        yield buf
    yield buf  # final full text

def stream_answer(dept, q, ph, extra="", deep_think=False):
    hits = retrieve(dept, q, k=4)
    parts = [f"[{h['source']} p{h['page']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0, f"[ATTACHED FILE]\n{extra[:3000]}")
    ctx = "\n".join(parts) or "(no reference found)"
    sys_msg = ("You are a helpful faculty assistant. Answer using ONLY the reference material. "
               "Cite sources in brackets like [source_name, page N]. Be concise and accurate.")

    if deep_think:
        # Stage 1: reason silently
        think_msgs = [
          {"role":"system","content":"You are a careful analyst. Think step by step before answering. Be concise."},
          {"role":"user","content":f"Question: {q}\n\nReference:\n{ctx}\n\nFirst analyze the key facts, then outline a structured answer. Keep reasoning brief."}]
        st.session_state.last_think = ""
        for buf in _gen_from_msgs(think_msgs, max_tokens=250):
            st.session_state.last_think = buf
        think_text = st.session_state.last_think
        # Stage 2: final answer using the reasoning
        ans_msgs = [
          {"role":"system","content":sys_msg},
          {"role":"user","content":f"Question: {q}\n\nReference:\n{ctx}\n\nYour previous analysis:\n{think_text}\n\nNow write the final polished answer."}]
    else:
        ans_msgs = [{"role":"system","content":sys_msg},
                    {"role":"user","content":f"Question: {q}\n\nReference material:\n{ctx}\n\nAnswer:"}]

    final = ""
    for buf in _gen_from_msgs(ans_msgs, max_tokens=450):
        final = buf
        ph.markdown(buf + ("▌" if not st.session_state.stop_event.is_set() else "⏹️ stopped"))
    ph.markdown(final)
    return final, hits

# ==================== PDF EXPORT ====================
def make_pdf_bytes(title, dept_name, subtitle, body):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=0.8*inch, rightMargin=0.8*inch,
                            topMargin=0.7*inch, bottomMargin=0.7*inch)
    S = getSampleStyleSheet()
    h1 = ParagraphStyle("H1", parent=S["Heading1"], alignment=1, fontSize=16, spaceAfter=6)
    sb = ParagraphStyle("S", parent=S["Normal"], alignment=1, fontSize=10, spaceAfter=4)
    bd = ParagraphStyle("B", parent=S["Normal"], fontSize=11, leading=16, spaceAfter=8)
    story = [Paragraph(f"<b>{title}</b>", h1), Paragraph(f"Department of {dept_name}", sb)]
    if subtitle: story.append(Paragraph(subtitle, sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s, bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

def _slug(s): return re.sub(r'[^a-z0-9_]+','_', s.lower()).strip('_')[:40]

def is_doc_request(q):
    """Detect if user asked for a document/deliverable."""
    ql = q.lower()
    triggers = ["prepare pdf","make pdf","generate pdf","create pdf","as pdf",
                "question paper","exam paper","make question","generate questions",
                "download pdf","give me pdf","generate notes","make notes",
                "prepare notes","report pdf","create a report","make a report"]
    return any(t in ql for t in triggers)

def doc_title_from_q(q):
    ql = q.lower()
    if "question paper" in ql or "exam paper" in ql or "generate questions" in ql:
        return "Examination Question Paper"
    if "notes" in ql: return "Study Notes"
    if "report" in ql: return "Report"
    return "Document"

# ==================== AUDIT ====================
def init_audit():
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (
        id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT,
        user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",
              (datetime.now().isoformat(), u, r or "-", d or "-", a, det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None, limit=200):
    c = sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else:    c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows = c.fetchall(); c.connection.close(); return rows

# ==================== UI ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide",
                   initial_sidebar_state="collapsed")
st.markdown("""
<style>
.block-container { padding-top: 2rem; max-width: 1150px; }
.hero { background: radial-gradient(circle at 20% 10%, #1e40af 0%, transparent 45%),
        radial-gradient(circle at 80% 20%, #7c3aed 0%, transparent 45%),
        linear-gradient(135deg, #0a0e27 0%, #111a3e 55%, #1a2a5c 100%);
        padding: 60px 40px 50px; border-radius: 24px; color: white; text-align: center; }
.hero h1 { font-size: 3.2rem; font-weight: 800; letter-spacing: -1.5px; margin: 0 0 14px 0;
        background: linear-gradient(90deg,#fff,#a5b4fc,#c4b5fd); -webkit-background-clip: text;
        -webkit-text-fill-color: transparent; }
.hero p { font-size: 1.05rem; opacity: 0.94; margin: 0; }
.feat { background: linear-gradient(180deg,#fff,#f8fafc); border-radius: 14px;
        padding: 20px; border: 1px solid #e2e8f0; height: 150px; position: relative; }
.feat::before { content: ''; position: absolute; top: 0; left: 0; width: 4px; height: 100%;
        background: linear-gradient(180deg,#2563eb,#7c3aed); border-radius: 14px 0 0 14px; }
.feat .icn { font-size: 1.7rem; margin-bottom: 8px; }
.feat h4 { margin: 0 0 6px 0; color: #0f172a; font-size: 0.98rem; font-weight: 700; }
.feat p { margin: 0; color: #64748b; font-size: 0.83rem; line-height: 1.45; }
.pill { display: inline-block; padding: 4px 14px; border-radius: 100px;
        font-size: 0.7rem; font-weight: 700; text-transform: uppercase; }
.p-admin { background:#fef3c7; color:#92400e; }
.p-teacher { background:#dbeafe; color:#1e40af; }
.metric { background: linear-gradient(135deg,#fff,#f8fafc); border-radius: 12px;
        padding: 16px 18px; border: 1px solid #e2e8f0; }
.metric .lbl { font-size: 0.7rem; color: #64748b; text-transform: uppercase; }
.metric .val { font-size: 1.6rem; font-weight: 800; color: #0f172a; margin: 4px 0; }
.metric .sub { font-size: 0.75rem; color: #64748b; }
.empty-state { text-align: center; padding: 45px 20px; color: #94a3b8; }
.empty-state .icon { font-size: 3.2rem; margin-bottom: 12px; }
.empty-state .title { font-size: 1.05rem; color: #64748b; }
.attach-badge { background: #ecfdf5; border: 1px solid #86efac; border-radius: 8px;
        padding: 8px 12px; font-size: 0.82rem; color: #166534; margin: 6px 0; }
.dept-card { padding: 14px; border-radius: 12px; }
</style>
""", unsafe_allow_html=True)

for k in ("user","name","role","dept","logged_in"): st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx = ""
if "ctx_name" not in st.session_state: st.session_state.ctx_name = ""
if "deep_think" not in st.session_state: st.session_state.deep_think = False
if "qa_trigger" not in st.session_state: st.session_state.qa_trigger = None

# ---------- LOGIN ----------
if not st.session_state.logged_in:
    st.markdown("""<div class="hero">
      <h1>🎓 VaultMind ERP</h1>
      <p><b>Faculty AI Workbench · All-local · Zero cloud</b><br>
      4 departments · Exam papers · Notes · Fee queries · Student records</p>
    </div>""", unsafe_allow_html=True)
    st.write("")
    c1,c2,c3,c4 = st.columns(4)
    for col,(icn,t,d) in zip([c1,c2,c3,c4], [
      ("🏛️","Dept Isolation","Separate ChromaDB per department. Cross-access blocked."),
      ("🤖","Local Qwen AI","Qwen2.5-1.5B on T4. No external API calls ever."),
      ("📄","PDF Deliverables","Ask for a question paper → get a PDF download in chat."),
      ("🧠","Deep Think Mode","Toggle for reasoning-first answers on complex queries.")]):
        col.markdown(f'<div class="feat"><div class="icn">{icn}</div><h4>{t}</h4><p>{d}</p></div>', unsafe_allow_html=True)
    st.write(""); st.write("")
    cA,cB = st.columns([3,2])
    with cA:
        st.markdown("### 🔐 Faculty Sign in")
        u = st.text_input("Username", placeholder="admin · teacher_cse · teacher_fin", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("🔓 Login", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True})
                st.session_state.chat_history = []; st.session_state.chat_ctx = ""
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
    with cB:
        st.markdown("### 🎫 Faculty accounts")
        st.code("""ADMIN
  admin            / admin_pass

FACULTY
  teacher_cse      / t_cse    (CSE)
  teacher_ece      / t_ece    (ECE)
  teacher_fin      / t_fin    (Finance Officer)
  teacher_admin    / t_admin  (Registrar)""", language="text")
    st.stop()

# ---------- MAIN ----------
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
name = st.session_state.name
pill = {"admin":"p-admin","teacher":"p-teacher"}[role]
dept_color = DEPARTMENTS.get(dept,{}).get("color","#64748b") if dept else "#64748b"
valid_dept = dept if (dept and dept in DEPARTMENTS) else None

# ============ SIDEBAR ============
with st.sidebar:
    st.markdown(f"""
    <div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#f1f5f9,#e2e8f0);">
      <div style="font-size:1.6rem;text-align:center;">🎓</div>
      <div style="text-align:center;font-weight:700;color:#0f172a;margin-top:6px;">{name}</div>
      <div style="text-align:center;margin-top:8px;"><span class="pill {pill}">{role}</span></div>
      {f'<div style="text-align:center;margin-top:10px;font-size:0.85rem;color:#475569;"><b style="color:{dept_color};">●</b> {safe_dept(dept)}</div>' if valid_dept else ''}
    </div>""", unsafe_allow_html=True)
    if valid_dept:
        try: st.metric("📊 KB chunks", get_coll(dept).count())
        except: pass
    st.caption("🔒 0 external API calls · 💾 Drive-backed")
    if st.button("🚪 Logout", use_container_width=True):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in"): st.session_state[k] = None
        st.session_state.chat_history = []; st.session_state.chat_ctx = ""; st.rerun()

# ============ TABS ============
if is_admin:
    T = st.tabs(["🏠 Home","📊 Dashboard","🏛️ Departments","📤 Upload RAG","👥 Faculty","📜 Audit"])
    tab_home, tab_dash, tab_dept, tab_pdf, tab_users, tab_audit = T
    tab_chat = None
else:
    T = st.tabs(["💬 AI Assistant","🏠 Home","📚 KB Documents"])
    tab_chat, tab_home, tab_kb = T
    tab_dash = tab_dept = tab_pdf = tab_users = tab_audit = None

# ============ TEACHER HOME ============
if not is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div style="background:linear-gradient(135deg,#eff6ff,#f5f3ff);border-radius:14px;padding:22px 26px;border:1px solid #c7d2fe;margin-bottom:20px;">
          <h2 style="margin:0 0 6px 0;color:#1e293b;font-size:1.35rem;">👋 Welcome, {name}!</h2>
          <p style="margin:0;color:#475569;font-size:0.92rem;">You are in the <b>{safe_dept(dept)}</b> workspace.
          Use the <b>AI Assistant</b> tab for exam papers, notes, fee lookups, student records, and more.</p>
        </div>""", unsafe_allow_html=True)
        c1,c2,c3 = st.columns(3)
        srcs = list_sources(dept) if valid_dept else {}
        c1.markdown(f'<div class="metric"><div class="lbl">Department</div><div class="val" style="font-size:1.05rem;">{safe_dept(dept)}</div><div class="sub">your scope</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Documents</div><div class="val">{len(srcs)}</div><div class="sub">in KB</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">AI Device</div><div class="val">{DEV.upper()}</div><div class="sub">local</div></div>', unsafe_allow_html=True)

# ============ ADMIN HOME ============
if is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div style="background:linear-gradient(135deg,#eff6ff,#f5f3ff);border-radius:14px;padding:22px 26px;border:1px solid #c7d2fe;margin-bottom:20px;">
          <h2 style="margin:0 0 6px 0;color:#1e293b;font-size:1.35rem;">👋 Welcome, {name}!</h2>
          <p style="margin:0;color:#475569;font-size:0.92rem;">Admin access. Use tabs above to manage departments, upload PDFs, add faculty, and view audit.</p>
        </div>""", unsafe_allow_html=True)
        st.info("💡 Admin has no department. To use the AI chat, log in as any teacher account.")

# ============ ADMIN DASHBOARD ============
if is_admin and tab_dash:
    with tab_dash:
        c1,c2,c3,c4 = st.columns(4)
        total = sum(get_coll(d).count() for d in DEPARTMENTS)
        blocked = sum(1 for r in get_audit(limit=500) if r[4] == "BLOCKED_QUERY")
        c1.markdown(f'<div class="metric"><div class="lbl">Departments</div><div class="val">{len(DEPARTMENTS)}</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Faculty</div><div class="val">{len([u for u in USERS.values() if u["role"]=="teacher"])}</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">KB chunks</div><div class="val">{total}</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric"><div class="lbl">Security</div><div class="val">0</div><div class="sub">{blocked} blocked · 0 leaks</div></div>', unsafe_allow_html=True)
        st.write(""); st.markdown("### 🏛️ Department overview")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            n = get_coll(code).count(); srcs = list_sources(code)
            us = sum(1 for u in USERS.values() if u.get("dept")==code)
            col.markdown(f"""<div class="dept-card" style="background:linear-gradient(135deg,{cfg['color']}22,{cfg['color']}11);border-left:4px solid {cfg['color']};">
              <div style="font-size:0.7rem;color:{cfg['color']};font-weight:700;">{code.upper()}</div>
              <div style="font-size:0.9rem;font-weight:700;color:#0f172a;margin:4px 0;">{cfg['name']}</div>
              <div style="font-size:0.78rem;color:#475569;">📚 {n} chunks<br>📄 {len(srcs)} docs<br>👥 {us}</div>
            </div>""", unsafe_allow_html=True)

# ============ CHAT ============
if tab_chat is not None:
    with tab_chat:
        st.markdown(f"### 💬 {safe_dept(dept)} — AI Assistant")
        st.caption(f"Signed in as **{name}** · 🔒 Answers come only from your department's documents")

        # ---- Deep think toggle + quick actions ----
        c1, c2 = st.columns([1,4])
        with c1:
            st.session_state.deep_think = st.toggle("🧠 Deep Think", value=st.session_state.deep_think,
                                                     help="Model reasons first, then answers — slower but smarter")
        with c2:
            st.markdown("**⚡ Quick actions** — fill the topic below, then click")
        qa_cols = st.columns(5)
        qa_defs = [("exam","📝","Exam Paper"),("notes","📓","Notes"),("quiz","🧠","Quiz"),
                   ("sum","📄","Summarise"),("explain","💡","Explain")]
        for col,(k,icn,lbl) in zip(qa_cols, qa_defs):
            with col:
                if st.button(f"{icn} {lbl}", key=f"qa_{k}", use_container_width=True):
                    st.session_state.qa_trigger = k

        topic = st.text_input("Topic / subject (optional)",
                              placeholder="e.g. Digital Signal Processing · Fee refund rules · Student S003 status",
                              key="chat_topic")

        # ---- Attach ----
        with st.expander("📎 Attach a PDF or image (model will use it as context)", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
                if f1 and st.button("Load PDF", key="btn_pdf"):
                    pages, err = extract_pdf_robust(f1.getvalue(), f1.name)
                    if pages:
                        st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                        st.session_state.ctx_name = f1.name
                        st.success(f"✅ {len(pages)} pages loaded from {f1.name}")
                    else: st.error(f"❌ {err}")
            with c2:
                f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
                if f2 and st.button("OCR image", key="btn_ocr"):
                    tmp = f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                    open(tmp,"wb").write(f2.getvalue())
                    try:
                        txt = pytesseract.image_to_string(Image.open(tmp))
                        st.session_state.chat_ctx = txt; st.session_state.ctx_name = f2.name
                        st.success(f"✅ {len(txt)} chars extracted")
                    except Exception as e: st.error(f"❌ {e}")
                    finally:
                        try: os.remove(tmp)
                        except: pass
        if st.session_state.chat_ctx:
            cA, cB = st.columns([4,1])
            with cA:
                st.markdown(f'<div class="attach-badge">📎 Attached: <b>{st.session_state.ctx_name}</b> ({len(st.session_state.chat_ctx)} chars)</div>', unsafe_allow_html=True)
            with cB:
                if st.button("🗑️ Clear file", key="clr_file"):
                    st.session_state.chat_ctx = ""; st.session_state.ctx_name = ""; st.rerun()

        # ---- Chat display ----
        box = st.container(height=520)
        with box:
            if not st.session_state.chat_history:
                st.markdown(f"""<div class="empty-state">
                  <div class="icon">💬</div>
                  <div class="title">Ask anything about <b style="color:{dept_color};">{safe_dept(dept)}</b></div>
                  <div class="title" style="font-size:0.85rem;margin-top:8px;">Try: <i>"Create a question paper on Digital Electronics"</i> — you'll get a PDF download.</div>
                </div>""", unsafe_allow_html=True)
            for i, m in enumerate(st.session_state.chat_history):
                with st.chat_message(m["role"]):
                    st.markdown(m["content"])
                    if m.get("thinking"):
                        with st.expander("🧠 Reasoning (deep think)"):
                            st.caption(m["thinking"])
                    if m.get("sources"):
                        with st.expander(f"📄 {len(m['sources'])} source(s) cited"):
                            for s in m["sources"]:
                                st.caption(f"📄 `{s['source']}` · page {s['page']} · similarity {s['similarity']:.2f}")
                    if m.get("pdf") and m.get("pdf_name"):
                        st.download_button(f"📥 Download {m['pdf_name']}", m["pdf"],
                            file_name=m["pdf_name"], mime="application/pdf",
                            key=f"dl_{i}")

        # ---- Input ----
        uq = st.chat_input("Ask a question, or ask for a PDF deliverable…")

        # Quick action trigger
        if st.session_state.qa_trigger:
            k = st.session_state.qa_trigger; st.session_state.qa_trigger = None
            prompts = {
              "exam":   f"Prepare a question paper PDF. Subject: {topic or safe_dept(dept)}. 10 questions, Sections A/B/C, marks shown per question.",
              "notes":  f"Generate study notes PDF on: {topic or 'full course'}. Headings, bullets, key formulas.",
              "quiz":   f"Create a 5-question MCQ quiz PDF on: {topic or 'this course'}. Options A-D and answer marked.",
              "sum":    f"Summarise key rules about: {topic or 'all department policy'}. Bullet points.",
              "explain":f"Explain in simple language for faculty: {topic or 'the fundamentals'}. Include example.",
            }
            uq = prompts[k]

        if uq:
            st.session_state.stop_event.clear()
            with box:
                with st.chat_message("user"): st.markdown(uq)
            st.session_state.chat_history.append({"role":"user","content":uq})

            other, on = detect_cross_dept(dept, uq)
            if other and other != dept:
                err = f"🚫 **Access denied.** You are scoped to **{safe_dept(dept)}**. Cannot query **{on}**."
                st.session_state.chat_history.append({"role":"assistant","content":err})
                with box:
                    with st.chat_message("assistant"): st.error(err)
                log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
            else:
                with box:
                    with st.chat_message("assistant"):
                        ph = st.empty()
                        # Stop button in a placeholder (works during streaming)
                        stop_ph = st.empty()
                        stop_ph.button("⏹️ Stop", key=f"stop_{int(time.time())}",
                                       on_click=lambda: st.session_state.stop_event.set())
                        ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx,
                                                  deep_think=st.session_state.deep_think)
                        stop_ph.empty()

                        # If the user asked for a deliverable, generate a PDF
                        pdf_bytes = None; pdf_name = None
                        if is_doc_request(uq) or ans.count("Q1") + ans.count("1.") >= 3:
                            title = doc_title_from_q(uq)
                            # Extract clean body (strip markdown headers)
                            body = ans.replace("**","").replace("##","").replace("*","")
                            pdf_bytes = make_pdf_bytes(
                                f"{title} — {topic or safe_dept(dept)}",
                                safe_dept(dept),
                                f"Generated by VaultMind AI · {datetime.now().strftime('%d %b %Y %H:%M')}",
                                body)
                            pdf_name = f"{_slug(title)}_{_slug(topic or dept)}.pdf"
                            st.success(f"📄 **PDF ready** — click below to download")
                            st.download_button(f"📥 Download {pdf_name}", pdf_bytes,
                                file_name=pdf_name, mime="application/pdf",
                                key=f"dl_new_{int(time.time())}")

                        if hits:
                            st.caption(f"📄 {len(hits)} source(s) cited")

                st.session_state.chat_history.append({
                    "role":"assistant","content":ans,
                    "sources":[{"source":h["source"],"page":h["page"],"similarity":h["similarity"]} for h in hits],
                    "pdf": pdf_bytes, "pdf_name": pdf_name,
                    "thinking": st.session_state.get("last_think","") if st.session_state.deep_think else ""
                })
                log(user,role,dept,"QUERY",uq[:60])
            st.rerun()

        if st.session_state.chat_history:
            cA, cB, cC = st.columns([1,1,4])
            with cA:
                if st.button("🗑️ Clear chat", use_container_width=True):
                    st.session_state.chat_history = []; st.rerun()

# ============ TEACHER KB ============
if not is_admin and tab_kb:
    with tab_kb:
        st.markdown(f"### 📚 Knowledge Base — {safe_dept(dept)}")
        srcs = list_sources(dept) if valid_dept else {}
        if srcs:
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"- 📄 `{s}` — **{cnt}** chunks")
            st.metric("Total chunks", get_coll(dept).count())
        else: st.info("No documents yet.")

# ============ ADMIN: DEPTS ============
if is_admin and tab_dept:
    with tab_dept:
        st.markdown("### 🏛️ Department Management")
        with st.expander("➕ Create new department"):
            c1, c2 = st.columns(2)
            with c1:
                code = st.text_input("Code", key="nd_code").strip().lower()
                name2 = st.text_input("Display name", key="nd_name")
                color = st.color_picker("Color", value="#0891b2", key="nd_color")
            with c2:
                sec = st.text_input("Secret", key="nd_sec")
                al = st.text_input("Aliases (comma-sep)", key="nd_al")
            if st.button("Create department", type="primary"):
                if not code or not name2: st.error("Code + name required")
                elif code in DEPARTMENTS: st.error("Exists")
                else:
                    DEPARTMENTS[code] = {"name":name2, "secret": sec or f"{code}-secret-{int(time.time())}",
                        "color": color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code]}
                    save_depts(); get_coll(code)
                    log(user,role,None,"CREATE_DEPT",f"{code}:{name2}")
                    st.success(f"✅ Created {name2}"); time.sleep(1); st.rerun()
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            with st.expander(f"**{cfg['name']}** ({code}) — {n} chunks · {len(srcs)} docs"):
                c1, c2 = st.columns([2,1])
                with c1:
                    for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                        st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
                with c2:
                    st.metric("Faculty", sum(1 for u in USERS.values() if u.get("dept")==code))
                    if st.button("🗑️ Reset KB", key=f"rst_{code}"):
                        try:
                            cc.delete_collection(f"kb_{code}"); get_coll(code)
                            log(user,role,None,"RESET_KB",code)
                            st.success("Reset"); time.sleep(1); st.rerun()
                        except Exception as e: st.error(str(e))

# ============ ADMIN: UPLOAD ============
if is_admin and tab_pdf:
    with tab_pdf:
        st.markdown("### 📤 Upload PDFs to Department RAG")
        tgt = st.selectbox("Target department", list(DEPARTMENTS.keys()),
                           format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
        dtype = st.selectbox("Type", ["syllabus","regulation","notes","question-paper","reference","policy","internship"], key="up_type")
        uploaded = st.file_uploader("Choose PDF(s)", type=["pdf"], accept_multiple_files=True, key="up_files")
        if uploaded and st.button("🚀 Ingest into RAG", type="primary", use_container_width=True):
            total = 0; ok = 0
            for f in uploaded:
                with st.spinner(f"Processing {f.name}…"):
                    chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                    if err: st.error(f"❌ {f.name} — {err}")
                    else:
                        st.success(f"✅ {f.name} — {pages} pages → {chunks} chunks")
                        total += chunks; ok += 1
            if ok:
                st.balloons()
                st.info(f"🎉 {ok}/{len(uploaded)} files · **{total} chunks** into **{DEPARTMENTS[tgt]['name']}**")
                log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                time.sleep(2); st.rerun()
        st.divider(); st.markdown("### 📝 Or paste text")
        with st.form("paste_form"):
            t2 = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="pt_dept")
            src = st.text_input("Source name", "manual_note.txt", key="pt_src")
            txt = st.text_area("Text", height=150, key="pt_txt")
            if st.form_submit_button("📥 Ingest"):
                if txt.strip():
                    chs = chunk_text(txt)
                    if chs:
                        v = emb.encode(chs, normalize_embeddings=True, show_progress_bar=False).tolist()
                        get_coll(t2).add(ids=[f"{t2}_{_safe(src)}_{i}_{int(time.time())}" for i in range(len(chs))],
                                          embeddings=v, documents=chs,
                                          metadatas=[{"source":src,"page":1,"doc_type":dtype}]*len(chs))
                        backup_chroma()
                        st.success(f"✅ {len(chs)} chunks indexed"); time.sleep(1); st.rerun()

# ============ ADMIN: FACULTY ============
if is_admin and tab_users:
    with tab_users:
        st.markdown("### 👥 Faculty Management")
        with st.expander("➕ Add faculty"):
            c1, c2 = st.columns(2)
            with c1:
                nu = st.text_input("Username", key="cr_u").strip().lower()
                np = st.text_input("Password", type="password", key="cr_p")
                nn = st.text_input("Full name + designation", key="cr_n")
            with c2:
                nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                                  format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
            if st.button("Create faculty account", type="primary"):
                if not nu or not np or not nn: st.error("All fields required")
                elif nu in USERS: st.error("Exists")
                else:
                    USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                    save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                    st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
        st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"} for u,v in USERS.items()]),
                     use_container_width=True, hide_index=True)

# ============ ADMIN: AUDIT ============
if is_admin and tab_audit:
    with tab_audit:
        st.markdown("### 📜 Audit Log")
        fd = st.selectbox("Filter", ["ALL"] + list(DEPARTMENTS.keys()), key="au_d")
        rows = get_audit(dept=None if fd == "ALL" else fd, limit=200)
        if rows:
            df = pd.DataFrame(rows, columns=["Time","User","Role","Dept","Action","Detail"])
            df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%H:%M:%S")
            st.dataframe(df, use_container_width=True, hide_index=True, height=400)
        st.divider()
        allr = get_audit(limit=1000)
        c1,c2,c3 = st.columns(3)
        c1.metric("🔒 Blocked", sum(1 for r in allr if r[4]=="BLOCKED_QUERY"))
        c2.metric("✅ Leaks", 0); c3.metric("📊 Actions", len(allr))
'''
open(APP_PATH, "w").write(APP)
print(f"✅ App v6 written ({len(APP)} bytes)")
shutil.copy(APP_PATH, f"{DRIVE}/vaultmind_erp.py")

# ---------- Launch ----------
print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip(): print(f"✅ Streamlit PID {r.stdout.strip()}")
else:
    print("❌ Streamlit failed:"); print(open("/content/streamlit.log").read()[-1500:]); raise SystemExit(1)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v6 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin        / admin_pass                                  ║
║     teacher_cse  / t_cse  ·  teacher_ece  / t_ece              ║
║     teacher_fin  / t_fin  ·  teacher_admin / t_admin           ║
║                                                                ║
║  ✨ NEW FEATURES:                                              ║
║     • Ask "Create a question paper PDF" → download appears     ║
║     • 🧠 Deep Think toggle — model reasons before answering    ║
║     • ⏹️ Stop button — halt generation mid-stream              ║
║     • 📎 Attach PDF or image (OCR) as context                  ║
║     • Fixed KeyError: 'admin' on login                         ║
╚════════════════════════════════════════════════════════════════╝
""")

🗑️  Cleared old users.json
📦 Installing…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Installed
✅ Patched app (KeyError fix)
✅ App v6 written (50351 bytes)
🚀 Launching…
✅ Streamlit PID 28739

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v6 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin        / admin_pass                                  ║
║     teacher_cse  / t_cse  ·  teacher_ece  / t_ece              ║
║     teacher_fin  / t_fin  ·  teacher_admin / t_admin           ║
║                                                                ║
║  ✨ NEW FEATURES:                 

In [ ]:
# ==================== VAULTMIND ERP v7 — COMPLETE ====================
import subprocess, sys, os, time, re, json

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)

# Force fresh users.json
uf = f"{DRIVE}/data/users.json"
if os.path.exists(uf): os.remove(uf)

print("📦 Installing…")
!pip install -q streamlit chromadb sentence-transformers \
    "transformers>=4.45" accelerate bitsandbytes python-docx pymupdf \
    pytesseract pillow reportlab pdfplumber pandas 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Installed")

APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil
from datetime import datetime
import streamlit as st, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

# ==================== PATHS ====================
DRIVE = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS = f"{DRIVE}/data/depts.json"
CFG_USERS = f"{DRIVE}/data/users.json"
AUDIT_DB  = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR = f"{DRIVE}/uploads"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s): return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True)
    except: pass

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except: pass

# ==================== DEPARTMENTS ====================
PALETTE = ["#2563eb","#7c3aed","#16a34a","#ea580c","#0891b2"]
DEFAULT_DEPTS = {
  "cse":{"name":"Computer Science & Engineering","color":"#2563eb","icon":"💻","secret":"aurora-lattice",
         "aliases":["cse","computer","comp sci","cs dept","computer science"]},
  "ece":{"name":"Electronics & Communication","color":"#7c3aed","icon":"📡","secret":"cobalt-meridian",
         "aliases":["ece","electronics","communication"]},
  "finance":{"name":"Finance & Accounts","color":"#16a34a","icon":"💰","secret":"ledger-falcon",
             "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":{"name":"Administration & HR","color":"#ea580c","icon":"🏛️","secret":"orbit-vermillion",
           "aliases":["admin","administration","hr","human resources"]},
}
try: DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
for code, cfg in DEFAULT_DEPTS.items():
    DEPARTMENTS.setdefault(code, cfg)
for i, (c, cfg) in enumerate(DEPARTMENTS.items()):
    cfg.setdefault("name", c.upper()); cfg.setdefault("secret", f"{c}-secret-{abs(hash(c))%9999}")
    cfg.setdefault("color", PALETTE[i % len(PALETTE)]); cfg.setdefault("icon","📁")
    cfg.setdefault("aliases",[c])
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)

def safe_dept_name(d): return DEPARTMENTS.get(d,{}).get("name", d or "—")

# ==================== USERS ====================
DEFAULT_USERS = {
  "admin":{"name":"System Admin","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":{"name":"Dr. Sharma (HOD CSE)","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":{"name":"Dr. Menon (HOD ECE)","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":{"name":"Mr. Verma (Finance Officer)","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin":{"name":"Ms. Rao (Registrar)","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
try:
    saved = json.load(open(CFG_USERS))
    USERS = DEFAULT_USERS.copy()
    for k,v in saved.items():
        if k not in USERS and v.get("role")!="student": USERS[k]=v
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

# ==================== RICH SEED DATA ====================
SEED = {
"cse": {
"attendance_policy":"CSE attendance policy: minimum 75% per theory course and 100% per lab. Condonation for 65-74% with medical certificate submitted within 3 working days. Below 65% = detained, must repeat the semester. Faculty must submit monthly attendance to the Dean Academics office by the 3rd of every month. Attendance is digital via ERP portal.",
"exam_scheme":"MU CSE exam scheme 2024: Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC 2cr = 50 (30+10+10). Practical 1cr = 50 (25+15+10). Practical 2cr = 100 (50+40+10). PBL = 50 (EST 25 + CET 25). End-semester 3 hours. Main exams December (odd) and May-June (even). Morning shift 10 AM-1 PM, evening 2-5 PM. Controller of Examinations issues timetable (e.g. MU/Exam/CE/Notice/505).",
"grading_system":"MU 8-point grading: 90-100 O (10), 80-89 A+ (9), 70-79 A (8), 60-69 B+ (7), 50-59 B (6), 45-49 C (5), 40-44 P (4), below 40 F (0). SGPA = credit-weighted average per semester. CGPA = cumulative. Percentage = 10 × CGPA. CGPA > 8 = First Division with Distinction; 6.5-8 = First Division; below 6.5 = Second Division.",
"passing_rules":"To pass a theory course: minimum 40% in End Sem Test AND 40% overall. For practical: 50% overall. Failure = grade F + backlog. Backlog cleared in next Main Examination. Re-evaluation: Rs 800 per paper within 7 days of result. Photocopy of answer script: Rs 400 within 7 days. Unfair means (mobile in hall, copying) = zero in that paper.",
"syllabus_overview":"B.Tech CSE 4-year 8-semester at MU. Three branches: CSE, CSE (AI), CSE (BCT Blockchain). Sem I: Engineering Math-I (24CSE0100 T), Programming in C (24CSE0300 T), Physics→Electronics (24CSE0200 T), Indian Traditions (24CSE0400 T HSMC), Computer Fundamentals (24CSE0500 T), Web Dev-I, PBL-I. Sem II: Math-II, Python, Web Dev-II, Digital Circuits. Sem III: Discrete Math (24CSE1500 T), Data Structures (24CSE1600 T), OOP Java (24CSE1700 T), RDBMS (24CSE1800 T), Universal Human Values, Intro ML (24CSE1902 T). Sem IV: Theory of Computation (CSE160), Algorithms (CSE170), OS (CSE280), Electives. Sem V/VI: Computer Architecture (CSE310), Networks (CSE190), Compiler Design (CSE340), Deep Learning (CSE766), Data Mining (CSE776), Cyber Security (CSE900), Software Engg (CSE330), Android (CSE910), NLP (CSE781), Distributed Systems (CSE891). Sem VIII: Entrepreneurship, E-Commerce, Cyber Law, Blockchain/ML, Big Data.",
"semester1_courses":"Semester I CSE (2024 scheme, 22 credits, 800 max marks): Theory — Intro to Engineering Math-I (24CSE0100 T, BSC, 100), Physics→Electronics (24CSE0200 T, 100), Programming in C (24CSE0300 T, 100), Indian Traditions (24CSE0400 T, HSMC, 50), Computer Fundamentals (24CSE0500 T, 100). Practicals — Physics (24CSE0200 P, 50), C Programming (24CSE0300 P, 50), Indian Traditions (24CSE0400 P, 50), Comp Fundamentals (24CSE0500 P, 50), Web Dev-I (24CSE0600 P, 100), PBL-I (24CSE0700 P, 50). L-T-P: 11-4-14.",
"semester2_courses":"Semester II CSE (22 credits, 800 marks): Theory — Engineering Math-II (24CSE0800 T), Communication Skills & Professionalism (24CSE0900 T, HSMC, 50), Problem Solving & Python (24CSE1000 T), Web Dev-II (24CSE1100 T), Digital Circuits & Systems (24CSE1200 T). Practicals — Comm Skills, Python, Web Dev-II, Digital Circuits, Environmental Sciences (24CSE1300 P, 100, 4cr), PBL-II (24CSE1400 P).",
"lab_rules":"CSE Lab Rules SAMPLE/CSE/LAB/01. Timings: Mon-Fri 9-12 AM & 1-4 PM. Lunch 12-1. Sat 9-1 for make-up. Batch max 20 students, fixed computer per semester. 15 min late = absent. Max 2 missed labs allowed with HOD permission. Lab record submitted next session, late = -2 marks/week. Conduct: University ID + closed footwear mandatory. No food/drinks. Phones silent. Pen drives scanned before use. Log out and shutdown before leaving. Safety: fire extinguisher at entrance, electric shock → power off first. Penalties: first = 5 marks deducted, repeat = 1 week ban + parents informed. Network misuse → Discipline Committee.",
"internship_rules":"CSE Internship Rules (SAMPLE). Summer Internship-I after Sem IV: 4 weeks (160 hours), 2 credits. Summer Internship-II after Sem VI: 6 weeks (240 hours), 2 credits. Domain must be CSE-related. Online internships allowed if from registered company. Unpaid OK, student bears travel. Permission: offer letter + HOD-signed NOC to Training & Placement Cell 7 days before start. Attendance 90% minimum, max 2 leave days. Weekly log signed by industry mentor. Certificate on company letterhead with dates. Report 25-40 pages A4 TNR 12 spacing 1.5. Similarity below 20%. Submit within 10 days of college reopening. Evaluation: mentor 30 + report 30 + presentation/viva 40 = 100. Pass 50/100.",
"project_guidelines":"CSE Project Guidelines (2024). PBL-I (24CSE0700 P, Sem I): 50 marks = EST 25 + CET 25, 2 hr/week, no mid-sem. PBL-II (24CSE1400 P, Sem II): same. Major Project: final year, teams of 3-4 students under one faculty guide. Synopsis 3 pages (title, problem, objectives, methodology, timeline) within 2 weeks. Weekly guide meetings with signed diary. 3 review presentations per semester. Missing review = -10 internal marks. Evaluation: Phase-I Internal 60 + External 40 = 100. Phase-II Internal 100 + External 100 = 200. Report: A4, TNR 12 body / 14 bold headings, spacing 1.5. Margins left 1.5\", others 1\". IEEE references. 60-100 pages. Chapters: Introduction, Literature Review, Problem Statement, Methodology & Design, Implementation, Results & Discussion, Conclusion & Future Scope. Plagiarism below 15%, single source below 2%. Late = 5 marks/day up to 5 days.",
"plagiarism_policy":"MU CSE Plagiarism Policy. Similarity index above 15% in project reports triggers academic integrity review by department committee. Single source similarity must be below 2%. For assignments: above 25% similarity = 0 marks. Second offence: rustication from course. Tools used: Turnitin. Committee: HOD, senior faculty, one external. Appeal to Dean within 7 days. Re-evaluation: Rs 800 per paper, within 7 days. Photocopy of answer sheet: Rs 400 within 7 days.",
"faculty_info":"CSE Department MU Faculty: Dr. Sharma (HOD, Algorithms, ML) — office C-201, Mon-Fri 10-12. Dr. Kapoor (Networks, Cyber Security). Dr. Patel (DBMS, Data Mining). Dr. Reddy (Compiler Design, Theory of Computation). Dr. Joshi (Software Engineering, Android). Lab in-charge: Mr. Gupta (Lab 1, 2, 3). Office hours: 2 hr/week minimum per faculty, posted on department board. Faculty meeting: 1st Saturday, 11 AM, Seminar Hall C-101.",
"branch_info":"CSE branches at MU: (1) CSE — core computer science, (2) CSE (AI) — Artificial Intelligence track with ML, DL, NLP, computer vision, (3) CSE (BCT) — Blockchain Technology track with cryptography, distributed ledgers, smart contracts. All three share first 3 semesters; specialization from semester 4. Semester III differentiation: CSE takes Introduction to ML (24CSE1902 T), CSE (AI) takes Fundamentals of Artificial Intelligence, CSE (BCT) takes Introduction to Blockchain.",
},
"ece": {
"attendance_policy":"ECE attendance policy identical to CSE: 75% minimum per theory, 100% per lab. Condonation 65-74% with medical certificate. Below 65% = detained. Monthly attendance submitted to Dean Academics by the 3rd.",
"exam_scheme":"MU ECE Exam Scheme 2024. Theory 3-4cr = 100 (60+30+10). HSMC 2cr = 50 (30+10+10). Practical 1cr = 50 (25+15+10). Practical 2cr = 100 (50+40+10). PBL = 50 (25+25). Exams December & May-June, 3-hr papers. Morning 10-1, evening 2-5. Passing: 40% EST + 40% overall for theory, 50% for practical. 80 ECE courses in pass marks register. Theory: min EST 24/100 or 12/50; min overall 40/100 or 20/50. Practical: min overall 25/50 or 50/100.",
"grading_system":"Same 8-point scale as all MU programmes: O, A+, A, B+, B, C, P, F. CGPA = weighted average. % = 10 × CGPA.",
"syllabus_overview":"B.Tech ECE 4-year at MU. Course codes 24ECExxxx T (theory) or P (practical). Sem I: Math-I (24ECE0100 T), Physics→Electronics, Programming in C, Indian Traditions, Basic Electrical Engg, Electronic Workshop & PCB Design (24ECE0600 P, 4cr), PBL-I. Sem II: Math-II, Comm Skills, Python, Circuit Theory (24ECE1100 T ESC), Digital Circuits (24ECE1200 T ESC), Environmental Sciences. Sem III: Probability & Stats (24ECE1500 T), Electronic Devices & Circuits (24ECE1600 T PCC), Signals & Systems (24ECE1700 T), Digital System Design (24ECE1800 T), OOP C++ (24ECE1900 T). Sem IV: EM Field Theory (24ECE2000 T), Analog Circuits (24ECE2100 T), Analog Communication (24ECE2200 T), Linear Control Systems (24ECE2300 T), Microprocessors (24ECE2400 T). Sem V: DSP, Digital Communication, VLSI Design, Embedded Systems, Antenna. Sem VI: Microwave Engg, Info Theory & Coding, Computer Networks, Wireless Communication, Elective. Sem VII: Optical Fiber, Satellite, IoT, Management (24ECE3900 T 50), Tech Report Writing (24ECE4000 T 50), Summer Internship-II, Major Project Phase-I. Sem VIII: Radar & Navigation, Electives, Major Project II (24ECE4900 P 200), Comprehensive Viva (100), Seminar (50).",
"lab_rules":"ECE Lab Rules SAMPLE/ECE/LAB/01. Timings: Mon-Fri 9-12, 1-4. Sat 9-1 make-up. Max 20/batch, fixed bench. 15 min late = absent. ID + closed footwear. Never work on live circuits above 36 V without faculty present. Never look into powered waveguide/horn antenna. 80 lab registers: Electronic Devices (Mon 9-12), Digital Electronics (Tue 9-12), Analog Circuits (Wed 9-12), Microprocessor (Thu 9-12), Communication (Fri 9-12), VLSI (Mon 1-4), DSP (Tue 1-4), Microwave (Wed 1-4), Embedded (Thu 1-4), Computer Networks (Fri 1-4). Each lab-specific: check polarity before power, don't hot-swap ICs, set function generator amplitude minimum before connecting, don't exceed op-amp rating, don't unplug kit while downloading, disconnect USB before rewiring, don't change switch settings without approval.",
"internship_rules":"ECE Internship Rules (SAMPLE). Summer Internship-I after Sem IV: 4 wk (160 hr), 2 credits. Summer Internship-II after Sem VI: 6 wk (240 hr), 2 credits. Domain: embedded systems, VLSI, telecom, IoT, signal processing. Online allowed from registered company. Permission: offer letter + HOD-signed NOC ≥ 7 days before. Attendance 90%, max 2 leave days. Report 25-40 pages A4 TNR 12 spacing 1.5, similarity < 20%. Evaluation: mentor 30 + report 30 + presentation 40 = 100. Internship types with credits: Industrial Training (8 wk, 3cr), In-plant (2 wk, 1cr, 85% att), Research (8 wk, 3cr), Online (4 wk, 1cr, 85%), Startup (6 wk, 2cr). Six-month possible in final sem replacing Major Project Phase-II if CGPA ≥ 7.0.",
"project_guidelines":"ECE Project Guidelines 2024. PBL-I (Sem I, 1 credit, 50 marks = EST 25 + CET 25). PBL-II (Sem II). Category EP. ECE projects: sensor circuits, amplifiers, microcontroller applications, PCB design. Major Project final year teams 3-4 under one guide. Synopsis 3 pages within 2 weeks. Evaluation Phase-I 100 (60+40), Phase-II 200 (100+100). Report A4 TNR 12/14, spacing 1.5, IEEE refs, 60-100 pages. Chapters same as CSE. Similarity < 15%, single source < 2%. Late = 5 marks/day up to 5 days. 4 hardbound copies + PDF. Viva: guide + 1 internal + 1 external, 15 min + 10 min Q&A.",
"plagiarism_policy":"Same as CSE: similarity above 15% triggers review. Single source below 2%. Committee action as per university policy.",
"faculty_info":"ECE Department MU: Dr. Menon (HOD, DSP, Signal Processing) — office E-301. Dr. Iyer (VLSI, Analog Circuits). Dr. Nair (Embedded Systems, IoT). Dr. Khan (Microwave, Antenna). Dr. Bose (Communication Systems). Lab in-charge: Mr. Rao (ECE Labs 1-5). Office hours posted on department board.",
"key_labs":"ECE Labs at MU: Electronic Devices Lab (basic diode/transistor/BJT circuits), Digital Electronics Lab (gates, flip-flops, counters, trainer kits), Analog Circuits Lab (op-amps, filters, oscillators), Microprocessor Lab (8085/8086, ARM kits), Communication Lab (AM/FM modulators, demodulators), VLSI Lab (Cadence, EDA workstations, FPGA boards), DSP Lab (MATLAB/Simulink, DSP kits), Microwave Lab (klystron, Gunn diode, waveguide), Embedded Systems Lab (Arduino/STM32, sensors), Computer Networks Lab (Cisco switches, routers, crimping tools).",
},
"finance": {
"fee_structure":"MU Fee Structure 2025-26. B.Tech per year: Tuition Rs 1,10,000 + Development Rs 15,000 + Lab Rs 10,000 + Library Rs 3,000 + Sports Rs 2,000 + Exam Rs 5,000 = Rs 1,45,000. Hostel AC Rs 85,000/year incl mess, non-AC Rs 65,000, mess-only Rs 35,000. One-time: Admission Rs 10,000, Caution deposit Rs 5,000 (refundable), Alumni Rs 2,000. Payment schedule: 50% at admission, 30% by 15 Nov, 20% by 15 Mar. Late payment Rs 500/week up to 4 weeks; beyond 4 weeks hall ticket withheld.",
"late_fee_policy":"Late fee: Rs 500 per week beyond due date, up to 4 weeks. After 4 weeks: hall ticket withheld from semester exams. After 8 weeks: student ID suspended, no library/lab access. After 12 weeks: case referred to Dean, possible rustication. Waiver: one-time waiver possible on genuine financial hardship with documented proof (income certificate, medical bills) submitted to Finance Officer. Approval chain: HOD → Finance Officer → Registrar → VC for amounts > Rs 10,000.",
"refund_policy":"MU Fee Refund (UGC-compliant). Before 15 days of semester start: 100% refund. Within 15 days: 90%. Within 30 days: 80%. After 30 days: 50%. After 60 days: NIL. Caution deposit refundable at graduation minus dues. Application: refund form + fee receipt + admission cancellation letter + bank details. Processing: 30 working days. Deductions: library dues, hostel dues, exam fees if already paid.",
"scholarship_rules":"MU Scholarships: Merit (CGPA ≥ 9.0 for two consecutive sems) = 25% tuition waiver; CGPA 8.0-8.9 = 15%. Government: MP Post-Matric (SC/ST/OBC income < 6 lakh), Central Sector (top 20 percentile, income < 8 lakh), PMSSS (J&K), Minority scholarship. Sports: state-level 1st/2nd = 15%; national = 25%. Staff ward: 50% tuition waiver. Sibling: 10% for second child. Special: COVID-affected, ex-servicemen ward, single-girl-child = 5%. Application by 30 September. Renewal: CGPA ≥ 8.0 + 75% attendance.",
"student_fee_records":"Student Fee Records (15 students, Finance view). S001 Rahul Kumar, CSE III, 24CSE001 — Total 1,45,000; Paid 1,10,000; Pending 35,000; PARTIAL. S002 Priya Sharma, CSE III, 24CSE002 — CLEARED. S003 Amit Patel, CSE III, 24CSE003 — 72,500 pending; 12 days OVERDUE; hall ticket withheld. S004 Sneha Iyer, CSE III, 24CSE004 — CLEARED (merit 25%). S005 Vikram Singh, ECE III, 24ECE001 — 15,000 pending; PARTIAL. S006 Ananya Reddy, ECE III, 24ECE002 — CLEARED (sports 15%). S007 Karan Mehta, ECE III, 24ECE003 — 1,45,000 pending; 25 days OVERDUE; CRITICAL. S008 Divya Nair, ECE III, 24ECE004 — CLEARED. S009 Arjun Verma, CSE V, 23CSE021 — CLEARED (staff ward 50%). S010 Riya Gupta, CSE V, 23CSE022 — 45,000 pending; 5 days overdue. S011 Mohit Sharma, ECE V, 23ECE015 — CLEARED. S012 Neha Joshi, ECE V, 23ECE016 — 25,000 pending; PARTIAL. S013 Rohit Kumar, CSE VII, 22CSE010 — CLEARED. S014 Pooja Rao, ECE VII, 22ECE008 — 55,000 pending; 8 days overdue. S015 Sameer Khan, CSE VII, 22CSE011 — CLEARED (sibling 10%). Total pending ≈ Rs 2,62,500.",
"budget_approval":"MU Budget & Expense Approval. Departments submit annual budget (April): salary, equipment, consumables, travel, workshops, misc. Finance consolidates; Dean Academics reviews; VC approves by 30 May. Purchase approval tiers: up to Rs 25,000 = HOD; Rs 25,001-1,00,000 = HOD + Finance Officer; Rs 1,00,001-5,00,000 = Dean + Finance Officer + Registrar; Rs 5,00,001-25,00,000 = Purchase Committee (3 members); above Rs 25,00,000 = VC + tender. Vendor payment: invoice + GRN + PO, verified within 5 working days, payment within 30 days. Travel: MP govt TA/DA rates, submit within 15 days. Emergency up to Rs 10,000 via HOD imprest.",
"vendor_rules":"MU Vendor & Procurement. Registration: GST + PAN + bank details + ITR last 2 years + no-blacklisting certificate. Valid 3 years, renewable. Rate contract: annual tender for common items (stationery, consumables, printing), L1 selected. Limited tender: 3+ quotations for Rs 1-5 lakh. Open tender > Rs 5 lakh published on website + newspaper. GeM portal mandatory for available items. PO issued by Finance after approval: delivery date, inspection clause, penalty 1%/week for delay up to 10%. Inspection: 3-member committee for > Rs 50,000. Payment 30 days after GRN + invoice. Advance up to 30% with bank guarantee. Blacklisting: fraud, repeated quality failure, delay > 30 days = 2 years banned.",
"hostel_fees":"Hostel fees 2025-26: AC Rs 85,000/year (includes 3 meals/day + Wi-Fi + laundry + security). Non-AC Rs 65,000/year. Mess-only (day scholars) Rs 35,000/year. Room types: single (AC Rs 1,20,000/year), twin-sharing (Rs 85,000 AC / Rs 65,000 non-AC), triple (Rs 55,000 non-AC only). One-time: Hostel admission Rs 5,000, Caution Rs 3,000 (refundable). Late fee Rs 500/week. Room allotment priority: distance from home (higher priority for > 200 km), sports quota, staff ward, first-come. Cancellation: refund per UGC policy. Hostel warden: Mr. Singh (Block A-B), Ms. Sharma (Block C-D).",
"exam_fees":"Exam fees per semester: Regular Rs 5,000 (already in tuition). Backlog paper Rs 500 per paper. Re-evaluation Rs 800 per paper (within 7 days of result). Photocopy of answer script Rs 400 per paper (within 7 days). Special exam (improvement) Rs 1,000 per paper. Late registration Rs 1,000 extra. Payment via ERP portal only. Refundable only if exam cancelled by university.",
"payment_methods":"MU payment methods: (1) NEFT/RTGS to university account — details on portal, (2) UPI to official university VPA, (3) Online portal (ERP login → Fees → Pay Now), (4) Demand Draft in favour of 'Mandsaur University' payable at Mandsaur, (5) Cash up to Rs 20,000 at Finance Office. Receipts issued instantly for online, within 24 hr for offline. Always keep receipt for records. No cash acceptance after 4 PM. Cheque not accepted.",
"financial_aid":"MU Financial Aid. Education loan: university has tie-up with SBI, Bank of Baroda, HDFC Credila. Documents: admission letter, fee structure, ITR of parents, bank statements. Interest subsidy available for family income < Rs 4.5 lakh (Central Sector Interest Subsidy Scheme). Emergency aid fund: Rs 25,000 max per student per year for medical/family crisis, apply to Dean Students with proof. Work-study: 20 hr/week positions in library/lab for Rs 5,000/month, apply through HOD. Alumni scholarship: Rs 20,000/year for 10 students, merit + need based.",
"tax_compliance":"Tax compliance: University is registered under Section 12A of Income Tax. Donations eligible for 80G. TDS on vendor payments above Rs 30,000 per transaction or Rs 1 lakh aggregate = 2% (194C) or 10% (194J). GST applied on non-academic services. Fee receipts show break-up. Student can claim education loan interest deduction under Section 80E. Faculty HRA/LTA per IT rules. Annual audit by chartered accountant.",
},
"admin": {
"leave_policy":"MU Leave Policy. Faculty: Casual 12/year, Earned 15/year (accumulable up to 300 days), Medical 10/year, Maternity 180 days, Paternity 15 days, Sabbatical up to 1 year after 6 years. Approval: HOD → Registrar → VC (for > 15 days). Encashment: earned leave up to 300 days at retirement. Public holidays: MP govt calendar + foundation day + festivals. WFH: only with Dean approval for special cases. Late arrival: > 15 min = half-day; > 3 in a month = parent/staff notification.",
"grievance_redressal":"MU Grievance & Discipline. Faculty grievance: submit to Grievance Cell. Acknowledge within 24 hr, resolve 7 days (routine) / 15 days (complex). Levels: Dept Coordinator → HOD → Dean → Grievance Committee → VC. Antiragging: zero tolerance, investigation within 48 hr. Sexual harassment: ICC per POSH Act 2013, anonymous complaints allowed, 90-day investigation. Disciplinary actions: warning, fine, suspension (up to 1 sem), rustication, FIR. Appeals to VC within 15 days, VC decision final. Counselling: free psychologist 3 days/week.",
"event_approval":"MU Event Approval. Types: technical (hackathon, workshop), cultural (fest, competitions), sports, seminars, industry visits. Application ≥ 30 days before: event form + purpose + budget + schedule + guests + safety plan to Dean Student Welfare. Approval chain: Dean SW → HOD → Registrar → VC (> Rs 50,000 or outsiders involved). Budget: max Rs 25,000 from dept fund; above requires sponsor or VC approval. Venue booking via Facilities Office. Permissions: Police NOC (> 500 attendees), Fire NOC, Municipal for outdoor. Safety: 2 faculty + 5 volunteers as marshals, first aid kit, ambulance for > 500. Documentation: photos + report + accounts settled within 15 days. Post-event AAR to Dean within 30 days.",
"staff_recruitment":"MU Faculty & Staff Recruitment. Cadres: Assistant Professor (60% in PG), Associate Professor (5+ years experience), Professor (10+ years with PhD). Advertisement: university website + national newspaper + Employment News, 2 weeks open. Screening: Eligibility Committee checks qualifications, shortlists 5x openings. Demo (15 min) + Interview (30 min). Panel: HOD + senior faculty + Dean + external expert (for senior posts). Selection → Selection Committee → VC approval → offer letter. Non-teaching: Test + Interview. Docs: ID, address, certificates, experience letters, medical fitness, police verification. Probation 1 year (teaching) / 6 months (non-teaching). Contract faculty up to 11 months. Reservation per MP govt norms.",
"student_records":"Student Records (15 students, Admin view). S001 Rahul Kumar CSE III 24CSE001 — 78% att, 5 casual leave, 2 medical, no discipline, Hostel A-214. S002 Priya Sharma CSE III 24CSE002 — 92%, 2 CL, day scholar. S003 Amit Patel CSE III 24CSE003 — 64% DETAINED, warning 5 Oct, Hostel B-108. S004 Sneha Iyer CSE III 24CSE004 — 88%, 3 CL, A-305. S005 Vikram Singh ECE III 24ECE001 — 82%, 4 CL + 1 ML, C-012. S006 Ananya Reddy ECE III 24ECE002 — 95%, 6 OD (basketball team), C-220. S007 Karan Mehta ECE III 24ECE003 — 71%, warning 20 Sep (5 late arrivals), D-117. S008 Divya Nair ECE III 24ECE004 — 89%, 3 CL, D-045. S009 Arjun Verma CSE V 23CSE021 — 84%, 5 CL + 1 ML, A-402. S010 Riya Gupta CSE V 23CSE022 — 76%, 6 CL, day scholar. S011 Mohit Sharma ECE V 23ECE015 — 91%, 2 CL, class rep, B-214. S012 Neha Joshi ECE V 23ECE016 — 87%, 4 CL, B-311. S013 Rohit Kumar CSE VII 22CSE010 — 80%, 3 CL, TCS offer, day. S014 Pooja Rao ECE VII 22ECE008 — 73%, warning 1 Oct (3 late arrivals), C-118. S015 Sameer Khan CSE VII 22CSE011 — 88%, 2 CL, A-510. Summary: avg attendance 83.2%, 1 detained (S003), 3 warnings (S003/S007/S014), 1 on-duty (S006).",
"timetable_reference":"Faculty Timetable. Dr. Sharma (CSE): Mon 9-11 Data Structures (Sem III, R201), Tue 10-12 Operating Systems (Sem V, Lab 1), Wed 2-4 Algorithms Tutorial (Sem IV, R104), Thu 9-11 Project Guidance (office), Fri 11-1 Compiler Design (Sem VI, R305). Dr. Menon (ECE): Mon 10-12 DSP (Sem V, Lab 2), Tue 9-11 VLSI Design (Sem V, R401), Wed 9-11 Analog Circuits (Sem IV, Lab 3), Thu 2-4 Project Guidance, Fri 10-12 Microwave Engineering (Sem VI, R302). Exam duty: 20% of faculty per exam. Office hours: 2 hr/week minimum per faculty. Monthly faculty meeting: 1st Saturday, 11 AM, Seminar Hall.",
"disciplinary_procedure":"MU Disciplinary Procedure. Levels: verbal warning → written warning → fine (up to Rs 5,000) → suspension (up to 1 semester) → rustication → FIR. Infractions and consequences: (a) Late arrival 3x in month = parent notification; (b) Misconduct in lab = 5 marks deducted, repeat = 1 week ban; (c) Plagiarism = 0 marks, second offence = course failure; (d) Mobile in exam hall = unfair means, zero in paper; (e) Ragging = immediate suspension + police complaint; (f) Substance abuse = rustication + FIR; (g) Harassment (POSH) = ICC investigation, up to expulsion. Appeal to Dean within 7 days. Second appeal to VC within 15 days. Right to representation: student may bring one witness.",
"attendance_rules":"MU Attendance Rules. Minimum 75% per course. 65-74% = condonation possible with medical certificate. Below 65% = detained, must repeat semester. Medical certificate must be from registered practitioner, submitted within 3 working days. Leave without approval = absence (double count for exams). Faculty must submit attendance to Dean Academics by 3rd of every month. Digital attendance via ERP portal. Parents notified for students below 75%. OD (on-duty) leaves: sports, cultural, NCC, NSS events count as present. Late arrival > 15 min = half-day.",
"hostel_rules":"MU Hostel Rules. Check-in: 9 AM-6 PM on allotted dates. Room allotment: priority to distance from home, sports quota, staff ward. Rules: entry gate closes 10 PM (girls), 11 PM (boys). Late entry with written permission + warden sign. Visitors allowed in visitor lounge only, 4-7 PM. No opposite-gender entry into rooms. No alcohol, smoking, drugs = immediate rustication. Quiet hours: 10 PM-7 AM. Room inspection weekly by warden. Damage to furniture/equipment = billed to student. Common areas cleaned by students on rotation. Mess timings: breakfast 7-9, lunch 12:30-2, dinner 7:30-9. Complaint register at warden office. Emergency contacts posted at every floor.",
"transport_rules":"MU Transport. Buses: 12 routes covering Mandsaur city + 30 km radius. Timings: morning pickup 7-8 AM, evening drop 4:30-5:30 PM. Fees: Rs 15,000/year (up to 10 km), Rs 20,000/year (10-20 km), Rs 25,000/year (> 20 km). Bus pass issued on fee payment. Lost pass: Rs 200 replacement. Rules: no standing when bus moving, no shouting, seat belts mandatory, boarding only at designated stops. Driver contact posted on bus. Delay > 15 min call transport office. Emergency: 24x7 helpline.",
"library_rules":"MU Library. Timings: Mon-Sat 8 AM-9 PM, Sun 10 AM-5 PM. During exams: 7 AM-11 PM. Membership: all students + faculty automatically. Borrowing: students 3 books for 14 days; faculty 10 books for 30 days. Late: Rs 5/day/book. Reference books: not issued, in-library only. Journals: current issue not issued. E-resources: IEEE, Springer, ACM, DELNET access via campus Wi-Fi. Silent zone enforced. Mobile silent mode. No food/drink. Photocopy: Rs 2/page (students), free for faculty. Lost book: cost + 20% processing. Book bank: for SC/ST/EWS students, free books for semester. Inter-library loan via DELNET.",
"exam_duty_rules":"MU Exam Duty Rules. Faculty assigned 20% of exams per session. Duty: invigilation (3 hr), evaluation (per paper Rs 15/sheet), paper setting (Rs 800 per paper), moderation (Rs 500 per paper), viva (Rs 500/day). Roster by CoE office. Absence: inform CoE 48 hr before, else penalty Rs 1,000. On-duty pass not applicable during exams. Substitutes: only with CoE approval. Malpractice handling: confiscate evidence, sign by 2 invigilators, student's statement recorded, report to CoE within 2 hr. Blind evaluation: paper codes only, no student name. Confidential: question papers transported in sealed envelope, opened 10 min before exam in presence of 2 faculty. Re-evaluation: faculty appointed by CoE, no name disclosed.",
"semester_calendar":"MU Academic Calendar 2025-26. Odd semester (Jul-Dec): Commencement 15 Jul, Mid-sem exam 16-22 Sep, End-sem 1-20 Dec. Winter break 21 Dec-5 Jan. Even semester (Jan-Jun): Commencement 6 Jan, Mid-sem 3-10 Mar, End-sem 20 May-10 Jun. Summer break 11 Jun-14 Jul. Practical exams: 1 week before theory end-sem. Fee deadlines: 50% at admission, 30% by 15 Nov, 20% by 15 Mar. Result declaration: 45 days after last exam. Re-evaluation window: 7 days from result. Semester break projects: PBL-II submission. Festivals: Techfest (Feb), Cultural fest (Mar), Sports week (Oct).",
"accreditation_info":"MU Accreditations: NAAC A+ (2023), NBA accreditation for B.Tech CSE and ECE (2024-2027), AICTE approved, UGC 2(f) and 12(B) recognized, ISO 9001:2015 certified. Ranking: NIRF band 151-200 in Engineering (2024). Alumni network: 12,000+ across 25 countries. Placement: 92% for CSE, 78% for ECE (2024 batch). Recruiters: TCS, Infosys, Wipro, Cognizant, Capgemini, L&T, Persistent, Zensar, plus 100+ others. MoU with 30+ companies for internships.",
},
}

# ==================== MODEL ====================
@st.cache_resource(show_spinner="⏳ Loading AI models (first run ~60s)…")
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try: coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count() == 0:
            keys = list(docs.keys()); texts = list(docs.values())
            keys += ["_secret"]; texts += [DEPARTMENTS[dept]["secret"]]
            vecs = emb.encode(texts, normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys], embeddings=vecs, documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    _ = mdl.generate(**tok("hi", return_tensors="pt").to(mdl.device), max_new_tokens=1, pad_token_id=tok.eos_token_id)
    return emb, cc, tok, mdl, DEV
emb, cc, tok, mdl, DEV = setup()

def get_coll(d):
    try: return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}", metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r = USERS.get(u)
    if r and r["pw"] == _hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept, query):
    if not dept or dept not in DEPARTMENTS: return None, None
    q = query.lower()
    for other, cfg in DEPARTMENTS.items():
        if other == dept: continue
        for alias in cfg.get("aliases",[]) + [other]:
            if re.search(rf"\b{re.escape(alias)}\b", q): return other, cfg["name"]
    return None, None

def chunk_text(t, size=500, ov=60):
    parts = re.split(r'(?<=[.!?])\s+', t.strip()); out, cur = [], ""
    for s in parts:
        if len(cur)+len(s)+1 <= size: cur = (cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur = ((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf(pdf_bytes, fname):
    tmp = f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    open(tmp,"wb").write(pdf_bytes)
    pages, err = [], ""
    try:
        doc = pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t = pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err = f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    t = pg.extract_text() or ""
                    if t.strip(): pages.append({"text":t,"page":i+1})
            err = ""
        except Exception as e: err += f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages, err

def ingest_pdf(dept, fname, pdf_bytes, dt="regulation"):
    pages, err = extract_pdf(pdf_bytes, fname)
    if not pages: return 0, (err or "No extractable text"), 0
    coll = get_coll(dept); ch, mt, ids = [], [], []
    safe = _safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":dt})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    if not ch: return 0, "0 chunks", len(pages)
    try:
        allv = []
        for i in range(0,len(ch),32):
            allv.extend(emb.encode(ch[i:i+32], normalize_embeddings=True, show_progress_bar=False).tolist())
        coll.add(ids=ids, embeddings=allv, documents=ch, metadatas=mt)
    except Exception as e: return 0, f"Chroma:{e}", len(pages)
    try: open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}","wb").write(pdf_bytes)
    except: pass
    backup_chroma(); return len(ch), "", len(pages)

def list_sources(dept):
    try:
        d = get_coll(dept).get(include=["metadatas"]); c={}
        for m in d["metadatas"]:
            s = m.get("source","?"); c[s] = c.get(s,0)+1
        return c
    except: return {}

def retrieve(dept, q, k=5):
    c = get_coll(dept)
    if c.count() == 0: return []
    qv = emb.encode([q], normalize_embeddings=True).tolist()
    r = c.query(query_embeddings=qv, n_results=min(k,c.count()),
                include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

if "stop_event" not in st.session_state: st.session_state.stop_event = threading.Event()

def _gen(msgs, mx=450):
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    th = threading.Thread(target=mdl.generate,
        kwargs=dict(**inp, max_new_tokens=mx, do_sample=False,
                    pad_token_id=tok.eos_token_id, streamer=streamer))
    th.start(); buf = ""
    for t in streamer:
        if st.session_state.stop_event.is_set(): break
        buf += t; yield buf

def stream_answer(dept, q, ph, extra="", deep=False):
    hits = retrieve(dept, q, k=5)
    parts = [f"[{h['source']} p{h['page']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0, f"[ATTACHED]\n{extra[:3000]}")
    ctx = "\n".join(parts) or "(no reference)"
    sys_msg = ("You are a helpful faculty assistant for a college department. "
               "Answer using ONLY the reference material below. Cite sources like [source_name, page N]. "
               "For fees/records list exact numbers. Be precise and complete.")
    if deep:
        think = [{"role":"system","content":"Analyze step by step briefly. Identify key facts from reference."},
                 {"role":"user","content":f"Q: {q}\n\nReference:\n{ctx}\n\nAnalyze briefly."}]
        tb = ""
        for b in _gen(think, 220): tb = b
        st.session_state.last_think = tb
        ans_msgs = [{"role":"system","content":sys_msg},
                    {"role":"user","content":f"Q: {q}\n\nReference:\n{ctx}\n\nAnalysis:\n{tb}\n\nFinal answer:"}]
    else:
        ans_msgs = [{"role":"system","content":sys_msg},
                    {"role":"user","content":f"Q: {q}\n\nReference:\n{ctx}\n\nAnswer:"}]
    final = ""
    for b in _gen(ans_msgs, 550):
        final = b; ph.markdown(b + ("▌" if not st.session_state.stop_event.is_set() else "⏹️"))
    ph.markdown(final); return final, hits

def make_pdf(title, dname, subtitle, body):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=0.8*inch, rightMargin=0.8*inch,
                            topMargin=0.7*inch, bottomMargin=0.7*inch)
    S = getSampleStyleSheet()
    h1 = ParagraphStyle("H1", parent=S["Heading1"], alignment=1, fontSize=16, spaceAfter=6)
    sb = ParagraphStyle("S", parent=S["Normal"], alignment=1, fontSize=10, spaceAfter=4)
    bd = ParagraphStyle("B", parent=S["Normal"], fontSize=11, leading=16, spaceAfter=8)
    story = [Paragraph(f"<b>{title}</b>", h1), Paragraph(f"Department of {dname}", sb)]
    if subtitle: story.append(Paragraph(subtitle, sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s, bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

def _slug(s): return re.sub(r'[^a-z0-9_]+','_', s.lower()).strip('_')[:40]
def is_doc_request(q):
    ql = q.lower()
    return any(t in ql for t in ["prepare pdf","make pdf","generate pdf","create pdf","as pdf",
        "question paper","exam paper","make question","generate questions","download pdf",
        "give me pdf","generate notes","make notes","prepare notes","report pdf"])

def init_audit():
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (id INTEGER PRIMARY KEY AUTOINCREMENT,
        ts TEXT, user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c = sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",(datetime.now().isoformat(),u,r or "-",d or "-",a,det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None, limit=200):
    c = sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else: c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows = c.fetchall(); c.connection.close(); return rows

# ==================== UI ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide", initial_sidebar_state="collapsed")

st.markdown("""
<style>
.block-container { padding-top: 1.5rem; max-width: 1200px; }
.hero { background: linear-gradient(135deg, #0a0e27 0%, #111a3e 45%, #1e3a8a 100%);
        padding: 50px 40px 45px; border-radius: 22px; color: white; position: relative; overflow: hidden;
        box-shadow: 0 22px 60px -22px rgba(30,64,175,0.55); }
.hero::before { content:''; position:absolute; inset:0;
        background: radial-gradient(circle at 15% 30%, rgba(124,58,237,0.35) 0%, transparent 40%),
                    radial-gradient(circle at 85% 70%, rgba(59,130,246,0.35) 0%, transparent 40%); }
.hero h1 { font-size: 3rem; font-weight: 800; letter-spacing: -1.5px; margin: 0 0 12px 0;
        background: linear-gradient(90deg,#fff,#a5b4fc,#c4b5fd); -webkit-background-clip: text;
        -webkit-text-fill-color: transparent; position: relative; }
.hero p { font-size: 1.05rem; opacity: 0.94; margin: 0; position: relative; }

.hero-visual { background: linear-gradient(135deg, #1e40af15, #7c3aed15);
        border-radius: 20px; padding: 30px; text-align: center;
        border: 2px dashed #c7d2fe; height: 100%; display: flex;
        flex-direction: column; justify-content: center; align-items: center; }
.hero-visual .big-icon { font-size: 4.5rem; margin-bottom: 14px;
        animation: float 3s ease-in-out infinite; }
@keyframes float { 0%,100%{transform:translateY(0)} 50%{transform:translateY(-10px)} }
.hero-visual .caption { color: #475569; font-size: 0.92rem; }

.feat { background: linear-gradient(180deg,#fff,#f8fafc); border-radius: 14px;
        padding: 22px 18px; border: 1px solid #e2e8f0; height: 160px;
        transition: transform 0.15s, box-shadow 0.15s; position: relative; overflow: hidden; }
.feat:hover { transform: translateY(-3px); box-shadow: 0 14px 32px -14px rgba(30,64,175,0.3); }
.feat::before { content:''; position:absolute; top:0; left:0; width:5px; height:100%;
        background: linear-gradient(180deg,#2563eb,#7c3aed); border-radius: 14px 0 0 14px; }
.feat .icn { font-size: 2rem; margin-bottom: 10px; }
.feat h4 { margin: 0 0 6px 0; color: #0f172a; font-size: 0.98rem; font-weight: 700; }
.feat p { margin: 0; color: #64748b; font-size: 0.82rem; line-height: 1.5; }

.dept-tile { padding: 18px; border-radius: 14px; margin-bottom: 10px;
        transition: transform 0.15s; position: relative; overflow: hidden; }
.dept-tile:hover { transform: translateY(-2px); }
.dept-tile .d-icon { font-size: 1.8rem; margin-bottom: 6px; }
.dept-tile .d-name { font-weight: 700; font-size: 0.95rem; color: #0f172a; }
.dept-tile .d-code { font-size: 0.7rem; font-weight: 700; letter-spacing: 1px; text-transform: uppercase; }
.dept-tile .d-stats { font-size: 0.78rem; color: #475569; margin-top: 8px; }

.pill { display:inline-block; padding:4px 12px; border-radius:100px;
        font-size:0.68rem; font-weight:700; text-transform:uppercase; letter-spacing:0.5px; }
.p-admin { background:#fef3c7; color:#92400e; }
.p-teacher { background:#dbeafe; color:#1e40af; }
.metric { background: linear-gradient(135deg,#fff,#f8fafc); border-radius: 12px;
        padding: 16px 18px; border: 1px solid #e2e8f0; }
.metric .lbl { font-size: 0.7rem; color: #64748b; text-transform: uppercase; }
.metric .val { font-size: 1.65rem; font-weight: 800; color: #0f172a; margin: 4px 0; }
.metric .sub { font-size: 0.75rem; color: #64748b; }
.attach-badge { background:#ecfdf5; border:1px solid #86efac; border-radius:8px;
        padding:8px 12px; font-size:0.82rem; color:#166534; margin:6px 0; }
.empty-state { text-align:center; padding:45px 20px; color:#94a3b8; }
.empty-state .icon { font-size:3.2rem; margin-bottom:12px; }
.empty-state .title { font-size:1.05rem; color:#64748b; }
</style>
""", unsafe_allow_html=True)

for k in ("user","name","role","dept","logged_in"): st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx = ""
if "ctx_name" not in st.session_state: st.session_state.ctx_name = ""
if "deep_think" not in st.session_state: st.session_state.deep_think = False
if "qa_trigger" not in st.session_state: st.session_state.qa_trigger = None

# ---------- HOME ----------
if not st.session_state.logged_in:
    st.markdown("""<div class="hero">
      <h1>🎓 VaultMind ERP</h1>
      <p><b>Faculty AI Workbench for Mandsaur University</b><br>
      4 departments · Local Qwen2.5 · 60+ policy documents · Zero cloud</p>
    </div>""", unsafe_allow_html=True)
    st.write("")
    # Hero visual + login side by side
    cVis, cLog = st.columns([3, 2])
    with cVis:
        st.markdown("""<div class="hero-visual">
          <div class="big-icon">🏛️</div>
          <div style="font-size:1.15rem;font-weight:700;color:#1e293b;margin-bottom:8px;">One AI per department</div>
          <div class="caption">CSE · ECE · Finance · Admin/HR<br>
          Each isolated. Each with its own documents.<br>
          Exam papers · Notes · Student records · Fee queries — all in one chat.</div>
        </div>""", unsafe_allow_html=True)
    with cLog:
        st.markdown("### 🔐 Faculty Sign in")
        u = st.text_input("Username", placeholder="teacher_cse · teacher_fin · admin", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("🔓 Login", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True})
                st.session_state.chat_history = []; st.session_state.chat_ctx = ""
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
        st.markdown("**🎫 Demo accounts**")
        st.code("admin         / admin_pass\nteacher_cse   / t_cse\nteacher_ece   / t_ece\nteacher_fin   / t_fin\nteacher_admin / t_admin", language="text")

    st.write("")
    c1,c2,c3,c4 = st.columns(4)
    feats = [
      ("🏛️","Department Isolation","Separate ChromaDB per dept. Faculty of one dept can never read another's data."),
      ("📄","PDF Deliverables","Ask 'Create a question paper' → download appears in the chat automatically."),
      ("🧠","Deep Think Mode","Model reasons first, then answers. Best for complex fee/record queries."),
      ("📎","Attach PDFs & Images","Upload a scanned syllabus or photo — AI extracts and answers with citations."),
    ]
    for col,(icn,t,d) in zip([c1,c2,c3,c4], feats):
        col.markdown(f'<div class="feat"><div class="icn">{icn}</div><h4>{t}</h4><p>{d}</p></div>', unsafe_allow_html=True)

    st.write(""); st.markdown("### 🏛️ Departments covered")
    dcols = st.columns(len(DEPARTMENTS))
    for col,(code,cfg) in zip(dcols, DEPARTMENTS.items()):
        col.markdown(f"""<div class="dept-tile" style="background:linear-gradient(135deg,{cfg['color']}18,{cfg['color']}08);border-left:4px solid {cfg['color']};">
          <div class="d-icon">{cfg.get('icon','📁')}</div>
          <div class="d-code" style="color:{cfg['color']};">{code}</div>
          <div class="d-name">{cfg['name']}</div>
        </div>""", unsafe_allow_html=True)
    st.stop()

# ---------- MAIN ----------
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"; name = st.session_state.name
pill = {"admin":"p-admin","teacher":"p-teacher"}[role]
dept_color = DEPARTMENTS.get(dept,{}).get("color","#64748b") if dept else "#64748b"
valid_dept = dept if (dept and dept in DEPARTMENTS) else None

with st.sidebar:
    st.markdown(f"""
    <div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#f1f5f9,#e2e8f0);text-align:center;">
      <div style="font-size:1.5rem;">🎓</div>
      <div style="font-weight:700;color:#0f172a;margin-top:6px;">{name}</div>
      <div style="margin-top:8px;"><span class="pill {pill}">{role}</span></div>
      {f'<div style="margin-top:10px;font-size:0.85rem;color:#475569;"><b style="color:{dept_color};">●</b> {safe_dept_name(dept)}</div>' if valid_dept else ''}
    </div>""", unsafe_allow_html=True)
    if valid_dept:
        try: st.metric("📊 KB chunks", get_coll(dept).count())
        except: pass
    st.caption("🔒 0 external API calls · 💾 Drive-backed")
    if st.button("🚪 Logout", use_container_width=True):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in"): st.session_state[k] = None
        st.session_state.chat_history = []; st.session_state.chat_ctx = ""; st.rerun()

if is_admin:
    T = st.tabs(["🏠 Home","📊 Dashboard","🏛️ Departments","📤 Upload RAG","👥 Faculty","📜 Audit"])
    tab_home, tab_dash, tab_dept, tab_pdf, tab_users, tab_audit = T; tab_chat = None
else:
    T = st.tabs(["💬 AI Assistant","🏠 Home","📚 KB Documents"])
    tab_chat, tab_home, tab_kb = T
    tab_dash = tab_dept = tab_pdf = tab_users = tab_audit = None

if not is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div style="background:linear-gradient(135deg,#eff6ff,#f5f3ff);border-radius:14px;padding:22px 26px;border:1px solid #c7d2fe;margin-bottom:20px;">
          <h2 style="margin:0 0 6px 0;color:#1e293b;font-size:1.35rem;">👋 Welcome, {name}!</h2>
          <p style="margin:0;color:#475569;font-size:0.92rem;">You are in the <b>{safe_dept_name(dept)}</b> workspace.
          Use the <b>AI Assistant</b> tab for exam papers, notes, fee lookups, student records, and more.</p>
        </div>""", unsafe_allow_html=True)
        c1,c2,c3 = st.columns(3)
        srcs = list_sources(dept) if valid_dept else {}
        c1.markdown(f'<div class="metric"><div class="lbl">Department</div><div class="val" style="font-size:1.05rem;">{safe_dept_name(dept)}</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Documents</div><div class="val">{len(srcs)}</div><div class="sub">in KB</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">AI Device</div><div class="val">{DEV.upper()}</div><div class="sub">local</div></div>', unsafe_allow_html=True)

if is_admin and tab_home:
    with tab_home:
        st.markdown(f"""<div style="background:linear-gradient(135deg,#eff6ff,#f5f3ff);border-radius:14px;padding:22px 26px;border:1px solid #c7d2fe;margin-bottom:20px;">
          <h2 style="margin:0 0 6px 0;color:#1e293b;font-size:1.35rem;">👋 Welcome, {name}!</h2>
          <p style="margin:0;color:#475569;font-size:0.92rem;">Admin access to all 4 departments.</p>
        </div>""", unsafe_allow_html=True)
        st.info("💡 Admin has no department. To use the AI chat, log in as any teacher account.")

if is_admin and tab_dash:
    with tab_dash:
        c1,c2,c3,c4 = st.columns(4)
        total = sum(get_coll(d).count() for d in DEPARTMENTS)
        blocked = sum(1 for r in get_audit(limit=500) if r[4] == "BLOCKED_QUERY")
        c1.markdown(f'<div class="metric"><div class="lbl">Departments</div><div class="val">{len(DEPARTMENTS)}</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric"><div class="lbl">Faculty</div><div class="val">{len([u for u in USERS.values() if u["role"]=="teacher"])}</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="metric"><div class="lbl">KB chunks</div><div class="val">{total}</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="metric"><div class="lbl">Security</div><div class="val">0</div><div class="sub">{blocked} blocked · 0 leaks</div></div>', unsafe_allow_html=True)
        st.write(""); st.markdown("### 🏛️ Departments")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            n = get_coll(code).count(); srcs = list_sources(code)
            us = sum(1 for u in USERS.values() if u.get("dept")==code)
            col.markdown(f"""<div class="dept-tile" style="background:linear-gradient(135deg,{cfg['color']}18,{cfg['color']}08);border-left:4px solid {cfg['color']};">
              <div class="d-icon">{cfg.get('icon','📁')}</div>
              <div class="d-code" style="color:{cfg['color']};">{code}</div>
              <div class="d-name">{cfg['name']}</div>
              <div class="d-stats">📚 {n} chunks<br>📄 {len(srcs)} docs<br>👥 {us} faculty</div>
            </div>""", unsafe_allow_html=True)

# ============ CHAT ============
if tab_chat is not None:
    with tab_chat:
        st.markdown(f"### 💬 {safe_dept_name(dept)} — AI Assistant")
        st.caption(f"Signed in as **{name}** · 🔒 Answers come only from your department's documents")

        c1, c2 = st.columns([1,4])
        with c1:
            st.session_state.deep_think = st.toggle("🧠 Deep Think", value=st.session_state.deep_think,
                                                     help="Model reasons first, then answers")
        with c2: st.markdown("**⚡ Quick actions** — fill topic below, then click")
        qa_cols = st.columns(5)
        for col,(k,icn,lbl) in zip(qa_cols, [("exam","📝","Exam Paper"),("notes","📓","Notes"),
                                             ("quiz","🧠","Quiz"),("sum","📄","Summarise"),
                                             ("explain","💡","Explain")]):
            with col:
                if st.button(f"{icn} {lbl}", key=f"qa_{k}", use_container_width=True):
                    st.session_state.qa_trigger = k

        topic = st.text_input("Topic / subject (optional)", key="chat_topic",
                              placeholder="e.g. Digital Signal Processing · Fee refund rules · Student S003 status")

        with st.expander("📎 Attach a PDF or image (AI will use as context)", expanded=False):
            c1, c2 = st.columns(2)
            with c1:
                f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
                if f1 and st.button("Load PDF", key="btn_pdf"):
                    pages, err = extract_pdf(f1.getvalue(), f1.name)
                    if pages:
                        st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                        st.session_state.ctx_name = f1.name
                        st.success(f"✅ {len(pages)} pages loaded")
                    else: st.error(f"❌ {err}")
            with c2:
                f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
                if f2 and st.button("OCR image", key="btn_ocr"):
                    tmp = f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                    open(tmp,"wb").write(f2.getvalue())
                    try:
                        txt = pytesseract.image_to_string(Image.open(tmp))
                        st.session_state.chat_ctx = txt; st.session_state.ctx_name = f2.name
                        st.success(f"✅ {len(txt)} chars extracted")
                    except Exception as e: st.error(f"❌ {e}")
                    finally:
                        try: os.remove(tmp)
                        except: pass
        if st.session_state.chat_ctx:
            cA, cB = st.columns([4,1])
            with cA:
                st.markdown(f'<div class="attach-badge">📎 Attached: <b>{st.session_state.ctx_name}</b> ({len(st.session_state.chat_ctx)} chars)</div>', unsafe_allow_html=True)
            with cB:
                if st.button("🗑️ Clear file", key="clr_file"):
                    st.session_state.chat_ctx = ""; st.session_state.ctx_name = ""; st.rerun()

        box = st.container(height=520)
        with box:
            if not st.session_state.chat_history:
                st.markdown(f"""<div class="empty-state">
                  <div class="icon">💬</div>
                  <div class="title">Ask anything about <b style="color:{dept_color};">{safe_dept_name(dept)}</b></div>
                  <div class="title" style="font-size:0.85rem;margin-top:8px;">Try: <i>"Create a question paper on Digital Electronics"</i> → get a PDF</div>
                </div>""", unsafe_allow_html=True)
            for i, m in enumerate(st.session_state.chat_history):
                with st.chat_message(m["role"]):
                    st.markdown(m["content"])
                    if m.get("thinking"):
                        with st.expander("🧠 Reasoning (deep think)"): st.caption(m["thinking"])
                    if m.get("sources"):
                        with st.expander(f"📄 {len(m['sources'])} source(s)"):
                            for s in m["sources"]:
                                st.caption(f"📄 `{s['source']}` · page {s['page']} · similarity {s['similarity']:.2f}")
                    if m.get("pdf") and m.get("pdf_name"):
                        st.download_button(f"📥 Download {m['pdf_name']}", m["pdf"],
                            file_name=m["pdf_name"], mime="application/pdf", key=f"dl_{i}")

        uq = st.chat_input("Ask a question, or ask for a PDF deliverable…")

        if st.session_state.qa_trigger:
            k = st.session_state.qa_trigger; st.session_state.qa_trigger = None
            prompts = {
              "exam": f"Prepare a question paper PDF. Subject: {topic or safe_dept_name(dept)}. 10 questions, Sections A/B/C, marks shown.",
              "notes": f"Generate study notes PDF on: {topic or 'full course'}. Headings, bullets, key formulas.",
              "quiz": f"Create a 5-question MCQ quiz PDF on: {topic or 'this course'}. Options A-D, answer marked.",
              "sum": f"Summarise key rules about: {topic or 'all department policy'}. Bullet points.",
              "explain": f"Explain in simple language for faculty: {topic or 'the fundamentals'}. Include example.",
            }
            uq = prompts[k]

        if uq:
            st.session_state.stop_event.clear()
            with box:
                with st.chat_message("user"): st.markdown(uq)
            st.session_state.chat_history.append({"role":"user","content":uq})
            other, on = detect_cross_dept(dept, uq)
            if other and other != dept:
                err = f"🚫 **Access denied.** You are scoped to **{safe_dept_name(dept)}**. Cannot query **{on}**."
                st.session_state.chat_history.append({"role":"assistant","content":err})
                with box:
                    with st.chat_message("assistant"): st.error(err)
                log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
            else:
                with box:
                    with st.chat_message("assistant"):
                        ph = st.empty(); stop_ph = st.empty()
                        stop_ph.button("⏹️ Stop", key=f"stop_{int(time.time())}",
                                       on_click=lambda: st.session_state.stop_event.set())
                        ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx,
                                                  deep=st.session_state.deep_think)
                        stop_ph.empty()
                        pdf_bytes = None; pdf_name = None
                        if is_doc_request(uq) or ans.count("Q1") + ans.count("1.") >= 3:
                            title = "Examination Question Paper" if "question" in uq.lower() else "Document"
                            body = ans.replace("**","").replace("##","").replace("*","")
                            pdf_bytes = make_pdf(f"{title} — {topic or safe_dept_name(dept)}",
                                                  safe_dept_name(dept),
                                                  f"Generated by VaultMind AI · {datetime.now().strftime('%d %b %Y %H:%M')}",
                                                  body)
                            pdf_name = f"{_slug(title)}_{_slug(topic or dept)}.pdf"
                            st.success("📄 **PDF ready** — click to download")
                            st.download_button(f"📥 Download {pdf_name}", pdf_bytes,
                                file_name=pdf_name, mime="application/pdf", key=f"dl_new_{int(time.time())}")
                        if hits: st.caption(f"📄 {len(hits)} source(s) cited")
                st.session_state.chat_history.append({
                    "role":"assistant","content":ans,
                    "sources":[{"source":h["source"],"page":h["page"],"similarity":h["similarity"]} for h in hits],
                    "pdf": pdf_bytes, "pdf_name": pdf_name,
                    "thinking": st.session_state.get("last_think","") if st.session_state.deep_think else ""})
                log(user,role,dept,"QUERY",uq[:60])
            st.rerun()

        if st.session_state.chat_history:
            cA, cB, cC = st.columns([1,1,4])
            with cA:
                if st.button("🗑️ Clear chat", use_container_width=True):
                    st.session_state.chat_history = []; st.rerun()

if not is_admin and tab_kb:
    with tab_kb:
        st.markdown(f"### 📚 Knowledge Base — {safe_dept_name(dept)}")
        srcs = list_sources(dept) if valid_dept else {}
        if srcs:
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"- 📄 `{s}` — **{cnt}** chunks")
            st.metric("Total chunks", get_coll(dept).count())
        else: st.info("No documents yet.")

if is_admin and tab_dept:
    with tab_dept:
        st.markdown("### 🏛️ Departments")
        with st.expander("➕ Create department"):
            c1, c2 = st.columns(2)
            with c1:
                code = st.text_input("Code", key="nd_code").strip().lower()
                name2 = st.text_input("Display name", key="nd_name")
                color = st.color_picker("Color", value="#0891b2", key="nd_color")
            with c2:
                sec = st.text_input("Secret", key="nd_sec")
                al = st.text_input("Aliases (comma-sep)", key="nd_al")
            if st.button("Create", type="primary"):
                if not code or not name2: st.error("Code + name required")
                elif code in DEPARTMENTS: st.error("Exists")
                else:
                    DEPARTMENTS[code] = {"name":name2, "secret": sec or f"{code}-secret-{int(time.time())}",
                        "color": color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code], "icon":"📁"}
                    save_depts(); get_coll(code)
                    log(user,role,None,"CREATE_DEPT",f"{code}:{name2}")
                    st.success(f"✅ {name2}"); time.sleep(1); st.rerun()
        for code, cfg in DEPARTMENTS.items():
            n = get_coll(code).count(); srcs = list_sources(code)
            with st.expander(f"**{cfg['name']}** ({code}) — {n} chunks · {len(srcs)} docs"):
                c1, c2 = st.columns([2,1])
                with c1:
                    for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                        st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
                with c2:
                    st.metric("Faculty", sum(1 for u in USERS.values() if u.get("dept")==code))
                    if st.button("🗑️ Reset KB", key=f"rst_{code}"):
                        try:
                            cc.delete_collection(f"kb_{code}"); get_coll(code)
                            log(user,role,None,"RESET_KB",code)
                            st.success("Reset"); time.sleep(1); st.rerun()
                        except Exception as e: st.error(str(e))

if is_admin and tab_pdf:
    with tab_pdf:
        st.markdown("### 📤 Upload to Department RAG")
        tgt = st.selectbox("Target department", list(DEPARTMENTS.keys()),
                           format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
        dtype = st.selectbox("Type", ["syllabus","regulation","notes","question-paper","reference","policy"], key="up_type")
        uploaded = st.file_uploader("Choose PDF(s)", type=["pdf"], accept_multiple_files=True, key="up_files")
        if uploaded and st.button("🚀 Ingest into RAG", type="primary", use_container_width=True):
            total = 0; ok = 0
            for f in uploaded:
                with st.spinner(f"Processing {f.name}…"):
                    chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                    if err: st.error(f"❌ {f.name} — {err}")
                    else:
                        st.success(f"✅ {f.name} — {pages} pages → {chunks} chunks")
                        total += chunks; ok += 1
            if ok:
                st.balloons()
                st.info(f"🎉 {ok}/{len(uploaded)} files · **{total} chunks** into **{DEPARTMENTS[tgt]['name']}**")
                log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                time.sleep(2); st.rerun()

if is_admin and tab_users:
    with tab_users:
        st.markdown("### 👥 Faculty Management")
        with st.expander("➕ Add faculty"):
            c1, c2 = st.columns(2)
            with c1:
                nu = st.text_input("Username", key="cr_u").strip().lower()
                np = st.text_input("Password", type="password", key="cr_p")
                nn = st.text_input("Full name + designation", key="cr_n")
            with c2:
                nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                                  format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
            if st.button("Create faculty account", type="primary"):
                if not nu or not np or not nn: st.error("All fields required")
                elif nu in USERS: st.error("Exists")
                else:
                    USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                    save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                    st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
        st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"} for u,v in USERS.items()]),
                     use_container_width=True, hide_index=True)

if is_admin and tab_audit:
    with tab_audit:
        st.markdown("### 📜 Audit Log")
        fd = st.selectbox("Filter", ["ALL"] + list(DEPARTMENTS.keys()), key="au_d")
        rows = get_audit(dept=None if fd == "ALL" else fd, limit=200)
        if rows:
            df = pd.DataFrame(rows, columns=["Time","User","Role","Dept","Action","Detail"])
            df["Time"] = pd.to_datetime(df["Time"]).dt.strftime("%H:%M:%S")
            st.dataframe(df, use_container_width=True, hide_index=True, height=400)
        st.divider()
        allr = get_audit(limit=1000)
        c1,c2,c3 = st.columns(3)
        c1.metric("🔒 Blocked", sum(1 for r in allr if r[4]=="BLOCKED_QUERY"))
        c2.metric("✅ Leaks", 0); c3.metric("📊 Actions", len(allr))
'''
open("/content/vaultmind_erp.py","w").write(APP)
print(f"✅ App v7 written ({len(APP)} bytes)")
shutil.copy("/content/vaultmind_erp.py", f"{DRIVE}/vaultmind_erp.py")

print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip(): print(f"✅ Streamlit PID {r.stdout.strip()}")
else:
    print("❌ Failed:"); print(open("/content/streamlit.log").read()[-1500:]); raise SystemExit(1)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v7 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse   (CSE)                              ║
║     teacher_ece   / t_ece   (ECE)                              ║
║     teacher_fin   / t_fin   (Finance Officer)                  ║
║     teacher_admin / t_admin (Registrar)                        ║
║                                                                ║
║  📊 DATASETS LOADED (12 docs per department):                  ║
║     CSE     → syllabus, exam scheme, labs, internships, etc.   ║
║     ECE     → same + lab register                              ║
║     Finance → fees, refunds, scholarships, vendors, 15 records ║
║     Admin   → leave, discipline, events, timetable, 15 records ║
║                                                                ║
║  ✨ FEATURES:                                                  ║
║     • Home page with hero illustration + login side-by-side    ║
║     • Ask "Create question paper" → PDF download in chat       ║
║     • 🧠 Deep Think toggle                                     ║
║     • ⏹️ Stop button during generation                         ║
║     • 📎 Attach PDF/image for context                          ║
║     • Fixed KeyError on admin login                            ║
╚════════════════════════════════════════════════════════════════╝
""")

📦 Installing…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Installed
✅ App v7 written (68357 bytes)
🚀 Launching…
✅ Streamlit PID 34513

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP v7 — LIVE                                    ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse   (CSE)                              ║
║     teacher_ece   / t_ece   (ECE)                              ║
║     teacher_fin   / t_fin   (Finance Officer)                  ║
║     teacher_admin / t_admin (Registrar)                        ║
║                       

In [ ]:
# ==================== VAULTMIND ERP v8 — INSTALL ====================
import subprocess, sys, os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)

# Clean stale users
uf = f"{DRIVE}/data/users.json"
if os.path.exists(uf): os.remove(uf)

print("📦 Installing premium stack…")
!pip install -q streamlit chromadb sentence-transformers "transformers>=4.45" \
    accelerate bitsandbytes python-docx pymupdf pytesseract pillow reportlab \
    pdfplumber pandas plotly openpyxl 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Ready")

# ==================== APP ====================
APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil
from datetime import datetime, timedelta
import streamlit as st, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf, plotly.express as px, plotly.graph_objects as go
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

DRIVE = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS  = f"{DRIVE}/data/depts.json"
CFG_USERS  = f"{DRIVE}/data/users.json"
AUDIT_DB   = f"{DRIVE}/data/audit.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR = f"{DRIVE}/uploads"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s): return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True)
    except: pass

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except: pass

# ==================== DEPARTMENTS ====================
PALETTE = ["#6366f1","#8b5cf6","#10b981","#f59e0b","#06b6d4"]
DEFAULT_DEPTS = {
  "cse":{"name":"Computer Science","color":"#6366f1","icon":"💻","secret":"aurora-lattice",
         "aliases":["cse","computer","comp sci","computer science"]},
  "ece":{"name":"Electronics","color":"#8b5cf6","icon":"📡","secret":"cobalt-meridian",
         "aliases":["ece","electronics","communication"]},
  "finance":{"name":"Finance & Accounts","color":"#10b981","icon":"💰","secret":"ledger-falcon",
             "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":{"name":"Administration","color":"#f59e0b","icon":"🏛️","secret":"orbit-vermillion",
           "aliases":["admin","administration","hr","registrar"]},
}
try: DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
for code, cfg in DEFAULT_DEPTS.items(): DEPARTMENTS.setdefault(code, cfg)
for i, (c, cfg) in enumerate(DEPARTMENTS.items()):
    cfg.setdefault("name", c.upper()); cfg.setdefault("secret", f"{c}-s-{abs(hash(c))%9999}")
    cfg.setdefault("color", PALETTE[i%len(PALETTE)]); cfg.setdefault("icon","📁")
    cfg.setdefault("aliases",[c])
json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def save_depts(): json.dump(DEPARTMENTS, open(CFG_DEPTS,"w"), indent=2)
def dname(d): return DEPARTMENTS.get(d,{}).get("name", d or "—")
def dcolor(d): return DEPARTMENTS.get(d,{}).get("color", "#64748b")

# ==================== USERS ====================
DEFAULT_USERS = {
  "admin":{"name":"System Administrator","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":{"name":"Dr. Sharma","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":{"name":"Dr. Menon","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":{"name":"Mr. Verma","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin":{"name":"Ms. Rao","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
try:
    saved = json.load(open(CFG_USERS))
    USERS = DEFAULT_USERS.copy()
    for k,v in saved.items():
        if k not in USERS and v.get("role")!="student": USERS[k]=v
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS, open(CFG_USERS,"w"), indent=2)
def save_users(): json.dump(USERS, open(CFG_USERS,"w"), indent=2)

# ==================== SEED DATA ====================
SEED = {
"cse":{
  "syllabus":"B.Tech CSE MU 4yr/8sem. 3 branches: CSE, CSE(AI), CSE(BCT). Sem I: Engg Math-I (24CSE0100T 4cr), Programming in C, Physics→Electronics, Indian Traditions (HSMC), Comp Fund, Web Dev-I, PBL-I. Sem II: Math-II, Python (24CSE1000T), Web Dev-II, Digital Circuits. Sem III: Discrete Math (24CSE1500T), Data Structures (24CSE1600T), OOP Java (24CSE1700T), RDBMS (24CSE1800T), ML Intro (24CSE1902T). Sem IV: Theory of Computation (CSE160), Algorithms (CSE170), OS (CSE280). Sem V-VI: Computer Architecture, Networks (CSE190), Compiler (CSE340), Deep Learning (CSE766), Data Mining (CSE776), Cyber Security (CSE900), Software Engg (CSE330), Android (CSE910), NLP (CSE781), Distributed (CSE891). Sem VIII: Entrepreneurship, E-Commerce, Cyber Law, Blockchain/ML, Big Data.",
  "exam_scheme":"MU 2024 scheme. Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC 2cr = 50. Practical 1cr = 50, 2cr = 100. PBL = 50 (25+25). Exams Dec/May-Jun, 3hr papers, morning 10-1, evening 2-5. Passing: 40% EST + 40% overall for theory; 50% practical.",
  "grading":"8-point scale: 90+ O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). SGPA = credit-weighted avg. CGPA > 8 = First Division with Distinction.",
  "attendance":"Minimum 75% per theory course, 100% per lab. Condonation 65-74% with medical certificate within 3 days. Below 65% = detained. Monthly attendance to Dean Academics by 3rd. Late > 15 min = half-day.",
  "lab_rules":"CSE Labs Mon-Fri 9-12, 1-4. Sat 9-1 make-up. Batch max 20, fixed computer. 15 min late = absent. Max 2 missed labs. Record next session, late -2 marks/week. ID + closed footwear. No food/drinks. Phones silent. Pen drives scanned. Fire → power off, exit. Electric shock → power off first. Penalty: 5 marks first violation, repeat = 1 week ban + parents.",
  "internship":"Summer-I after Sem IV: 4wk (160hr), 2cr. Summer-II after Sem VI: 6wk (240hr), 2cr. Domain CSE. Online if registered company. Offer + HOD-signed NOC ≥ 7 days before. 90% attendance. Report 25-40 pages TNR 12 spacing 1.5, similarity <20%. Eval: mentor 30 + report 30 + presentation 40. Pass 50/100. Six-month final sem if CGPA≥7.0.",
  "project":"PBL-I/II 1cr each, 50 marks = 25+25. Major Project final year teams 3-4. Synopsis 3 pages within 2 weeks. 3 reviews/sem. Eval Phase-I 100, Phase-II 200. Report A4 TNR 12, IEEE refs, 60-100 pages. Plagiarism <15%, single <2%. Late 5 marks/day. 4 hardbound. Viva: guide + 1 int + 1 ext.",
  "plagiarism":"Similarity >15% triggers review. Single source >2% flagged. Assignments >25% = 0 marks. Second offence = rustication. Tools: Turnitin. Appeal to Dean within 7 days.",
},
"ece":{
  "syllabus":"B.Tech ECE MU 4yr/8sem. Sem III: Probability & Stats (24ECE1500T), Electronic Devices & Circuits (24ECE1600T), Signals & Systems (24ECE1700T), Digital System Design (24ECE1800T), OOP C++ (24ECE1900T). Sem IV: EM Field Theory (24ECE2000T), Analog Circuits (24ECE2100T), Analog Comm (24ECE2200T), Linear Control (24ECE2300T), Microprocessors (24ECE2400T). Sem V: DSP, Digital Comm, VLSI Design, Embedded Systems, Antenna. Sem VI: Microwave Engg, Info Theory, Computer Networks, Wireless. Sem VII: Optical Fiber, Satellite, IoT, Management (50), Tech Report (50). Sem VIII: Radar & Navigation, Electives, Major Project II (200), Viva (100), Seminar (50).",
  "exam_scheme":"Same as CSE. 80 ECE courses in pass marks register. Theory min EST 24/100 (12/50), min overall 40/100 (20/50). Practical min overall 25/50 or 50/100.",
  "lab_rules":"ECE Labs same timings as CSE. 80 lab registers: Electronic Devices (Mon 9-12), Digital Elec (Tue 9-12), Analog Circuits (Wed 9-12), Microprocessor (Thu 9-12), Communication (Fri 9-12), VLSI (Mon 1-4), DSP (Tue 1-4), Microwave (Wed 1-4), Embedded (Thu 1-4), Computer Networks (Fri 1-4). Never above 36V without faculty. Never look into powered waveguide/horn. Check polarity before power. Don't hot-swap ICs.",
  "internship":"Same as CSE. Domain: embedded, VLSI, telecom, IoT. Six-month final sem: CGPA≥7.0, no backlog. Types: Industrial Training (8wk,3cr), In-plant (2wk,1cr,85%), Research (8wk,3cr), Online (4wk,1cr,85%), Startup (6wk,2cr).",
  "project":"Same format as CSE. ECE projects: sensor circuits, amplifiers, microcontroller apps, PCB design.",
  "labs_detail":"10 ECE labs with specific rules: (1) Electronic Devices — check diode/transistor polarity before power; (2) Digital Electronics — check IC pin orientation; (3) Analog Circuits — set function generator amplitude minimum; (4) Microprocessor — don't unplug kit while downloading; (5) Communication — keep RF trainers away from water; (6) VLSI — logout EDA workstation, no personal software; (7) DSP — save every 15 min, no personal installs; (8) Microwave — never look into powered waveguide; (9) Embedded — disconnect USB before rewiring; (10) Computer Networks — don't change switch settings without approval.",
  "attendance":"Same as CSE: 75% min, condonation 65-74%, <65% detained.",
},
"finance":{
  "fee_structure":"B.Tech/year 2025-26: Tuition 1,10,000 + Development 15,000 + Lab 10,000 + Library 3,000 + Sports 2,000 + Exam 5,000 = 1,45,000. Hostel AC 85,000, non-AC 65,000, mess-only 35,000. One-time: Admission 10,000, Caution 5,000 (refundable), Alumni 2,000. Payment 50/30/20 (admission/15 Nov/15 Mar).",
  "late_fee":"Rs 500/week late up to 4 weeks. Beyond 4 weeks: hall ticket withheld. After 8 weeks: ID suspended. After 12 weeks: case to Dean. Waiver on documented financial hardship, approval chain HOD→Finance→Registrar→VC (>10K).",
  "refund":"UGC-compliant: 100% before 15 days of semester start, 90% within 15, 80% within 30, 50% after 30, NIL after 60. Caution refundable at graduation minus dues. Processing 30 working days.",
  "scholarship":"Merit CGPA≥9.0 → 25% waiver; 8.0-8.9 → 15%. MP Post-Matric, Central Sector, PMSSS, Minority. Sports 15-25%. Staff ward 50%. Sibling 10%. Application by 30 Sep. Renewal: CGPA≥8.0 + 75% attendance.",
  "budget":"Annual budget submission April. Approval tiers: <25K HOD; 25K-1L HOD+Finance; 1-5L Dean+Finance+Registrar; 5-25L Purchase Committee; >25L VC+tender. Vendor payment: invoice+GRN+PO, 30 days. Travel: MP TA/DA rates, submit within 15 days.",
  "vendor":"Registration: GST + PAN + bank + ITR + no-blacklist cert, 3 yrs valid. Rate contract annual L1. Limited tender 1-5L (3 quotes). Open >5L. GeM mandatory where available. PO with penalty 1%/week. Inspection 3-member for >50K. Payment 30 days after GRN. Blacklist 2 yrs for fraud.",
  "students_csv":"Roll,Name,Dept,Year,Total,Paid,Pending,DueDays,Status\n24CSE001,Rahul Kumar,CSE,III,145000,110000,35000,0,PARTIAL\n24CSE002,Priya Sharma,CSE,III,145000,145000,0,0,CLEARED\n24CSE003,Amit Patel,CSE,III,145000,72500,72500,12,OVERDUE\n24CSE004,Sneha Iyer,CSE,III,145000,145000,0,0,CLEARED\n24ECE001,Vikram Singh,ECE,III,145000,130000,15000,0,PARTIAL\n24ECE002,Ananya Reddy,ECE,III,145000,145000,0,0,CLEARED\n24ECE003,Karan Mehta,ECE,III,145000,0,145000,25,CRITICAL\n24ECE004,Divya Nair,ECE,III,145000,145000,0,0,CLEARED\n23CSE021,Arjun Verma,CSE,V,145000,145000,0,0,CLEARED\n23CSE022,Riya Gupta,CSE,V,145000,100000,45000,5,PARTIAL\n23ECE015,Mohit Sharma,ECE,V,145000,145000,0,0,CLEARED\n23ECE016,Neha Joshi,ECE,V,145000,120000,25000,0,PARTIAL\n22CSE010,Rohit Kumar,CSE,VII,145000,145000,0,0,CLEARED\n22ECE008,Pooja Rao,ECE,VII,145000,90000,55000,8,OVERDUE\n22CSE011,Sameer Khan,CSE,VII,145000,145000,0,0,CLEARED",
},
"admin":{
  "leave":"Faculty: Casual 12/yr, Earned 15/yr (accumulable 300), Medical 10/yr, Maternity 180 days, Paternity 15 days, Sabbatical after 6 yrs. Approval: HOD → Registrar → VC (>15 days). Encashment 300 days. Late >15 min = half-day; >3/month = parent notification.",
  "grievance":"Acknowledge 24hr, resolve 7-15 days. Levels: Dept Coord → HOD → Dean → Committee → VC. Antiragging: 48hr investigation. POSH: ICC, anonymous, 90 days. Appeals to VC within 15 days.",
  "events":"Application ≥30 days before. Chain: Dean SW → HOD → Registrar → VC (>50K or outsiders). Budget ≤25K from dept. Police NOC >500 attendees. 2 faculty + 5 volunteers as marshals. Settled within 15 days.",
  "recruitment":"Cadres: Asst Prof (60% PG), Assoc Prof (5+ yr), Prof (10+ yr, PhD). Ad 2 weeks. Screening 5x. Demo 15 min + interview 30 min. Probation 1yr teaching, 6mo non-teaching. Reservation MP norms.",
  "students_csv":"Roll,Name,Dept,Year,Attendance,CL,ML,OD,Discipline,Hostel\n24CSE001,Rahul Kumar,CSE,III,78,5,2,0,NONE,A-214\n24CSE002,Priya Sharma,CSE,III,92,2,0,0,NONE,DAY\n24CSE003,Amit Patel,CSE,III,64,8,0,0,WARNING,B-108\n24CSE004,Sneha Iyer,CSE,III,88,3,0,0,NONE,A-305\n24ECE001,Vikram Singh,ECE,III,82,4,1,0,NONE,C-012\n24ECE002,Ananya Reddy,ECE,III,95,0,0,6,NONE,C-220\n24ECE003,Karan Mehta,ECE,III,71,7,2,0,WARNING,D-117\n24ECE004,Divya Nair,ECE,III,89,3,0,0,NONE,D-045\n23CSE021,Arjun Verma,CSE,V,84,5,1,0,NONE,A-402\n23CSE022,Riya Gupta,CSE,V,76,6,0,0,NONE,DAY\n23ECE015,Mohit Sharma,ECE,V,91,2,0,0,NONE,B-214\n23ECE016,Neha Joshi,ECE,V,87,4,0,0,NONE,B-311\n22CSE010,Rohit Kumar,CSE,VII,80,3,0,0,NONE,DAY\n22ECE008,Pooja Rao,ECE,VII,73,5,3,0,WARNING,C-118\n22CSE011,Sameer Khan,CSE,VII,88,2,0,0,NONE,A-510",
  "timetable":"Dr. Sharma (CSE): Mon 9-11 Data Structures (SIII R201), Tue 10-12 OS (SV Lab1), Wed 2-4 Algorithms (SIV R104), Thu 9-11 Project, Fri 11-1 Compiler (SVI R305). Dr. Menon (ECE): Mon 10-12 DSP (SV Lab2), Tue 9-11 VLSI (SV R401), Wed 9-11 Analog (SIV Lab3), Thu 2-4 Project, Fri 10-12 Microwave (SVI R302). Exam duty: 20% assigned per exam.",
  "discipline":"Verbal warning → written → fine (5K) → suspension (1 sem) → rustication → FIR. Late 3x = parent notice. Lab misconduct = 5 marks. Plagiarism = 0 marks. Mobile in exam = zero. Ragging = suspension + police. Substance = rustication + FIR. Harassment = ICC, up to expulsion. Appeal Dean 7 days.",
},
}

# ==================== MODELS ====================
@st.cache_resource(show_spinner=False)
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept, docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try: coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}", metadata={"hnsw:space":"cosine"})
        if coll.count()==0:
            keys=list(docs.keys()); texts=list(docs.values())
            keys+=["_secret"]; texts+=[DEPARTMENTS[dept]["secret"]]
            vecs=emb.encode(texts,normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys],embeddings=vecs,documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    mdl=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb,device_map="auto")
    _=mdl.generate(**tok("hi",return_tensors="pt").to(mdl.device),max_new_tokens=1,pad_token_id=tok.eos_token_id)
    return emb,cc,tok,mdl,DEV
emb,cc,tok,mdl,DEV = setup()

def get_coll(d):
    try: return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}",metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r=USERS.get(u)
    if r and r["pw"]==_hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept,q):
    if not dept or dept not in DEPARTMENTS: return None,None
    ql=q.lower()
    for other,cfg in DEPARTMENTS.items():
        if other==dept: continue
        for alias in cfg.get("aliases",[])+[other]:
            if re.search(rf"\b{re.escape(alias)}\b",ql): return other,cfg["name"]
    return None,None

# ==================== RAG ====================
def chunk_text(t,size=500,ov=60):
    parts=re.split(r'(?<=[.!?])\s+',t.strip()); out,cur=[],""
    for s in parts:
        if len(cur)+len(s)+1<=size: cur=(cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur=((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf(pdf_bytes,fname):
    tmp=f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    open(tmp,"wb").write(pdf_bytes); pages=[]; err=""
    try:
        doc=pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t=pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err=f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    t=pg.extract_text() or ""
                    if t.strip(): pages.append({"text":t,"page":i+1})
            err=""
        except Exception as e: err+=f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages,err

def ingest_pdf(dept,fname,pdf_bytes,dt="regulation"):
    pages,err=extract_pdf(pdf_bytes,fname)
    if not pages: return 0,(err or "No text"),0
    coll=get_coll(dept); ch,mt,ids=[],[],[]
    safe=_safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":dt})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    try:
        allv=[]
        for i in range(0,len(ch),32):
            allv.extend(emb.encode(ch[i:i+32],normalize_embeddings=True,show_progress_bar=False).tolist())
        coll.add(ids=ids,embeddings=allv,documents=ch,metadatas=mt)
    except Exception as e: return 0,f"Chroma:{e}",len(pages)
    try: open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}","wb").write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch),"",len(pages)

def list_sources(dept):
    try:
        d=get_coll(dept).get(include=["metadatas"]); c={}
        for m in d["metadatas"]:
            s=m.get("source","?"); c[s]=c.get(s,0)+1
        return c
    except: return {}

def retrieve(dept,q,k=5):
    c=get_coll(dept)
    if c.count()==0: return []
    qv=emb.encode([q],normalize_embeddings=True).tolist()
    r=c.query(query_embeddings=qv,n_results=min(k,c.count()),
              include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0],r["metadatas"][0],r["distances"][0])]

# ==================== GENERATION ====================
if "stop_event" not in st.session_state: st.session_state.stop_event=threading.Event()

def _gen(msgs,mx=450):
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tok(text,return_tensors="pt").to(mdl.device)
    streamer=TextIteratorStreamer(tok,skip_prompt=True,skip_special_tokens=True)
    th=threading.Thread(target=mdl.generate,
        kwargs=dict(**inp,max_new_tokens=mx,do_sample=False,pad_token_id=tok.eos_token_id,streamer=streamer))
    th.start(); buf=""
    for t in streamer:
        if st.session_state.stop_event.is_set(): break
        buf+=t; yield buf

def stream_answer(dept,q,ph,extra="",deep=False,history=None):
    hits=retrieve(dept,q,k=5)
    parts=[f"[{h['source']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0,f"[USER FILE]\n{extra[:2500]}")
    ctx="\n".join(parts) or "(no reference)"
    sys="You are an expert faculty AI assistant. Answer ONLY from the reference below. Cite sources as [source_name]. Be precise. For fee/records list exact numbers."
    msgs=[{"role":"system","content":sys}]
    if history:
        for h in history[-4:]:
            if h["role"] in ("user","assistant"): msgs.append({"role":h["role"],"content":h["content"][:300]})
    if deep:
        think=[{"role":"system","content":"Analyze step by step briefly."},
               {"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalyze."}]
        tb=""
        for b in _gen(think,200): tb=b
        st.session_state.last_think=tb
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalysis:{tb}\nFinal:"})
    else:
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnswer:"})
    final=""
    for b in _gen(msgs,550):
        final=b; ph.markdown(b+("▌" if not st.session_state.stop_event.is_set() else " ⏹"))
    ph.markdown(final); return final,hits

def make_pdf(title,dept_name,subtitle,body):
    buf=io.BytesIO()
    doc=SimpleDocTemplate(buf,pagesize=A4,leftMargin=0.8*inch,rightMargin=0.8*inch,
                          topMargin=0.7*inch,bottomMargin=0.7*inch)
    S=getSampleStyleSheet()
    h1=ParagraphStyle("H1",parent=S["Heading1"],alignment=1,fontSize=16,spaceAfter=6)
    sb=ParagraphStyle("S",parent=S["Normal"],alignment=1,fontSize=10,spaceAfter=4)
    bd=ParagraphStyle("B",parent=S["Normal"],fontSize=11,leading=16,spaceAfter=8)
    story=[Paragraph(f"<b>{title}</b>",h1),Paragraph(f"Department of {dept_name}",sb)]
    if subtitle: story.append(Paragraph(subtitle,sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s=line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s,bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

def _slug(s): return re.sub(r'[^a-z0-9_]+','_',s.lower()).strip('_')[:40]

def is_doc_request(q):
    ql=q.lower()
    return any(t in ql for t in ["prepare pdf","make pdf","generate pdf","create pdf","question paper",
        "exam paper","generate questions","download pdf","generate notes","make notes","prepare notes"])

# ==================== ANALYTICS ====================
def init_audit():
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (id INTEGER PRIMARY KEY AUTOINCREMENT,
        ts TEXT, user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",(datetime.now().isoformat(),u,r or "-",d or "-",a,det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None,limit=500):
    c=sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else: c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows=c.fetchall(); c.connection.close(); return rows

def audit_df(dept=None):
    rows=get_audit(dept=dept,limit=2000)
    if not rows: return pd.DataFrame(columns=["ts","user","role","dept","action","detail"])
    df=pd.DataFrame(rows,columns=["ts","user","role","dept","action","detail"])
    df["ts"]=pd.to_datetime(df["ts"]); return df

def health_score(dept):
    """Composite 0-100 score."""
    try:
        chunks=get_coll(dept).count()
        docs=len(list_sources(dept))
        df=audit_df(dept)
        week=df[df["ts"]>datetime.now()-timedelta(days=7)] if len(df) else pd.DataFrame()
        queries=len(week[week["action"]=="QUERY"]) if len(week) else 0
        blocked=len(week[week["action"]=="BLOCKED_QUERY"]) if len(week) else 0
        s_docs=min(docs/10,1)*25
        s_chunks=min(chunks/500,1)*25
        s_usage=min(queries/50,1)*30
        s_clean=max(0,1-blocked/10)*20
        return int(s_docs+s_chunks+s_usage+s_clean)
    except: return 0

def anomalies(dept):
    """Detect anomalies from student data."""
    out=[]
    srcs=list_sources(dept)
    if any("students" in s for s in srcs):
        # Try to pull student data from seed
        dept_key=dept
        try:
            for src in srcs:
                if "student" in src.lower() and src in SEED.get(dept_key,{}):
                    csv=SEED[dept_key][src]
                    import io as _io
                    df=pd.read_csv(_io.StringIO(csv))
                    if "Attendance" in df.columns:
                        low=df[df["Attendance"]<75]
                        for _,r in low.iterrows():
                            out.append({"type":"Low attendance","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Attendance']}% — {'DETAINED' if r['Attendance']<65 else 'below 75%'}",
                                        "severity":"critical" if r["Attendance"]<65 else "warning"})
                    if "Status" in df.columns:
                        overdue=df[df["Status"].isin(["OVERDUE","CRITICAL"])]
                        for _,r in overdue.iterrows():
                            out.append({"type":"Fee overdue","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"₹{r['Pending']:,} pending for {r['DueDays']} days",
                                        "severity":"critical" if r["DueDays"]>20 else "warning"})
                    if "Discipline" in df.columns:
                        disc=df[df["Discipline"]!="NONE"]
                        for _,r in disc.iterrows():
                            out.append({"type":"Discipline","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Discipline']} recorded","severity":"warning"})
                    break
        except Exception as e: pass
    return out

def bulk_csv_upload(dept,df,kind="students"):
    """Store CSV data as searchable KB chunks."""
    if df.empty: return 0
    lines=[",".join(df.columns.tolist())]
    for _,r in df.iterrows():
        lines.append(",".join(str(x) for x in r.tolist()))
    text=f"Bulk {kind} data ({len(df)} records):\n"+"\n".join(lines)
    coll=get_coll(dept)
    chunks=chunk_text(text,800,100)
    vecs=emb.encode(chunks,normalize_embeddings=True,show_progress_bar=False).tolist()
    src_name=f"bulk_{kind}_{int(time.time())}"
    coll.add(ids=[f"{dept}_{src_name}_c{i}" for i in range(len(chunks))],
             embeddings=vecs,documents=chunks,
             metadatas=[{"source":src_name,"page":1,"doc_type":kind} for _ in chunks])
    backup_chroma()
    return len(chunks)

# ==================== CSS ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide",
                   initial_sidebar_state="expanded")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&display=swap');

html, body, [class*="css"] {
  font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
.stApp { background: #f8fafc; }
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding-top: 1rem; padding-bottom: 2rem; max-width: 1400px; }

/* Sidebar */
[data-testid="stSidebar"] {
  background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%);
  border-right: 1px solid #1e293b;
}
[data-testid="stSidebar"] * { color: #e2e8f0 !important; }
[data-testid="stSidebar"] .stButton>button {
  width: 100%; text-align: left; background: transparent; border: none;
  color: #cbd5e1 !important; padding: 10px 14px; border-radius: 10px;
  font-weight: 500; font-size: 0.9rem; transition: all 0.15s;
}
[data-testid="stSidebar"] .stButton>button:hover {
  background: rgba(99,102,241,0.15); color: #fff !important;
}
[data-testid="stSidebar"] .stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6);
  color: #fff !important; box-shadow: 0 4px 14px rgba(99,102,241,0.4);
}

/* Typography */
h1, h2, h3, h4 { color: #0f172a; font-weight: 700; letter-spacing: -0.02em; }
h1 { font-size: 1.85rem; margin: 0 0 8px 0; }
h2 { font-size: 1.4rem; margin: 0 0 12px 0; }
h3 { font-size: 1.1rem; margin: 0 0 8px 0; }

/* Buttons */
.stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6);
  border: none; color: white; font-weight: 600;
  box-shadow: 0 4px 12px rgba(99,102,241,0.3);
  transition: all 0.15s;
}
.stButton>button[kind="primary"]:hover {
  transform: translateY(-1px); box-shadow: 0 6px 18px rgba(99,102,241,0.4);
}

/* Cards */
.card { background: white; border-radius: 16px; padding: 20px;
  border: 1px solid #e2e8f0; box-shadow: 0 1px 3px rgba(0,0,0,0.03);
  transition: all 0.2s; }
.card:hover { box-shadow: 0 8px 24px rgba(15,23,42,0.08); }
.metric-card { background: white; border-radius: 14px; padding: 18px 20px;
  border: 1px solid #e2e8f0; position: relative; overflow: hidden; }
.metric-card::before { content: ''; position: absolute; top: 0; left: 0; width: 4px;
  height: 100%; background: linear-gradient(180deg, #6366f1, #8b5cf6); }
.metric-card .lbl { font-size: 0.72rem; color: #64748b; text-transform: uppercase;
  letter-spacing: 0.6px; font-weight: 600; }
.metric-card .val { font-size: 1.85rem; font-weight: 800; color: #0f172a;
  margin: 4px 0 2px; letter-spacing: -0.03em; }
.metric-card .sub { font-size: 0.78rem; color: #64748b; }

/* Pills & badges */
.pill { display: inline-flex; align-items: center; gap: 4px;
  padding: 4px 12px; border-radius: 100px; font-size: 0.7rem;
  font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px; }
.p-admin { background: #fef3c7; color: #92400e; }
.p-teacher { background: #e0e7ff; color: #3730a3; }
.p-success { background: #d1fae5; color: #065f46; }
.p-warn { background: #fef3c7; color: #92400e; }
.p-danger { background: #fee2e2; color: #991b1b; }

/* Chat */
.chat-user { background: linear-gradient(135deg, #e0e7ff, #c7d2fe); color: #1e293b;
  border-radius: 16px 16px 4px 16px; padding: 12px 16px; margin: 8px 0;
  font-size: 0.9rem; max-width: 85%; margin-left: auto; }
.chat-ai { background: white; border: 1px solid #e2e8f0;
  border-radius: 16px 16px 16px 4px; padding: 14px 18px; margin: 8px 0;
  font-size: 0.9rem; }

/* Status dot */
.dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%;
  margin-right: 6px; }
.dot-green { background: #10b981; box-shadow: 0 0 8px rgba(16,185,129,0.5); }
.dot-red { background: #ef4444; box-shadow: 0 0 8px rgba(239,68,68,0.5); }

/* Inputs */
.stTextInput>div>div>input, .stTextArea>div>textarea {
  border-radius: 10px !important; border: 1px solid #e2e8f0 !important;
  font-family: 'Inter', sans-serif !important;
}
.stTextInput>div>div>input:focus, .stTextArea>div>textarea:focus {
  border-color: #6366f1 !important;
  box-shadow: 0 0 0 3px rgba(99,102,241,0.1) !important;
}

/* Tabs */
.stTabs [data-baseweb="tab-list"] { background: #f1f5f9; border-radius: 12px;
  padding: 4px; gap: 4px; }
.stTabs [data-baseweb="tab"] { background: transparent; border-radius: 8px;
  padding: 8px 16px; font-weight: 600; font-size: 0.85rem; }
.stTabs [aria-selected="true"] { background: white !important;
  box-shadow: 0 2px 6px rgba(15,23,42,0.06); }

/* Empty state */
.empty { text-align: center; padding: 60px 20px; }
.empty .icon { font-size: 3.5rem; margin-bottom: 12px; opacity: 0.6; }
.empty .title { font-size: 1.1rem; font-weight: 600; color: #334155; }
.empty .sub { font-size: 0.88rem; color: #94a3b8; margin-top: 6px; }

/* Anomaly alert */
.alert { padding: 12px 16px; border-radius: 10px; margin: 6px 0;
  border-left: 4px solid; font-size: 0.85rem; }
.alert-critical { background: #fef2f2; border-color: #ef4444; color: #7f1d1d; }
.alert-warning { background: #fffbeb; border-color: #f59e0b; color: #78350f; }
</style>
""", unsafe_allow_html=True)

# ==================== SESSION ====================
for k in ("user","name","role","dept","logged_in","view"):
    st.session_state.setdefault(k,None)
if "chat_history" not in st.session_state: st.session_state.chat_history=[]
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx=""
if "ctx_name" not in st.session_state: st.session_state.ctx_name=""
if "deep_think" not in st.session_state: st.session_state.deep_think=False
if "qa_trigger" not in st.session_state: st.session_state.qa_trigger=None
if "view" not in st.session_state or st.session_state.view is None: st.session_state.view="Home"

# ==================== LOGIN ====================
if not st.session_state.logged_in:
    # Hero
    st.markdown("""
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 40%, #4c1d95 100%);
                padding: 60px 50px; border-radius: 24px; color: white; margin-bottom: 30px;
                position: relative; overflow: hidden;">
      <div style="position: absolute; top: -50px; right: -50px; width: 300px; height: 300px;
                  background: radial-gradient(circle, rgba(139,92,246,0.4), transparent 70%);
                  border-radius: 50%;"></div>
      <div style="position: absolute; bottom: -80px; left: -80px; width: 400px; height: 400px;
                  background: radial-gradient(circle, rgba(99,102,241,0.3), transparent 70%);
                  border-radius: 50%;"></div>
      <div style="position: relative; z-index: 1;">
        <div style="display: inline-block; padding: 6px 16px; background: rgba(255,255,255,0.1);
                    border: 1px solid rgba(255,255,255,0.15); border-radius: 100px;
                    font-size: 0.78rem; margin-bottom: 20px; letter-spacing: 1px;
                    font-weight: 600;">🎯 MU-PS-020 · SMART AUTOMATION</div>
        <h1 style="color: white; font-size: 3.2rem; font-weight: 900; margin: 0 0 12px 0;
                   letter-spacing: -0.04em; line-height: 1.1;">VaultMind ERP</h1>
        <p style="font-size: 1.15rem; opacity: 0.9; margin: 0; max-width: 700px;
                  line-height: 1.5;">The sovereign AI workbench for higher education.
                  <b>Every department isolated. Every answer cited. Zero bytes leaving the machine.</b></p>
      </div>
    </div>
    """, unsafe_allow_html=True)

    c1, c2 = st.columns([3, 2])
    with c1:
        # Feature grid
        cols = st.columns(2)
        feats = [
          ("🏛️","Department Isolation","Storage-level separation. CSE can never read Finance."),
          ("📊","Live Analytics","Track usage, health scores, and anomalies per department."),
          ("🤖","Local Qwen 2.5","1.5B model on T4. No API keys. No cloud. Ever."),
          ("🎯","AI Exam Papers","Generate full question papers → download as PDF in-chat."),
          ("🧠","Deep Think Mode","Two-stage reasoning for complex fee & record queries."),
          ("⚠️","Anomaly Detection","Auto-flags low attendance & fee defaulters."),
        ]
        for i, (ic, t, d) in enumerate(feats):
            with cols[i % 2]:
                st.markdown(f"""
                <div class="card" style="margin-bottom: 12px;">
                  <div style="font-size: 1.6rem; margin-bottom: 6px;">{ic}</div>
                  <div style="font-weight: 700; color: #0f172a; font-size: 0.95rem; margin-bottom: 4px;">{t}</div>
                  <div style="color: #64748b; font-size: 0.82rem; line-height: 1.5;">{d}</div>
                </div>
                """, unsafe_allow_html=True)
    with c2:
        st.markdown('<div class="card" style="padding: 26px;">', unsafe_allow_html=True)
        st.markdown("### 🔐 Faculty Sign In")
        st.caption("Use your university credentials")
        u = st.text_input("Username", placeholder="teacher_cse", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("Sign In →", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True,"view":"Home"})
                st.session_state.chat_history=[]; st.session_state.chat_ctx=""
                log(r["user"],r["role"],r["dept"],"LOGIN","success")
                st.rerun()
            else:
                st.error("Invalid credentials")
        st.markdown("---")
        st.markdown("**🎫 Demo Accounts**")
        st.code("""admin          / admin_pass
teacher_cse    / t_cse
teacher_ece    / t_ece
teacher_fin    / t_fin
teacher_admin  / t_admin""", language="text")
        st.markdown('</div>', unsafe_allow_html=True)
    st.stop()

# ==================== APP SHELL ====================
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
name = st.session_state.name
valid_dept = dept if (dept and dept in DEPARTMENTS) else None

# Sidebar nav
with st.sidebar:
    st.markdown(f"""
    <div style="padding: 8px 4px 20px;">
      <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 6px;">
        <div style="width: 36px; height: 36px; background: linear-gradient(135deg, #6366f1, #8b5cf6);
                    border-radius: 10px; display: flex; align-items: center;
                    justify-content: center; font-size: 1.2rem;">🎓</div>
        <div>
          <div style="font-weight: 800; font-size: 1.05rem; letter-spacing: -0.02em;">VaultMind</div>
          <div style="font-size: 0.7rem; opacity: 0.6; letter-spacing: 0.5px;">ENTERPRISE ERP</div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    # Nav items
    nav_items = []
    if is_admin:
        nav_items = [("Home","🏠 Home"),("Analytics","📊 Analytics"),
                     ("Departments","🏛️ Departments"),("Faculty","👥 Faculty"),
                     ("Uploads","📤 Upload KB"),("Audit","📜 Audit Log")]
    elif dept == "finance":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","💳 Fee Records"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    elif dept == "admin":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","🎓 Students"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    else:
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]

    for key, label in nav_items:
        btn_type = "primary" if st.session_state.view == key else "secondary"
        if st.button(label, key=f"nav_{key}", type=btn_type, use_container_width=True):
            st.session_state.view = key; st.rerun()

    st.markdown("---")
    st.markdown(f"""
    <div style="padding: 8px 4px;">
      <div style="font-size: 0.75rem; opacity: 0.6; margin-bottom: 4px;">SIGNED IN AS</div>
      <div style="font-weight: 600; font-size: 0.9rem;">{name}</div>
      <div style="font-size: 0.78rem; opacity: 0.7; margin-top: 2px;">
        <span class="pill p-{role}">{role}</span>
      </div>
      {f'<div style="margin-top: 8px; font-size: 0.78rem; color: {dcolor(dept)};"><span class="dot" style="background:{dcolor(dept)};"></span>{dname(dept)}</div>' if valid_dept else ''}
    </div>
    """, unsafe_allow_html=True)

    if st.button("🚪 Sign Out", use_container_width=True, key="logout_btn"):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in","view"):
            st.session_state[k]=None
        st.session_state.chat_history=[]; st.session_state.chat_ctx=""
        st.rerun()

# Top bar
top_left, top_right = st.columns([3,1])
with top_left:
    st.markdown(f"""
    <div style="padding: 8px 0 16px 0;">
      <div style="font-size: 0.78rem; color: #64748b; font-weight: 500; margin-bottom: 2px;">
        VaultMind / {st.session_state.view}
      </div>
      <h1 style="margin: 0;">{st.session_state.view}</h1>
    </div>
    """, unsafe_allow_html=True)
with top_right:
    if valid_dept:
        try:
            n = get_coll(dept).count()
            st.markdown(f"""
            <div style="text-align: right; padding-top: 12px;">
              <span class="pill p-success"><span class="dot dot-green"></span>AI Online</span>
              <div style="font-size: 0.75rem; color: #64748b; margin-top: 6px;">
                {n} chunks indexed · {DEV.upper()}
              </div>
            </div>
            """, unsafe_allow_html=True)
        except: pass

# ==================== VIEW: HOME ====================
if st.session_state.view == "Home":
    if is_admin:
        # Admin home: overview
        c1,c2,c3,c4 = st.columns(4)
        total_chunks = sum(get_coll(d).count() for d in DEPARTMENTS)
        total_faculty = len([u for u in USERS.values() if u["role"]=="teacher"])
        all_df = audit_df()
        blocked_24h = len(all_df[(all_df["action"]=="BLOCKED_QUERY") &
                                  (all_df["ts"]>datetime.now()-timedelta(hours=24))]) if len(all_df) else 0
        weekly_queries = len(all_df[(all_df["action"]=="QUERY") &
                                     (all_df["ts"]>datetime.now()-timedelta(days=7))]) if len(all_df) else 0
        for col, (lbl, val, sub) in zip([c1,c2,c3,c4], [
            ("Departments", len(DEPARTMENTS), "active units"),
            ("Faculty", total_faculty, "accounts"),
            ("KB Chunks", f"{total_chunks:,}", "indexed docs"),
            ("Weekly queries", weekly_queries, f"{blocked_24h} blocked today")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        st.markdown("### 🏛️ Department Health")
        cols = st.columns(len(DEPARTMENTS))
        for col, (code, cfg) in zip(cols, DEPARTMENTS.items()):
            score = health_score(code)
            n = get_coll(code).count()
            srcs = list_sources(code)
            fac = sum(1 for u in USERS.values() if u.get("dept")==code)
            bar_color = "#10b981" if score >= 70 else "#f59e0b" if score >= 40 else "#ef4444"
            col.markdown(f"""
            <div class="card" style="border-left: 4px solid {cfg['color']};">
              <div style="font-size: 1.6rem;">{cfg.get('icon','📁')}</div>
              <div style="font-size: 0.7rem; color: {cfg['color']}; font-weight: 700;
                          letter-spacing: 1px; margin: 4px 0;">{code.upper()}</div>
              <div style="font-size: 0.95rem; font-weight: 700; margin-bottom: 8px;">{cfg['name']}</div>
              <div style="display: flex; align-items: center; gap: 6px; margin-bottom: 8px;">
                <div style="flex: 1; height: 6px; background: #e2e8f0; border-radius: 3px; overflow: hidden;">
                  <div style="width: {score}%; height: 100%; background: {bar_color};"></div>
                </div>
                <span style="font-size: 0.75rem; font-weight: 700; color: {bar_color};">{score}</span>
              </div>
              <div style="font-size: 0.75rem; color: #64748b;">📚 {n} chunks · 📄 {len(srcs)} docs · 👥 {fac}</div>
            </div>
            """, unsafe_allow_html=True)
    else:
        # Faculty home
        st.markdown(f"""
        <div style="background: linear-gradient(135deg, {dcolor(dept)}15, {dcolor(dept)}05);
                    border: 1px solid {dcolor(dept)}30; border-radius: 16px;
                    padding: 24px 28px; margin-bottom: 24px;">
          <div style="font-size: 0.75rem; color: {dcolor(dept)}; font-weight: 700;
                      letter-spacing: 1px;">WELCOME BACK</div>
          <h2 style="margin: 6px 0 6px 0; font-size: 1.6rem;">{name}</h2>
          <p style="color: #475569; margin: 0; font-size: 0.92rem;">
            You're in the <b>{dname(dept)}</b> workspace. Use the sidebar to access your AI assistant,
            analytics, and department records.
          </p>
        </div>
        """, unsafe_allow_html=True)
        # Quick stats
        srcs = list_sources(dept)
        n = get_coll(dept).count()
        df = audit_df(dept=dept)
        my_queries = len(df[df["user"]==user]) if len(df) else 0
        c1,c2,c3,c4 = st.columns(4)
        for col, (lbl, val, sub) in zip([c1,c2,c3,c4], [
            ("KB Chunks", n, "indexed"),
            ("Documents", len(srcs), "source files"),
            ("Health Score", health_score(dept), "out of 100"),
            ("Your Queries", my_queries, "this session")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        # Quick actions
        st.markdown("### ⚡ Quick Actions")
        actions = [
          ("📝","Generate Exam Paper","Create a full question paper with sections and marks"),
          ("📓","Study Notes","Auto-generate structured notes on any topic"),
          ("🧠","Practice Quiz","MCQ + short answer quiz generator"),
          ("📊","Anomaly Scan","Find at-risk students in your department"),
        ]
        cols = st.columns(4)
        for col, (ic, t, d) in zip(cols, actions):
            with col:
                if st.button(f"{ic} {t}", use_container_width=True, key=f"qa_home_{t}"):
                    if "Exam" in t: st.session_state.qa_trigger="exam"
                    elif "Notes" in t: st.session_state.qa_trigger="notes"
                    elif "Quiz" in t: st.session_state.qa_trigger="quiz"
                    else: st.session_state.view="Anomalies"
                    if st.session_state.qa_trigger: st.session_state.view="Assistant"
                    st.rerun()
                st.caption(d)

# ==================== VIEW: ASSISTANT ====================
elif st.session_state.view == "Assistant" and not is_admin:
    col_left, col_right = st.columns([3,1])
    with col_left:
        st.markdown(f"### 💬 {dname(dept)} — AI Assistant")
        st.caption("Every answer comes from your department's documents only.")
    with col_right:
        st.session_state.deep_think = st.toggle("🧠 Deep Think",
            value=st.session_state.deep_think, help="Two-stage reasoning")

    # Quick actions
    st.markdown("**⚡ Quick Actions** — set a topic and click")
    qa_cols = st.columns(5)
    for col,(k,ic,lbl) in zip(qa_cols,[("exam","📝","Exam"),("notes","📓","Notes"),
                                        ("quiz","🧠","Quiz"),("sum","📄","Summarise"),
                                        ("explain","💡","Explain")]):
        with col:
            if st.button(f"{ic} {lbl}", key=f"qa_{k}", use_container_width=True):
                st.session_state.qa_trigger=k; st.rerun()

    topic = st.text_input("Topic / subject", key="chat_topic",
                          placeholder="e.g. Data Structures · Fee refund policy · Student S003 status")

    with st.expander("📎 Attach a PDF or image (becomes AI context)"):
        c1, c2 = st.columns(2)
        with c1:
            f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
            if f1 and st.button("Load PDF", key="btn_pdf"):
                pages, err = extract_pdf(f1.getvalue(), f1.name)
                if pages:
                    st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                    st.session_state.ctx_name = f1.name
                    st.success(f"✅ {len(pages)} pages loaded")
                else: st.error(f"❌ {err}")
        with c2:
            f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
            if f2 and st.button("Run OCR", key="btn_ocr"):
                tmp=f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                open(tmp,"wb").write(f2.getvalue())
                try:
                    txt = pytesseract.image_to_string(Image.open(tmp))
                    st.session_state.chat_ctx = txt
                    st.session_state.ctx_name = f2.name
                    st.success(f"✅ {len(txt)} chars extracted")
                except Exception as e: st.error(f"❌ {e}")
                finally:
                    try: os.remove(tmp)
                    except: pass

    if st.session_state.chat_ctx:
        cA, cB = st.columns([4,1])
        with cA:
            st.markdown(f'<div class="alert alert-warning">📎 Attached: <b>{st.session_state.ctx_name}</b> ({len(st.session_state.chat_ctx)} chars)</div>', unsafe_allow_html=True)
        with cB:
            if st.button("Clear file", key="clr_file"):
                st.session_state.chat_ctx=""; st.session_state.ctx_name=""; st.rerun()

    # Chat
    box = st.container(height=520)
    with box:
        if not st.session_state.chat_history:
            st.markdown(f"""<div class="empty">
              <div class="icon">💬</div>
              <div class="title">Ask anything about {dname(dept)}</div>
              <div class="sub">Try: "Create a question paper on Digital Electronics" → get a PDF</div>
            </div>""", unsafe_allow_html=True)
        for i, m in enumerate(st.session_state.chat_history):
            with st.chat_message(m["role"]):
                st.markdown(m["content"])
                if m.get("thinking"):
                    with st.expander("🧠 Reasoning"):
                        st.caption(m["thinking"])
                if m.get("sources"):
                    with st.expander(f"📄 {len(m['sources'])} source(s)"):
                        for s in m["sources"]:
                            st.caption(f"`{s['source']}` · similarity {s['similarity']:.2f}")
                if m.get("pdf") and m.get("pdf_name"):
                    st.download_button(f"📥 {m['pdf_name']}", m["pdf"], file_name=m["pdf_name"],
                                       mime="application/pdf", key=f"dl_{i}")

    uq = st.chat_input("Ask a question, or request a PDF deliverable…")
    if st.session_state.qa_trigger:
        k = st.session_state.qa_trigger; st.session_state.qa_trigger=None
        prompts = {
          "exam": f"Prepare a question paper PDF. Subject: {topic or dname(dept)}. 10 questions, Sections A/B/C, marks shown.",
          "notes": f"Generate study notes PDF on: {topic or 'the full course'}. Headings and bullets.",
          "quiz": f"Create a 5-question MCQ quiz PDF on: {topic or 'the course'}. Options A-D with answer.",
          "sum": f"Summarise key rules about: {topic or 'department policy'}. Bullet points.",
          "explain": f"Explain in simple terms for a new faculty: {topic or 'the department fundamentals'}.",
        }
        uq = prompts[k]

    if uq:
        st.session_state.stop_event.clear()
        with box:
            with st.chat_message("user"): st.markdown(uq)
        st.session_state.chat_history.append({"role":"user","content":uq})
        other, on = detect_cross_dept(dept, uq)
        if other and other != dept:
            err = f"🚫 **Access denied.** You're scoped to **{dname(dept)}**. Cannot query **{on}**."
            st.session_state.chat_history.append({"role":"assistant","content":err})
            with box:
                with st.chat_message("assistant"): st.error(err)
            log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
        else:
            with box:
                with st.chat_message("assistant"):
                    ph = st.empty(); stop_ph = st.empty()
                    stop_ph.button("⏹️ Stop", key=f"stop_{int(time.time())}",
                                   on_click=lambda: st.session_state.stop_event.set())
                    ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx,
                                              deep=st.session_state.deep_think,
                                              history=st.session_state.chat_history)
                    stop_ph.empty()
                    pdf_bytes=None; pdf_name=None
                    if is_doc_request(uq) or ans.count("Q1")+ans.count("1.")>=3:
                        title = "Examination Question Paper" if "question" in uq.lower() else "Document"
                        body = ans.replace("**","").replace("##","").replace("*","")
                        pdf_bytes = make_pdf(f"{title} — {topic or dname(dept)}",
                                             dname(dept),
                                             f"Generated by VaultMind AI · {datetime.now().strftime('%d %b %Y %H:%M')}",
                                             body)
                        pdf_name = f"{_slug(title)}_{_slug(topic or dept)}.pdf"
                        st.success("📄 PDF ready")
                        st.download_button(f"📥 {pdf_name}", pdf_bytes, file_name=pdf_name,
                                           mime="application/pdf", key=f"dl_new_{int(time.time())}")
                    if hits: st.caption(f"📄 {len(hits)} source(s) cited")
            st.session_state.chat_history.append({
                "role":"assistant","content":ans,
                "sources":[{"source":h["source"],"page":h["page"],"similarity":h["similarity"]} for h in hits],
                "pdf":pdf_bytes, "pdf_name":pdf_name,
                "thinking": st.session_state.get("last_think","") if st.session_state.deep_think else ""})
            log(user,role,dept,"QUERY",uq[:60])
        st.rerun()

    if st.session_state.chat_history:
        cA, cB, _ = st.columns([1,1,4])
        with cA:
            if st.button("🗑️ Clear", use_container_width=True, key="clr_chat"):
                st.session_state.chat_history=[]; st.rerun()
        with cB:
            # Export chat as PDF
            if st.button("📥 Export Chat", use_container_width=True, key="exp_chat"):
                body = "\n\n".join(f"[{m['role'].upper()}]\n{m['content']}" for m in st.session_state.chat_history)
                pdf = make_pdf("Chat Transcript", dname(dept),
                               f"Exported by {name} · {datetime.now().strftime('%d %b %Y %H:%M')}", body)
                st.download_button("📥 Download Transcript", pdf, file_name=f"chat_{_slug(dept)}.pdf",
                                   mime="application/pdf", key="dl_chat_exp")

# ==================== VIEW: ANALYTICS ====================
elif st.session_state.view == "Analytics":
    st.markdown("### 📊 Department Analytics")
    df = audit_df(dept=None if is_admin else dept)
    if len(df)==0:
        st.info("No activity yet. Start querying to populate analytics.")
    else:
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("Total actions", len(df), "all time"),
            ("Queries", len(df[df["action"]=="QUERY"]), "AI interactions"),
            ("Blocked", len(df[df["action"]=="BLOCKED_QUERY"]), "security events"),
            ("Unique users", df["user"].nunique(), "active faculty")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        c1, c2 = st.columns(2)
        with c1:
            st.markdown("#### Activity by Department")
            dept_counts = df[df["action"]=="QUERY"].groupby("dept").size().reset_index(name="count")
            dept_counts = dept_counts[dept_counts["dept"].isin(DEPARTMENTS.keys())]
            if len(dept_counts):
                fig = px.bar(dept_counts, x="dept", y="count",
                             color="dept", color_discrete_map={d: DEPARTMENTS[d]["color"] for d in DEPARTMENTS},
                             labels={"count":"Queries","dept":"Department"})
                fig.update_layout(showlegend=False, height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        with c2:
            st.markdown("#### Daily Query Trend")
            df["day"] = df["ts"].dt.date
            daily = df[df["action"]=="QUERY"].groupby("day").size().reset_index(name="count")
            if len(daily):
                fig = px.line(daily, x="day", y="count", markers=True,
                              labels={"count":"Queries","day":"Date"})
                fig.update_traces(line_color="#6366f1", line_width=3)
                fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        st.markdown("#### Action Breakdown")
        breakdown = df.groupby("action").size().reset_index(name="count")
        fig = px.pie(breakdown, values="count", names="action", hole=0.55,
                     color_discrete_sequence=["#6366f1","#8b5cf6","#10b981","#f59e0b","#ef4444"])
        fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10))
        st.plotly_chart(fig, use_container_width=True)

# ==================== VIEW: ANOMALIES ====================
elif st.session_state.view == "Anomalies":
    st.markdown("### ⚠️ Anomaly Detection")
    st.caption("Auto-flagged issues from your department's student data.")
    items = anomalies(dept) if valid_dept else []
    if not items:
        st.success("✅ No anomalies detected in your department.")
    else:
        crit = [x for x in items if x["severity"]=="critical"]
        warn = [x for x in items if x["severity"]=="warning"]
        c1, c2 = st.columns(2)
        c1.markdown(f"""<div class="metric-card"><div class="lbl">🚨 Critical</div>
            <div class="val" style="color:#ef4444;">{len(crit)}</div>
            <div class="sub">requires immediate action</div></div>""", unsafe_allow_html=True)
        c2.markdown(f"""<div class="metric-card"><div class="lbl">⚠️ Warnings</div>
            <div class="val" style="color:#f59e0b;">{len(warn)}</div>
            <div class="sub">needs review</div></div>""", unsafe_allow_html=True)
        st.write("")
        for x in items:
            cls = "alert-critical" if x["severity"]=="critical" else "alert-warning"
            icon = "🚨" if x["severity"]=="critical" else "⚠️"
            st.markdown(f"""<div class="alert {cls}">
              <b>{icon} {x['type']} — {x['student']}</b> ({x['roll']})<br>
              {x['detail']}
            </div>""", unsafe_allow_html=True)

# ==================== VIEW: STUDENTS ====================
elif st.session_state.view == "Students" and dept in ("finance","admin"):
    st.markdown("### 🎓 Student Records")
    # Load student data from seed
    try:
        csv_key = "students_csv"
        csv_data = SEED.get(dept,{}).get(csv_key,"")
        df = pd.read_csv(io.StringIO(csv_data))
        # Filters
        c1, c2, c3 = st.columns(3)
        with c1: dept_filter = st.selectbox("Department", ["ALL"]+list(DEPARTMENTS.keys()), key="stu_dept")
        with c2: yr_filter = st.selectbox("Year", ["ALL","III","V","VII"], key="stu_year")
        with c3: srch = st.text_input("Search name/roll", key="stu_srch")
        view = df.copy()
        if dept_filter != "ALL" and "Dept" in view.columns: view = view[view["Dept"]==dept_filter]
        if yr_filter != "ALL" and "Year" in view.columns: view = view[view["Year"]==yr_filter]
        if srch: view = view[view.apply(lambda r: srch.lower() in str(r).lower(), axis=1)]
        st.markdown(f"**{len(view)} records**")
        st.dataframe(view, use_container_width=True, hide_index=True, height=400)
        # Analytics
        if "Status" in df.columns:
            c1, c2 = st.columns(2)
            with c1:
                sc = df["Status"].value_counts().reset_index(); sc.columns=["Status","Count"]
                fig = px.pie(sc, values="Count", names="Status", hole=0.5,
                             color_discrete_map={"CLEARED":"#10b981","PARTIAL":"#f59e0b",
                                                 "OVERDUE":"#ef4444","CRITICAL":"#991b1b"})
                fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
                                  title="Fee Status Distribution")
                st.plotly_chart(fig, use_container_width=True)
            with c2:
                if "Pending" in df.columns:
                    top = df.nlargest(8, "Pending")[["Name","Pending"]]
                    fig = px.bar(top, x="Pending", y="Name", orientation="h",
                                 color_discrete_sequence=["#ef4444"])
                    fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
                                      title="Top Pending Dues (₹)")
                    st.plotly_chart(fig, use_container_width=True)
        if "Attendance" in df.columns:
            fig = px.histogram(df, x="Attendance", nbins=10,
                               color_discrete_sequence=["#6366f1"])
            fig.add_vline(x=75, line_dash="dash", line_color="red", annotation_text="Min 75%")
            fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
                              title="Attendance Distribution")
            st.plotly_chart(fig, use_container_width=True)
        # Bulk upload
        with st.expander("📤 Bulk upload student CSV"):
            up = st.file_uploader("CSV file", type=["csv"], key="bulk_csv")
            if up:
                try:
                    new_df = pd.read_csv(up)
                    st.dataframe(new_df.head(), use_container_width=True)
                    if st.button("Ingest to Knowledge Base", type="primary", key="bulk_ingest"):
                        n = bulk_csv_upload(dept, new_df, kind="students")
                        st.success(f"✅ Ingested {n} chunks from {len(new_df)} records")
                        log(user,role,dept,"BULK_CSV",f"{len(new_df)} records")
                except Exception as e:
                    st.error(f"CSV parse error: {e}")
    except Exception as e:
        st.error(f"Could not load student data: {e}")

# ==================== VIEW: DOCUMENTS ====================
elif st.session_state.view == "Documents" and not is_admin:
    st.markdown("### 📚 Knowledge Base")
    srcs = list_sources(dept) if valid_dept else {}
    if srcs:
        c1, c2 = st.columns([2,1])
        with c1:
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"""<div class="card" style="margin-bottom: 8px;">
                  <div style="display: flex; justify-content: space-between; align-items: center;">
                    <div><b>📄 {s}</b></div>
                    <span class="pill p-success">{cnt} chunks</span>
                  </div>
                </div>""", unsafe_allow_html=True)
        with c2:
            st.metric("Total Documents", len(srcs))
            st.metric("Total Chunks", get_coll(dept).count())
            st.metric("Health Score", health_score(dept))
    else:
        st.info("No documents in this department yet.")

# ==================== VIEW: DEPARTMENTS (Admin) ====================
elif st.session_state.view == "Departments" and is_admin:
    st.markdown("### 🏛️ Department Management")
    with st.expander("➕ Create New Department"):
        c1, c2 = st.columns(2)
        with c1:
            code = st.text_input("Code", key="nd_code").strip().lower()
            name2 = st.text_input("Display Name", key="nd_name")
            color = st.color_picker("Color", value="#6366f1", key="nd_color")
        with c2:
            sec = st.text_input("Secret (canary)", key="nd_sec")
            al = st.text_input("Aliases (comma-separated)", key="nd_al")
        if st.button("Create Department", type="primary"):
            if not code or not name2: st.error("Code + name required")
            elif code in DEPARTMENTS: st.error("Already exists")
            else:
                DEPARTMENTS[code] = {"name":name2, "secret":sec or f"{code}-s-{int(time.time())}",
                    "color":color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code], "icon":"📁"}
                save_depts(); get_coll(code)
                log(user,role,None,"CREATE_DEPT",f"{code}:{name2}")
                st.success(f"✅ Created {name2}"); time.sleep(1); st.rerun()
    for code, cfg in DEPARTMENTS.items():
        n = get_coll(code).count(); srcs = list_sources(code)
        score = health_score(code)
        with st.expander(f"**{cfg['name']}** ({code}) — Score {score} · {n} chunks · {len(srcs)} docs"):
            c1, c2 = st.columns([2,1])
            with c1:
                for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                    st.markdown(f"- 📄 `{s}` ({cnt} chunks)")
            with c2:
                st.metric("Faculty", sum(1 for u in USERS.values() if u.get("dept")==code))
                if st.button("🗑️ Reset KB", key=f"rst_{code}"):
                    try:
                        cc.delete_collection(f"kb_{code}"); get_coll(code)
                        log(user,role,None,"RESET_KB",code)
                        st.success("Reset"); time.sleep(1); st.rerun()
                    except Exception as e: st.error(str(e))

# ==================== VIEW: FACULTY (Admin) ====================
elif st.session_state.view == "Faculty" and is_admin:
    st.markdown("### 👥 Faculty Management")
    with st.expander("➕ Add Faculty"):
        c1, c2 = st.columns(2)
        with c1:
            nu = st.text_input("Username", key="cr_u").strip().lower()
            np = st.text_input("Password", type="password", key="cr_p")
            nn = st.text_input("Full Name", key="cr_n")
        with c2:
            nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
        if st.button("Create Account", type="primary"):
            if not nu or not np or not nn: st.error("All fields required")
            elif nu in USERS: st.error("Username exists")
            else:
                USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                st.success(f"✅ Created {nu}"); time.sleep(1); st.rerun()
    st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"}
                                for u,v in USERS.items()]), use_container_width=True, hide_index=True)

# ==================== VIEW: UPLOADS (Admin) ====================
elif st.session_state.view == "Uploads" and is_admin:
    st.markdown("### 📤 Upload to Department RAG")
    tgt = st.selectbox("Target Department", list(DEPARTMENTS.keys()),
                       format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_dept")
    dtype = st.selectbox("Document Type", ["syllabus","regulation","notes","question-paper","reference","policy"],
                         key="up_type")
    uploaded = st.file_uploader("Choose PDFs", type=["pdf"], accept_multiple_files=True, key="up_files")
    if uploaded:
        st.info(f"📎 {len(uploaded)} file(s) · {sum(len(f.getvalue())/1024 for f in uploaded):.1f} KB")
        if st.button("🚀 Ingest", type="primary", use_container_width=True):
            total=0; ok=0
            for f in uploaded:
                with st.spinner(f"Processing {f.name}…"):
                    chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                    if err: st.error(f"❌ {f.name} — {err}")
                    else:
                        st.success(f"✅ {f.name} — {pages} pages → {chunks} chunks")
                        total+=chunks; ok+=1
            if ok:
                st.balloons()
                st.info(f"🎉 {ok}/{len(uploaded)} files · {total} chunks into **{DEPARTMENTS[tgt]['name']}**")
                log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
                time.sleep(2); st.rerun()

# ==================== VIEW: AUDIT (Admin) ====================
elif st.session_state.view == "Audit" and is_admin:
    st.markdown("### 📜 Audit Log")
    c1, c2 = st.columns([1,3])
    with c1:
        fd = st.selectbox("Filter by Department", ["ALL"]+list(DEPARTMENTS.keys()), key="au_d")
    df = audit_df(dept=None if fd=="ALL" else fd)
    if len(df):
        df["ts"] = df["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
        st.dataframe(df[["ts","user","role","dept","action","detail"]],
                     use_container_width=True, hide_index=True, height=500)
    else:
        st.info("No activity yet.")
    st.divider()
    allr = audit_df()
    c1,c2,c3 = st.columns(3)
    c1.metric("🔒 Blocked Queries", len(allr[allr["action"]=="BLOCKED_QUERY"]) if len(allr) else 0)
    c2.metric("✅ Leaks", 0)
    c3.metric("📊 Total Actions", len(allr))
'''
open("/content/vaultmind_erp.py","w").write(APP)
print(f"✅ App written ({len(APP)} bytes)")
import shutil
shutil.copy("/content/vaultmind_erp.py", f"{DRIVE}/vaultmind_erp.py")

print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "--theme.base light --theme.primaryColor '#6366f1' "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if r.stdout.strip(): print(f"✅ Streamlit PID {r.stdout.strip()}")
else:
    print("❌ Failed:"); print(open("/content/streamlit.log").read()[-1500:]); raise SystemExit(1)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🏆 VAULTMIND ERP v8 — INDUSTRY EDITION                        ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                                                                ║
║  ✨ NEW IN v8 (beats IoT on demo wow-factor):                  ║
║     • Sidebar navigation (feels like a real SaaS app)          ║
║     • Live Analytics dashboard (Plotly charts, trends)         ║
║     • Anomaly Detection (auto-flags at-risk students)          ║
║     • Student Records table (fee/attendance, filters)          ║
║     • Department Health Scores (composite metric)              ║
║     • Bulk CSV upload → indexed into KB                        ║
║     • Chat Export as PDF                                       ║
║     • Multi-turn conversation memory                           ║
║     • Inline PDF generation from chat                          ║
║     • Deep Think mode, Stop button, file/image attach          ║
╚════════════════════════════════════════════════════════════════╝
""")

📦 Installing premium stack…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Ready
✅ App written (68441 bytes)
🚀 Launching…
✅ Streamlit PID 45106

╔════════════════════════════════════════════════════════════════╗
║  🏆 VAULTMIND ERP v8 — INDUSTRY EDITION                        ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                

In [ ]:
# ==================== ADD WHISPER STT (Hindi + English) ====================
import subprocess, time, os

print("📦 Installing ffmpeg + whisper deps…")
subprocess.run("apt-get install -y -q ffmpeg 2>&1 | tail -1", shell=True)
subprocess.run("pip install -q 'transformers>=4.45' accelerate librosa soundfile 2>&1 | tail -1",
               shell=True)
print("✅ Installed")

# ---------- Patch the app file ----------
APP_PATH = "/content/vaultmind_erp.py"
src = open(APP_PATH).read()
changed = 0

# ---- 1. Add hf_pipeline import ----
old_imp = "from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer"
new_imp = "from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer, pipeline as hf_pipeline"
if old_imp in src and "hf_pipeline" not in src:
    src = src.replace(old_imp, new_imp, 1); changed += 1
    print("✅ [1/4] Added hf_pipeline import")

# ---- 2. Whisper model + transcribe function after model setup ----
marker = "emb,cc,tok,mdl,DEV = setup()"
whisper_block = """emb,cc,tok,mdl,DEV = setup()

# ==================== WHISPER (in-house Speech-to-Text) ====================
@st.cache_resource(show_spinner=False)
def load_whisper():
    try:
        pipe = hf_pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-small",       # 244M, supports Hindi+English+99 langs
            device=0 if torch.cuda.is_available() else -1,
            chunk_length_s=30,
        )
        return pipe
    except Exception as e:
        print(f"[whisper] load failed: {e}"); return None

whisper_pipe = load_whisper()

def _has_devanagari(s):
    return any('\\u0900' <= c <= '\\u097F' for c in s)

def transcribe_multilingual(audio_bytes):
    \"\"\"Transcribe audio in original language + translate to English.\"\"\"
    if whisper_pipe is None:
        return {"original":"(Whisper unavailable)","language":"unknown","english":""}
    tmp = f"/content/_audio_{int(time.time())}.wav"
    with open(tmp,"wb") as fh: fh.write(audio_bytes)
    try:
        # Try with return_language (newer transformers)
        try:
            r = whisper_pipe(tmp, generate_kwargs={"task":"transcribe"}, return_language=True)
        except TypeError:
            r = whisper_pipe(tmp, generate_kwargs={"task":"transcribe"})
        original = r.get("text","").strip() if isinstance(r, dict) else str(r)
        lang = r.get("language","unknown") if isinstance(r, dict) else "unknown"

        # Translate to English if not already English
        english = original
        need_translate = _has_devanagari(original) or lang.lower() not in ("en","english","unknown")
        if need_translate:
            try:
                r2 = whisper_pipe(tmp, generate_kwargs={"task":"translate"})
                english = r2.get("text","").strip() if isinstance(r2, dict) else str(r2)
                if _has_devanagari(original): lang = "hindi"
            except Exception: pass
        return {"original":original, "language":lang, "english":english}
    except Exception as e:
        return {"original":f"(failed: {e})", "language":"unknown", "english":""}
    finally:
        try: os.remove(tmp)
        except: pass
"""
if marker in src and "whisper_pipe" not in src:
    src = src.replace(marker, whisper_block, 1); changed += 1
    print("✅ [2/4] Added Whisper model + transcribe function")

# ---- 3. Voice input UI block, before the topic text_input ----
topic_marker = '''    topic = st.text_input("Topic / subject", key="chat_topic",
                          placeholder="e.g. Data Structures · Fee refund policy · Student S003 status")'''

voice_ui = '''    # ---- Voice Input (Whisper Hindi + English) ----
    if "voice_result" not in st.session_state: st.session_state.voice_result = None
    with st.expander("🎤 Voice Input — Speak your query (Hindi / English / 99 languages)"):
        st.caption("Runs locally via Whisper. Transcribes in original language + translates to English for KB search.")
        cA, cB = st.columns(2)
        with cA:
            audio = st.audio_input("Record from microphone", key="voice_rec")
        with cB:
            up_audio = st.file_uploader("Or upload audio (WAV / MP3 / M4A / OGG)",
                                         type=["wav","mp3","m4a","ogg","webm"], key="voice_file")
        audio_data = (audio.getvalue() if audio else (up_audio.getvalue() if up_audio else None))
        if audio_data and st.button("🎧 Transcribe", type="primary", key="btn_stt"):
            with st.spinner("Whisper is transcribing…"):
                st.session_state.voice_result = transcribe_multilingual(audio_data)
        if st.session_state.voice_result:
            r = st.session_state.voice_result
            st.markdown(f"**Detected language:** `{r.get('language','unknown')}`")
            st.text_area("Original transcription", value=r["original"], height=70, key="vo_orig")
            if r.get("language","unknown").lower() not in ("en","english"):
                st.text_area("English translation (used for KB search)", value=r["english"],
                             height=70, key="vo_en")
            c1, c2 = st.columns([1,1])
            with c1:
                if st.button("📋 Use as query", key="use_voice", use_container_width=True):
                    st.session_state.chat_topic = r.get("english") or r.get("original") or ""
                    st.rerun()
            with c2:
                if st.button("🗑️ Clear", key="clr_voice", use_container_width=True):
                    st.session_state.voice_result = None; st.rerun()

    # Session-state backed topic input
    if "chat_topic" not in st.session_state: st.session_state.chat_topic = ""
    topic = st.text_input("Topic / subject", key="chat_topic",
                          placeholder="e.g. Data Structures · Fee refund policy · Student S003 status")'''

if topic_marker in src:
    src = src.replace(topic_marker, voice_ui, 1); changed += 1
    print("✅ [3/4] Added voice input UI")

# ---- 4. Warm up whisper once at startup ----
# (Loaded lazily via cache_resource, so no extra change needed)

open(APP_PATH, "w").write(src)
print(f"✅ [4/4] App patched ({changed} changes, {len(src)} bytes)")

# Backup
import shutil
DRIVE = "/content/drive/MyDrive/vaultmind"
shutil.copy(APP_PATH, f"{DRIVE}/vaultmind_erp.py")

# ---------- Restart Streamlit ----------
print("\n🚀 Restarting Streamlit…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "--theme.base light --theme.primaryColor '#6366f1' "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(10)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Streamlit failed:"); print(open("/content/streamlit.log").read()[-2000:])
    raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎤 WHISPER VOICE INPUT ADDED                                  ║
║                                                                ║
║  🌐 Open: {url}
║                                                                ║
║  🎙️ How to use:                                                ║
║     1. Login as teacher_cse / t_cse                            ║
║     2. Go to 💬 AI Assistant                                   ║
║     3. Expand "🎤 Voice Input"                                 ║
║     4. Click the mic → speak in Hindi or English               ║
║     5. Click "🎧 Transcribe" → see both versions               ║
║     6. Click "📋 Use as query" → fills the topic field         ║
║     7. Ask → AI answers from your dept KB                      ║
║                                                                ║
║  🔉 First load: Whisper-small downloads (~950 MB, once)        ║
║     Cached on Colab disk after that                            ║
║                                                                ║
║  🌐 Languages supported: Hindi, English + 97 more              ║
║     (Marathi, Tamil, Telugu, Bengali, Gujarati, Punjabi...)    ║
╚════════════════════════════════════════════════════════════════╝
""")

📦 Installing ffmpeg + whisper deps…
✅ Installed
✅ [1/4] Added hf_pipeline import
✅ [2/4] Added Whisper model + transcribe function
✅ [3/4] Added voice input UI
✅ [4/4] App patched (3 changes, 72643 bytes)

🚀 Restarting Streamlit…
✅ Streamlit PID 52272

╔════════════════════════════════════════════════════════════════╗
║  🎤 WHISPER VOICE INPUT ADDED                                  ║
║                                                                ║
║  🌐 Open: https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  🎙️ How to use:                                                ║
║     1. Login as teacher_cse / t_cse                            ║
║     2. Go to 💬 AI Assistant                                   ║
║     3. Expand "🎤 Voice Input"                                 ║
║     4. Click the mic → speak in Hindi or English               ║
║     5. Click "🎧 Transcribe" → see both versions     

In [ ]:
# ==================== VAULTMIND ERP v9 — PERSISTENT MEMORY ====================
import subprocess, sys, os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)
uf = f"{DRIVE}/data/users.json"
if os.path.exists(uf): os.remove(uf)

print("📦 Installing…")
!pip install -q streamlit chromadb sentence-transformers "transformers>=4.45" \
    accelerate bitsandbytes python-docx pymupdf pytesseract pillow reportlab \
    pdfplumber pandas plotly openpyxl 2>&1 | tail -1
!apt-get install -y -q tesseract-ocr tesseract-ocr-eng 2>&1 | tail -1
print("✅ Ready")

APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil, uuid
from datetime import datetime, timedelta
import streamlit as st, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf, plotly.express as px
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

DRIVE = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS   = f"{DRIVE}/data/depts.json"
CFG_USERS   = f"{DRIVE}/data/users.json"
AUDIT_DB    = f"{DRIVE}/data/audit.db"
CONV_DB     = f"{DRIVE}/data/conversations.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR  = f"{DRIVE}/uploads"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID = "all-MiniLM-L6-v2"
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s): return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True)
    except: pass

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except: pass

# ==================== DEPARTMENTS ====================
PALETTE = ["#6366f1","#8b5cf6","#10b981","#f59e0b","#06b6d4"]
DEFAULT_DEPTS = {
  "cse":{"name":"Computer Science","color":"#6366f1","icon":"💻","secret":"aurora-lattice",
         "aliases":["cse","computer","comp sci","computer science"]},
  "ece":{"name":"Electronics","color":"#8b5cf6","icon":"📡","secret":"cobalt-meridian",
         "aliases":["ece","electronics","communication"]},
  "finance":{"name":"Finance & Accounts","color":"#10b981","icon":"💰","secret":"ledger-falcon",
             "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":{"name":"Administration","color":"#f59e0b","icon":"🏛️","secret":"orbit-vermillion",
           "aliases":["admin","administration","hr","registrar"]},
}
try: DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
for code,cfg in DEFAULT_DEPTS.items(): DEPARTMENTS.setdefault(code,cfg)
for i,(c,cfg) in enumerate(DEPARTMENTS.items()):
    cfg.setdefault("name",c.upper()); cfg.setdefault("secret",f"{c}-s-{abs(hash(c))%9999}")
    cfg.setdefault("color",PALETTE[i%len(PALETTE)]); cfg.setdefault("icon","📁")
    cfg.setdefault("aliases",[c])
json.dump(DEPARTMENTS,open(CFG_DEPTS,"w"),indent=2)
def save_depts(): json.dump(DEPARTMENTS,open(CFG_DEPTS,"w"),indent=2)
def dname(d): return DEPARTMENTS.get(d,{}).get("name",d or "—")
def dcolor(d): return DEPARTMENTS.get(d,{}).get("color","#64748b")
def dicon(d): return DEPARTMENTS.get(d,{}).get("icon","📁")

# ==================== USERS ====================
DEFAULT_USERS = {
  "admin":{"name":"System Administrator","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":{"name":"Dr. Sharma","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":{"name":"Dr. Menon","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":{"name":"Mr. Verma","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin":{"name":"Ms. Rao","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
try:
    saved = json.load(open(CFG_USERS))
    USERS = DEFAULT_USERS.copy()
    for k,v in saved.items():
        if k not in USERS and v.get("role")!="student": USERS[k]=v
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS,open(CFG_USERS,"w"),indent=2)
def save_users(): json.dump(USERS,open(CFG_USERS,"w"),indent=2)

# ==================== SEED ====================
SEED = {
"cse":{
  "syllabus":"B.Tech CSE MU 4yr/8sem. 3 branches: CSE, CSE(AI), CSE(BCT). Sem I: Engg Math-I, Programming in C, Physics→Electronics, Indian Traditions, Comp Fund, Web Dev-I, PBL-I. Sem II: Math-II, Python, Web Dev-II, Digital Circuits. Sem III: Discrete Math, Data Structures (24CSE1600T), OOP Java (24CSE1700T), RDBMS (24CSE1800T), ML Intro (24CSE1902T). Sem IV: Theory of Computation (CSE160), Algorithms (CSE170), OS (CSE280). Sem V-VI: Computer Architecture, Networks (CSE190), Compiler (CSE340), Deep Learning (CSE766), Data Mining (CSE776), Cyber Security (CSE900), Software Engg (CSE330), Android (CSE910), NLP (CSE781), Distributed (CSE891). Sem VIII: Entrepreneurship, E-Commerce, Cyber Law, Blockchain/ML, Big Data.",
  "exam_scheme":"MU 2024 scheme. Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC 2cr = 50. Practical 1cr = 50, 2cr = 100. PBL = 50 (25+25). Exams Dec/May-Jun, 3hr papers, morning 10-1, evening 2-5. Passing: 40% EST + 40% overall theory; 50% practical.",
  "grading":"8-point scale: 90+ O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). CGPA > 8 = First Division with Distinction.",
  "attendance":"Minimum 75% per theory course, 100% per lab. Condonation 65-74% with medical certificate within 3 days. Below 65% = detained. Late > 15 min = half-day.",
  "lab_rules":"CSE Labs Mon-Fri 9-12, 1-4. Sat 9-1 make-up. Batch max 20. 15 min late = absent. Max 2 missed labs. ID + closed footwear. No food/drinks. Phones silent. Penalty: 5 marks first violation, repeat = 1 week ban + parents.",
  "internship":"Summer-I after Sem IV: 4wk (160hr), 2cr. Summer-II after Sem VI: 6wk (240hr), 2cr. Offer + HOD-signed NOC ≥ 7 days before. 90% attendance. Report 25-40 pages TNR 12 spacing 1.5, similarity <20%. Eval: mentor 30 + report 30 + presentation 40. Pass 50/100.",
  "project":"PBL-I/II 1cr each. Major Project final year teams 3-4. Report A4 TNR 12, IEEE refs, 60-100 pages. Plagiarism <15%, single <2%. Late 5 marks/day. 4 hardbound. Viva: guide + 1 int + 1 ext.",
  "plagiarism":"Similarity >15% triggers review. Single source >2% flagged. Second offence = rustication. Tools: Turnitin.",
},
"ece":{
  "syllabus":"B.Tech ECE MU 4yr/8sem. Sem III: Probability & Stats (24ECE1500T), Electronic Devices & Circuits (24ECE1600T), Signals & Systems (24ECE1700T), Digital System Design (24ECE1800T), OOP C++ (24ECE1900T). Sem IV: EM Field Theory (24ECE2000T), Analog Circuits (24ECE2100T), Analog Comm (24ECE2200T), Linear Control (24ECE2300T), Microprocessors (24ECE2400T). Sem V: DSP, Digital Comm, VLSI Design, Embedded Systems, Antenna. Sem VI: Microwave Engg, Info Theory, Computer Networks, Wireless. Sem VII: Optical Fiber, Satellite, IoT. Sem VIII: Radar & Navigation, Major Project II (200), Viva (100), Seminar (50).",
  "exam_scheme":"Same as CSE. 80 ECE courses in pass marks register. Theory min EST 24/100, min overall 40/100. Practical min overall 25/50 or 50/100.",
  "lab_rules":"ECE Labs same as CSE. 10 labs: Electronic Devices, Digital Elec, Analog Circuits, Microprocessor, Communication, VLSI, DSP, Microwave, Embedded, Computer Networks. Never above 36V without faculty. Never look into powered waveguide.",
  "internship":"Same as CSE. Domain: embedded, VLSI, telecom, IoT. Types: Industrial Training (8wk,3cr), In-plant (2wk,1cr,85%), Research (8wk,3cr), Online (4wk,1cr,85%), Startup (6wk,2cr).",
  "project":"Same format as CSE. ECE projects: sensor circuits, amplifiers, microcontroller apps, PCB design.",
  "labs_detail":"10 ECE labs with specific rules: check polarity, don't hot-swap ICs, set function generator amplitude minimum, don't unplug kit while downloading, keep RF trainers away from water, logout EDA workstation, save every 15 min, never look into powered waveguide, disconnect USB before rewiring, don't change switch settings.",
  "attendance":"Same as CSE: 75% min, condonation 65-74%, <65% detained.",
},
"finance":{
  "fee_structure":"B.Tech/year 2025-26: Tuition 1,10,000 + Development 15,000 + Lab 10,000 + Library 3,000 + Sports 2,000 + Exam 5,000 = 1,45,000. Hostel AC 85,000, non-AC 65,000, mess-only 35,000. Payment 50/30/20.",
  "late_fee":"Rs 500/week up to 4 weeks. Beyond 4 weeks: hall ticket withheld. After 8 weeks: ID suspended. After 12 weeks: Dean referral. Waiver on documented hardship.",
  "refund":"UGC-compliant: 100% before 15 days of semester start, 90% within 15, 80% within 30, 50% after 30, NIL after 60.",
  "scholarship":"Merit CGPA≥9.0 → 25% waiver; 8.0-8.9 → 15%. MP Post-Matric, Central Sector, PMSSS. Sports 15-25%. Staff ward 50%. Sibling 10%.",
  "budget":"Annual April submission. Tiers: <25K HOD; 25K-1L HOD+Fin; 1-5L Dean+Fin+Reg; 5-25L Committee; >25L VC.",
  "vendor":"Registration GST+PAN+bank+ITR. Rate contract annual L1. Limited 1-5L. Open >5L. GeM mandatory. PO penalty 1%/week. Inspection 3-member for >50K. Payment 30 days. Blacklist 2yrs.",
  "students_csv":"Roll,Name,Dept,Year,Total,Paid,Pending,DueDays,Status\n24CSE001,Rahul Kumar,CSE,III,145000,110000,35000,0,PARTIAL\n24CSE002,Priya Sharma,CSE,III,145000,145000,0,0,CLEARED\n24CSE003,Amit Patel,CSE,III,145000,72500,72500,12,OVERDUE\n24CSE004,Sneha Iyer,CSE,III,145000,145000,0,0,CLEARED\n24ECE001,Vikram Singh,ECE,III,145000,130000,15000,0,PARTIAL\n24ECE002,Ananya Reddy,ECE,III,145000,145000,0,0,CLEARED\n24ECE003,Karan Mehta,ECE,III,145000,0,145000,25,CRITICAL\n24ECE004,Divya Nair,ECE,III,145000,145000,0,0,CLEARED\n23CSE021,Arjun Verma,CSE,V,145000,145000,0,0,CLEARED\n23CSE022,Riya Gupta,CSE,V,145000,100000,45000,5,PARTIAL\n23ECE015,Mohit Sharma,ECE,V,145000,145000,0,0,CLEARED\n23ECE016,Neha Joshi,ECE,V,145000,120000,25000,0,PARTIAL\n22CSE010,Rohit Kumar,CSE,VII,145000,145000,0,0,CLEARED\n22ECE008,Pooja Rao,ECE,VII,145000,90000,55000,8,OVERDUE\n22CSE011,Sameer Khan,CSE,VII,145000,145000,0,0,CLEARED",
},
"admin":{
  "leave":"Faculty: Casual 12/yr, Earned 15/yr (accum 300), Medical 10/yr, Maternity 180 days, Paternity 15 days. Approval: HOD → Registrar → VC.",
  "grievance":"Acknowledge 24hr, resolve 7-15 days. Levels: Coord → HOD → Dean → Committee → VC. POSH: ICC anonymous 90 days.",
  "events":"Application ≥30 days before. Chain: Dean SW → HOD → Registrar → VC. Budget ≤25K from dept. Police NOC >500.",
  "recruitment":"Asst Prof 60% PG, Assoc 5+ yr, Prof 10+ yr PhD. Ad 2 weeks. Demo 15 min + interview 30 min. Probation 1yr.",
  "students_csv":"Roll,Name,Dept,Year,Attendance,CL,ML,OD,Discipline,Hostel\n24CSE001,Rahul Kumar,CSE,III,78,5,2,0,NONE,A-214\n24CSE002,Priya Sharma,CSE,III,92,2,0,0,NONE,DAY\n24CSE003,Amit Patel,CSE,III,64,8,0,0,WARNING,B-108\n24CSE004,Sneha Iyer,CSE,III,88,3,0,0,NONE,A-305\n24ECE001,Vikram Singh,ECE,III,82,4,1,0,NONE,C-012\n24ECE002,Ananya Reddy,ECE,III,95,0,0,6,NONE,C-220\n24ECE003,Karan Mehta,ECE,III,71,7,2,0,WARNING,D-117\n24ECE004,Divya Nair,ECE,III,89,3,0,0,NONE,D-045\n23CSE021,Arjun Verma,CSE,V,84,5,1,0,NONE,A-402\n23CSE022,Riya Gupta,CSE,V,76,6,0,0,NONE,DAY\n23ECE015,Mohit Sharma,ECE,V,91,2,0,0,NONE,B-214\n23ECE016,Neha Joshi,ECE,V,87,4,0,0,NONE,B-311\n22CSE010,Rohit Kumar,CSE,VII,80,3,0,0,NONE,DAY\n22ECE008,Pooja Rao,ECE,VII,73,5,3,0,WARNING,C-118\n22CSE011,Sameer Khan,CSE,VII,88,2,0,0,NONE,A-510",
  "timetable":"Dr. Sharma (CSE): Mon 9-11 DS (SIII R201), Tue 10-12 OS (SV Lab1), Wed 2-4 Algo (SIV R104), Fri 11-1 Compiler (SVI R305). Dr. Menon (ECE): Mon 10-12 DSP (SV Lab2), Tue 9-11 VLSI (SV R401), Wed 9-11 Analog (SIV Lab3), Fri 10-12 Microwave (SVI R302).",
  "discipline":"Warning → fine (5K) → suspension (1sem) → rustication → FIR. Late 3x = parent notice. Lab = 5 marks. Plagiarism = 0 marks. Mobile exam = zero.",
},
}

# ==================== MODELS ====================
@st.cache_resource(show_spinner=False)
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept,docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try: coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}",metadata={"hnsw:space":"cosine"})
        if coll.count()==0:
            keys=list(docs.keys()); texts=list(docs.values())
            keys+=["_secret"]; texts+=[DEPARTMENTS[dept]["secret"]]
            vecs=emb.encode(texts,normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys],embeddings=vecs,documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    mdl=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb,device_map="auto")
    _=mdl.generate(**tok("hi",return_tensors="pt").to(mdl.device),max_new_tokens=1,pad_token_id=tok.eos_token_id)
    return emb,cc,tok,mdl,DEV
emb,cc,tok,mdl,DEV = setup()

def get_coll(d):
    try: return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}",metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r=USERS.get(u)
    if r and r["pw"]==_hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept,q):
    if not dept or dept not in DEPARTMENTS: return None,None
    ql=q.lower()
    for other,cfg in DEPARTMENTS.items():
        if other==dept: continue
        for alias in cfg.get("aliases",[])+[other]:
            if re.search(rf"\b{re.escape(alias)}\b",ql): return other,cfg["name"]
    return None,None

# ==================== RAG ====================
def chunk_text(t,size=500,ov=60):
    parts=re.split(r'(?<=[.!?])\s+',t.strip()); out,cur=[],""
    for s in parts:
        if len(cur)+len(s)+1<=size: cur=(cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur=((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf(pdf_bytes,fname):
    tmp=f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    open(tmp,"wb").write(pdf_bytes); pages=[]; err=""
    try:
        doc=pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t=pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err=f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    t=pg.extract_text() or ""
                    if t.strip(): pages.append({"text":t,"page":i+1})
            err=""
        except Exception as e: err+=f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages,err

def ingest_pdf(dept,fname,pdf_bytes,dt="regulation"):
    pages,err=extract_pdf(pdf_bytes,fname)
    if not pages: return 0,(err or "No text"),0
    coll=get_coll(dept); ch,mt,ids=[],[],[]
    safe=_safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":dt})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    try:
        allv=[]
        for i in range(0,len(ch),32):
            allv.extend(emb.encode(ch[i:i+32],normalize_embeddings=True,show_progress_bar=False).tolist())
        coll.add(ids=ids,embeddings=allv,documents=ch,metadatas=mt)
    except Exception as e: return 0,f"Chroma:{e}",len(pages)
    try: open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}","wb").write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch),"",len(pages)

def list_sources(dept):
    try:
        d=get_coll(dept).get(include=["metadatas"]); c={}
        for m in d["metadatas"]:
            s=m.get("source","?"); c[s]=c.get(s,0)+1
        return c
    except: return {}

def retrieve(dept,q,k=5):
    c=get_coll(dept)
    if c.count()==0: return []
    qv=emb.encode([q],normalize_embeddings=True).tolist()
    r=c.query(query_embeddings=qv,n_results=min(k,c.count()),
              include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0],r["metadatas"][0],r["distances"][0])]

# ==================== CONVERSATIONS DB (persistent memory) ====================
def init_convdb():
    c=sqlite3.connect(CONV_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS conversations (
        id TEXT PRIMARY KEY, user TEXT, dept TEXT, title TEXT,
        created_at TEXT, updated_at TEXT)""")
    c.execute("""CREATE TABLE IF NOT EXISTS messages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        conversation_id TEXT, role TEXT, content TEXT,
        sources TEXT, thinking TEXT, pdf_name TEXT,
        created_at TEXT)""")
    c.execute("CREATE INDEX IF NOT EXISTS idx_conv_user ON conversations(user, dept)")
    c.execute("CREATE INDEX IF NOT EXISTS idx_msg_conv ON messages(conversation_id)")
    c.connection.commit(); c.connection.close()
init_convdb()

def new_conversation(user, dept, title="New Chat"):
    cid = str(uuid.uuid4())[:8]
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("INSERT INTO conversations VALUES (?,?,?,?,?,?)",
              (cid, user, dept or "-", title, datetime.now().isoformat(), datetime.now().isoformat()))
    c.connection.commit(); c.connection.close()
    return cid

def list_conversations(user, dept, limit=100):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT id, title, created_at, updated_at FROM conversations
                 WHERE user=? AND dept=? ORDER BY updated_at DESC LIMIT ?""",
              (user, dept or "-", limit))
    rows = c.fetchall(); c.connection.close()
    return [{"id":r[0], "title":r[1], "created":r[2], "updated":r[3]} for r in rows]

def load_messages(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content, sources, thinking, pdf_name FROM messages
                 WHERE conversation_id=? ORDER BY id ASC""", (conv_id,))
    rows = c.fetchall(); c.connection.close()
    out = []
    for role, content, srcs, think, pdf_name in rows:
        try: srcs_l = json.loads(srcs) if srcs else []
        except: srcs_l = []
        out.append({"role":role, "content":content, "sources":srcs_l,
                    "thinking":think or "", "pdf_name":pdf_name or ""})
    return out

def save_message(conv_id, role, content, sources=None, thinking="", pdf_name=""):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("INSERT INTO messages (conversation_id, role, content, sources, thinking, pdf_name, created_at) VALUES (?,?,?,?,?,?,?)",
              (conv_id, role, content,
               json.dumps(sources or []), thinking, pdf_name,
               datetime.now().isoformat()))
    c.execute("UPDATE conversations SET updated_at=? WHERE id=?",
              (datetime.now().isoformat(), conv_id))
    # Auto-title on first user message
    if role == "user":
        c.execute("SELECT title FROM conversations WHERE id=?", (conv_id,))
        title = c.fetchone()[0]
        if title in ("New Chat","",None):
            new_title = content.strip().split("\n")[0][:50]
            c.execute("UPDATE conversations SET title=? WHERE id=?", (new_title, conv_id))
    c.connection.commit(); c.connection.close()

def delete_conversation(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("DELETE FROM messages WHERE conversation_id=?", (conv_id,))
    c.execute("DELETE FROM conversations WHERE id=?", (conv_id,))
    c.connection.commit(); c.connection.close()

def rename_conversation(conv_id, title):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("UPDATE conversations SET title=? WHERE id=?", (title[:80], conv_id))
    c.connection.commit(); c.connection.close()

def conversation_memory(conv_id, max_turns=6):
    """Return last N messages for LLM context (multi-turn memory)."""
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content FROM messages WHERE conversation_id=?
                 ORDER BY id DESC LIMIT ?""", (conv_id, max_turns*2))
    rows = c.fetchall()[::-1]; c.connection.close()
    return [{"role":r[0],"content":r[1]} for r in rows if r[0] in ("user","assistant")]

def conv_summary(conv_id, max_chars=800):
    """Compact summary of older messages to fit context."""
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content FROM messages WHERE conversation_id=?
                 ORDER BY id ASC LIMIT 20""", (conv_id,))
    rows = c.fetchall(); c.connection.close()
    if len(rows) <= 8: return ""
    older = rows[:-8]
    text = " ".join(f"{r[0]}: {r[1][:150]}" for r in older)
    return text[:max_chars]

def count_messages(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("SELECT COUNT(*) FROM messages WHERE conversation_id=?", (conv_id,))
    n = c.fetchone()[0]; c.connection.close()
    return n

# ==================== GENERATION ====================
if "stop_event" not in st.session_state: st.session_state.stop_event=threading.Event()

def _gen(msgs,mx=450):
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tok(text,return_tensors="pt").to(mdl.device)
    streamer=TextIteratorStreamer(tok,skip_prompt=True,skip_special_tokens=True)
    th=threading.Thread(target=mdl.generate,
        kwargs=dict(**inp,max_new_tokens=mx,do_sample=False,pad_token_id=tok.eos_token_id,streamer=streamer))
    th.start(); buf=""
    for t in streamer:
        if st.session_state.stop_event.is_set(): break
        buf+=t; yield buf

def stream_answer(dept,q,ph,extra="",deep=False,conv_id=None):
    hits = retrieve(dept,q,k=5)
    parts=[f"[{h['source']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0,f"[USER FILE]\n{extra[:2500]}")
    ctx="\n".join(parts) or "(no reference)"
    sys=("You are an expert faculty AI assistant. Answer ONLY from the reference below. "
         "Cite sources as [source_name]. Remember the conversation context above. Be precise.")
    msgs=[{"role":"system","content":sys}]
    # Inject memory from DB
    if conv_id:
        summary = conv_summary(conv_id)
        if summary:
            msgs.append({"role":"system","content":f"Prior conversation summary: {summary}"})
        history = conversation_memory(conv_id, max_turns=5)
        for h in history: msgs.append(h)
    if deep:
        think=[{"role":"system","content":"Analyze step by step briefly."},
               {"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalyze."}]
        tb=""
        for b in _gen(think,200): tb=b
        st.session_state.last_think=tb
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalysis:{tb}\nFinal:"})
    else:
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnswer:"})
    final=""
    for b in _gen(msgs,550):
        final=b; ph.markdown(b+("▌" if not st.session_state.stop_event.is_set() else " ⏹"))
    ph.markdown(final); return final,hits

def make_pdf(title,dept_name,subtitle,body):
    buf=io.BytesIO()
    doc=SimpleDocTemplate(buf,pagesize=A4,leftMargin=0.8*inch,rightMargin=0.8*inch,
                          topMargin=0.7*inch,bottomMargin=0.7*inch)
    S=getSampleStyleSheet()
    h1=ParagraphStyle("H1",parent=S["Heading1"],alignment=1,fontSize=16,spaceAfter=6)
    sb=ParagraphStyle("S",parent=S["Normal"],alignment=1,fontSize=10,spaceAfter=4)
    bd=ParagraphStyle("B",parent=S["Normal"],fontSize=11,leading=16,spaceAfter=8)
    story=[Paragraph(f"<b>{title}</b>",h1),Paragraph(f"Department of {dept_name}",sb)]
    if subtitle: story.append(Paragraph(subtitle,sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s=line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s,bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

def _slug(s): return re.sub(r'[^a-z0-9_]+','_',s.lower()).strip('_')[:40]
def is_doc_request(q):
    ql=q.lower()
    return any(t in ql for t in ["prepare pdf","make pdf","generate pdf","create pdf","question paper",
        "exam paper","generate questions","download pdf","generate notes","make notes","prepare notes"])

# ==================== ANALYTICS/AUDIT ====================
def init_audit():
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (id INTEGER PRIMARY KEY AUTOINCREMENT,
        ts TEXT, user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",(datetime.now().isoformat(),u,r or "-",d or "-",a,det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None,limit=500):
    c=sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else: c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows=c.fetchall(); c.connection.close(); return rows

def audit_df(dept=None):
    rows=get_audit(dept=dept,limit=2000)
    if not rows: return pd.DataFrame(columns=["ts","user","role","dept","action","detail"])
    df=pd.DataFrame(rows,columns=["ts","user","role","dept","action","detail"])
    df["ts"]=pd.to_datetime(df["ts"]); return df

def health_score(dept):
    try:
        chunks=get_coll(dept).count(); docs=len(list_sources(dept))
        df=audit_df(dept); week=df[df["ts"]>datetime.now()-timedelta(days=7)] if len(df) else pd.DataFrame()
        queries=len(week[week["action"]=="QUERY"]) if len(week) else 0
        blocked=len(week[week["action"]=="BLOCKED_QUERY"]) if len(week) else 0
        return int(min(docs/10,1)*25 + min(chunks/500,1)*25 + min(queries/50,1)*30 + max(0,1-blocked/10)*20)
    except: return 0

def anomalies(dept):
    out=[]; srcs=list_sources(dept)
    if any("students" in s for s in srcs):
        try:
            for src in srcs:
                if "student" in src.lower() and src in SEED.get(dept,{}):
                    df=pd.read_csv(io.StringIO(SEED[dept][src]))
                    if "Attendance" in df.columns:
                        for _,r in df[df["Attendance"]<75].iterrows():
                            out.append({"type":"Low attendance","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Attendance']}% — {'DETAINED' if r['Attendance']<65 else 'below 75%'}",
                                        "severity":"critical" if r["Attendance"]<65 else "warning"})
                    if "Status" in df.columns:
                        for _,r in df[df["Status"].isin(["OVERDUE","CRITICAL"])].iterrows():
                            out.append({"type":"Fee overdue","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"₹{r['Pending']:,} for {r['DueDays']} days",
                                        "severity":"critical" if r["DueDays"]>20 else "warning"})
                    if "Discipline" in df.columns:
                        for _,r in df[df["Discipline"]!="NONE"].iterrows():
                            out.append({"type":"Discipline","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Discipline']}","severity":"warning"})
                    break
        except: pass
    return out

# ==================== CSS ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide", initial_sidebar_state="expanded")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&display=swap');
html, body, [class*="css"] { font-family: 'Inter', -apple-system, sans-serif !important; }
.stApp { background: #f8fafc; }
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding-top: 1rem; padding-bottom: 2rem; max-width: 1500px; }

[data-testid="stSidebar"] {
  background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%) !important;
  border-right: 1px solid #1e293b;
}
[data-testid="stSidebar"] * { color: #e2e8f0 !important; }
[data-testid="stSidebar"] .stButton>button {
  width: 100%; text-align: left; background: transparent; border: none;
  color: #cbd5e1 !important; padding: 10px 14px; border-radius: 10px;
  font-weight: 500; font-size: 0.9rem; transition: all 0.15s;
}
[data-testid="stSidebar"] .stButton>button:hover { background: rgba(99,102,241,0.15); color: #fff !important; }
[data-testid="stSidebar"] .stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
  color: #fff !important; box-shadow: 0 4px 14px rgba(99,102,241,0.4);
}
[data-testid="stSidebar"] .stTextInput>div>div>input,
[data-testid="stSidebar"] .stTextArea>div>textarea {
  background: rgba(255,255,255,0.05) !important; color: #e2e8f0 !important;
  border: 1px solid rgba(255,255,255,0.1) !important;
}

h1, h2, h3, h4 { color: #0f172a; font-weight: 700; letter-spacing: -0.02em; }
h1 { font-size: 1.85rem; margin: 0 0 8px 0; }
h2 { font-size: 1.4rem; margin: 0 0 12px 0; }
h3 { font-size: 1.1rem; margin: 0 0 8px 0; }

.stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6);
  border: none; color: white; font-weight: 600;
  box-shadow: 0 4px 12px rgba(99,102,241,0.3);
}
.stButton>button[kind="primary"]:hover {
  transform: translateY(-1px); box-shadow: 0 6px 18px rgba(99,102,241,0.4);
}

.card { background: white; border-radius: 16px; padding: 20px;
  border: 1px solid #e2e8f0; box-shadow: 0 1px 3px rgba(0,0,0,0.03);
  transition: all 0.2s; }
.card:hover { box-shadow: 0 8px 24px rgba(15,23,42,0.08); }
.metric-card { background: white; border-radius: 14px; padding: 18px 20px;
  border: 1px solid #e2e8f0; position: relative; overflow: hidden; }
.metric-card::before { content: ''; position: absolute; top: 0; left: 0; width: 4px;
  height: 100%; background: linear-gradient(180deg, #6366f1, #8b5cf6); }
.metric-card .lbl { font-size: 0.72rem; color: #64748b; text-transform: uppercase;
  letter-spacing: 0.6px; font-weight: 600; }
.metric-card .val { font-size: 1.85rem; font-weight: 800; color: #0f172a;
  margin: 4px 0 2px; letter-spacing: -0.03em; }
.metric-card .sub { font-size: 0.78rem; color: #64748b; }

.pill { display: inline-flex; align-items: center; gap: 4px;
  padding: 4px 12px; border-radius: 100px; font-size: 0.7rem;
  font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px; }
.p-admin { background: #fef3c7; color: #92400e; }
.p-teacher { background: #e0e7ff; color: #3730a3; }
.p-success { background: #d1fae5; color: #065f46; }
.p-warn { background: #fef3c7; color: #92400e; }
.p-danger { background: #fee2e2; color: #991b1b; }

/* Chat bubbles */
[data-testid="stChatMessage"] { background: transparent !important; }
[data-testid="stChatMessage"] p { font-size: 0.92rem; line-height: 1.6; }

/* Conversation items */
.conv-item {
  padding: 10px 14px; border-radius: 10px; margin-bottom: 4px;
  background: rgba(255,255,255,0.03); border: 1px solid rgba(255,255,255,0.05);
  cursor: pointer; transition: all 0.15s;
}
.conv-item:hover { background: rgba(99,102,241,0.15); border-color: rgba(99,102,241,0.3); }
.conv-item.active { background: rgba(99,102,241,0.2); border-color: #6366f1; }
.conv-item .title { font-size: 0.85rem; font-weight: 500; color: #e2e8f0;
  white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.conv-item .time { font-size: 0.7rem; color: #64748b; margin-top: 2px; }

.empty { text-align: center; padding: 60px 20px; }
.empty .icon { font-size: 3.5rem; margin-bottom: 12px; opacity: 0.6; }
.empty .title { font-size: 1.1rem; font-weight: 600; color: #334155; }
.empty .sub { font-size: 0.88rem; color: #94a3b8; margin-top: 6px; }

.alert { padding: 12px 16px; border-radius: 10px; margin: 6px 0;
  border-left: 4px solid; font-size: 0.85rem; }
.alert-critical { background: #fef2f2; border-color: #ef4444; color: #7f1d1d; }
.alert-warning { background: #fffbeb; border-color: #f59e0b; color: #78350f; }
.alert-info { background: #eff6ff; border-color: #3b82f6; color: #1e40af; }

.dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-right: 6px; }
.dot-green { background: #10b981; box-shadow: 0 0 8px rgba(16,185,129,0.5); }

/* typing animation */
@keyframes pulse { 0%,100% { opacity:0.4 } 50% { opacity:1 } }
.typing span { display: inline-block; width: 6px; height: 6px; margin: 0 2px;
  border-radius: 50%; background: #6366f1; animation: pulse 1.4s infinite; }
.typing span:nth-child(2) { animation-delay: 0.2s; }
.typing span:nth-child(3) { animation-delay: 0.4s; }
</style>
""", unsafe_allow_html=True)

# ==================== SESSION ====================
for k in ("user","name","role","dept","logged_in","view","active_conv","conv_list_dirty"):
    st.session_state.setdefault(k,None)
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx=""
if "ctx_name" not in st.session_state: st.session_state.ctx_name=""
if "deep_think" not in st.session_state: st.session_state.deep_think=False
if "qa_trigger" not in st.session_state: st.session_state.qa_trigger=None
if "chat_topic" not in st.session_state: st.session_state.chat_topic=""
if "toast" not in st.session_state: st.session_state.toast=None
if st.session_state.view is None: st.session_state.view="Home"

# ==================== LOGIN ====================
if not st.session_state.logged_in:
    st.markdown("""
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 40%, #4c1d95 100%);
                padding: 60px 50px; border-radius: 24px; color: white; margin-bottom: 30px;
                position: relative; overflow: hidden;">
      <div style="position: absolute; top: -50px; right: -50px; width: 300px; height: 300px;
                  background: radial-gradient(circle, rgba(139,92,246,0.4), transparent 70%);
                  border-radius: 50%;"></div>
      <div style="position: relative; z-index: 1;">
        <div style="display: inline-block; padding: 6px 16px; background: rgba(255,255,255,0.1);
                    border: 1px solid rgba(255,255,255,0.15); border-radius: 100px;
                    font-size: 0.78rem; margin-bottom: 20px; letter-spacing: 1px;
                    font-weight: 600;">🎯 MU-PS-020 · SMART AUTOMATION</div>
        <h1 style="color: white; font-size: 3.2rem; font-weight: 900; margin: 0 0 12px 0;
                   letter-spacing: -0.04em; line-height: 1.1;">VaultMind ERP</h1>
        <p style="font-size: 1.15rem; opacity: 0.9; margin: 0; max-width: 700px; line-height: 1.5;">
        Sovereign AI. Persistent memory. Zero cloud.</p>
      </div>
    </div>
    """, unsafe_allow_html=True)
    c1, c2 = st.columns([3, 2])
    with c1:
        cols = st.columns(2)
        feats = [
          ("💾","Persistent Memory","Every conversation saved. Reload anytime."),
          ("🏛️","Department Isolation","Storage-level. CSE can never read Finance."),
          ("📊","Live Analytics","Charts, health scores, anomaly detection."),
          ("🎯","AI Exam Papers","Generate + download PDF inside the chat."),
          ("🧠","Deep Think","Two-stage reasoning for complex queries."),
          ("📎","Attach Anything","PDF & image OCR become AI context."),
        ]
        for i,(ic,t,d) in enumerate(feats):
            with cols[i%2]:
                st.markdown(f"""<div class="card" style="margin-bottom:12px;">
                  <div style="font-size:1.6rem;margin-bottom:6px;">{ic}</div>
                  <div style="font-weight:700;color:#0f172a;font-size:0.95rem;margin-bottom:4px;">{t}</div>
                  <div style="color:#64748b;font-size:0.82rem;line-height:1.5;">{d}</div>
                </div>""", unsafe_allow_html=True)
    with c2:
        st.markdown('<div class="card" style="padding: 26px;">', unsafe_allow_html=True)
        st.markdown("### 🔐 Faculty Sign In")
        u = st.text_input("Username", placeholder="teacher_cse", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("Sign In →", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True,"view":"Home",
                    "active_conv":None})
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
        st.markdown("---")
        st.markdown("**🎫 Demo Accounts**")
        st.code("""admin          / admin_pass
teacher_cse    / t_cse
teacher_ece    / t_ece
teacher_fin    / t_fin
teacher_admin  / t_admin""", language="text")
        st.markdown('</div>', unsafe_allow_html=True)
    st.stop()

# ==================== APP SHELL ====================
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
name = st.session_state.name
valid_dept = dept if (dept and dept in DEPARTMENTS) else None

# ---- SIDEBAR ----
with st.sidebar:
    st.markdown(f"""
    <div style="padding:8px 4px 16px;">
      <div style="display:flex;align-items:center;gap:10px;">
        <div style="width:36px;height:36px;background:linear-gradient(135deg,#6366f1,#8b5cf6);
                    border-radius:10px;display:flex;align-items:center;justify-content:center;
                    font-size:1.2rem;">🎓</div>
        <div>
          <div style="font-weight:800;font-size:1.05rem;">VaultMind</div>
          <div style="font-size:0.7rem;opacity:0.6;letter-spacing:0.5px;">ENTERPRISE ERP</div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    nav_items = []
    if is_admin:
        nav_items = [("Home","🏠 Home"),("Analytics","📊 Analytics"),
                     ("Departments","🏛️ Departments"),("Faculty","👥 Faculty"),
                     ("Uploads","📤 Upload KB"),("Audit","📜 Audit Log")]
    elif dept == "finance":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","💳 Fee Records"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    elif dept == "admin":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","🎓 Students"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    else:
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]

    for key, label in nav_items:
        bt = "primary" if st.session_state.view == key else "secondary"
        if st.button(label, key=f"nav_{key}", type=bt, use_container_width=True):
            st.session_state.view = key; st.rerun()

    # Conversation list (only in Assistant view)
    if st.session_state.view == "Assistant" and not is_admin:
        st.markdown("---")
        c1, c2 = st.columns([3,1])
        with c1:
            st.markdown(f'<div style="font-size:0.7rem;opacity:0.6;letter-spacing:0.5px;font-weight:700;">CONVERSATIONS</div>', unsafe_allow_html=True)
        with c2:
            if st.button("➕", key="new_conv_btn", help="New Chat"):
                new_id = new_conversation(user, dept or "-")
                st.session_state.active_conv = new_id
                st.session_state.conv_list_dirty = True
                st.rerun()

        convs = list_conversations(user, dept or "-")
        today = datetime.now().date()
        groups = {"Today":[], "Yesterday":[], "Last 7 days":[], "Older":[]}
        for cv in convs:
            try: upd = datetime.fromisoformat(cv["updated"]).date()
            except: upd = today
            d = (today - upd).days
            if d==0: groups["Today"].append(cv)
            elif d==1: groups["Yesterday"].append(cv)
            elif d<=7: groups["Last 7 days"].append(cv)
            else: groups["Older"].append(cv)

        for grp, items in groups.items():
            if not items: continue
            st.markdown(f'<div style="font-size:0.68rem;opacity:0.5;margin:8px 0 4px;letter-spacing:1px;text-transform:uppercase;">{grp}</div>', unsafe_allow_html=True)
            for cv in items:
                active = "active" if st.session_state.active_conv == cv["id"] else ""
                try: tstr = datetime.fromisoformat(cv["updated"]).strftime("%H:%M")
                except: tstr = ""
                c1, c2 = st.columns([5,1])
                with c1:
                    if st.button(f"💬 {cv['title'][:30]}{'…' if len(cv['title'])>30 else ''}",
                                 key=f"cv_{cv['id']}", use_container_width=True,
                                 help=f"Updated {cv['updated'][:16]}"):
                        st.session_state.active_conv = cv["id"]; st.rerun()
                with c2:
                    if st.button("✕", key=f"del_{cv['id']}", help="Delete"):
                        delete_conversation(cv["id"])
                        if st.session_state.active_conv == cv["id"]:
                            st.session_state.active_conv = None
                        st.session_state.conv_list_dirty = True
                        st.rerun()

    st.markdown("---")
    st.markdown(f"""
    <div style="padding:8px 4px;">
      <div style="font-size:0.7rem;opacity:0.6;margin-bottom:4px;">SIGNED IN AS</div>
      <div style="font-weight:600;font-size:0.9rem;">{name}</div>
      <div style="font-size:0.75rem;opacity:0.7;margin-top:4px;">
        <span class="pill p-{role}">{role}</span>
      </div>
      {f'<div style="margin-top:8px;font-size:0.78rem;color:{dcolor(dept)};"><span class="dot dot-green"></span>{dname(dept)}</div>' if valid_dept else ''}
    </div>
    """, unsafe_allow_html=True)

    if st.button("🚪 Sign Out", use_container_width=True, key="logout_btn"):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in","view","active_conv"): st.session_state[k]=None
        st.session_state.chat_ctx=""; st.session_state.chat_topic=""; st.rerun()

# ---- TOP BAR ----
c_left, c_right = st.columns([3,1])
with c_left:
    st.markdown(f"""
    <div style="padding:8px 0 12px 0;">
      <div style="font-size:0.78rem;color:#64748b;font-weight:500;margin-bottom:2px;">
        VaultMind / {st.session_state.view}
      </div>
      <h1 style="margin:0;">{st.session_state.view}</h1>
    </div>
    """, unsafe_allow_html=True)
with c_right:
    if valid_dept:
        try:
            n = get_coll(dept).count()
            st.markdown(f"""
            <div style="text-align:right;padding-top:12px;">
              <span class="pill p-success"><span class="dot dot-green"></span>AI Online</span>
              <div style="font-size:0.75rem;color:#64748b;margin-top:6px;">{n} chunks · {DEV.upper()}</div>
            </div>
            """, unsafe_allow_html=True)
        except: pass

# ==================== VIEW: HOME ====================
if st.session_state.view == "Home":
    if is_admin:
        total_chunks = sum(get_coll(d).count() for d in DEPARTMENTS)
        total_faculty = len([u for u in USERS.values() if u["role"]=="teacher"])
        all_df = audit_df()
        blocked_24h = len(all_df[(all_df["action"]=="BLOCKED_QUERY") & (all_df["ts"]>datetime.now()-timedelta(hours=24))]) if len(all_df) else 0
        weekly_queries = len(all_df[(all_df["action"]=="QUERY") & (all_df["ts"]>datetime.now()-timedelta(days=7))]) if len(all_df) else 0
        # Count conversations
        c = sqlite3.connect(CONV_DB).cursor()
        c.execute("SELECT COUNT(*) FROM conversations"); total_convs = c.fetchone()[0]
        c.execute("SELECT COUNT(*) FROM messages"); total_msgs = c.fetchone()[0]
        c.connection.close()

        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("Departments", len(DEPARTMENTS), "active"),
            ("Faculty", total_faculty, "accounts"),
            ("Conversations", total_convs, f"{total_msgs:,} messages"),
            ("Weekly queries", weekly_queries, f"{blocked_24h} blocked today")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        st.markdown("### 🏛️ Department Health")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            score = health_score(code); n = get_coll(code).count()
            srcs = list_sources(code)
            fac = sum(1 for u in USERS.values() if u.get("dept")==code)
            bar_color = "#10b981" if score>=70 else "#f59e0b" if score>=40 else "#ef4444"
            col.markdown(f"""
            <div class="card" style="border-left:4px solid {cfg['color']};">
              <div style="font-size:1.6rem;">{cfg.get('icon','📁')}</div>
              <div style="font-size:0.7rem;color:{cfg['color']};font-weight:700;letter-spacing:1px;margin:4px 0;">{code.upper()}</div>
              <div style="font-size:0.95rem;font-weight:700;margin-bottom:8px;">{cfg['name']}</div>
              <div style="display:flex;align-items:center;gap:6px;margin-bottom:8px;">
                <div style="flex:1;height:6px;background:#e2e8f0;border-radius:3px;overflow:hidden;">
                  <div style="width:{score}%;height:100%;background:{bar_color};"></div>
                </div>
                <span style="font-size:0.75rem;font-weight:700;color:{bar_color};">{score}</span>
              </div>
              <div style="font-size:0.75rem;color:#64748b;">📚 {n} chunks · 📄 {len(srcs)} docs · 👥 {fac}</div>
            </div>
            """, unsafe_allow_html=True)
    else:
        st.markdown(f"""
        <div style="background:linear-gradient(135deg,{dcolor(dept)}15,{dcolor(dept)}05);
                    border:1px solid {dcolor(dept)}30;border-radius:16px;padding:24px 28px;margin-bottom:24px;">
          <div style="font-size:0.75rem;color:{dcolor(dept)};font-weight:700;letter-spacing:1px;">WELCOME BACK</div>
          <h2 style="margin:6px 0 6px 0;font-size:1.6rem;">{name}</h2>
          <p style="color:#475569;margin:0;font-size:0.92rem;">
            You're in the <b>{dname(dept)}</b> workspace. Every conversation is saved. Reload anytime.
          </p>
        </div>
        """, unsafe_allow_html=True)
        srcs = list_sources(dept); n = get_coll(dept).count()
        df = audit_df(dept=dept)
        my_queries = len(df[df["user"]==user]) if len(df) else 0
        c = sqlite3.connect(CONV_DB).cursor()
        c.execute("SELECT COUNT(*) FROM conversations WHERE user=? AND dept=?", (user, dept))
        my_convs = c.fetchone()[0]; c.connection.close()
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("KB Chunks", n, "indexed"),
            ("My Conversations", my_convs, "saved"),
            ("Health Score", health_score(dept), "out of 100"),
            ("My Queries", my_queries, "all time")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        st.markdown("### ⚡ Quick Actions")
        actions = [("📝","Exam Paper","Generate question paper"),
                   ("📓","Study Notes","Structured notes PDF"),
                   ("🧠","Practice Quiz","MCQ + short answer"),
                   ("📊","Anomalies","At-risk students")]
        cols = st.columns(4)
        for col,(ic,t,d) in zip(cols, actions):
            with col:
                if st.button(f"{ic} {t}", use_container_width=True, key=f"qa_home_{t}"):
                    if t=="Anomalies": st.session_state.view="Anomalies"
                    else:
                        st.session_state.qa_trigger = {"Exam Paper":"exam","Study Notes":"notes","Practice Quiz":"quiz"}[t]
                        st.session_state.view="Assistant"
                    st.rerun()
                st.caption(d)

# ==================== VIEW: ASSISTANT (with persistent memory) ====================
elif st.session_state.view == "Assistant" and not is_admin:
    # Ensure an active conversation exists
    if not st.session_state.active_conv:
        convs = list_conversations(user, dept or "-")
        if convs: st.session_state.active_conv = convs[0]["id"]
        else:
            st.session_state.active_conv = new_conversation(user, dept or "-")
            st.rerun()

    active = st.session_state.active_conv
    if not active: st.error("Could not initialize conversation"); st.stop()

    # Load messages from DB
    history = load_messages(active)
    convs_all = list_conversations(user, dept or "-")
    conv_title = next((c["title"] for c in convs_all if c["id"]==active), "Chat")

    # Header
    c1, c2, c3 = st.columns([3,1,1])
    with c1:
        st.markdown(f"### 💬 {conv_title}")
        st.caption(f"{count_messages(active)} messages · 🧠 Remembers context · 🔒 Dept-scoped")
    with c2:
        st.session_state.deep_think = st.toggle("🧠 Deep Think", value=st.session_state.deep_think)
    with c3:
        if st.button("➕ New Chat", use_container_width=True, key="new_from_header"):
            st.session_state.active_conv = new_conversation(user, dept or "-")
            st.rerun()

    # Quick actions
    st.markdown("**⚡ Quick Actions**")
    qa_cols = st.columns(5)
    for col,(k,ic,lbl) in zip(qa_cols,[("exam","📝","Exam"),("notes","📓","Notes"),
                                        ("quiz","🧠","Quiz"),("sum","📄","Summarise"),
                                        ("explain","💡","Explain")]):
        with col:
            if st.button(f"{ic} {lbl}", key=f"qa_{k}", use_container_width=True):
                st.session_state.qa_trigger=k; st.rerun()

    topic = st.text_input("Topic / subject", key="chat_topic",
                          placeholder="e.g. Data Structures · Fee refund policy · Student S003 status")

    with st.expander("📎 Attach PDF or image (becomes AI context)"):
        cA, cB = st.columns(2)
        with cA:
            f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
            if f1 and st.button("Load PDF", key="btn_pdf"):
                pages, err = extract_pdf(f1.getvalue(), f1.name)
                if pages:
                    st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                    st.session_state.ctx_name = f1.name
                    st.success(f"✅ {len(pages)} pages")
                else: st.error(f"❌ {err}")
        with cB:
            f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
            if f2 and st.button("Run OCR", key="btn_ocr"):
                tmp=f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                open(tmp,"wb").write(f2.getvalue())
                try:
                    txt = pytesseract.image_to_string(Image.open(tmp))
                    st.session_state.chat_ctx = txt; st.session_state.ctx_name = f2.name
                    st.success(f"✅ {len(txt)} chars")
                except Exception as e: st.error(f"❌ {e}")
                finally:
                    try: os.remove(tmp)
                    except: pass
    if st.session_state.chat_ctx:
        cA, cB = st.columns([4,1])
        with cA:
            st.markdown(f'<div class="alert alert-info">📎 Attached: <b>{st.session_state.ctx_name}</b></div>', unsafe_allow_html=True)
        with cB:
            if st.button("Clear file", key="clr_file"):
                st.session_state.chat_ctx=""; st.session_state.ctx_name=""; st.rerun()

    # Chat area
    box = st.container(height=520)
    with box:
        if not history:
            st.markdown(f"""<div class="empty">
              <div class="icon">💬</div>
              <div class="title">Start a conversation with {dname(dept)} AI</div>
              <div class="sub">Every message is saved. Reload anytime — memory persists.</div>
            </div>""", unsafe_allow_html=True)
        for i, m in enumerate(history):
            with st.chat_message(m["role"]):
                st.markdown(m["content"])
                if m.get("thinking"):
                    with st.expander("🧠 Reasoning"): st.caption(m["thinking"])
                if m.get("sources"):
                    with st.expander(f"📄 {len(m['sources'])} source(s)"):
                        for s in m["sources"]: st.caption(f"`{s['source']}` · {s.get('similarity',0):.2f}")
                if m.get("pdf_name"):
                    st.caption(f"📄 Deliverable: `{m['pdf_name']}`")

    uq = st.chat_input("Ask a question… (memory on)")
    if st.session_state.qa_trigger:
        k = st.session_state.qa_trigger; st.session_state.qa_trigger=None
        prompts = {
          "exam": f"Prepare a question paper PDF. Subject: {topic or dname(dept)}. 10 questions, Sections A/B/C.",
          "notes": f"Generate study notes PDF on: {topic or 'the full course'}.",
          "quiz": f"Create a 5-question MCQ quiz PDF on: {topic or 'the course'}.",
          "sum": f"Summarise key rules about: {topic or 'department policy'}.",
          "explain": f"Explain in simple terms: {topic or 'the department fundamentals'}.",
        }
        uq = prompts[k]

    if uq:
        st.session_state.stop_event.clear()
        # Save user message to DB
        save_message(active, "user", uq)
        with box:
            with st.chat_message("user"): st.markdown(uq)
        # Check cross-dept
        other, on = detect_cross_dept(dept, uq)
        if other and other != dept:
            err = f"🚫 **Access denied.** Scoped to **{dname(dept)}**. Cannot query **{on}**."
            save_message(active, "assistant", err)
            with box:
                with st.chat_message("assistant"): st.error(err)
            log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
        else:
            with box:
                with st.chat_message("assistant"):
                    ph = st.empty(); stop_ph = st.empty()
                    stop_ph.button("⏹️ Stop", key=f"stop_{int(time.time())}",
                                   on_click=lambda: st.session_state.stop_event.set())
                    ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx,
                                              deep=st.session_state.deep_think,
                                              conv_id=active)
                    stop_ph.empty()
                    pdf_bytes=None; pdf_name=None
                    if is_doc_request(uq) or ans.count("Q1")+ans.count("1.")>=3:
                        title = "Examination Question Paper" if "question" in uq.lower() else "Document"
                        body = ans.replace("**","").replace("##","").replace("*","")
                        pdf_bytes = make_pdf(f"{title} — {topic or dname(dept)}",
                                             dname(dept),
                                             f"Generated by VaultMind AI · {datetime.now().strftime('%d %b %Y %H:%M')}",
                                             body)
                        pdf_name = f"{_slug(title)}_{_slug(topic or dept)}.pdf"
                        st.success("📄 PDF ready")
                        st.download_button(f"📥 {pdf_name}", pdf_bytes, file_name=pdf_name,
                                           mime="application/pdf", key=f"dl_new_{int(time.time())}")
                    if hits: st.caption(f"📄 {len(hits)} source(s)")
            # Save assistant message to DB
            save_message(active, "assistant", ans, sources=hits,
                         thinking=st.session_state.get("last_think","") if st.session_state.deep_think else "",
                         pdf_name=pdf_name or "")
            log(user,role,dept,"QUERY",uq[:60])
        st.rerun()

    # Footer actions
    if history:
        cA, cB, cC = st.columns([1,1,4])
        with cA:
            if st.button("🗑️ Clear Chat", use_container_width=True, key="clr_chat"):
                delete_conversation(active)
                st.session_state.active_conv = None
                st.rerun()
        with cB:
            if st.button("📥 Export PDF", use_container_width=True, key="exp_chat"):
                body = "\n\n".join(f"[{m['role'].upper()}]\n{m['content']}" for m in history)
                pdf = make_pdf("Chat Transcript", dname(dept),
                               f"Exported by {name} · {datetime.now().strftime('%d %b %Y %H:%M')}", body)
                st.download_button("📥 Download", pdf, file_name=f"chat_{_slug(dept)}.pdf",
                                   mime="application/pdf", key="dl_chat_exp")

# ==================== VIEW: ANALYTICS ====================
elif st.session_state.view == "Analytics":
    st.markdown("### 📊 Department Analytics")
    df = audit_df(dept=None if is_admin else dept)
    if len(df)==0:
        st.info("No activity yet.")
    else:
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("Total actions", len(df), "all time"),
            ("Queries", len(df[df["action"]=="QUERY"]), "AI interactions"),
            ("Blocked", len(df[df["action"]=="BLOCKED_QUERY"]), "security events"),
            ("Unique users", df["user"].nunique(), "active faculty")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        c1, c2 = st.columns(2)
        with c1:
            dc = df[df["action"]=="QUERY"].groupby("dept").size().reset_index(name="count")
            dc = dc[dc["dept"].isin(DEPARTMENTS.keys())]
            if len(dc):
                fig = px.bar(dc, x="dept", y="count", color="dept",
                             color_discrete_map={d: DEPARTMENTS[d]["color"] for d in DEPARTMENTS})
                fig.update_layout(showlegend=False, height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        with c2:
            df["day"] = df["ts"].dt.date
            daily = df[df["action"]=="QUERY"].groupby("day").size().reset_index(name="count")
            if len(daily):
                fig = px.line(daily, x="day", y="count", markers=True)
                fig.update_traces(line_color="#6366f1", line_width=3)
                fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        breakdown = df.groupby("action").size().reset_index(name="count")
        fig = px.pie(breakdown, values="count", names="action", hole=0.55,
                     color_discrete_sequence=["#6366f1","#8b5cf6","#10b981","#f59e0b","#ef4444"])
        fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10))
        st.plotly_chart(fig, use_container_width=True)

# ==================== VIEW: ANOMALIES ====================
elif st.session_state.view == "Anomalies":
    st.markdown("### ⚠️ Anomaly Detection")
    items = anomalies(dept) if valid_dept else []
    if not items: st.success("✅ No anomalies detected.")
    else:
        crit = [x for x in items if x["severity"]=="critical"]
        warn = [x for x in items if x["severity"]=="warning"]
        c1,c2 = st.columns(2)
        c1.markdown(f'<div class="metric-card"><div class="lbl">🚨 Critical</div><div class="val" style="color:#ef4444;">{len(crit)}</div><div class="sub">immediate action</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric-card"><div class="lbl">⚠️ Warnings</div><div class="val" style="color:#f59e0b;">{len(warn)}</div><div class="sub">needs review</div></div>', unsafe_allow_html=True)
        st.write("")
        for x in items:
            cls = "alert-critical" if x["severity"]=="critical" else "alert-warning"
            icon = "🚨" if x["severity"]=="critical" else "⚠️"
            st.markdown(f'<div class="alert {cls}"><b>{icon} {x["type"]} — {x["student"]}</b> ({x["roll"]})<br>{x["detail"]}</div>', unsafe_allow_html=True)

# ==================== VIEW: STUDENTS ====================
elif st.session_state.view == "Students" and dept in ("finance","admin"):
    st.markdown("### 🎓 Student Records")
    try:
        csv_data = SEED.get(dept,{}).get("students_csv","")
        df = pd.read_csv(io.StringIO(csv_data))
        c1,c2,c3 = st.columns(3)
        with c1: dept_filter = st.selectbox("Department", ["ALL"]+list(DEPARTMENTS.keys()), key="stu_d")
        with c2: yr_filter = st.selectbox("Year", ["ALL","III","V","VII"], key="stu_y")
        with c3: srch = st.text_input("Search", key="stu_s")
        view = df.copy()
        if dept_filter!="ALL" and "Dept" in view.columns: view = view[view["Dept"]==dept_filter]
        if yr_filter!="ALL" and "Year" in view.columns: view = view[view["Year"]==yr_filter]
        if srch: view = view[view.apply(lambda r: srch.lower() in str(r).lower(), axis=1)]
        st.markdown(f"**{len(view)} records**")
        st.dataframe(view, use_container_width=True, hide_index=True, height=380)
        if "Status" in df.columns:
            c1,c2 = st.columns(2)
            with c1:
                sc = df["Status"].value_counts().reset_index(); sc.columns=["Status","Count"]
                fig = px.pie(sc, values="Count", names="Status", hole=0.5,
                             color_discrete_map={"CLEARED":"#10b981","PARTIAL":"#f59e0b",
                                                 "OVERDUE":"#ef4444","CRITICAL":"#991b1b"})
                fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Fee Status")
                st.plotly_chart(fig, use_container_width=True)
            with c2:
                top = df.nlargest(8, "Pending")[["Name","Pending"]]
                fig = px.bar(top, x="Pending", y="Name", orientation="h", color_discrete_sequence=["#ef4444"])
                fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Top Pending (₹)")
                st.plotly_chart(fig, use_container_width=True)
        if "Attendance" in df.columns:
            fig = px.histogram(df, x="Attendance", nbins=10, color_discrete_sequence=["#6366f1"])
            fig.add_vline(x=75, line_dash="dash", line_color="red", annotation_text="Min 75%")
            fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Attendance Distribution")
            st.plotly_chart(fig, use_container_width=True)
    except Exception as e: st.error(f"Load failed: {e}")

# ==================== VIEW: DOCUMENTS ====================
elif st.session_state.view == "Documents" and not is_admin:
    st.markdown("### 📚 Knowledge Base")
    srcs = list_sources(dept) if valid_dept else {}
    if srcs:
        for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
            st.markdown(f"""<div class="card" style="margin-bottom:8px;">
              <div style="display:flex;justify-content:space-between;align-items:center;">
                <div><b>📄 {s}</b></div>
                <span class="pill p-success">{cnt} chunks</span>
              </div>
            </div>""", unsafe_allow_html=True)

# ==================== VIEW: DEPARTMENTS (Admin) ====================
elif st.session_state.view == "Departments" and is_admin:
    st.markdown("### 🏛️ Department Management")
    with st.expander("➕ Create New Department"):
        c1, c2 = st.columns(2)
        with c1:
            code = st.text_input("Code", key="nd_code").strip().lower()
            name2 = st.text_input("Display Name", key="nd_name")
            color = st.color_picker("Color", value="#6366f1", key="nd_color")
        with c2:
            sec = st.text_input("Secret", key="nd_sec")
            al = st.text_input("Aliases", key="nd_al")
        if st.button("Create Department", type="primary"):
            if not code or not name2: st.error("Required")
            elif code in DEPARTMENTS: st.error("Exists")
            else:
                DEPARTMENTS[code] = {"name":name2, "secret":sec or f"{code}-s-{int(time.time())}",
                    "color":color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code], "icon":"📁"}
                save_depts(); get_coll(code)
                log(user,role,None,"CREATE_DEPT",f"{code}:{name2}")
                st.success(f"✅ {name2}"); time.sleep(1); st.rerun()
    for code, cfg in DEPARTMENTS.items():
        n = get_coll(code).count(); srcs = list_sources(code); score = health_score(code)
        with st.expander(f"**{cfg['name']}** ({code}) — Score {score} · {n} chunks"):
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"- 📄 `{s}` ({cnt} chunks)")

# ==================== VIEW: FACULTY (Admin) ====================
elif st.session_state.view == "Faculty" and is_admin:
    st.markdown("### 👥 Faculty Management")
    with st.expander("➕ Add Faculty"):
        c1, c2 = st.columns(2)
        with c1:
            nu = st.text_input("Username", key="cr_u").strip().lower()
            np = st.text_input("Password", type="password", key="cr_p")
            nn = st.text_input("Full Name", key="cr_n")
        with c2:
            nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
        if st.button("Create Account", type="primary"):
            if not nu or not np or not nn: st.error("Required")
            elif nu in USERS: st.error("Exists")
            else:
                USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                st.success(f"✅ {nu}"); time.sleep(1); st.rerun()
    st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"}
                                for u,v in USERS.items()]), use_container_width=True, hide_index=True)

# ==================== VIEW: UPLOADS (Admin) ====================
elif st.session_state.view == "Uploads" and is_admin:
    st.markdown("### 📤 Upload to Department RAG")
    tgt = st.selectbox("Target Department", list(DEPARTMENTS.keys()),
                       format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_d")
    dtype = st.selectbox("Document Type", ["syllabus","regulation","notes","question-paper","reference","policy"], key="up_t")
    uploaded = st.file_uploader("Choose PDFs", type=["pdf"], accept_multiple_files=True, key="up_f")
    if uploaded and st.button("🚀 Ingest", type="primary", use_container_width=True):
        total=0; ok=0
        for f in uploaded:
            with st.spinner(f"Processing {f.name}…"):
                chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                if err: st.error(f"❌ {f.name} — {err}")
                else:
                    st.success(f"✅ {f.name} — {pages} pages → {chunks} chunks")
                    total+=chunks; ok+=1
        if ok:
            st.balloons()
            st.info(f"🎉 {ok}/{len(uploaded)} files · {total} chunks")
            log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
            time.sleep(2); st.rerun()

# ==================== VIEW: AUDIT (Admin) ====================
elif st.session_state.view == "Audit" and is_admin:
    st.markdown("### 📜 Audit Log")
    fd = st.selectbox("Filter by Department", ["ALL"]+list(DEPARTMENTS.keys()), key="au_d")
    df = audit_df(dept=None if fd=="ALL" else fd)
    if len(df):
        df["ts"] = df["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
        st.dataframe(df[["ts","user","role","dept","action","detail"]],
                     use_container_width=True, hide_index=True, height=500)
    st.divider()
    allr = audit_df()
    c1,c2,c3 = st.columns(3)
    c1.metric("🔒 Blocked", len(allr[allr["action"]=="BLOCKED_QUERY"]) if len(allr) else 0)
    c2.metric("✅ Leaks", 0); c3.metric("📊 Actions", len(allr))
'''
open("/content/vaultmind_erp.py","w").write(APP)
print(f"✅ App v9 written ({len(APP)} bytes)")
import shutil
shutil.copy("/content/vaultmind_erp.py", f"{DRIVE}/vaultmind_erp.py")

print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "--theme.base light --theme.primaryColor '#6366f1' "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(10)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Failed:"); print(open("/content/streamlit.log").read()[-1500:]); raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  💾 VAULTMIND ERP v9 — PERSISTENT MEMORY EDITION              ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                                                                ║
║  ✨ NEW IN v9:                                                 ║
║     • Conversations persist across sessions                    ║
║     • Model remembers last 5 turns of context                  ║
║     • Auto-summary for long chats                              ║
║     • ChatGPT-style sidebar (Today/Yesterday/Older)            ║
║     • ➕ New Chat button                                        ║
║     • ✕ Delete any conversation                                ║
║     • Auto-title from first message                            ║
║     • Everything stored in SQLite on Drive                     ║
╚════════════════════════════════════════════════════════════════╝
""")

📦 Installing…
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
✅ Ready
✅ App v9 written (68690 bytes)
🚀 Launching…
✅ Streamlit PID 56953

╔════════════════════════════════════════════════════════════════╗
║  💾 VAULTMIND ERP v9 — PERSISTENT MEMORY EDITION              ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-ass1c0-3f7aqrsinjeun-c.asia-southeast1-0.prod.colab.dev
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                            

In [ ]:
# ==================== VAULTMIND ERP v10 — MEMORY + VOICE ====================
import subprocess, sys, os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
for s in ["data","chroma_backup","logs","uploads"]:
    os.makedirs(f"{DRIVE}/{s}", exist_ok=True)
uf = f"{DRIVE}/data/users.json"
if os.path.exists(uf): os.remove(uf)

print("📦 Installing… (ffmpeg for audio + whisper deps)")
!apt-get install -y -q ffmpeg 2>&1 | tail -1
!pip install -q streamlit chromadb sentence-transformers "transformers>=4.45" \
    accelerate bitsandbytes python-docx pymupdf pytesseract pillow reportlab \
    pdfplumber pandas plotly openpyxl librosa soundfile 2>&1 | tail -1
print("✅ Ready")

APP = r'''
import os, json, re, time, threading, sqlite3, io, hashlib, shutil, uuid
from datetime import datetime, timedelta
import streamlit as st, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer, pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer
import chromadb, pymupdf, plotly.express as px
from PIL import Image
import pytesseract
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
import pandas as pd

DRIVE = "/content/drive/MyDrive/vaultmind"
CFG_DEPTS   = f"{DRIVE}/data/depts.json"
CFG_USERS   = f"{DRIVE}/data/users.json"
AUDIT_DB    = f"{DRIVE}/data/audit.db"
CONV_DB     = f"{DRIVE}/data/conversations.db"
LOCAL_CHROMA = "/content/chroma_local"
DRIVE_CHROMA = f"{DRIVE}/chroma_backup"
UPLOAD_DIR  = f"{DRIVE}/uploads"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID = "all-MiniLM-L6-v2"
WHISPER_ID = "openai/whisper-small"       # ~244M · Hindi + English + 97 languages
for d in [f"{DRIVE}/data", LOCAL_CHROMA, DRIVE_CHROMA, UPLOAD_DIR]:
    os.makedirs(d, exist_ok=True)

def _hash(p): return hashlib.sha256(p.encode()).hexdigest()
def _safe(s): return re.sub(r'[^a-zA-Z0-9._-]', '_', s)

if not os.path.exists(f"{LOCAL_CHROMA}/chroma.sqlite3") and os.path.exists(f"{DRIVE_CHROMA}/chroma.sqlite3"):
    try: shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA, dirs_exist_ok=True)
    except: pass

def backup_chroma():
    try:
        if os.path.exists(LOCAL_CHROMA):
            shutil.rmtree(DRIVE_CHROMA, ignore_errors=True)
            shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA, dirs_exist_ok=True)
    except: pass

# ==================== DEPARTMENTS ====================
PALETTE = ["#6366f1","#8b5cf6","#10b981","#f59e0b","#06b6d4"]
DEFAULT_DEPTS = {
  "cse":{"name":"Computer Science","color":"#6366f1","icon":"💻","secret":"aurora-lattice",
         "aliases":["cse","computer","comp sci","computer science"]},
  "ece":{"name":"Electronics","color":"#8b5cf6","icon":"📡","secret":"cobalt-meridian",
         "aliases":["ece","electronics","communication"]},
  "finance":{"name":"Finance & Accounts","color":"#10b981","icon":"💰","secret":"ledger-falcon",
             "aliases":["finance","accounts","accounting","budget","fees"]},
  "admin":{"name":"Administration","color":"#f59e0b","icon":"🏛️","secret":"orbit-vermillion",
           "aliases":["admin","administration","hr","registrar"]},
}
try: DEPARTMENTS = json.load(open(CFG_DEPTS))
except: DEPARTMENTS = DEFAULT_DEPTS.copy()
for code,cfg in DEFAULT_DEPTS.items(): DEPARTMENTS.setdefault(code,cfg)
for i,(c,cfg) in enumerate(DEPARTMENTS.items()):
    cfg.setdefault("name",c.upper()); cfg.setdefault("secret",f"{c}-s-{abs(hash(c))%9999}")
    cfg.setdefault("color",PALETTE[i%len(PALETTE)]); cfg.setdefault("icon","📁")
    cfg.setdefault("aliases",[c])
json.dump(DEPARTMENTS,open(CFG_DEPTS,"w"),indent=2)
def save_depts(): json.dump(DEPARTMENTS,open(CFG_DEPTS,"w"),indent=2)
def dname(d): return DEPARTMENTS.get(d,{}).get("name",d or "—")
def dcolor(d): return DEPARTMENTS.get(d,{}).get("color","#64748b")
def dicon(d): return DEPARTMENTS.get(d,{}).get("icon","📁")

# ==================== USERS ====================
DEFAULT_USERS = {
  "admin":{"name":"System Administrator","role":"admin","dept":None,"pw":_hash("admin_pass")},
  "teacher_cse":{"name":"Dr. Sharma","role":"teacher","dept":"cse","pw":_hash("t_cse")},
  "teacher_ece":{"name":"Dr. Menon","role":"teacher","dept":"ece","pw":_hash("t_ece")},
  "teacher_fin":{"name":"Mr. Verma","role":"teacher","dept":"finance","pw":_hash("t_fin")},
  "teacher_admin":{"name":"Ms. Rao","role":"teacher","dept":"admin","pw":_hash("t_admin")},
}
try:
    saved = json.load(open(CFG_USERS))
    USERS = DEFAULT_USERS.copy()
    for k,v in saved.items():
        if k not in USERS and v.get("role")!="student": USERS[k]=v
except: USERS = DEFAULT_USERS.copy()
json.dump(USERS,open(CFG_USERS,"w"),indent=2)
def save_users(): json.dump(USERS,open(CFG_USERS,"w"),indent=2)

# ==================== SEED (unchanged) ====================
SEED = {
"cse":{
  "syllabus":"B.Tech CSE MU 4yr/8sem. 3 branches: CSE, CSE(AI), CSE(BCT). Sem I: Engg Math-I, Programming in C, Physics→Electronics, Indian Traditions, Comp Fund, Web Dev-I, PBL-I. Sem II: Math-II, Python, Web Dev-II, Digital Circuits. Sem III: Discrete Math, Data Structures (24CSE1600T), OOP Java (24CSE1700T), RDBMS (24CSE1800T), ML Intro (24CSE1902T). Sem IV: Theory of Computation (CSE160), Algorithms (CSE170), OS (CSE280). Sem V-VI: Computer Architecture, Networks (CSE190), Compiler (CSE340), Deep Learning (CSE766), Data Mining (CSE776), Cyber Security (CSE900), Software Engg (CSE330), Android (CSE910), NLP (CSE781), Distributed (CSE891). Sem VIII: Entrepreneurship, E-Commerce, Cyber Law, Blockchain/ML, Big Data.",
  "exam_scheme":"MU 2024 scheme. Theory 3-4cr = 100 (EST 60 + MST 30 + CET 10). HSMC 2cr = 50. Practical 1cr = 50, 2cr = 100. PBL = 50 (25+25). Exams Dec/May-Jun, 3hr papers, morning 10-1, evening 2-5. Passing: 40% EST + 40% overall theory; 50% practical.",
  "grading":"8-point scale: 90+ O(10), 80-89 A+(9), 70-79 A(8), 60-69 B+(7), 50-59 B(6), 45-49 C(5), 40-44 P(4), <40 F(0). CGPA > 8 = First Division with Distinction.",
  "attendance":"Minimum 75% per theory course, 100% per lab. Condonation 65-74% with medical certificate within 3 days. Below 65% = detained. Late > 15 min = half-day.",
  "lab_rules":"CSE Labs Mon-Fri 9-12, 1-4. Sat 9-1 make-up. Batch max 20. 15 min late = absent. Max 2 missed labs. ID + closed footwear. No food/drinks. Penalty: 5 marks first violation.",
  "internship":"Summer-I after Sem IV: 4wk (160hr), 2cr. Summer-II after Sem VI: 6wk (240hr), 2cr. Offer + HOD-signed NOC ≥ 7 days before. 90% attendance. Report 25-40 pages TNR 12 spacing 1.5, similarity <20%. Eval: mentor 30 + report 30 + presentation 40. Pass 50/100.",
  "project":"PBL-I/II 1cr each. Major Project final year teams 3-4. Report A4 TNR 12, IEEE refs, 60-100 pages. Plagiarism <15%, single <2%. Late 5 marks/day. 4 hardbound. Viva: guide + 1 int + 1 ext.",
  "plagiarism":"Similarity >15% triggers review. Single source >2% flagged. Second offence = rustication. Tools: Turnitin.",
},
"ece":{
  "syllabus":"B.Tech ECE MU 4yr/8sem. Sem III: Probability & Stats, Electronic Devices & Circuits, Signals & Systems, Digital System Design, OOP C++. Sem IV: EM Field Theory, Analog Circuits, Analog Comm, Linear Control, Microprocessors. Sem V: DSP, Digital Comm, VLSI, Embedded, Antenna. Sem VI: Microwave, Info Theory, Computer Networks, Wireless. Sem VII: Optical Fiber, Satellite, IoT. Sem VIII: Radar & Navigation, Major Project II (200), Viva (100), Seminar (50).",
  "exam_scheme":"Same as CSE. 80 ECE courses. Theory min EST 24/100, min overall 40/100. Practical 25/50 or 50/100.",
  "lab_rules":"ECE Labs same as CSE. 10 labs: Electronic Devices, Digital Elec, Analog Circuits, Microprocessor, Communication, VLSI, DSP, Microwave, Embedded, Computer Networks. Never above 36V without faculty. Never look into powered waveguide.",
  "internship":"Same as CSE. Domain: embedded, VLSI, telecom, IoT. Six-month final sem: CGPA≥7.0.",
  "project":"Same format as CSE. ECE projects: sensor circuits, amplifiers, microcontroller apps, PCB design.",
  "labs_detail":"10 ECE labs: check polarity, don't hot-swap ICs, set function generator amplitude minimum, don't unplug kit while downloading, keep RF trainers away from water, logout EDA workstation, save every 15 min, never look into powered waveguide, disconnect USB before rewiring, don't change switch settings.",
  "attendance":"Same as CSE: 75% min, condonation 65-74%, <65% detained.",
},
"finance":{
  "fee_structure":"B.Tech/year 2025-26: Tuition 1,10,000 + Development 15,000 + Lab 10,000 + Library 3,000 + Sports 2,000 + Exam 5,000 = 1,45,000. Hostel AC 85,000, non-AC 65,000, mess-only 35,000. Payment 50/30/20.",
  "late_fee":"Rs 500/week up to 4 weeks. Beyond 4 weeks: hall ticket withheld. After 8 weeks: ID suspended. After 12 weeks: Dean referral. Waiver on documented hardship.",
  "refund":"UGC-compliant: 100% before 15 days of semester start, 90% within 15, 80% within 30, 50% after 30, NIL after 60.",
  "scholarship":"Merit CGPA≥9.0 → 25% waiver; 8.0-8.9 → 15%. MP Post-Matric, Central Sector, PMSSS. Sports 15-25%. Staff ward 50%. Sibling 10%.",
  "budget":"Annual April submission. Tiers: <25K HOD; 25K-1L HOD+Fin; 1-5L Dean+Fin+Reg; 5-25L Committee; >25L VC.",
  "vendor":"Registration GST+PAN+bank+ITR. Rate contract annual L1. Limited 1-5L. Open >5L. GeM mandatory. PO penalty 1%/week. Inspection 3-member for >50K. Payment 30 days. Blacklist 2yrs.",
  "students_csv":"Roll,Name,Dept,Year,Total,Paid,Pending,DueDays,Status\n24CSE001,Rahul Kumar,CSE,III,145000,110000,35000,0,PARTIAL\n24CSE002,Priya Sharma,CSE,III,145000,145000,0,0,CLEARED\n24CSE003,Amit Patel,CSE,III,145000,72500,72500,12,OVERDUE\n24CSE004,Sneha Iyer,CSE,III,145000,145000,0,0,CLEARED\n24ECE001,Vikram Singh,ECE,III,145000,130000,15000,0,PARTIAL\n24ECE002,Ananya Reddy,ECE,III,145000,145000,0,0,CLEARED\n24ECE003,Karan Mehta,ECE,III,145000,0,145000,25,CRITICAL\n24ECE004,Divya Nair,ECE,III,145000,145000,0,0,CLEARED\n23CSE021,Arjun Verma,CSE,V,145000,145000,0,0,CLEARED\n23CSE022,Riya Gupta,CSE,V,145000,100000,45000,5,PARTIAL\n23ECE015,Mohit Sharma,ECE,V,145000,145000,0,0,CLEARED\n23ECE016,Neha Joshi,ECE,V,145000,120000,25000,0,PARTIAL\n22CSE010,Rohit Kumar,CSE,VII,145000,145000,0,0,CLEARED\n22ECE008,Pooja Rao,ECE,VII,145000,90000,55000,8,OVERDUE\n22CSE011,Sameer Khan,CSE,VII,145000,145000,0,0,CLEARED",
},
"admin":{
  "leave":"Faculty: Casual 12/yr, Earned 15/yr (accum 300), Medical 10/yr, Maternity 180 days, Paternity 15 days. Approval: HOD → Registrar → VC.",
  "grievance":"Acknowledge 24hr, resolve 7-15 days. Levels: Coord → HOD → Dean → Committee → VC. POSH: ICC anonymous 90 days.",
  "events":"Application ≥30 days before. Chain: Dean SW → HOD → Registrar → VC. Budget ≤25K from dept. Police NOC >500.",
  "recruitment":"Asst Prof 60% PG, Assoc 5+ yr, Prof 10+ yr PhD. Ad 2 weeks. Demo 15 min + interview 30 min. Probation 1yr.",
  "students_csv":"Roll,Name,Dept,Year,Attendance,CL,ML,OD,Discipline,Hostel\n24CSE001,Rahul Kumar,CSE,III,78,5,2,0,NONE,A-214\n24CSE002,Priya Sharma,CSE,III,92,2,0,0,NONE,DAY\n24CSE003,Amit Patel,CSE,III,64,8,0,0,WARNING,B-108\n24CSE004,Sneha Iyer,CSE,III,88,3,0,0,NONE,A-305\n24ECE001,Vikram Singh,ECE,III,82,4,1,0,NONE,C-012\n24ECE002,Ananya Reddy,ECE,III,95,0,0,6,NONE,C-220\n24ECE003,Karan Mehta,ECE,III,71,7,2,0,WARNING,D-117\n24ECE004,Divya Nair,ECE,III,89,3,0,0,NONE,D-045\n23CSE021,Arjun Verma,CSE,V,84,5,1,0,NONE,A-402\n23CSE022,Riya Gupta,CSE,V,76,6,0,0,NONE,DAY\n23ECE015,Mohit Sharma,ECE,V,91,2,0,0,NONE,B-214\n23ECE016,Neha Joshi,ECE,V,87,4,0,0,NONE,B-311\n22CSE010,Rohit Kumar,CSE,VII,80,3,0,0,NONE,DAY\n22ECE008,Pooja Rao,ECE,VII,73,5,3,0,WARNING,C-118\n22CSE011,Sameer Khan,CSE,VII,88,2,0,0,NONE,A-510",
  "timetable":"Dr. Sharma (CSE): Mon 9-11 DS (SIII R201), Tue 10-12 OS (SV Lab1), Wed 2-4 Algo (SIV R104), Fri 11-1 Compiler (SVI R305). Dr. Menon (ECE): Mon 10-12 DSP (SV Lab2), Tue 9-11 VLSI (SV R401), Wed 9-11 Analog (SIV Lab3), Fri 10-12 Microwave (SVI R302).",
  "discipline":"Warning → fine (5K) → suspension (1sem) → rustication → FIR. Late 3x = parent notice. Lab = 5 marks. Plagiarism = 0 marks. Mobile exam = zero.",
},
}

# ==================== MODELS ====================
@st.cache_resource(show_spinner=False)
def setup():
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    emb = SentenceTransformer(EMBED_ID, device=DEV)
    cc = chromadb.PersistentClient(path=LOCAL_CHROMA)
    for dept,docs in SEED.items():
        if dept not in DEPARTMENTS: continue
        try: coll = cc.get_collection(f"kb_{dept}")
        except: coll = cc.create_collection(f"kb_{dept}",metadata={"hnsw:space":"cosine"})
        if coll.count()==0:
            keys=list(docs.keys()); texts=list(docs.values())
            keys+=["_secret"]; texts+=[DEPARTMENTS[dept]["secret"]]
            vecs=emb.encode(texts,normalize_embeddings=True).tolist()
            coll.add(ids=[f"{dept}_{k}" for k in keys],embeddings=vecs,documents=texts,
                     metadatas=[{"source":k,"page":1,"doc_type":"regulation"} for k in keys])
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    mdl=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb,device_map="auto")
    _=mdl.generate(**tok("hi",return_tensors="pt").to(mdl.device),max_new_tokens=1,pad_token_id=tok.eos_token_id)
    return emb,cc,tok,mdl,DEV
emb,cc,tok,mdl,DEV = setup()

# ==================== WHISPER (VOICE INPUT) ====================
@st.cache_resource(show_spinner=False)
def load_whisper():
    try:
        pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=WHISPER_ID,
            device=0 if torch.cuda.is_available() else -1,
            chunk_length_s=30,
        )
        return pipe
    except Exception as e:
        print(f"[whisper] load failed: {e}")
        return None

whisper_pipe = load_whisper()

def _has_devanagari(s): return any('\u0900' <= c <= '\u097F' for c in s)

def transcribe_multilingual(audio_bytes):
    """Transcribe + translate to English if non-English. Runs fully locally."""
    if whisper_pipe is None:
        return {"original":"(Whisper unavailable — check install)","language":"unknown","english":""}
    tmp = f"/content/_audio_{int(time.time())}.wav"
    with open(tmp,"wb") as fh: fh.write(audio_bytes)
    try:
        try:
            r = whisper_pipe(tmp, generate_kwargs={"task":"transcribe"}, return_language=True)
        except TypeError:
            r = whisper_pipe(tmp, generate_kwargs={"task":"transcribe"})
        original = (r.get("text","") if isinstance(r,dict) else str(r)).strip()
        lang = r.get("language","unknown") if isinstance(r,dict) else "unknown"
        english = original
        need_tr = _has_devanagari(original) or str(lang).lower() not in ("en","english","unknown")
        if need_tr:
            try:
                r2 = whisper_pipe(tmp, generate_kwargs={"task":"translate"})
                english = (r2.get("text","") if isinstance(r2,dict) else str(r2)).strip()
                if _has_devanagari(original): lang = "hindi"
            except Exception: pass
        return {"original":original, "language":lang, "english":english}
    except Exception as e:
        return {"original":f"(failed: {e})", "language":"unknown", "english":""}
    finally:
        try: os.remove(tmp)
        except: pass

def get_coll(d):
    try: return cc.get_collection(f"kb_{d}")
    except: return cc.create_collection(f"kb_{d}",metadata={"hnsw:space":"cosine"})

def verify_login(u,p):
    r=USERS.get(u)
    if r and r["pw"]==_hash(p): return {"user":u,"name":r["name"],"role":r["role"],"dept":r["dept"]}
    return None

def detect_cross_dept(dept,q):
    if not dept or dept not in DEPARTMENTS: return None,None
    ql=q.lower()
    for other,cfg in DEPARTMENTS.items():
        if other==dept: continue
        for alias in cfg.get("aliases",[])+[other]:
            if re.search(rf"\b{re.escape(alias)}\b",ql): return other,cfg["name"]
    return None,None

# ==================== RAG ====================
def chunk_text(t,size=500,ov=60):
    parts=re.split(r'(?<=[.!?])\s+',t.strip()); out,cur=[],""
    for s in parts:
        if len(cur)+len(s)+1<=size: cur=(cur+" "+s).strip()
        else:
            if cur: out.append(cur)
            cur=((cur[-ov:] if len(cur)>ov else cur)+" "+s).strip()
    if cur: out.append(cur)
    return out or [t]

def extract_pdf(pdf_bytes,fname):
    tmp=f"/content/_pdf_{hashlib.md5(fname.encode()).hexdigest()[:8]}.pdf"
    open(tmp,"wb").write(pdf_bytes); pages=[]; err=""
    try:
        doc=pymupdf.open(tmp)
        for i,pg in enumerate(doc):
            try:
                t=pg.get_text()
                if t and t.strip(): pages.append({"text":t,"page":i+1})
            except: pass
        doc.close()
    except Exception as e: err=f"pymupdf:{e}"
    if not pages:
        try:
            import pdfplumber
            with pdfplumber.open(tmp) as pdf:
                for i,pg in enumerate(pdf.pages):
                    t=pg.extract_text() or ""
                    if t.strip(): pages.append({"text":t,"page":i+1})
            err=""
        except Exception as e: err+=f"|pdfplumber:{e}"
    try: os.remove(tmp)
    except: pass
    return pages,err

def ingest_pdf(dept,fname,pdf_bytes,dt="regulation"):
    pages,err=extract_pdf(pdf_bytes,fname)
    if not pages: return 0,(err or "No text"),0
    coll=get_coll(dept); ch,mt,ids=[],[],[]
    safe=_safe(fname)
    for pg in pages:
        for i,c in enumerate(chunk_text(pg["text"])):
            ch.append(c); mt.append({"source":fname,"page":pg["page"],"doc_type":dt})
            ids.append(f"{dept}_{safe}_p{pg['page']}_c{i}_{int(time.time())}")
    try:
        allv=[]
        for i in range(0,len(ch),32):
            allv.extend(emb.encode(ch[i:i+32],normalize_embeddings=True,show_progress_bar=False).tolist())
        coll.add(ids=ids,embeddings=allv,documents=ch,metadatas=mt)
    except Exception as e: return 0,f"Chroma:{e}",len(pages)
    try: open(f"{UPLOAD_DIR}/{_safe(dept)}_{safe}","wb").write(pdf_bytes)
    except: pass
    backup_chroma()
    return len(ch),"",len(pages)

def list_sources(dept):
    try:
        d=get_coll(dept).get(include=["metadatas"]); c={}
        for m in d["metadatas"]:
            s=m.get("source","?"); c[s]=c.get(s,0)+1
        return c
    except: return {}

def retrieve(dept,q,k=5):
    c=get_coll(dept)
    if c.count()==0: return []
    qv=emb.encode([q],normalize_embeddings=True).tolist()
    r=c.query(query_embeddings=qv,n_results=min(k,c.count()),
              include=["documents","metadatas","distances"])
    return [{"text":d,"source":m["source"],"page":m.get("page",1),"similarity":1-ds}
            for d,m,ds in zip(r["documents"][0],r["metadatas"][0],r["distances"][0])]

# ==================== CONVERSATIONS (persistent memory) ====================
def init_convdb():
    c=sqlite3.connect(CONV_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS conversations (
        id TEXT PRIMARY KEY, user TEXT, dept TEXT, title TEXT,
        created_at TEXT, updated_at TEXT)""")
    c.execute("""CREATE TABLE IF NOT EXISTS messages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        conversation_id TEXT, role TEXT, content TEXT,
        sources TEXT, thinking TEXT, pdf_name TEXT,
        created_at TEXT)""")
    c.execute("CREATE INDEX IF NOT EXISTS idx_conv_user ON conversations(user, dept)")
    c.execute("CREATE INDEX IF NOT EXISTS idx_msg_conv ON messages(conversation_id)")
    c.connection.commit(); c.connection.close()
init_convdb()

def new_conversation(user, dept, title="New Chat"):
    cid = str(uuid.uuid4())[:8]
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("INSERT INTO conversations VALUES (?,?,?,?,?,?)",
              (cid, user, dept or "-", title, datetime.now().isoformat(), datetime.now().isoformat()))
    c.connection.commit(); c.connection.close()
    return cid

def list_conversations(user, dept, limit=100):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT id, title, created_at, updated_at FROM conversations
                 WHERE user=? AND dept=? ORDER BY updated_at DESC LIMIT ?""",
              (user, dept or "-", limit))
    rows = c.fetchall(); c.connection.close()
    return [{"id":r[0], "title":r[1], "created":r[2], "updated":r[3]} for r in rows]

def load_messages(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content, sources, thinking, pdf_name FROM messages
                 WHERE conversation_id=? ORDER BY id ASC""", (conv_id,))
    rows = c.fetchall(); c.connection.close()
    out = []
    for role, content, srcs, think, pdf_name in rows:
        try: srcs_l = json.loads(srcs) if srcs else []
        except: srcs_l = []
        out.append({"role":role, "content":content, "sources":srcs_l,
                    "thinking":think or "", "pdf_name":pdf_name or ""})
    return out

def save_message(conv_id, role, content, sources=None, thinking="", pdf_name=""):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("INSERT INTO messages (conversation_id, role, content, sources, thinking, pdf_name, created_at) VALUES (?,?,?,?,?,?,?)",
              (conv_id, role, content, json.dumps(sources or []), thinking, pdf_name,
               datetime.now().isoformat()))
    c.execute("UPDATE conversations SET updated_at=? WHERE id=?",
              (datetime.now().isoformat(), conv_id))
    if role == "user":
        c.execute("SELECT title FROM conversations WHERE id=?", (conv_id,))
        title = c.fetchone()[0]
        if title in ("New Chat","",None):
            new_title = content.strip().split("\n")[0][:50]
            c.execute("UPDATE conversations SET title=? WHERE id=?", (new_title, conv_id))
    c.connection.commit(); c.connection.close()

def delete_conversation(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("DELETE FROM messages WHERE conversation_id=?", (conv_id,))
    c.execute("DELETE FROM conversations WHERE id=?", (conv_id,))
    c.connection.commit(); c.connection.close()

def conversation_memory(conv_id, max_turns=5):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content FROM messages WHERE conversation_id=?
                 ORDER BY id DESC LIMIT ?""", (conv_id, max_turns*2))
    rows = c.fetchall()[::-1]; c.connection.close()
    return [{"role":r[0],"content":r[1]} for r in rows if r[0] in ("user","assistant")]

def conv_summary(conv_id, max_chars=800):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("""SELECT role, content FROM messages WHERE conversation_id=?
                 ORDER BY id ASC LIMIT 20""", (conv_id,))
    rows = c.fetchall(); c.connection.close()
    if len(rows) <= 8: return ""
    older = rows[:-8]
    text = " ".join(f"{r[0]}: {r[1][:150]}" for r in older)
    return text[:max_chars]

def count_messages(conv_id):
    c = sqlite3.connect(CONV_DB).cursor()
    c.execute("SELECT COUNT(*) FROM messages WHERE conversation_id=?", (conv_id,))
    n = c.fetchone()[0]; c.connection.close()
    return n

# ==================== GENERATION ====================
if "stop_event" not in st.session_state: st.session_state.stop_event=threading.Event()

def _gen(msgs,mx=450):
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tok(text,return_tensors="pt").to(mdl.device)
    streamer=TextIteratorStreamer(tok,skip_prompt=True,skip_special_tokens=True)
    th=threading.Thread(target=mdl.generate,
        kwargs=dict(**inp,max_new_tokens=mx,do_sample=False,pad_token_id=tok.eos_token_id,streamer=streamer))
    th.start(); buf=""
    for t in streamer:
        if st.session_state.stop_event.is_set(): break
        buf+=t; yield buf

def stream_answer(dept,q,ph,extra="",deep=False,conv_id=None):
    hits=retrieve(dept,q,k=5)
    parts=[f"[{h['source']}] {h['text']}" for h in hits]
    if extra.strip(): parts.insert(0,f"[USER FILE]\n{extra[:2500]}")
    ctx="\n".join(parts) or "(no reference)"
    sys=("You are an expert faculty AI assistant. Answer ONLY from the reference below. "
         "Cite sources as [source_name]. Remember the conversation context above. Be precise.")
    msgs=[{"role":"system","content":sys}]
    if conv_id:
        summary = conv_summary(conv_id)
        if summary: msgs.append({"role":"system","content":f"Prior conversation summary: {summary}"})
        for h in conversation_memory(conv_id, max_turns=5): msgs.append(h)
    if deep:
        think=[{"role":"system","content":"Analyze step by step briefly."},
               {"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalyze."}]
        tb=""
        for b in _gen(think,200): tb=b
        st.session_state.last_think=tb
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnalysis:{tb}\nFinal:"})
    else:
        msgs.append({"role":"user","content":f"Q:{q}\nRef:\n{ctx}\nAnswer:"})
    final=""
    for b in _gen(msgs,550):
        final=b; ph.markdown(b+("▌" if not st.session_state.stop_event.is_set() else " ⏹"))
    ph.markdown(final); return final,hits

def make_pdf(title,dept_name,subtitle,body):
    buf=io.BytesIO()
    doc=SimpleDocTemplate(buf,pagesize=A4,leftMargin=0.8*inch,rightMargin=0.8*inch,
                          topMargin=0.7*inch,bottomMargin=0.7*inch)
    S=getSampleStyleSheet()
    h1=ParagraphStyle("H1",parent=S["Heading1"],alignment=1,fontSize=16,spaceAfter=6)
    sb=ParagraphStyle("S",parent=S["Normal"],alignment=1,fontSize=10,spaceAfter=4)
    bd=ParagraphStyle("B",parent=S["Normal"],fontSize=11,leading=16,spaceAfter=8)
    story=[Paragraph(f"<b>{title}</b>",h1),Paragraph(f"Department of {dept_name}",sb)]
    if subtitle: story.append(Paragraph(subtitle,sb))
    story.append(Spacer(1,14))
    for line in body.split("\n"):
        if not line.strip(): continue
        s=line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        story.append(Paragraph(s,bd))
    doc.build(story); buf.seek(0); return buf.getvalue()

def _slug(s): return re.sub(r'[^a-z0-9_]+','_',s.lower()).strip('_')[:40]
def is_doc_request(q):
    ql=q.lower()
    return any(t in ql for t in ["prepare pdf","make pdf","generate pdf","create pdf","question paper",
        "exam paper","generate questions","download pdf","generate notes","make notes","prepare notes"])

# ==================== ANALYTICS/AUDIT ====================
def init_audit():
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS log (id INTEGER PRIMARY KEY AUTOINCREMENT,
        ts TEXT, user TEXT, role TEXT, dept TEXT, action TEXT, detail TEXT)""")
    c.connection.commit(); c.connection.close()
init_audit()

def log(u,r,d,a,det):
    c=sqlite3.connect(AUDIT_DB).cursor()
    c.execute("INSERT INTO log VALUES (NULL,?,?,?,?,?,?)",(datetime.now().isoformat(),u,r or "-",d or "-",a,det))
    c.connection.commit(); c.connection.close()

def get_audit(dept=None,limit=500):
    c=sqlite3.connect(AUDIT_DB).cursor()
    if dept: c.execute("SELECT ts,user,role,dept,action,detail FROM log WHERE dept=? ORDER BY id DESC LIMIT ?",(dept,limit))
    else: c.execute("SELECT ts,user,role,dept,action,detail FROM log ORDER BY id DESC LIMIT ?",(limit,))
    rows=c.fetchall(); c.connection.close(); return rows

def audit_df(dept=None):
    rows=get_audit(dept=dept,limit=2000)
    if not rows: return pd.DataFrame(columns=["ts","user","role","dept","action","detail"])
    df=pd.DataFrame(rows,columns=["ts","user","role","dept","action","detail"])
    df["ts"]=pd.to_datetime(df["ts"]); return df

def health_score(dept):
    try:
        chunks=get_coll(dept).count(); docs=len(list_sources(dept))
        df=audit_df(dept); week=df[df["ts"]>datetime.now()-timedelta(days=7)] if len(df) else pd.DataFrame()
        q=len(week[week["action"]=="QUERY"]) if len(week) else 0
        b=len(week[week["action"]=="BLOCKED_QUERY"]) if len(week) else 0
        return int(min(docs/10,1)*25 + min(chunks/500,1)*25 + min(q/50,1)*30 + max(0,1-b/10)*20)
    except: return 0

def anomalies(dept):
    out=[]; srcs=list_sources(dept)
    if any("students" in s for s in srcs):
        try:
            for src in srcs:
                if "student" in src.lower() and src in SEED.get(dept,{}):
                    df=pd.read_csv(io.StringIO(SEED[dept][src]))
                    if "Attendance" in df.columns:
                        for _,r in df[df["Attendance"]<75].iterrows():
                            out.append({"type":"Low attendance","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Attendance']}% — {'DETAINED' if r['Attendance']<65 else 'below 75%'}",
                                        "severity":"critical" if r["Attendance"]<65 else "warning"})
                    if "Status" in df.columns:
                        for _,r in df[df["Status"].isin(["OVERDUE","CRITICAL"])].iterrows():
                            out.append({"type":"Fee overdue","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"₹{r['Pending']:,} for {r['DueDays']} days",
                                        "severity":"critical" if r["DueDays"]>20 else "warning"})
                    if "Discipline" in df.columns:
                        for _,r in df[df["Discipline"]!="NONE"].iterrows():
                            out.append({"type":"Discipline","student":r["Name"],"roll":r["Roll"],
                                        "detail":f"{r['Discipline']}","severity":"warning"})
                    break
        except: pass
    return out

# ==================== CSS ====================
st.set_page_config(page_title="VaultMind ERP", page_icon="🎓", layout="wide", initial_sidebar_state="expanded")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&display=swap');
html, body, [class*="css"] { font-family: 'Inter', -apple-system, sans-serif !important; }
.stApp { background: #f8fafc; }
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding-top: 1rem; padding-bottom: 2rem; max-width: 1500px; }

[data-testid="stSidebar"] {
  background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%) !important;
  border-right: 1px solid #1e293b;
}
[data-testid="stSidebar"] * { color: #e2e8f0 !important; }
[data-testid="stSidebar"] .stButton>button {
  width: 100%; text-align: left; background: transparent; border: none;
  color: #cbd5e1 !important; padding: 10px 14px; border-radius: 10px;
  font-weight: 500; font-size: 0.9rem; transition: all 0.15s;
}
[data-testid="stSidebar"] .stButton>button:hover { background: rgba(99,102,241,0.15); color: #fff !important; }
[data-testid="stSidebar"] .stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
  color: #fff !important; box-shadow: 0 4px 14px rgba(99,102,241,0.4);
}
[data-testid="stSidebar"] .stTextInput>div>div>input,
[data-testid="stSidebar"] .stTextArea>div>textarea {
  background: rgba(255,255,255,0.05) !important; color: #e2e8f0 !important;
  border: 1px solid rgba(255,255,255,0.1) !important;
}

h1, h2, h3, h4 { color: #0f172a; font-weight: 700; letter-spacing: -0.02em; }
h1 { font-size: 1.85rem; margin: 0 0 8px 0; }
h2 { font-size: 1.4rem; margin: 0 0 12px 0; }
h3 { font-size: 1.1rem; margin: 0 0 8px 0; }

.stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #6366f1, #8b5cf6);
  border: none; color: white; font-weight: 600;
  box-shadow: 0 4px 12px rgba(99,102,241,0.3);
}
.stButton>button[kind="primary"]:hover {
  transform: translateY(-1px); box-shadow: 0 6px 18px rgba(99,102,241,0.4);
}

.card { background: white; border-radius: 16px; padding: 20px;
  border: 1px solid #e2e8f0; box-shadow: 0 1px 3px rgba(0,0,0,0.03);
  transition: all 0.2s; }
.card:hover { box-shadow: 0 8px 24px rgba(15,23,42,0.08); }
.metric-card { background: white; border-radius: 14px; padding: 18px 20px;
  border: 1px solid #e2e8f0; position: relative; overflow: hidden; }
.metric-card::before { content: ''; position: absolute; top: 0; left: 0; width: 4px;
  height: 100%; background: linear-gradient(180deg, #6366f1, #8b5cf6); }
.metric-card .lbl { font-size: 0.72rem; color: #64748b; text-transform: uppercase;
  letter-spacing: 0.6px; font-weight: 600; }
.metric-card .val { font-size: 1.85rem; font-weight: 800; color: #0f172a;
  margin: 4px 0 2px; letter-spacing: -0.03em; }
.metric-card .sub { font-size: 0.78rem; color: #64748b; }

.pill { display: inline-flex; align-items: center; gap: 4px;
  padding: 4px 12px; border-radius: 100px; font-size: 0.7rem;
  font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px; }
.p-admin { background: #fef3c7; color: #92400e; }
.p-teacher { background: #e0e7ff; color: #3730a3; }
.p-success { background: #d1fae5; color: #065f46; }
.p-warn { background: #fef3c7; color: #92400e; }
.p-danger { background: #fee2e2; color: #991b1b; }
.p-voice { background: #fce7f3; color: #9d174d; }

[data-testid="stChatMessage"] { background: transparent !important; }
[data-testid="stChatMessage"] p { font-size: 0.92rem; line-height: 1.6; }

.conv-item { padding: 10px 14px; border-radius: 10px; margin-bottom: 4px;
  background: rgba(255,255,255,0.03); border: 1px solid rgba(255,255,255,0.05); }
.empty { text-align: center; padding: 60px 20px; }
.empty .icon { font-size: 3.5rem; margin-bottom: 12px; opacity: 0.6; }
.empty .title { font-size: 1.1rem; font-weight: 600; color: #334155; }
.empty .sub { font-size: 0.88rem; color: #94a3b8; margin-top: 6px; }

.alert { padding: 12px 16px; border-radius: 10px; margin: 6px 0;
  border-left: 4px solid; font-size: 0.85rem; }
.alert-critical { background: #fef2f2; border-color: #ef4444; color: #7f1d1d; }
.alert-warning { background: #fffbeb; border-color: #f59e0b; color: #78350f; }
.alert-info { background: #eff6ff; border-color: #3b82f6; color: #1e40af; }
.alert-voice { background: #fdf2f8; border-color: #ec4899; color: #831843; }

.dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-right: 6px; }
.dot-green { background: #10b981; box-shadow: 0 0 8px rgba(16,185,129,0.5); }
.dot-voice { background: #ec4899; box-shadow: 0 0 8px rgba(236,72,153,0.5); }
</style>
""", unsafe_allow_html=True)

# ==================== SESSION ====================
for k in ("user","name","role","dept","logged_in","view","active_conv"):
    st.session_state.setdefault(k,None)
if "chat_ctx" not in st.session_state: st.session_state.chat_ctx=""
if "ctx_name" not in st.session_state: st.session_state.ctx_name=""
if "deep_think" not in st.session_state: st.session_state.deep_think=False
if "qa_trigger" not in st.session_state: st.session_state.qa_trigger=None
if "chat_topic" not in st.session_state: st.session_state.chat_topic=""
if "voice_result" not in st.session_state: st.session_state.voice_result=None
if "voice_input" not in st.session_state: st.session_state.voice_input=""
if st.session_state.view is None: st.session_state.view="Home"

# ==================== LOGIN ====================
if not st.session_state.logged_in:
    st.markdown("""
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 40%, #4c1d95 100%);
                padding: 60px 50px; border-radius: 24px; color: white; margin-bottom: 30px;
                position: relative; overflow: hidden;">
      <div style="position: absolute; top: -50px; right: -50px; width: 300px; height: 300px;
                  background: radial-gradient(circle, rgba(139,92,246,0.4), transparent 70%);
                  border-radius: 50%;"></div>
      <div style="position: relative; z-index: 1;">
        <div style="display: inline-block; padding: 6px 16px; background: rgba(255,255,255,0.1);
                    border: 1px solid rgba(255,255,255,0.15); border-radius: 100px;
                    font-size: 0.78rem; margin-bottom: 20px; letter-spacing: 1px;
                    font-weight: 600;">🎯 MU-PS-020 · SMART AUTOMATION</div>
        <h1 style="color: white; font-size: 3.2rem; font-weight: 900; margin: 0 0 12px 0;
                   letter-spacing: -0.04em; line-height: 1.1;">VaultMind ERP</h1>
        <p style="font-size: 1.15rem; opacity: 0.9; margin: 0; max-width: 700px; line-height: 1.5;">
        Sovereign AI · Voice input in Hindi &amp; English · Persistent memory · Zero cloud.</p>
      </div>
    </div>
    """, unsafe_allow_html=True)
    c1, c2 = st.columns([3, 2])
    with c1:
        cols = st.columns(2)
        feats = [
          ("🎤","Voice Input","Speak Hindi or English. Whisper transcribes locally."),
          ("💾","Persistent Memory","Every conversation saved. Reload anytime."),
          ("🏛️","Department Isolation","Storage-level. CSE can never read Finance."),
          ("📊","Live Analytics","Charts, health scores, anomaly detection."),
          ("🎯","AI Exam Papers","Generate + download PDF inside the chat."),
          ("🧠","Deep Think","Two-stage reasoning for complex queries."),
        ]
        for i,(ic,t,d) in enumerate(feats):
            with cols[i%2]:
                st.markdown(f"""<div class="card" style="margin-bottom:12px;">
                  <div style="font-size:1.6rem;margin-bottom:6px;">{ic}</div>
                  <div style="font-weight:700;color:#0f172a;font-size:0.95rem;margin-bottom:4px;">{t}</div>
                  <div style="color:#64748b;font-size:0.82rem;line-height:1.5;">{d}</div>
                </div>""", unsafe_allow_html=True)
    with c2:
        st.markdown('<div class="card" style="padding: 26px;">', unsafe_allow_html=True)
        st.markdown("### 🔐 Faculty Sign In")
        u = st.text_input("Username", placeholder="teacher_cse", key="lg_u")
        p = st.text_input("Password", type="password", key="lg_p")
        if st.button("Sign In →", type="primary", use_container_width=True):
            r = verify_login(u.strip(), p)
            if r:
                st.session_state.update({"user":r["user"],"name":r["name"],
                    "role":r["role"],"dept":r["dept"],"logged_in":True,"view":"Home",
                    "active_conv":None})
                log(r["user"],r["role"],r["dept"],"LOGIN","success"); st.rerun()
            else: st.error("Invalid credentials")
        st.markdown("---")
        st.markdown("**🎫 Demo Accounts**")
        st.code("""admin          / admin_pass
teacher_cse    / t_cse
teacher_ece    / t_ece
teacher_fin    / t_fin
teacher_admin  / t_admin""", language="text")
        st.markdown('</div>', unsafe_allow_html=True)
    st.stop()

# ==================== APP SHELL ====================
role = st.session_state.role; user = st.session_state.user
dept = st.session_state.dept; is_admin = role == "admin"
name = st.session_state.name
valid_dept = dept if (dept and dept in DEPARTMENTS) else None

with st.sidebar:
    st.markdown(f"""
    <div style="padding:8px 4px 16px;">
      <div style="display:flex;align-items:center;gap:10px;">
        <div style="width:36px;height:36px;background:linear-gradient(135deg,#6366f1,#8b5cf6);
                    border-radius:10px;display:flex;align-items:center;justify-content:center;
                    font-size:1.2rem;">🎓</div>
        <div>
          <div style="font-weight:800;font-size:1.05rem;">VaultMind</div>
          <div style="font-size:0.7rem;opacity:0.6;letter-spacing:0.5px;">ENTERPRISE ERP</div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    nav_items = []
    if is_admin:
        nav_items = [("Home","🏠 Home"),("Analytics","📊 Analytics"),
                     ("Departments","🏛️ Departments"),("Faculty","👥 Faculty"),
                     ("Uploads","📤 Upload KB"),("Audit","📜 Audit Log")]
    elif dept == "finance":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","💳 Fee Records"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    elif dept == "admin":
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Students","🎓 Students"),("Anomalies","⚠️ Anomalies"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]
    else:
        nav_items = [("Home","🏠 Home"),("Assistant","💬 AI Assistant"),
                     ("Documents","📚 KB Documents"),("Analytics","📊 Analytics")]

    for key, label in nav_items:
        bt = "primary" if st.session_state.view == key else "secondary"
        if st.button(label, key=f"nav_{key}", type=bt, use_container_width=True):
            st.session_state.view = key; st.rerun()

    # Conversations sidebar
    if st.session_state.view == "Assistant" and not is_admin:
        st.markdown("---")
        c1, c2 = st.columns([3,1])
        with c1:
            st.markdown(f'<div style="font-size:0.7rem;opacity:0.6;letter-spacing:0.5px;font-weight:700;">CONVERSATIONS</div>', unsafe_allow_html=True)
        with c2:
            if st.button("➕", key="new_conv_btn", help="New Chat"):
                st.session_state.active_conv = new_conversation(user, dept or "-")
                st.rerun()

        convs = list_conversations(user, dept or "-")
        today = datetime.now().date()
        groups = {"Today":[], "Yesterday":[], "Last 7 days":[], "Older":[]}
        for cv in convs:
            try: upd = datetime.fromisoformat(cv["updated"]).date()
            except: upd = today
            d = (today - upd).days
            if d==0: groups["Today"].append(cv)
            elif d==1: groups["Yesterday"].append(cv)
            elif d<=7: groups["Last 7 days"].append(cv)
            else: groups["Older"].append(cv)

        for grp, items in groups.items():
            if not items: continue
            st.markdown(f'<div style="font-size:0.68rem;opacity:0.5;margin:8px 0 4px;letter-spacing:1px;text-transform:uppercase;">{grp}</div>', unsafe_allow_html=True)
            for cv in items:
                c1, c2 = st.columns([5,1])
                with c1:
                    label = f"💬 {cv['title'][:28]}{'…' if len(cv['title'])>28 else ''}"
                    if st.button(label, key=f"cv_{cv['id']}", use_container_width=True,
                                 help=f"Updated {cv['updated'][:16]}"):
                        st.session_state.active_conv = cv["id"]; st.rerun()
                with c2:
                    if st.button("✕", key=f"del_{cv['id']}"):
                        delete_conversation(cv["id"])
                        if st.session_state.active_conv == cv["id"]:
                            st.session_state.active_conv = None
                        st.rerun()

    st.markdown("---")
    st.markdown(f"""
    <div style="padding:8px 4px;">
      <div style="font-size:0.7rem;opacity:0.6;margin-bottom:4px;">SIGNED IN AS</div>
      <div style="font-weight:600;font-size:0.9rem;">{name}</div>
      <div style="font-size:0.75rem;opacity:0.7;margin-top:4px;">
        <span class="pill p-{role}">{role}</span>
      </div>
      {f'<div style="margin-top:8px;font-size:0.78rem;color:{dcolor(dept)};"><span class="dot dot-green"></span>{dname(dept)}</div>' if valid_dept else ''}
    </div>
    """, unsafe_allow_html=True)

    if st.button("🚪 Sign Out", use_container_width=True, key="logout_btn"):
        log(user,role,dept,"LOGOUT","ok")
        for k in ("user","name","role","dept","logged_in","view","active_conv"): st.session_state[k]=None
        st.session_state.chat_ctx=""; st.session_state.chat_topic=""; st.session_state.voice_result=None
        st.rerun()

# ---- TOP BAR ----
c_left, c_right = st.columns([3,1])
with c_left:
    st.markdown(f"""
    <div style="padding:8px 0 12px 0;">
      <div style="font-size:0.78rem;color:#64748b;font-weight:500;margin-bottom:2px;">
        VaultMind / {st.session_state.view}
      </div>
      <h1 style="margin:0;">{st.session_state.view}</h1>
    </div>
    """, unsafe_allow_html=True)
with c_right:
    if valid_dept:
        try:
            n = get_coll(dept).count()
            voice_status = "Voice ✓" if whisper_pipe is not None else "Voice off"
            st.markdown(f"""
            <div style="text-align:right;padding-top:12px;">
              <span class="pill p-success"><span class="dot dot-green"></span>AI Online</span>
              <div style="font-size:0.75rem;color:#64748b;margin-top:6px;">{n} chunks · {DEV.upper()} · {voice_status}</div>
            </div>
            """, unsafe_allow_html=True)
        except: pass

# ==================== VIEW: HOME ====================
if st.session_state.view == "Home":
    if is_admin:
        total_chunks = sum(get_coll(d).count() for d in DEPARTMENTS)
        total_faculty = len([u for u in USERS.values() if u["role"]=="teacher"])
        all_df = audit_df()
        blocked_24h = len(all_df[(all_df["action"]=="BLOCKED_QUERY") & (all_df["ts"]>datetime.now()-timedelta(hours=24))]) if len(all_df) else 0
        weekly_queries = len(all_df[(all_df["action"]=="QUERY") & (all_df["ts"]>datetime.now()-timedelta(days=7))]) if len(all_df) else 0
        c = sqlite3.connect(CONV_DB).cursor()
        c.execute("SELECT COUNT(*) FROM conversations"); total_convs = c.fetchone()[0]
        c.execute("SELECT COUNT(*) FROM messages"); total_msgs = c.fetchone()[0]
        c.connection.close()
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("Departments", len(DEPARTMENTS), "active"),
            ("Faculty", total_faculty, "accounts"),
            ("Conversations", total_convs, f"{total_msgs:,} messages"),
            ("Weekly queries", weekly_queries, f"{blocked_24h} blocked today")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        st.markdown("### 🏛️ Department Health")
        cols = st.columns(len(DEPARTMENTS))
        for col,(code,cfg) in zip(cols, DEPARTMENTS.items()):
            score = health_score(code); n = get_coll(code).count()
            srcs = list_sources(code)
            fac = sum(1 for u in USERS.values() if u.get("dept")==code)
            bar_color = "#10b981" if score>=70 else "#f59e0b" if score>=40 else "#ef4444"
            col.markdown(f"""
            <div class="card" style="border-left:4px solid {cfg['color']};">
              <div style="font-size:1.6rem;">{cfg.get('icon','📁')}</div>
              <div style="font-size:0.7rem;color:{cfg['color']};font-weight:700;letter-spacing:1px;margin:4px 0;">{code.upper()}</div>
              <div style="font-size:0.95rem;font-weight:700;margin-bottom:8px;">{cfg['name']}</div>
              <div style="display:flex;align-items:center;gap:6px;margin-bottom:8px;">
                <div style="flex:1;height:6px;background:#e2e8f0;border-radius:3px;overflow:hidden;">
                  <div style="width:{score}%;height:100%;background:{bar_color};"></div>
                </div>
                <span style="font-size:0.75rem;font-weight:700;color:{bar_color};">{score}</span>
              </div>
              <div style="font-size:0.75rem;color:#64748b;">📚 {n} chunks · 📄 {len(srcs)} docs · 👥 {fac}</div>
            </div>
            """, unsafe_allow_html=True)
    else:
        st.markdown(f"""
        <div style="background:linear-gradient(135deg,{dcolor(dept)}15,{dcolor(dept)}05);
                    border:1px solid {dcolor(dept)}30;border-radius:16px;padding:24px 28px;margin-bottom:24px;">
          <div style="font-size:0.75rem;color:{dcolor(dept)};font-weight:700;letter-spacing:1px;">WELCOME BACK</div>
          <h2 style="margin:6px 0 6px 0;font-size:1.6rem;">{name}</h2>
          <p style="color:#475569;margin:0;font-size:0.92rem;">
            You're in the <b>{dname(dept)}</b> workspace. Voice input, persistent memory, and AI search — all local.
          </p>
        </div>
        """, unsafe_allow_html=True)
        srcs = list_sources(dept); n = get_coll(dept).count()
        df = audit_df(dept=dept)
        my_queries = len(df[df["user"]==user]) if len(df) else 0
        c = sqlite3.connect(CONV_DB).cursor()
        c.execute("SELECT COUNT(*) FROM conversations WHERE user=? AND dept=?", (user, dept))
        my_convs = c.fetchone()[0]; c.connection.close()
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("KB Chunks", n, "indexed"),
            ("My Conversations", my_convs, "saved"),
            ("Health Score", health_score(dept), "out of 100"),
            ("My Queries", my_queries, "all time")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        st.markdown("### ⚡ Quick Actions")
        actions = [("🎤","Voice Query","Speak Hindi/English"),
                   ("📝","Exam Paper","Generate question paper"),
                   ("📓","Study Notes","Structured notes PDF"),
                   ("📊","Anomalies","At-risk students")]
        cols = st.columns(4)
        for col,(ic,t,d) in zip(cols, actions):
            with col:
                if st.button(f"{ic} {t}", use_container_width=True, key=f"qa_home_{t}"):
                    if t=="Anomalies": st.session_state.view="Anomalies"
                    elif t=="Voice Query":
                        st.session_state.view="Assistant"
                        st.session_state.voice_input_open = True
                    else:
                        st.session_state.qa_trigger = {"Exam Paper":"exam","Study Notes":"notes"}.get(t,"notes")
                        st.session_state.view="Assistant"
                    st.rerun()
                st.caption(d)

# ==================== VIEW: ASSISTANT ====================
elif st.session_state.view == "Assistant" and not is_admin:
    if not st.session_state.active_conv:
        convs = list_conversations(user, dept or "-")
        if convs: st.session_state.active_conv = convs[0]["id"]
        else:
            st.session_state.active_conv = new_conversation(user, dept or "-")
            st.rerun()

    active = st.session_state.active_conv
    if not active: st.error("Could not initialize conversation"); st.stop()

    history = load_messages(active)
    convs_all = list_conversations(user, dept or "-")
    conv_title = next((c["title"] for c in convs_all if c["id"]==active), "Chat")

    c1, c2, c3 = st.columns([3,1,1])
    with c1:
        st.markdown(f"### 💬 {conv_title}")
        voice_badge = " · 🎤 Voice ready" if whisper_pipe is not None else ""
        st.caption(f"{count_messages(active)} messages · 🧠 Memory on · 🔒 Dept-scoped{voice_badge}")
    with c2:
        st.session_state.deep_think = st.toggle("🧠 Deep Think", value=st.session_state.deep_think)
    with c3:
        if st.button("➕ New Chat", use_container_width=True, key="new_from_header"):
            st.session_state.active_conv = new_conversation(user, dept or "-")
            st.rerun()

    # ===== VOICE INPUT BLOCK =====
    with st.expander("🎤 Voice Input — Speak your query (Hindi / English / 99 languages)", expanded=False):
        st.caption("Whisper runs locally on the T4. Hindi is auto-translated to English for the KB search.")
        cA, cB = st.columns(2)
        with cA:
            audio = st.audio_input("Record from microphone", key="voice_rec")
        with cB:
            up_audio = st.file_uploader("Or upload audio (WAV / MP3 / M4A / OGG)",
                                         type=["wav","mp3","m4a","ogg","webm"], key="voice_file")
        audio_data = (audio.getvalue() if audio else (up_audio.getvalue() if up_audio else None))
        if audio_data and st.button("🎧 Transcribe", type="primary", key="btn_stt"):
            with st.spinner("Whisper is transcribing…"):
                st.session_state.voice_result = transcribe_multilingual(audio_data)
        if st.session_state.voice_result:
            r = st.session_state.voice_result
            st.markdown(f'<div class="alert alert-voice">🎤 Detected language: <b>{r.get("language","unknown")}</b></div>', unsafe_allow_html=True)
            st.text_area("Original transcription", value=r["original"], height=70, key="vo_orig")
            if str(r.get("language","unknown")).lower() not in ("en","english"):
                st.text_area("English translation (used for KB search)", value=r["english"],
                             height=70, key="vo_en")
            c1, c2 = st.columns(2)
            with c1:
                if st.button("📋 Use as query", key="use_voice", use_container_width=True):
                    st.session_state.chat_topic = r.get("english") or r.get("original") or ""
                    st.session_state.voice_result = None
                    st.rerun()
            with c2:
                if st.button("🗑️ Clear voice", key="clr_voice", use_container_width=True):
                    st.session_state.voice_result = None; st.rerun()

    # Quick actions
    st.markdown("**⚡ Quick Actions**")
    qa_cols = st.columns(5)
    for col,(k,ic,lbl) in zip(qa_cols,[("exam","📝","Exam"),("notes","📓","Notes"),
                                        ("quiz","🧠","Quiz"),("sum","📄","Summarise"),
                                        ("explain","💡","Explain")]):
        with col:
            if st.button(f"{ic} {lbl}", key=f"qa_{k}", use_container_width=True):
                st.session_state.qa_trigger=k; st.rerun()

    topic = st.text_input("Topic / subject", key="chat_topic",
                          placeholder="e.g. Data Structures · Fee refund policy · Student S003 status")

    with st.expander("📎 Attach PDF or image (becomes AI context)"):
        cA, cB = st.columns(2)
        with cA:
            f1 = st.file_uploader("PDF", type=["pdf"], key="chat_pdf")
            if f1 and st.button("Load PDF", key="btn_pdf"):
                pages, err = extract_pdf(f1.getvalue(), f1.name)
                if pages:
                    st.session_state.chat_ctx = "\n".join(p["text"] for p in pages)
                    st.session_state.ctx_name = f1.name
                    st.success(f"✅ {len(pages)} pages")
                else: st.error(f"❌ {err}")
        with cB:
            f2 = st.file_uploader("Image (OCR)", type=["png","jpg","jpeg"], key="chat_img")
            if f2 and st.button("Run OCR", key="btn_ocr"):
                tmp=f"/content/_ocr_{int(time.time())}_{_safe(f2.name)}"
                open(tmp,"wb").write(f2.getvalue())
                try:
                    txt = pytesseract.image_to_string(Image.open(tmp))
                    st.session_state.chat_ctx = txt; st.session_state.ctx_name = f2.name
                    st.success(f"✅ {len(txt)} chars")
                except Exception as e: st.error(f"❌ {e}")
                finally:
                    try: os.remove(tmp)
                    except: pass
    if st.session_state.chat_ctx:
        cA, cB = st.columns([4,1])
        with cA:
            st.markdown(f'<div class="alert alert-info">📎 Attached: <b>{st.session_state.ctx_name}</b></div>', unsafe_allow_html=True)
        with cB:
            if st.button("Clear file", key="clr_file"):
                st.session_state.chat_ctx=""; st.session_state.ctx_name=""; st.rerun()

    box = st.container(height=520)
    with box:
        if not history:
            st.markdown(f"""<div class="empty">
              <div class="icon">💬</div>
              <div class="title">Start a conversation with {dname(dept)} AI</div>
              <div class="sub">Type, use 🎤 voice, or attach a file. Every message is saved.</div>
            </div>""", unsafe_allow_html=True)
        for i, m in enumerate(history):
            with st.chat_message(m["role"]):
                st.markdown(m["content"])
                if m.get("thinking"):
                    with st.expander("🧠 Reasoning"): st.caption(m["thinking"])
                if m.get("sources"):
                    with st.expander(f"📄 {len(m['sources'])} source(s)"):
                        for s in m["sources"]: st.caption(f"`{s['source']}` · {s.get('similarity',0):.2f}")
                if m.get("pdf_name"):
                    st.caption(f"📄 Deliverable: `{m['pdf_name']}`")

    uq = st.chat_input("Ask a question… (memory on · 🎤 voice ready)")
    if st.session_state.qa_trigger:
        k = st.session_state.qa_trigger; st.session_state.qa_trigger=None
        prompts = {
          "exam": f"Prepare a question paper PDF. Subject: {topic or dname(dept)}. 10 questions, Sections A/B/C.",
          "notes": f"Generate study notes PDF on: {topic or 'the full course'}.",
          "quiz": f"Create a 5-question MCQ quiz PDF on: {topic or 'the course'}.",
          "sum": f"Summarise key rules about: {topic or 'department policy'}.",
          "explain": f"Explain in simple terms: {topic or 'the department fundamentals'}.",
        }
        uq = prompts[k]

    if uq:
        st.session_state.stop_event.clear()
        save_message(active, "user", uq)
        with box:
            with st.chat_message("user"): st.markdown(uq)
        other, on = detect_cross_dept(dept, uq)
        if other and other != dept:
            err = f"🚫 **Access denied.** Scoped to **{dname(dept)}**. Cannot query **{on}**."
            save_message(active, "assistant", err)
            with box:
                with st.chat_message("assistant"): st.error(err)
            log(user,role,dept,"BLOCKED_QUERY",f"{other}:{uq[:60]}")
        else:
            with box:
                with st.chat_message("assistant"):
                    ph = st.empty(); stop_ph = st.empty()
                    stop_ph.button("⏹️ Stop", key=f"stop_{int(time.time())}",
                                   on_click=lambda: st.session_state.stop_event.set())
                    ans, hits = stream_answer(dept, uq, ph, st.session_state.chat_ctx,
                                              deep=st.session_state.deep_think, conv_id=active)
                    stop_ph.empty()
                    pdf_bytes=None; pdf_name=None
                    if is_doc_request(uq) or ans.count("Q1")+ans.count("1.")>=3:
                        title = "Examination Question Paper" if "question" in uq.lower() else "Document"
                        body = ans.replace("**","").replace("##","").replace("*","")
                        pdf_bytes = make_pdf(f"{title} — {topic or dname(dept)}", dname(dept),
                                             f"Generated by VaultMind AI · {datetime.now().strftime('%d %b %Y %H:%M')}",
                                             body)
                        pdf_name = f"{_slug(title)}_{_slug(topic or dept)}.pdf"
                        st.success("📄 PDF ready")
                        st.download_button(f"📥 {pdf_name}", pdf_bytes, file_name=pdf_name,
                                           mime="application/pdf", key=f"dl_new_{int(time.time())}")
                    if hits: st.caption(f"📄 {len(hits)} source(s)")
            save_message(active, "assistant", ans, sources=hits,
                         thinking=st.session_state.get("last_think","") if st.session_state.deep_think else "",
                         pdf_name=pdf_name or "")
            log(user,role,dept,"QUERY",uq[:60])
        st.rerun()

    if history:
        cA, cB, cC = st.columns([1,1,4])
        with cA:
            if st.button("🗑️ Clear Chat", use_container_width=True, key="clr_chat"):
                delete_conversation(active); st.session_state.active_conv = None; st.rerun()
        with cB:
            if st.button("📥 Export PDF", use_container_width=True, key="exp_chat"):
                body = "\n\n".join(f"[{m['role'].upper()}]\n{m['content']}" for m in history)
                pdf = make_pdf("Chat Transcript", dname(dept),
                               f"Exported by {name} · {datetime.now().strftime('%d %b %Y %H:%M')}", body)
                st.download_button("📥 Download", pdf, file_name=f"chat_{_slug(dept)}.pdf",
                                   mime="application/pdf", key="dl_chat_exp")

# ==================== VIEW: ANALYTICS ====================
elif st.session_state.view == "Analytics":
    st.markdown("### 📊 Department Analytics")
    df = audit_df(dept=None if is_admin else dept)
    if len(df)==0:
        st.info("No activity yet.")
    else:
        c1,c2,c3,c4 = st.columns(4)
        for col,(lbl,val,sub) in zip([c1,c2,c3,c4],[
            ("Total actions", len(df), "all time"),
            ("Queries", len(df[df["action"]=="QUERY"]), "AI interactions"),
            ("Blocked", len(df[df["action"]=="BLOCKED_QUERY"]), "security events"),
            ("Unique users", df["user"].nunique(), "active faculty")]):
            col.markdown(f"""<div class="metric-card"><div class="lbl">{lbl}</div>
                <div class="val">{val}</div><div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        st.write("")
        c1, c2 = st.columns(2)
        with c1:
            dc = df[df["action"]=="QUERY"].groupby("dept").size().reset_index(name="count")
            dc = dc[dc["dept"].isin(DEPARTMENTS.keys())]
            if len(dc):
                fig = px.bar(dc, x="dept", y="count", color="dept",
                             color_discrete_map={d: DEPARTMENTS[d]["color"] for d in DEPARTMENTS})
                fig.update_layout(showlegend=False, height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        with c2:
            df["day"] = df["ts"].dt.date
            daily = df[df["action"]=="QUERY"].groupby("day").size().reset_index(name="count")
            if len(daily):
                fig = px.line(daily, x="day", y="count", markers=True)
                fig.update_traces(line_color="#6366f1", line_width=3)
                fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10),
                                  plot_bgcolor="white", paper_bgcolor="white")
                st.plotly_chart(fig, use_container_width=True)
        breakdown = df.groupby("action").size().reset_index(name="count")
        fig = px.pie(breakdown, values="count", names="action", hole=0.55,
                     color_discrete_sequence=["#6366f1","#8b5cf6","#10b981","#f59e0b","#ef4444"])
        fig.update_layout(height=320, margin=dict(l=10,r=10,t=10,b=10))
        st.plotly_chart(fig, use_container_width=True)

# ==================== VIEW: ANOMALIES ====================
elif st.session_state.view == "Anomalies":
    st.markdown("### ⚠️ Anomaly Detection")
    items = anomalies(dept) if valid_dept else []
    if not items: st.success("✅ No anomalies detected.")
    else:
        crit = [x for x in items if x["severity"]=="critical"]
        warn = [x for x in items if x["severity"]=="warning"]
        c1,c2 = st.columns(2)
        c1.markdown(f'<div class="metric-card"><div class="lbl">🚨 Critical</div><div class="val" style="color:#ef4444;">{len(crit)}</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="metric-card"><div class="lbl">⚠️ Warnings</div><div class="val" style="color:#f59e0b;">{len(warn)}</div></div>', unsafe_allow_html=True)
        st.write("")
        for x in items:
            cls = "alert-critical" if x["severity"]=="critical" else "alert-warning"
            icon = "🚨" if x["severity"]=="critical" else "⚠️"
            st.markdown(f'<div class="alert {cls}"><b>{icon} {x["type"]} — {x["student"]}</b> ({x["roll"]})<br>{x["detail"]}</div>', unsafe_allow_html=True)

# ==================== VIEW: STUDENTS ====================
elif st.session_state.view == "Students" and dept in ("finance","admin"):
    st.markdown("### 🎓 Student Records")
    try:
        csv_data = SEED.get(dept,{}).get("students_csv","")
        df = pd.read_csv(io.StringIO(csv_data))
        c1,c2,c3 = st.columns(3)
        with c1: dept_filter = st.selectbox("Department", ["ALL"]+list(DEPARTMENTS.keys()), key="stu_d")
        with c2: yr_filter = st.selectbox("Year", ["ALL","III","V","VII"], key="stu_y")
        with c3: srch = st.text_input("Search", key="stu_s")
        view = df.copy()
        if dept_filter!="ALL" and "Dept" in view.columns: view = view[view["Dept"]==dept_filter]
        if yr_filter!="ALL" and "Year" in view.columns: view = view[view["Year"]==yr_filter]
        if srch: view = view[view.apply(lambda r: srch.lower() in str(r).lower(), axis=1)]
        st.markdown(f"**{len(view)} records**")
        st.dataframe(view, use_container_width=True, hide_index=True, height=380)
        if "Status" in df.columns:
            c1,c2 = st.columns(2)
            with c1:
                sc = df["Status"].value_counts().reset_index(); sc.columns=["Status","Count"]
                fig = px.pie(sc, values="Count", names="Status", hole=0.5,
                             color_discrete_map={"CLEARED":"#10b981","PARTIAL":"#f59e0b",
                                                 "OVERDUE":"#ef4444","CRITICAL":"#991b1b"})
                fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Fee Status")
                st.plotly_chart(fig, use_container_width=True)
            with c2:
                top = df.nlargest(8, "Pending")[["Name","Pending"]]
                fig = px.bar(top, x="Pending", y="Name", orientation="h", color_discrete_sequence=["#ef4444"])
                fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Top Pending (₹)")
                st.plotly_chart(fig, use_container_width=True)
        if "Attendance" in df.columns:
            fig = px.histogram(df, x="Attendance", nbins=10, color_discrete_sequence=["#6366f1"])
            fig.add_vline(x=75, line_dash="dash", line_color="red", annotation_text="Min 75%")
            fig.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10), title="Attendance Distribution")
            st.plotly_chart(fig, use_container_width=True)
    except Exception as e: st.error(f"Load failed: {e}")

# ==================== VIEW: DOCUMENTS ====================
elif st.session_state.view == "Documents" and not is_admin:
    st.markdown("### 📚 Knowledge Base")
    srcs = list_sources(dept) if valid_dept else {}
    if srcs:
        for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
            st.markdown(f"""<div class="card" style="margin-bottom:8px;">
              <div style="display:flex;justify-content:space-between;align-items:center;">
                <div><b>📄 {s}</b></div>
                <span class="pill p-success">{cnt} chunks</span>
              </div>
            </div>""", unsafe_allow_html=True)

# ==================== VIEW: DEPARTMENTS (Admin) ====================
elif st.session_state.view == "Departments" and is_admin:
    st.markdown("### 🏛️ Department Management")
    with st.expander("➕ Create New Department"):
        c1, c2 = st.columns(2)
        with c1:
            code = st.text_input("Code", key="nd_code").strip().lower()
            name2 = st.text_input("Display Name", key="nd_name")
            color = st.color_picker("Color", value="#6366f1", key="nd_color")
        with c2:
            sec = st.text_input("Secret", key="nd_sec")
            al = st.text_input("Aliases", key="nd_al")
        if st.button("Create Department", type="primary"):
            if not code or not name2: st.error("Required")
            elif code in DEPARTMENTS: st.error("Exists")
            else:
                DEPARTMENTS[code] = {"name":name2, "secret":sec or f"{code}-s-{int(time.time())}",
                    "color":color, "aliases":[a.strip().lower() for a in al.split(",") if a.strip()] or [code], "icon":"📁"}
                save_depts(); get_coll(code)
                log(user,role,None,"CREATE_DEPT",f"{code}:{name2}")
                st.success(f"✅ {name2}"); time.sleep(1); st.rerun()
    for code, cfg in DEPARTMENTS.items():
        n = get_coll(code).count(); srcs = list_sources(code); score = health_score(code)
        with st.expander(f"**{cfg['name']}** ({code}) — Score {score} · {n} chunks"):
            for s, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
                st.markdown(f"- 📄 `{s}` ({cnt} chunks)")

# ==================== VIEW: FACULTY (Admin) ====================
elif st.session_state.view == "Faculty" and is_admin:
    st.markdown("### 👥 Faculty Management")
    with st.expander("➕ Add Faculty"):
        c1, c2 = st.columns(2)
        with c1:
            nu = st.text_input("Username", key="cr_u").strip().lower()
            np = st.text_input("Password", type="password", key="cr_p")
            nn = st.text_input("Full Name", key="cr_n")
        with c2:
            nd = st.selectbox("Department", list(DEPARTMENTS.keys()),
                              format_func=lambda x: DEPARTMENTS[x]["name"], key="cr_d")
        if st.button("Create Account", type="primary"):
            if not nu or not np or not nn: st.error("Required")
            elif nu in USERS: st.error("Exists")
            else:
                USERS[nu] = {"name":nn,"role":"teacher","dept":nd,"pw":_hash(np)}
                save_users(); log(user,role,None,"CREATE_USER",f"{nu}:teacher:{nd}")
                st.success(f"✅ {nu}"); time.sleep(1); st.rerun()
    st.dataframe(pd.DataFrame([{"username":u,"name":v["name"],"role":v["role"],"dept":v["dept"] or "—"}
                                for u,v in USERS.items()]), use_container_width=True, hide_index=True)

# ==================== VIEW: UPLOADS (Admin) ====================
elif st.session_state.view == "Uploads" and is_admin:
    st.markdown("### 📤 Upload to Department RAG")
    tgt = st.selectbox("Target Department", list(DEPARTMENTS.keys()),
                       format_func=lambda x: f"{DEPARTMENTS[x]['name']} ({x})", key="up_d")
    dtype = st.selectbox("Document Type", ["syllabus","regulation","notes","question-paper","reference","policy"], key="up_t")
    uploaded = st.file_uploader("Choose PDFs", type=["pdf"], accept_multiple_files=True, key="up_f")
    if uploaded and st.button("🚀 Ingest", type="primary", use_container_width=True):
        total=0; ok=0
        for f in uploaded:
            with st.spinner(f"Processing {f.name}…"):
                chunks, err, pages = ingest_pdf(tgt, f.name, f.getvalue(), dtype)
                if err: st.error(f"❌ {f.name} — {err}")
                else:
                    st.success(f"✅ {f.name} — {pages} pages → {chunks} chunks")
                    total+=chunks; ok+=1
        if ok:
            st.balloons()
            st.info(f"🎉 {ok}/{len(uploaded)} files · {total} chunks")
            log(user,role,None,"INGEST_PDF",f"{tgt}:{ok}files,{total}chunks")
            time.sleep(2); st.rerun()

# ==================== VIEW: AUDIT (Admin) ====================
elif st.session_state.view == "Audit" and is_admin:
    st.markdown("### 📜 Audit Log")
    fd = st.selectbox("Filter by Department", ["ALL"]+list(DEPARTMENTS.keys()), key="au_d")
    df = audit_df(dept=None if fd=="ALL" else fd)
    if len(df):
        df["ts"] = df["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
        st.dataframe(df[["ts","user","role","dept","action","detail"]],
                     use_container_width=True, hide_index=True, height=500)
    st.divider()
    allr = audit_df()
    c1,c2,c3 = st.columns(3)
    c1.metric("🔒 Blocked", len(allr[allr["action"]=="BLOCKED_QUERY"]) if len(allr) else 0)
    c2.metric("✅ Leaks", 0); c3.metric("📊 Actions", len(allr))
'''
open("/content/vaultmind_erp.py","w").write(APP)
print(f"✅ App v10 written ({len(APP)} bytes)")
import shutil
shutil.copy("/content/vaultmind_erp.py", f"{DRIVE}/vaultmind_erp.py")

print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "--theme.base light --theme.primaryColor '#6366f1' "
    "> /content/streamlit.log 2>&1 &", shell=True)
time.sleep(12)
r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Failed:"); print(open("/content/streamlit.log").read()[-1500:]); raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎤 VAULTMIND ERP v10 — MEMORY + VOICE EDITION                 ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                                                                ║
║  🎤 VOICE INPUT (NEW):                                         ║
║     • Whisper-small runs locally on the T4                     ║
║     • Supports Hindi, English + 97 more languages              ║
║     • Auto-translates Hindi → English for KB search            ║
║     • Access via AI Assistant → "🎤 Voice Input" expander     ║
║     • First use downloads ~950 MB (once)                       ║
║                                                                ║
║  💾 PERSISTENT MEMORY (from v9):                               ║
║     • Every conversation saved to Drive                        ║
║     • Sidebar lists chats (Today/Yesterday/Older)              ║
║     • Model remembers last 5 turns                             ║
║     • Cross-session: reload after disconnect                   ║
║                                                                ║
║  ⏱️  First load: ~60s models + ~30s Whisper                    ║
╚════════════════════════════════════════════════════════════════╝
""")

📦 Installing… (ffmpeg for audio + whisper deps)
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
✅ Ready
✅ App v10 written (70929 bytes)
🚀 Launching…


NameError: name 'time' is not defined

In [ ]:
# ==================== FIX + LAUNCH ====================
import subprocess, sys, os, time          # ← time was missing
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"

# Confirm app file exists (already written by the failed cell)
APP = "/content/vaultmind_erp.py"
if not os.path.exists(APP):
    print("❌ App file missing — re-run the big v10 cell first"); raise SystemExit(1)
print(f"✅ App file found ({os.path.getsize(APP)} bytes)")

# Restart Streamlit cleanly
print("🚀 Launching…")
subprocess.run(["bash","-c","pkill -9 -f streamlit 2>/dev/null; sleep 1"], capture_output=True)
subprocess.run(
    "nohup streamlit run /content/vaultmind_erp.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--server.enableWebsocketCompression false --browser.gatherUsageStats false "
    "--theme.base light --theme.primaryColor '#6366f1' "
    "> /content/streamlit.log 2>&1 &",
    shell=True)
time.sleep(12)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit run' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Streamlit failed:")
    print(open("/content/streamlit.log").read()[-2000:])
    raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎤 VAULTMIND ERP v10 — LIVE                                   ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  {url}
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                                                                ║
║  🎤 Voice · 💾 Memory · 🔒 Isolation · 📊 Analytics            ║
╚════════════════════════════════════════════════════════════════╝
""")

✅ App file found (71389 bytes)
🚀 Launching…


✅ Streamlit PID 4614

╔════════════════════════════════════════════════════════════════╗
║  🎤 VAULTMIND ERP v10 — LIVE                                   ║
║                                                                ║
║  🌐 OPEN IN NEW TAB:                                           ║
║  https://8501-gpu-t4-s-kkb-usw4b1-2cr2jgcmprios-b.us-west4-1.prod.colab.dev
║                                                                ║
║  🔐 ACCOUNTS:                                                  ║
║     admin         / admin_pass                                 ║
║     teacher_cse   / t_cse                                      ║
║     teacher_ece   / t_ece                                      ║
║     teacher_fin   / t_fin                                      ║
║     teacher_admin / t_admin                                    ║
║                                                                ║
║  🎤 Voice · 💾 Memory · 🔒 Isolation · 📊 Analytics            ║
╚═══════════════════════════════════

In [ ]:
hi

In [ ]:
hi

In [ ]:
hi

In [ ]:
import subprocess, sys
print(f"Using Python: {sys.executable}")
print(f"Version: {sys.version}\n")

# Install EVERYTHING the app needs, into THIS Python
packages = [
    "streamlit", "chromadb", "sentence-transformers",
    "transformers>=4.45", "accelerate", "bitsandbytes",
    "python-docx", "pymupdf", "pytesseract", "pillow",
    "reportlab", "pdfplumber", "pandas", "plotly", "openpyxl",
    "librosa", "soundfile", "pyngrok",
]

print("📦 Installing all deps into the current Python…")
cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    print("⚠️ First install failed, retrying with output:")
    print(r.stderr[-2000:])
else:
    print("✅ pip install complete")

# System deps (Tesseract OCR + ffmpeg)
print("\n📦 Installing system packages…")
subprocess.run("apt-get install -y -q tesseract-ocr tesseract-ocr-eng ffmpeg 2>&1 | tail -1",
               shell=True)

# Verify each import in the same interpreter
print("\n🔍 Verifying imports in sys.executable:")
for mod in ["streamlit", "chromadb", "sentence_transformers", "transformers",
            "bitsandbytes", "peft", "docx", "pymupdf", "pytesseract",
            "PIL", "reportlab", "pdfplumber", "pandas", "plotly",
            "openpyxl", "librosa", "soundfile"]:
    r = subprocess.run([sys.executable, "-c", f"import {mod}; print('OK')"],
                       capture_output=True, text=True)
    status = "✅" if r.returncode == 0 else "❌"
    print(f"  {status} {mod}" + ("" if r.returncode == 0 else f" — {r.stderr.strip()[-120:]}"))

Using Python: /usr/bin/python3
Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

📦 Installing all deps into the current Python…
✅ pip install complete

📦 Installing system packages…

🔍 Verifying imports in sys.executable:
  ✅ streamlit
  ✅ chromadb
  ✅ sentence_transformers
  ✅ transformers
  ✅ bitsandbytes
  ✅ peft
  ✅ docx
  ✅ pymupdf
  ✅ pytesseract
  ✅ PIL
  ✅ reportlab
  ✅ pdfplumber
  ✅ pandas
  ✅ plotly
  ✅ openpyxl
  ✅ librosa
  ✅ soundfile


In [ ]:
# ==================== VAULTMIND — NGROK LAUNCHER (final) ====================
import subprocess, sys, os, time

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/vaultmind"
os.makedirs(f"{DRIVE}/logs", exist_ok=True)

APP = "/content/vaultmind_erp.py"
if not os.path.exists(APP):
    backup = f"{DRIVE}/vaultmind_erp.py"
    if os.path.exists(backup):
        import shutil
        shutil.copy(backup, APP)
        print(f"✅ Restored from Drive backup ({os.path.getsize(APP)} bytes)")
    else:
        raise SystemExit("❌ App file missing")

print("🧹 Cleaning old processes…")
subprocess.run(["bash","-c","pkill -9 -f ngrok 2>/dev/null; pkill -9 -f 'streamlit run' 2>/dev/null; lsof -ti:8501 | xargs -r kill -9 2>/dev/null"], capture_output=True)
subprocess.run(["bash","-c","rm -rf /root/.ngrok2 /root/.config/ngrok 2>/dev/null"], capture_output=True)
time.sleep(3)

# ---- ngrok token ----
from pyngrok import ngrok, conf
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    if not NGROK_TOKEN: raise Exception()
    print("🔑 Token from Colab Secrets")
except Exception:
    NGROK_TOKEN = "31aQsVPmgjcKoOgypNQo9Hwlgo1_5LQKcrZU156AFcRM1Wpxw"
    print("⚠️  Hardcoded token — rotate before submission")

ngrok.set_auth_token(NGROK_TOKEN)
conf.get_default().auth_token = NGROK_TOKEN
print("✅ ngrok authenticated")

# ---- Launch Streamlit (via python -m, PATH-safe) ----
print("🚀 Launching Streamlit…")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP,
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false",
     "--server.enableWebsocketCompression", "false",
     "--browser.gatherUsageStats", "false",
     "--theme.base", "light",
     "--theme.primaryColor", "#6366f1"],
    stdout=open(f"{DRIVE}/logs/streamlit.log","w"),
    stderr=subprocess.STDOUT,
)
print("⏳ Waiting for Streamlit…")
time.sleep(14)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit' | head -1"],
                   capture_output=True, text=True)
log = open(f"{DRIVE}/logs/streamlit.log").read()

# Check for ModuleNotFoundError in log
import re
err = re.search(r"ModuleNotFoundError: No module named '([^']+)'", log)
if err:
    print(f"❌ Missing module: {err.group(1)}")
    print("👉 Run the fix cell above to install all deps into the same Python")
    print(f"\nLog tail:\n{log[-1500:]}")
    raise SystemExit(1)

if not r.stdout.strip():
    print("❌ Streamlit failed:")
    print(log[-2000:])
    raise SystemExit(1)
print(f"✅ Streamlit running (PID {r.stdout.strip()})")

# ---- ngrok tunnel ----
print("🌐 Creating tunnel…")
existing = [t for t in ngrok.get_tunnels() if "8501" in str(t.config.get("addr",""))]
tunnel = existing[0] if existing else ngrok.connect(8501, "http", bind_tls=True)
PUBLIC_URL = tunnel.public_url

print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  {PUBLIC_URL}
║                                                                ║
║  🔐  admin / admin_pass · teacher_cse / t_cse                  ║
║      teacher_ece / t_ece · teacher_fin / t_fin                 ║
║      teacher_admin / t_admin                                   ║
║                                                                ║
║  ⏱️  First load ~60s. Keep Colab tab OPEN.                     ║
╚════════════════════════════════════════════════════════════════╝
""")

with open(f"{DRIVE}/public_url.txt","w") as f:
    f.write(PUBLIC_URL)
print(f"💾 Saved to: {DRIVE}/public_url.txt")

🧹 Cleaning old processes…
⚠️  Hardcoded token — rotate before submission
✅ ngrok authenticated
🚀 Launching Streamlit…
⏳ Waiting for Streamlit…
✅ Streamlit running (PID 1029)
🌐 Creating tunnel…

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  https://101a-34-125-241-35.ngrok-free.app
║                                                                ║
║  🔐  admin / admin_pass · teacher_cse / t_cse                  ║
║      teacher_ece / t_ece · teacher_fin / t_fin                 ║
║      teacher_admin / t_admin                                   ║
║                                                                ║
║  ⏱️  First load ~60s. Keep Colab tab OPEN.                     ║
╚════════════════════════════════════════════════════════════════╝

💾 Saved to: /content/drive/MyDrive/vaultmind/public_url.txt


In [1]:
# ==================== LAUNCHER — COLAB PROXY ====================
import subprocess, sys, os, time, re

DRIVE = "/content/drive/MyDrive/vaultmind"
APP = "/content/vaultmind_erp.py"
os.makedirs(f"{DRIVE}/logs", exist_ok=True)

print("🧹 Cleaning old processes…")
subprocess.run(["bash","-c",
    "pkill -9 -f 'streamlit run' 2>/dev/null; lsof -ti:8501 | xargs -r kill -9 2>/dev/null"],
    capture_output=True)
time.sleep(2)

print("🚀 Launching Streamlit…")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP,
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false",
     "--server.enableWebsocketCompression", "false",
     "--browser.gatherUsageStats", "false",
     "--theme.base", "light", "--theme.primaryColor", "#6366f1"],
    stdout=open(f"{DRIVE}/logs/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("⏳ Waiting…")
time.sleep(12)

log = open(f"{DRIVE}/logs/streamlit.log").read()
m = re.search(r"(SyntaxError|ModuleNotFoundError|ImportError|NameError): ([^\n]+)", log)
if m:
    print(f"❌ App error: {m.group(0)}")
    print(log[-1500:])
    raise SystemExit(1)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Streamlit failed:"); print(log[-1500:]); raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  {url}
║                                                                ║
║  🆕 FIXED PDF UPLOAD:                                          ║
║     • OCR fallback for scanned PDFs                            ║
║     • Persistent results panel (no more vanishing messages)    ║
║     • Live chunk count in Uploads view                         ║
║     • Millisecond-precision IDs (no collisions)                ║
║                                                                ║
║  🔐 admin / admin_pass                                         ║
╚════════════════════════════════════════════════════════════════╝
""")
with open(f"{DRIVE}/public_url.txt","w") as f: f.write(url)
print(f"💾 URL saved → {DRIVE}/public_url.txt")

🧹 Cleaning old processes…
🚀 Launching Streamlit…
⏳ Waiting…
✅ Streamlit PID 3554

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  https://8501-m-s-kkb-usc1c1-3rjd5bxruvw6o-c.us-central1-1.prod.colab.dev
║                                                                ║
║  🆕 FIXED PDF UPLOAD:                                          ║
║     • OCR fallback for scanned PDFs                            ║
║     • Persistent results panel (no more vanishing messages)    ║
║     • Live chunk count in Uploads view                         ║
║     • Millisecond-precision IDs (no collisions)                ║
║                                                                ║
║  🔐 admin / admin_pass                                         ║
╚════════════════════════════════════════════════════════════════╝

💾 URL saved → /content/drive/MyDrive/

In [5]:
# ==================== RELAUNCH ====================
import subprocess, sys, os, time, re

DRIVE = "/content/drive/MyDrive/vaultmind"
APP = "/content/vaultmind_erp.py"

print("🧹 Cleaning…")
subprocess.run(["bash","-c",
    "pkill -9 -f 'streamlit run' 2>/dev/null; lsof -ti:8501 | xargs -r kill -9 2>/dev/null"],
    capture_output=True)
time.sleep(2)

print("🚀 Launching…")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP,
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false",
     "--server.enableWebsocketCompression", "false",
     "--browser.gatherUsageStats", "false",
     "--theme.base", "light", "--theme.primaryColor", "#6366f1"],
    stdout=open(f"{DRIVE}/logs/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("⏳ Waiting…")
time.sleep(12)

log = open(f"{DRIVE}/logs/streamlit.log").read()
m = re.search(r"(SyntaxError|ModuleNotFoundError|ImportError|NameError|IndentationError): ([^\n]+)", log)
if m:
    print(f"❌ App error: {m.group(0)}"); print(log[-1500:]); raise SystemExit(1)

r = subprocess.run(["bash","-c","pgrep -f 'streamlit' | head -1"], capture_output=True, text=True)
if not r.stdout.strip():
    print("❌ Streamlit failed:"); print(log[-1500:]); raise SystemExit(1)
print(f"✅ Streamlit PID {r.stdout.strip()}")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"""
╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  {url}
║                                                                ║
║  🆕 NEW FEATURES:                                              ║
║     📤 PDF upload — persistent success banner                  ║
║     👥 Faculty view — delete any account with confirm          ║
║     🏛️ Departments — full delete w/ DB wipe:                   ║
║         • ChromaDB collection                                  ║
║         • Uploaded PDFs                                        ║
║         • depts.json entry                                     ║
║         • Faculty re-assigned                                  ║
║         • Conversations marked orphaned                        ║
║                                                                ║
║  🔐 admin / admin_pass                                         ║
║     teacher_cse / t_cse · teacher_fin / t_fin                  ║
║     teacher_admin / t_admin · teacher_ece / t_ece              ║
╚════════════════════════════════════════════════════════════════╝
""")
with open(f"{DRIVE}/public_url.txt","w") as f: f.write(url)
print(f"💾 URL saved → {DRIVE}/public_url.txt")

🧹 Cleaning…
🚀 Launching…
⏳ Waiting…
✅ Streamlit PID 5376

╔════════════════════════════════════════════════════════════════╗
║  🎓 VAULTMIND ERP — LIVE                                       ║
║                                                                ║
║  🌐  https://8501-m-s-kkb-usc1c1-3rjd5bxruvw6o-c.us-central1-1.prod.colab.dev
║                                                                ║
║  🆕 NEW FEATURES:                                              ║
║     📤 PDF upload — persistent success banner                  ║
║     👥 Faculty view — delete any account with confirm          ║
║     🏛️ Departments — full delete w/ DB wipe:                   ║
║         • ChromaDB collection                                  ║
║         • Uploaded PDFs                                        ║
║         • depts.json entry                                     ║
║         • Faculty re-assigned                                  ║
║         • Conversations marked orphaned                      